# Logistic Regression — Chapter 4 (Math on Screen)

Chapter 4 wraps any plot animation in a **fixed template**: plot upper-left, math rows on the right, formula columns below.

## Abstractions (use for every clip)

| Module | Purpose |
|--------|---------|
| `tutorial_template.py` | `TutorialComposer`, `TutorialScene`, `TutorialTheme`, layout/typography/export |
| `handwrite_tutorial.py` | Patrick Hand + subscripts + symbol font + write-on reveal |
| `ch4_layout.py` | Chapter 4 defaults + `ch4_compose_tutorial_frame()` |

```python
scene = TutorialScene(plot=frame, math_right_blocks=..., math_bottom_blocks=...)
composer = make_composer("dark_rails")  # or any theme name
img = composer.render_scene(scene, write_progress=t)
```

All clips share **≈16.89×9.5 in @ 200 DPI → 3378×1900 px** (16:9 width at legacy height). Plot:right-rail width ratio **11:3**. Typography and spacing live in `TutorialTypography`; colors/gradients in `TutorialTheme`.

## Exports

| Cell | Output |
|------|--------|
| **`ch4_setup`** | Layout reload + Ch3 builders + all export definitions (run once) |
| **`ch4-likelihood-*`** | One cell per clip — see list below |


In [26]:
# --- Ch4 setup: layout reload + Ch3 builders + export definitions ---

import importlib
import json
from pathlib import Path

import handwrite_tutorial
import tutorial_template
import ch4_layout

importlib.reload(handwrite_tutorial)
importlib.reload(tutorial_template)
importlib.reload(ch4_layout)
from ch4_layout import *

# Reuse all Chapter 3 builders (datasets, knobs, triptych, strip, duo, …).
_CH3_NB = Path("logistic-regression-chap3.ipynb")
if not _CH3_NB.is_file():
    raise FileNotFoundError(_CH3_NB.resolve())
_ch3_src = "".join(json.loads(_CH3_NB.read_text())["cells"][1]["source"])
exec(compile(_ch3_src, str(_CH3_NB), "exec"), globals())
del _CH3_NB, _ch3_src

def ch4_sample_mistakes_triptych_frame():
    """One representative frame from the ch3_25 mistakes triptych (split-screen) family.

    Returns ``(plot_img, w_st, w_el, b)`` for math-rail values.
    """
    spec = CH3_LOSS_SPECS["mistakes"]
    loss_fn = spec["fn"]
    study, exam, y = study_sep, exam_sep, y_sep
    wr, we, br = ch3_triptych_script_weights()
    seqs, triples, losses = {}, {}, {}
    for which in ("st", "el", "b"):
        c = ch3_active_value(which, wr, we, br)
        seqs[which] = ch3_quad_sweep(c, CH3_LOSS_SYM_DELTA, CH3_SWEEP_NSEG)
        triples[which] = [ch3_triplet(which, float(v)) for v in seqs[which]]
        losses[which] = [loss_fn(ws, we, bb, study, exam, y) for ws, we, bb in triples[which]]
    all_l = np.concatenate([losses["st"], losses["el"], losses["b"]])
    pad_y = 0.06 * max(1e-6, float(np.nanmax(all_l) - np.nanmin(all_l)))
    y_lo = float(np.nanmin(all_l) - pad_y)
    y_hi = float(np.nanmax(all_l) + pad_y)

    def _xlim(seq):
        span = float(np.max(seq) - np.min(seq))
        pad_x = max(0.06 * span, 0.08)
        return float(np.min(seq) - pad_x), float(np.max(seq) + pad_x)

    ws, we, bb = triples["el"][len(triples["el"]) // 2]
    n_st = len(seqs["st"])
    n_el = len(seqs["el"])
    n_b = max(2, len(seqs["b"]) // 2)
    rots = _ch3_triptych_pack_knob_rots(ws, we, bb, wr, we, br)
    frame = ch3_triptych_frame(
        ws,
        we,
        bb,
        study,
        exam,
        y,
        spec,
        show_colormap=bool(spec["colormap"]),
        xs_st=seqs["st"][:n_st],
        ys_st=losses["st"][:n_st],
        xs_el=seqs["el"][:n_el],
        ys_el=losses["el"][:n_el],
        xs_b=seqs["b"][:n_b],
        ys_b=losses["b"][:n_b],
        x_lim_st=_xlim(seqs["st"]),
        x_lim_el=_xlim(seqs["el"]),
        x_lim_b=_xlim(seqs["b"]),
        y_lo=y_lo,
        y_hi=y_hi,
        knob_rots=rots,
        knob_scales=[1.0, float(CH3_KNOB_ACTIVE_SCALE), 1.0],
        arrows=None,
        panel_visible=(True, True, True),
        emphasize_knob="el",
    )
    return frame, float(ws), float(we), float(bb)


def ch4_math_right_blocks(w_st, w_el, b, study, exam, y):
    """Right-rail rows: current weights, gradient, and NLL at ``(w_st, w_el, b)``."""
    z = logits_plane(w_st, w_el, b, study, exam)
    p = sigmoid(z)
    yy = y.astype(float)
    eps = 1e-12
    nll = float(-np.sum(yy * np.log(p + eps) + (1.0 - yy) * np.log(1.0 - p + eps)))
    resid = p - yy
    gw1 = float(np.sum(resid * study))
    gw2 = float(np.sum(resid * exam))
    gb = float(np.sum(resid))
    w_text = rf"$w_1={w_st:.2f}$" + "\n" + rf"$w_2={w_el:.2f}$" + "\n" + rf"$b={b:.2f}$"
    g_text = (
        rf"$\partial w_1={gw1:.2f}$"
        + "\n"
        + rf"$\partial w_2={gw2:.2f}$"
        + "\n"
        + rf"$\partial b={gb:.2f}$"
    )
    return [
        {"label": r"$w,\,b$", "text": w_text, "bold_lhs": True, "role": "weights"},
        {"label": r"$\nabla\mathrm{NLL}$", "text": g_text, "bold_lhs": True, "role": "gradient"},
        {"label": "NLL", "text": rf"$NLL={nll:.2f}$", "bold_lhs": True, "role": "nll"},
    ]

CH4_BOTTOM_FORMULA_BLOCKS = [
    {"text": r"$p(y_i \mid x_i)=\hat p_i^{\,y_i}(1-\hat p_i)^{1-y_i}$", "bold_lhs": True, "role": "formula"},
    {"text": r"$\mathrm{NLL}(w)=-\sum_i \log p(y_i \mid x_i)$", "bold_lhs": True, "role": "formula"},
    {"text": r"$\nabla_w\,\mathrm{NLL}=\sum_i(\hat p_i-y_i)\,x_i$", "bold_lhs": True, "role": "formula"},
]


def ch4_tutorial_scene():
    """Shared plot + math blocks for static PNG and animated MP4."""
    plot, w1, w2, b = ch4_sample_mistakes_triptych_frame()
    return {
        "plot": plot,
        "math_right_blocks": ch4_math_right_blocks(w1, w2, b, study_sep, exam_sep, y_sep),
        "math_bottom_blocks": CH4_BOTTOM_FORMULA_BLOCKS,
    }


def ch4_render_tutorial_frame(scene, write_progress=1.0, *, theme=None, composer=None):
    return ch4_compose_tutorial_frame(
        scene["plot"],
        math_right_blocks=scene["math_right_blocks"],
        math_bottom_blocks=scene["math_bottom_blocks"],
        write_progress=write_progress,
        theme=theme,
        composer=composer,
    )


def ch4_export_handwrite_demo_mp4(n_frames=32, ms_per_frame=100):
    scene = ch4_tutorial_scene()
    return CH4_COMPOSER.export_mp4(
        ch4_scene_from_dict(scene),
        "ch4_01_handwrite_demo.mp4",
        save_mp4=save_mp4,
        output_dir=OUTPUT_DIR,
        n_frames=n_frames,
        ms_per_frame=ms_per_frame,
    )


def ch4_export_all_theme_demos(n_frames=32, ms_per_frame=100):
    """Export handwrite demo for each of the 5 color themes."""
    return ch4_export_theme_demos(
        ch4_tutorial_scene(),
        save_mp4=save_mp4,
        n_frames=n_frames,
        ms_per_frame=ms_per_frame,
    )


































































































































































































































# --- ch4_02: likelihood(w1,w2) landscape — duo left, tall 3D right ---
CH3_LIK_W12_W_ST0 = float(CH3_SCRIPT_K1_W_ST)
CH3_LIK_W12_W_EL0 = float(CH3_SCRIPT_K1_W_EL)
CH3_LIK_W12_B0 = float(CH3_SCRIPT_K1_B)
CH3_LIK_W12_W1_LO = float(CH3_SCRIPT_W1_WIDE[0])
CH3_LIK_W12_W1_HI = float(CH3_SCRIPT_W1_WIDE[1])
CH3_LIK_W12_W2_LO = float(CH3_LIK_W12_W_EL0 - CH3_SCRIPT_K1_STOP_SWEEP_DELTA)
CH3_LIK_W12_W2_HI = float(CH3_LIK_W12_W_EL0 + CH3_SCRIPT_K1_STOP_SWEEP_DELTA)
CH3_LIK_W12_B_HALF = float(CH3_SCRIPT_K1_TIGHT_SWEEP_DELTA * 20.0)
CH3_LIK_W12_FIGSIZE = CH4_DUO_FIGSIZE
CH3_LIK_W12_WIDTH_RATIOS = (1.22, 1.42)
CH3_LIK_W12_GRID_N = 52 if not _CH3_DRAFT else 28
CH3_LIK_W12_GRID_N_COARSE = 22 if not _CH3_DRAFT else 14
CH3_LIK_W12_GRID_N_FINE = 72 if not _CH3_DRAFT else 36
CH3_LIK_W12_QUAD_MID = 0.0
CH3_LIK_W12_CURVE_LW = 2.4
# Split-complementary likelihood mesh (ch4_02/03) — deeper red vs 2D FAIL_COLOR.
CH4_LIK_SURFACE_COLOR = "#b71c1c"
CH4_LIK_SURFACE_ALPHA = 0.80
CH3_LIK_W12_SURFACE_ALPHA = CH4_LIK_SURFACE_ALPHA
CH3_LIK_W12_SLICE_ALPHA = 0.42 * 0.9
CH3_LIK_W12_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_W12_N_HOLD = max(10, CH3_SCRIPT_N_HOLD // 4)
CH3_LIK_W12_N_KNOB = max(28, _smooth_n(22))
CH3_LIK_W12_N_ROT = max(32, _smooth_n(24))
CH3_LIK_W12_N_REVEAL = max(40, _smooth_n(30))
CH3_LIK_W12_N_FILL_LINES = CH3_LIK_W12_GRID_N
CH3_LIK_W12_N_FILL_TRACE = max(48, _smooth_n(36)) if not _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_W12_N_B_SLICES = 7 if not _CH3_DRAFT else 4
CH3_LIK_W12_N_ORBIT = max(48, _smooth_n(36))
CH3_LIK_W12_AZIM_W1 = 90.0
CH3_LIK_W12_AZIM_W2 = 180.0
CH3_LIK_W12_AZIM_BOTH = -54.0
CH3_LIK_W12_AZIM_STACK = 90.0
CH3_LIK_W12_ELEV_W1 = 0.0
CH3_LIK_W12_ELEV_W2 = 0.0
CH3_LIK_W12_ELEV_BOTH = 26.0
CH3_LIK_W12_ELEV_STACK = 0.0
CH3_LIK_W12_N_SQUISH = max(28, _smooth_n(22))
CH3_LIK_W12_CORNER_W1 = CH3_LIK_W12_W1_LO
CH3_LIK_W12_CORNER_W2 = CH3_LIK_W12_W2_LO
# 3-D knob 1 / knob 2 axis display (matches ch4_03 ±4 cube)
CH3_LIK_W12_3D_LO = -4.0
CH3_LIK_W12_3D_HI = 4.0
CH3_LIK_W12_CT_ELEV = 24.0
CH3_LIK_W12_CT_AZIM = -128.0
CH3_LIK_W12_ZOOM_GRID = 36 if not _CH3_DRAFT else 20
# Lift scatter slightly above the mesh so it draws in front (mpl 3D depth sort).
CH3_LIK_MARKER_Z_BUMP_FRAC = 0.018


def _ch3_lik_w12_axis_refined(lo, hi, mid, n_coarse, n_fine, *, fine_lo_half):
    """Denser samples on one half-axis; ``mid`` is the split (typically 0)."""
    lo, hi, mid = float(lo), float(hi), float(mid)
    nc, nf = max(2, int(n_coarse)), max(2, int(n_fine))
    if fine_lo_half:
        g_a = np.linspace(lo, mid, nf, dtype=np.float64)
        g_b = np.linspace(mid, hi, nc, dtype=np.float64)
    else:
        g_a = np.linspace(lo, mid, nc, dtype=np.float64)
        g_b = np.linspace(mid, hi, nf, dtype=np.float64)
    return np.unique(np.concatenate([g_a, g_b[1:]]))


def ch3_lik_w12_mesh_pack(
    study, exam, y, b, *, w1_lo, w1_hi, w2_lo, w2_hi, grid_n=None, quadrant_fine=False,
):
    if quadrant_fine:
        mid = float(CH3_LIK_W12_QUAD_MID)
        nc = int(CH3_LIK_W12_GRID_N_COARSE)
        nf = int(CH3_LIK_W12_GRID_N_FINE)
        g1 = _ch3_lik_w12_axis_refined(w1_lo, w1_hi, mid, nc, nf, fine_lo_half=False)
        g2 = _ch3_lik_w12_axis_refined(w2_lo, w2_hi, mid, nc, nf, fine_lo_half=True)
    else:
        gn = int(CH3_LIK_W12_GRID_N if grid_n is None else grid_n)
        g1 = np.linspace(float(w1_lo), float(w1_hi), gn, dtype=np.float64)
        g2 = np.linspace(float(w2_lo), float(w2_hi), gn, dtype=np.float64)
    W1m, W2m = np.meshgrid(g1, g2, indexing="ij")
    bf = np.full(W1m.size, float(b), dtype=np.float64)
    Zf = _ch3_likelihood_on_flat_w12_grid(study, exam, y, W1m.ravel(), W2m.ravel(), bf)
    Z = Zf.reshape(W1m.shape)
    return {
        "W1m": W1m, "W2m": W2m, "Z": Z,
        "w1_lo": float(w1_lo), "w1_hi": float(w1_hi),
        "w2_lo": float(w2_lo), "w2_hi": float(w2_hi),
    }


def ch3_figure_lik_w12_3d(*, duo_width_ratios=None):
    """Split screen: dataset+knobs (left), tall 3D likelihood ridge (right)."""
    wr = CH3_LIK_W12_WIDTH_RATIOS if duo_width_ratios is None else duo_width_ratios
    fig = plt.figure(figsize=CH3_LIK_W12_FIGSIZE)
    gs = fig.add_gridspec(
        1, 2, width_ratios=wr, wspace=CH3_DUO_WSPACE,
    )
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE,
    )
    ax_data = fig.add_subplot(g_left[0,  0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    ax3d = fig.add_subplot(gs[0, 1], projection="3d")
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    from ch4_layout import ch4_duo_plot_layout_tune, ch4_duo_knob_layout_tune

    ch4_duo_plot_layout_tune(fig, ax_data, ax3d)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    ch4_duo_knob_layout_tune(fig, ax_data, axes_k)
    return fig, ax_data, ax3d, axes_k


def ch4_figure_duo_weight3d():
    """Ch4 duo layout — legacy 2D/knob column, wider square 3D."""
    from ch4_layout import CH4_DUO_WIDTH_RATIOS

    return ch3_figure_lik_w12_3d(duo_width_ratios=CH4_DUO_WIDTH_RATIOS)


def ch3_lik_w12_z_limits(Z, *, scale=1.0):
    z_hi = float(np.nanmax(Z)) * float(scale)
    pad = 0.10 * max(z_hi, 1e-15)
    return 0.0, z_hi + pad


def ch3_lik_w12_facecolors_full(W1m, W2m, reveal_u, *, rgba):
    t = float(np.clip(float(reveal_u), 0.0, 1.0))
    fc = np.empty(W1m.shape + (4,), dtype=float)
    rgba = mpl.colors.to_rgba(rgba)
    fc[..., :] = rgba
    fc[..., 3] = rgba[3] * t
    return fc


def ch3_lik_w12_stack_z_lim(z_lik_hi, *, pad_frac=0.06):
    """Fixed display box: all stacked surfaces compress into [0, z_hi]."""
    z_hi = float(z_lik_hi)
    pad = float(pad_frac) * max(z_hi, 1.0)
    return 0.0, z_hi + pad


def ch3_lik_w12_squish_z(Z, layer_i, n_layers, z_lo, z_hi, z_ref):
    """Map mistake height into layer_i of n_layers equal slots in [z_lo, z_hi]."""
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    z_base = float(z_lo) + float(layer_i) * slot
    scale = slot / max(float(z_ref), 1e-9)
    return z_base + np.asarray(Z, dtype=float) * scale


def ch3_lik_w12_squish_scalar(z_val, layer_i, n_layers, z_lo, z_hi, z_ref):
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    z_base = float(z_lo) + float(layer_i) * slot
    scale = slot / max(float(z_ref), 1e-9)
    return z_base + float(z_val) * scale


def _ch3_lik_w12_z_at(W1m, W2m, Z, w1, w2):
    d = (np.asarray(W1m, dtype=float) - float(w1)) ** 2 + (np.asarray(W2m, dtype=float) - float(w2)) ** 2
    return float(np.ravel(np.asarray(Z, dtype=float))[int(np.nanargmin(d))])


def _ch3_lik_w12_crop_mesh_xy(W1m, W2m, Z, *, xlo, xhi, ylo, yhi, fc=None):
    """Subset a meshgrid surface to an (x, y) window so zoomed axes don't show out-of-box geometry."""
    W1m = np.asarray(W1m, dtype=float)
    W2m = np.asarray(W2m, dtype=float)
    Z = np.asarray(Z, dtype=float)
    g1 = W1m[:, 0]
    g2 = W2m[0, :]
    i_idx = np.flatnonzero((g1 >= float(xlo) - 1e-12) & (g1 <= float(xhi) + 1e-12))
    j_idx = np.flatnonzero((g2 >= float(ylo) - 1e-12) & (g2 <= float(yhi) + 1e-12))
    if i_idx.size < 2 or j_idx.size < 2:
        return W1m, W2m, Z, fc
    i0, i1 = int(i_idx[0]), int(i_idx[-1]) + 1
    j0, j1 = int(j_idx[0]), int(j_idx[-1]) + 1
    W1c = W1m[i0:i1, j0:j1]
    W2c = W2m[i0:i1, j0:j1]
    Zc = Z[i0:i1, j0:j1]
    fcc = fc[i0:i1, j0:j1] if fc is not None else None
    return W1c, W2c, Zc, fcc


def ch3_lik_w12_knob3_z_ticks(stack_layers, n_layers, z_lo, z_hi):
    """Tick positions at stacked-layer centers; labels are Knob 3 (b) values."""
    if not stack_layers:
        return None, None
    n = max(float(n_layers), 1.0)
    span = float(z_hi) - float(z_lo)
    slot = span / n
    layers = sorted(
        (
            sl for sl in stack_layers
            if float(sl.get("reveal", 1.0)) > 1e-4 and sl.get("b") is not None
        ),
        key=lambda sl: int(sl.get("layer_i", 0)),
    )
    if not layers:
        return None, None
    tick_z, tick_lbl = [], []
    for sl in layers:
        li = int(sl.get("layer_i", 0))
        tick_z.append(float(z_lo) + (float(li) + 0.5) * slot)
        tick_lbl.append(f"{float(sl['b']):.2g}")
    return tick_z, tick_lbl


def ch3_lik_w12_facecolors_diag(W1m, W2m, reveal_u, *, w1_lo, w1_hi, w2_lo, w2_hi, rgba, origin="lo_lo"):
    """Diagonal surface reveal; ``origin='lo_hi'`` starts at (w1_lo, w2_hi)."""
    u1 = (W1m - float(w1_lo)) / max(float(w1_hi) - float(w1_lo), 1e-9)
    if str(origin) == "lo_hi":
        u2 = (float(w2_hi) - W2m) / max(float(w2_hi) - float(w2_lo), 1e-9)
    else:
        u2 = (W2m - float(w2_lo)) / max(float(w2_hi) - float(w2_lo), 1e-9)
    t = float(np.clip(float(reveal_u), 0.0, 1.0))
    mask = (u1 + u2) <= 2.0 * t + 1e-9
    fc = np.empty(W1m.shape + (4,), dtype=float)
    rgba = mpl.colors.to_rgba(rgba)
    fc[..., :] = rgba
    fc[..., 3] = rgba[3] * mask.astype(float)
    return fc


def _ch3_lik_w12_here_annotation(
    fig, ax3d, mx, my, z_mark, *,
    label="WE ARE HERE",
    label_fig=None,
    color=None,
    edgecolor=None,
):
    """Arrow + label in figure space, upper-right of the 3D panel."""
    from matplotlib.patches import FancyArrowPatch

    fig.canvas.draw()
    px, py = ch3_ax3d_to_fig_xy(fig, ax3d, float(mx), float(my), float(z_mark))
    if label_fig is None:
        label_fig = CH3_LIK_RIDGE_LABEL_FIG
    lx, ly = float(label_fig[0]), float(label_fig[1])
    mcol = FAIL_COLOR if color is None else color
    medge = "white" if edgecolor is None else edgecolor
    fig.text(
        lx, ly, str(label),
        transform=fig.transFigure,
        fontsize=CH3_LIK_RIDGE_HERE_FS,
        color=mcol,
        fontweight="bold",
        ha="right",
        va="top",
        zorder=60,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="black", edgecolor=medge, alpha=0.95),
    )
    arrow = FancyArrowPatch(
        (lx, ly - 0.05), (px, py),
        transform=fig.transFigure,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=2.4,
        color=mcol,
        shrinkA=6,
        shrinkB=6,
        zorder=59,
    )
    fig.patches.append(arrow)


def ch3_frame_lik_w12_3d(
    study,
    exam,
    y,
    w_st,
    w_el,
    b,
    *,
    mesh_pack,
    z_lim,
    curves,
    elev,
    azim,
    emphasize_knob="st",
    landscape_reveal=0.0,
    landscape_reveal_origin="lo_lo",
    landscape_rgba=None,
    b_slices=None,
    slice_reveal=0.0,
    show_curves=True,
    marker=True,
    marker_z_offset=0.0,
    stack_layers=None,
    stack_n_layers=1.0,
    z_lik_ref=None,
    show_axis_labels=True,
    flat_surface=None,
    z_label=None,
    knob_pack=None,
    knob_scales=None,
    weight_axis_labels=False,
    marker_ws=None,
    marker_we=None,
    marker_z=None,
    marker_color=None,
    marker_edgecolors="white",
    here_annotation=False,
    here_label="WE ARE HERE",
    here_label_fig=None,
    ax3d_xlim=None,
    ax3d_ylim=None,
):
    spec = CH3_LOSS_SPECS["likelihood"]
    loss_fn = spec["fn"]
    w_st, w_el, b = float(w_st), float(w_el), float(b)
    W1m, W2m, Z = mesh_pack["W1m"], mesh_pack["W2m"], mesh_pack["Z"]
    w1_lo, w1_hi = mesh_pack["w1_lo"], mesh_pack["w1_hi"]
    w2_lo, w2_hi = mesh_pack["w2_lo"], mesh_pack["w2_hi"]
    z_lo_ax, z_hi_ax = float(z_lim[0]), float(z_lim[1])
    z_ref = float(np.nanmax(Z) if z_lik_ref is None else z_lik_ref)
    n_stack = max(float(stack_n_layers), 1.0)

    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(w_st, w_el, b, emphasize_knob)
    ch3_draw_left_panel(
        ax_data, w_st, w_el, b, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    if knob_pack is None:
        knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    else:
        knob_rgbs, canvas_sides = knob_pack
    if knob_scales is None:
        scales = ch3_knob_scales_emphasize(emphasize_knob, CH3_KNOB_ACTIVE_SCALE)
    else:
        scales = list(knob_scales)
    ch3_draw_knob_row(
        fig, axes_k, w_st, w_el, b, emphasize_knob,
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(w_st, w_el, b),
        knob_scales=scales, ax_data=ax_data,
    )

    ax3d.cla()
    ax3d.computed_zorder = False
    lr = landscape_rgba if landscape_rgba is not None else (CH4_LIK_SURFACE_COLOR, CH3_LIK_W12_SURFACE_ALPHA)
    z_pt = None
    mark_x = mark_y = mark_z = None

    def _maybe_crop_xy(W1, W2, Z, fc=None):
        if ax3d_xlim is None and ax3d_ylim is None:
            return W1, W2, Z, fc
        xlo = float(w1_lo if ax3d_xlim is None else ax3d_xlim[0])
        xhi = float(w1_hi if ax3d_xlim is None else ax3d_xlim[1])
        ylo = float(w2_lo if ax3d_ylim is None else ax3d_ylim[0])
        yhi = float(w2_hi if ax3d_ylim is None else ax3d_ylim[1])
        W1, W2, Z, fc = _ch3_lik_w12_crop_mesh_xy(
            W1, W2, Z, xlo=xlo, xhi=xhi, ylo=ylo, yhi=yhi, fc=fc,
        )
        z_lo_c, z_hi_c = float(z_lim[0]), float(z_lim[1])
        Z = np.clip(np.asarray(Z, dtype=float), z_lo_c, z_hi_c)
        return W1, W2, Z, fc

    if flat_surface is not None:
        Wb = flat_surface["W1m"]
        W2b = flat_surface["W2m"]
        Zb = np.asarray(flat_surface["Z"], dtype=float)
        if flat_surface.get("nll_heatmap") is not None:
            Wb, W2b, Zb, _ = _maybe_crop_xy(Wb, W2b, Zb)
            _ch4_nll_heatmap_plot_surface(
                ax3d, Wb, W2b, Zb, flat_surface["nll_heatmap"],
            )
        elif flat_surface.get("facecolors") is not None:
            fc_b = flat_surface["facecolors"]
            Wb, W2b, Zb, fc_b = _maybe_crop_xy(Wb, W2b, Zb, fc_b)
            ax3d.plot_surface(
                Wb, W2b, Zb, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
            )
        else:
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, W2b, 1.0,
                rgba=(CH4_LIK_SURFACE_COLOR, CH3_LIK_W12_SURFACE_ALPHA * float(flat_surface.get("alpha_scale", 1.0))),
            )
            Wb, W2b, Zb, fc_b = _maybe_crop_xy(Wb, W2b, Zb, fc_b)
            ax3d.plot_surface(
                Wb, W2b, Zb, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
            )
        z_pt = float(
            flat_surface["marker_z"]
            if flat_surface.get("marker_z") is not None
            else _ch3_lik_w12_z_at(Wb, W2b, Zb, w_st, w_el)
        )
    elif float(landscape_reveal) > 1e-4:
        fc = ch3_lik_w12_facecolors_diag(
            W1m, W2m, landscape_reveal,
            w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi, rgba=lr,
            origin=landscape_reveal_origin,
        )
        W1p, W2p, Zp, fc = _maybe_crop_xy(W1m, W2m, Z, fc)
        ax3d.plot_surface(
            W1p, W2p, Zp, facecolors=fc, shade=False,
            linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=1,
        )
    if stack_layers:
        for sl in stack_layers:
            rev = float(sl.get("reveal", 1.0)) * float(slice_reveal)
            if rev < 1e-4:
                continue
            pack_b = sl["pack"]
            Wb, Zb = pack_b["W1m"], pack_b["Z"]
            li = int(sl.get("layer_i", 0))
            Zplot = ch3_lik_w12_squish_z(Zb, li, n_stack, z_lo_ax, z_hi_ax, z_ref)
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, pack_b["W2m"], rev,
                rgba=(CH4_LIK_SURFACE_COLOR, CH3_LIK_W12_SURFACE_ALPHA * float(sl.get("alpha_scale", 1.0))),
            )
            ax3d.plot_surface(
                Wb, pack_b["W2m"], Zplot, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=int(sl.get("zorder", 3)),
            )
    elif b_slices:
        for sl in b_slices:
            bb = float(sl["b"])
            rev = float(sl.get("reveal", 1.0)) * float(slice_reveal)
            if rev < 1e-4:
                continue
            pack_b = sl.get("pack")
            if pack_b is None:
                pack_b = ch3_lik_w12_mesh_pack(
                    study, exam, y, bb, w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi,
                )
            Wb, Zb = pack_b["W1m"], pack_b["Z"]
            z0 = float(sl.get("z_base", 0.0))
            fc_b = ch3_lik_w12_facecolors_full(
                Wb, pack_b["W2m"], rev,
                rgba=(CH4_LIK_SURFACE_COLOR, CH3_LIK_W12_SLICE_ALPHA * float(sl.get("alpha_scale", 1.0))),
            )
            ax3d.plot_surface(
                Wb, pack_b["W2m"], Zb + z0, facecolors=fc_b, shade=False,
                linewidth=0, antialiased=False, rstride=1, cstride=1, zorder=3,
            )
    if show_curves and curves:
        for cw1, cw2, cz in curves:
            cw1 = np.asarray(cw1, dtype=float)
            cw2 = np.asarray(cw2, dtype=float)
            cz = np.asarray(cz, dtype=float) + float(marker_z_offset)
            if cw1.size >= 2:
                ax3d.plot(
                    cw1, cw2, cz, color=CH4_LIK_SURFACE_COLOR, linewidth=CH3_LIK_W12_CURVE_LW,
                    alpha=0.95, zorder=8,
                )
    if z_pt is None:
        z_raw = float(loss_fn(w_st, w_el, b, study, exam, y))
        if stack_layers:
            top_i = max(int(sl.get("layer_i", 0)) for sl in stack_layers if float(sl.get("reveal", 0)) > 1e-4)
            z_pt = ch3_lik_w12_squish_scalar(z_raw, top_i, n_stack, z_lo_ax, z_hi_ax, z_ref)
        else:
            z_pt = z_raw + float(marker_z_offset)
    if marker:
        mx = float(w_st if marker_ws is None else marker_ws)
        my = float(w_el if marker_we is None else marker_we)
        if marker_z is not None:
            z_pt = float(marker_z)
        elif marker_ws is not None or marker_we is not None:
            if flat_surface is not None:
                Wb = flat_surface["W1m"]
                W2b = flat_surface["W2m"]
                Zb = np.asarray(flat_surface["Z"], dtype=float)
                z_pt = _ch3_lik_w12_z_at(Wb, W2b, Zb, mx, my)
            else:
                z_pt = float(loss_fn(mx, my, b, study, exam, y)) + float(marker_z_offset)
        mcol = FAIL_COLOR if marker_color is None else marker_color
        z_span = max(z_hi_ax - z_lo_ax, 1e-12)
        z_mark = float(z_pt) + float(CH3_LIK_MARKER_Z_BUMP_FRAC) * z_span
        mark_x, mark_y, mark_z = mx, my, z_mark
        ax3d.scatter(
            [mx], [my], [z_mark],
            color=mcol, edgecolors=marker_edgecolors, linewidths=2.0,
            s=260.0, depthshade=False, zorder=30,
        )
    if ax3d_xlim is not None:
        ax3d.set_xlim(float(ax3d_xlim[0]), float(ax3d_xlim[1]))
    else:
        ax3d.set_xlim(w1_lo, w1_hi)
    if ax3d_ylim is not None:
        ax3d.set_ylim(float(ax3d_ylim[0]), float(ax3d_ylim[1]))
    else:
        ax3d.set_ylim(w2_lo, w2_hi)
    ax3d.set_zlim(float(z_lim[0]), float(z_lim[1]))
    knob3_zticks, knob3_zlabels = ch3_lik_w12_knob3_z_ticks(stack_layers, n_stack, z_lo_ax, z_hi_ax)
    if show_axis_labels:
        fs_ax = float(AXIS_LABEL_SIZE) * float(CH3_LIK_3D_AXIS_LABEL_SCALE)
        if weight_axis_labels:
            ax3d.set_xlabel(r"$w_{\mathrm{ST}}$", fontsize=fs_ax, labelpad=10)
            ax3d.set_ylabel(r"$w_{\mathrm{EL}}$", fontsize=fs_ax, labelpad=10)
        else:
            ax3d.set_xlabel("Knob 1", fontsize=AXIS_LABEL_SIZE, labelpad=10)
            ax3d.set_ylabel("Knob 2", fontsize=AXIS_LABEL_SIZE, labelpad=10)
        if knob3_zticks:
            ax3d.set_zlabel("Knob 3", fontsize=AXIS_LABEL_SIZE, labelpad=10)
            ax3d.set_zticks(knob3_zticks)
            ax3d.set_zticklabels(knob3_zlabels)
        elif weight_axis_labels:
            from matplotlib.ticker import MaxNLocator

            ax3d.set_zlabel(str(z_label or r"$b$"), fontsize=fs_ax, labelpad=10)
            ax3d.zaxis.set_major_locator(MaxNLocator(nbins=5))
        else:
            ax3d.set_zlabel(str(z_label or "Likelihood"), fontsize=AXIS_LABEL_SIZE, labelpad=10)
        ax3d.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
        ax3d.grid(True)
    else:
        ax3d.set_xlabel("")
        ax3d.set_ylabel("")
        ax3d.set_zlabel("")
        ax3d.set_xticklabels([])
        ax3d.set_yticklabels([])
        ax3d.set_zticklabels([])
    ax3d.view_init(elev=float(elev), azim=float(azim))
    if bool(here_annotation) and mark_x is not None:
        _ch3_lik_w12_here_annotation(
            fig, ax3d, mark_x, mark_y, mark_z,
            label=here_label,
            label_fig=here_label_fig,
            color=marker_color,
            edgecolor=marker_edgecolors,
        )
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)


def _ch3_lik_w12_azim_shortest_delta(az0, az1):
    """Signed azimuth delta in (-180, 180] — shortest rotation."""
    return (float(az1) - float(az0) + 180.0) % 360.0 - 180.0


def _ch3_lik_w12_lerp_azim_shortest(az0, az1, u):
    return float(az0) + _ch3_lik_w12_azim_shortest_delta(az0, az1) * float(u)


def ch3_lik_w12_frame_opening(mesh_pack, z_lim, *, w_st=None, w_el=None):
    """Opening frame: empty 3-D at ``(w_st, w_el)`` (defaults: script corner)."""
    w1_c = float(CH3_LIK_W12_CORNER_W1 if w_st is None else w_st)
    w2_c = float(CH3_LIK_W12_CORNER_W2 if w_el is None else w_el)
    b0 = float(CH3_LIK_W12_B0)
    return ch3_frame_lik_w12_3d(
        study_sep, exam_sep, y_sep, w1_c, w2_c, b0,
        mesh_pack=mesh_pack, z_lim=z_lim, curves=[],
        elev=CH3_LIK_W12_ELEV_W1, azim=CH3_LIK_W12_AZIM_W1,
        emphasize_knob="st", show_curves=False, marker=False,
    )


def _ch3_lik_w12_trace_knob1(study, exam, y, w2_fix, b, w1_from, w1_to, n):
    w1s = np.linspace(float(w1_from), float(w1_to), int(n), dtype=float)
    z = [_ch3_likelihood_on_flat_w12_grid(study, exam, y, [w], [w2_fix], [b])[0] for w in w1s]
    return w1s, np.full_like(w1s, float(w2_fix)), np.asarray(z, dtype=float)


def _ch3_lik_w12_trace_knob2(study, exam, y, w1_fix, b, w2_from, w2_to, n):
    w2s = np.linspace(float(w2_from), float(w2_to), int(n), dtype=float)
    z = [_ch3_likelihood_on_flat_w12_grid(study, exam, y, [w1_fix], [w], [b])[0] for w in w2s]
    return np.full_like(w2s, float(w1_fix)), w2s, np.asarray(z, dtype=float)


def _ch4_lik_02_03_handoff_state():
    """Shared end-of-02 / start-of-03 pose: weights (3, -3, 0), ±3 mesh, no marker."""
    study, exam, y = study_sep, exam_sep, y_sep
    lo = float(CH3_LIK_W12_3D_LO)
    hi = float(CH3_LIK_W12_3D_HI)
    b0 = float(CH3_LIK_W12_B0)
    mesh = ch3_lik_w12_mesh_pack(
        study, exam, y, b0,
        w1_lo=lo, w1_hi=hi, w2_lo=lo, w2_hi=hi,
        quadrant_fine=True,
    )
    return {
        "study": study, "exam": exam, "y": y,
        "w_st": hi, "w_el": lo, "b": b0,
        "mesh": mesh, "z_lim": ch3_lik_w12_z_limits(mesh["Z"], scale=1.0),
    }


def _ch4_lik_resolve_knob_pack(knob_labeled_blend):
    """Stable numbered / labeled / blended knob row — avoids blend flicker at u≈1."""
    if knob_labeled_blend is None:
        return None
    blends = tuple(float(x) for x in knob_labeled_blend)
    if all(b >= 1.0 - 1e-6 for b in blends):
        from ch4_layout import ch4_knob_asset_pack

        return ch4_knob_asset_pack()
    if all(b <= 1e-6 for b in blends):
        return ch3_knob_asset_pack()
    from ch4_layout import ch4_knob_asset_pack, ch4_knob_asset_pack_blended

    return ch4_knob_asset_pack_blended(
        ch3_knob_asset_pack(),
        ch4_knob_asset_pack(),
        blends,
    )


def _ch4_lik_02_03_handoff_plot(
    *,
    knob_labeled_blend=None,
    emphasize_knob=None,
    display_ws=None,
    display_we=None,
    display_b=None,
    marker=False,
    marker_ws=None,
    marker_we=None,
    marker_color=None,
    marker_edgecolors="white",
    ax3d_xlim=None,
    ax3d_ylim=None,
    elev=None,
    azim=None,
):
    """Full likelihood landscape at CT view — matches 02 end and 03 opening."""
    st = _ch4_lik_02_03_handoff_state()
    ws = float(display_ws if display_ws is not None else st["w_st"])
    we = float(display_we if display_we is not None else st["w_el"])
    bb = float(display_b if display_b is not None else st["b"])
    labeled = (
        knob_labeled_blend is not None
        and all(float(x) >= 1.0 - 1e-6 for x in knob_labeled_blend)
    )
    emp = "all" if labeled else ("st" if emphasize_knob is None else emphasize_knob)
    knob_pack = _ch4_lik_resolve_knob_pack(knob_labeled_blend)
    el = float(CH3_LIK_W12_CT_ELEV if elev is None else elev)
    az = float(CH3_LIK_W12_CT_AZIM if azim is None else azim)
    mesh = st["mesh"]
    z_lim = st["z_lim"]
    if ax3d_xlim is not None and ax3d_ylim is not None:
        xlo, xhi = float(ax3d_xlim[0]), float(ax3d_xlim[1])
        ylo, yhi = float(ax3d_ylim[0]), float(ax3d_ylim[1])
        mesh = ch3_lik_w12_mesh_pack(
            st["study"], st["exam"], st["y"], bb,
            w1_lo=xlo, w1_hi=xhi, w2_lo=ylo, w2_hi=yhi,
            grid_n=int(CH3_LIK_W12_ZOOM_GRID),
            quadrant_fine=False,
        )
        z_lim = ch3_lik_w12_z_limits(mesh["Z"], scale=1.0)
    return ch3_frame_lik_w12_3d(
        st["study"], st["exam"], st["y"], ws, we, bb,
        mesh_pack=mesh, z_lim=z_lim, curves=[],
        elev=el, azim=az, emphasize_knob=emp,
        landscape_reveal=1.0, landscape_reveal_origin="lo_hi",
        show_curves=False, marker=marker,
        knob_pack=knob_pack,
        knob_scales=[1.0, 1.0, 1.0],
        weight_axis_labels=labeled,
        z_label="Likelihood",
        marker_ws=marker_ws,
        marker_we=marker_we,
        marker_color=marker_color,
        marker_edgecolors=marker_edgecolors,
        ax3d_xlim=ax3d_xlim,
        ax3d_ylim=ax3d_ylim,
    )


def _ch4_02_compose_plot(plot_img):
    """Same shell as ch4_03 opening (layout_u=0, full-width plot, no rails)."""
    return _ch4_lik_03_opening_compose(plot_img, layout_u=0.0)


def ch3_build_frames_likelihood_w12_landscape_story():
    study, exam, y = study_sep, exam_sep, y_sep
    b0 = float(CH3_LIK_W12_B0)
    lo = float(CH3_LIK_W12_3D_LO)
    hi = float(CH3_LIK_W12_3D_HI)
    w1s, w2s = hi, lo  # start (knob1, knob2) = (3, -3)
    el_ct = float(CH3_LIK_W12_CT_ELEV)
    az_ct = float(CH3_LIK_W12_CT_AZIM)
    el_w1 = float(CH3_LIK_W12_ELEV_W1)
    az_w1 = float(CH3_LIK_W12_AZIM_W1)
    el_w2 = float(CH3_LIK_W12_ELEV_W2)
    az_w2 = float(CH3_LIK_W12_AZIM_W2)

    mesh0 = ch3_lik_w12_mesh_pack(
        study, exam, y, b0,
        w1_lo=lo, w1_hi=hi,
        w2_lo=lo, w2_hi=hi,
        quadrant_fine=True,
    )
    z_lik_hi = float(np.nanmax(mesh0["Z"]))
    z_lim_full = ch3_lik_w12_z_limits(mesh0["Z"], scale=1.0)
    n_trace = max(24, CH3_LIK_W12_N_KNOB)

    frames = []
    curves = []

    def emit(
        ws, we, bb, *, elev, azim, emp="st", lrev=0.0, show_curves=True,
        z_lim=None, marker=True, srev=0.0, marker_z_offset=0.0, stack_layers=None,
        stack_n_layers=1.0, z_lik_ref=None, landscape_reveal_origin="lo_lo",
    ):
        fr = ch3_frame_lik_w12_3d(
            study, exam, y, ws, we, bb,
            mesh_pack=mesh0, z_lim=z_lim or z_lim_full,
            curves=list(curves) if show_curves else [],
            elev=elev, azim=azim, emphasize_knob=emp,
            landscape_reveal=lrev, landscape_reveal_origin=landscape_reveal_origin,
            slice_reveal=srev,
            show_curves=show_curves, marker=marker,
            marker_z_offset=marker_z_offset, stack_layers=stack_layers,
            stack_n_layers=stack_n_layers,
            z_lik_ref=z_lik_hi if z_lik_ref is None else z_lik_ref,
        )
        frames.append(_ch4_02_compose_plot(fr))

    def hold(ws, we, bb, n, **kw):
        for _ in range(int(n)):
            emit(ws, we, bb, **kw)

    # 1–2: empty 3D at (3, -3)
    open_fr = ch3_lik_w12_frame_opening(mesh0, z_lim_full, w_st=w1s, w_el=w2s)
    open_comp = _ch4_02_compose_plot(open_fr)
    for _ in range(CH3_LIK_W12_N_HOLD):
        frames.append(open_comp.copy())

    # 3: knob 1 — w1: 3 → -3 (w2 = -3)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w1c = ch3_lerp(hi, lo, u)
        cw1, cw2, cz = _ch3_lik_w12_trace_knob1(study, exam, y, w2s, b0, hi, w1c, n_trace)
        if curves:
            curves[-1] = (cw1, cw2, cz)
        else:
            curves.append((cw1, cw2, cz))
        emit(w1c, w2s, b0, elev=el_w1, azim=az_w1, emp="st")
    hold(lo, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w1, azim=az_w1)
    curves[-1] = _ch3_lik_w12_trace_knob1(study, exam, y, w2s, b0, hi, lo, n_trace)

    # 3b: knob 1 — w1: -3 → 3
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w1c = ch3_lerp(lo, hi, u)
        emit(w1c, w2s, b0, elev=el_w1, azim=az_w1, emp="st")
    hold(w1s, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w1, azim=az_w1)

    # 4: shortest rotation to knob-2 view
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_ROT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1s, w2s, b0,
            elev=ch3_lerp(el_w1, el_w2, u),
            azim=_ch3_lik_w12_lerp_azim_shortest(az_w1, az_w2, u),
        )

    # 5: knob 2 — w2: -3 → 3 (w1 = 3)
    curves.append(_ch3_lik_w12_trace_knob2(study, exam, y, w1s, b0, w2s, w2s, 2))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w2c = ch3_lerp(w2s, hi, u)
        cw1, cw2, cz = _ch3_lik_w12_trace_knob2(study, exam, y, w1s, b0, w2s, w2c, n_trace)
        curves[-1] = (cw1, cw2, cz)
        emit(w1s, w2c, b0, elev=el_w2, azim=az_w2, emp="el")
    hold(w1s, hi, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w2, azim=az_w2)

    # 5b: knob 2 — w2: 3 → -3
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_KNOB, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        w2c = ch3_lerp(hi, w2s, u)
        emit(w1s, w2c, b0, elev=el_w2, azim=az_w2, emp="el")
    hold(w1s, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_w2, azim=az_w2)

    # 6: shortest rotation to ch4_03 CT view — keep for fill + landscape
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_ROT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1s, w2s, b0,
            elev=ch3_lerp(el_w2, el_ct, u),
            azim=_ch3_lik_w12_lerp_azim_shortest(az_w2, az_ct, u),
        )

    # 7: raster fill — evenly spaced curves (surface mesh stays finer in one quadrant)
    w1_vals = np.linspace(lo, hi, int(CH3_LIK_W12_N_FILL_LINES), dtype=np.float64)
    w2_vals = np.linspace(lo, hi, int(CH3_LIK_W12_N_FILL_LINES), dtype=np.float64)
    n_trace_fill = int(CH3_LIK_W12_N_FILL_TRACE)
    for w1v in w1_vals:
        cw1, cw2, cz = _ch3_lik_w12_trace_knob2(
            study, exam, y, float(w1v), b0, lo, hi, n_trace_fill,
        )
        curves.append((cw1, cw2, cz))
        emit(w1s, w2s, b0, elev=el_ct, azim=az_ct, emp="st", marker=True)
    for w2v in w2_vals:
        cw1, cw2, cz = _ch3_lik_w12_trace_knob1(
            study, exam, y, float(w2v), b0, lo, hi, n_trace_fill,
        )
        curves.append((cw1, cw2, cz))
        emit(w1s, w2s, b0, elev=el_ct, azim=az_ct, emp="el", marker=True)

    hold(w1s, w2s, b0, CH3_LIK_W12_N_HOLD // 2, elev=el_ct, azim=az_ct, marker=True)

    # 8: diagonal landscape reveal from (-3, 3); then drop marker
    for tv in np.linspace(0.0, 1.0, CH3_LIK_W12_N_REVEAL, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            w1s, w2s, b0, elev=el_ct, azim=az_ct,
            lrev=u, show_curves=True, marker=False,
            landscape_reveal_origin="lo_hi",
        )
    emit(
        w1s, w2s, b0, elev=el_ct, azim=az_ct, lrev=1.0,
        show_curves=False, marker=False, landscape_reveal_origin="lo_hi",
    )
    handoff_comp = _ch4_02_compose_plot(_ch4_lik_02_03_handoff_plot())
    for _ in range(CH3_LIK_W12_N_HOLD):
        frames.append(handoff_comp.copy())
    return frames


def ch4_preview_likelihood_w12_landscape_last_frame():
    return _ch4_02_compose_plot(_ch4_lik_02_03_handoff_plot())


def ch4_export_likelihood_w12_landscape():
    frames = ch3_build_frames_likelihood_w12_landscape_story()
    fn = "ch4_02_likelihood_w12_landscape.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_W12_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


# --- ch4_02a: wide 2D prob labels → slide to duo-left → labels fade → ch4_02 opening ---

CH4_02A_W_ST = float(CH3_LIK_W12_3D_HI)
CH4_02A_W_EL = float(CH3_LIK_W12_3D_LO)
CH4_02A_B = float(CH3_LIK_W12_B0)
CH4_02A_MS = 68 if not _CH3_DRAFT else 110
CH4_02A_N_HOLD = 8 if _CH3_DRAFT else 16
CH4_02A_N_CMAP_REVEAL = 4 if _CH3_DRAFT else max(24, _smooth_n(18))
CH4_02A_N_SLIDE = 6 if _CH3_DRAFT else max(40, _smooth_n(30))  # layout slide + knob fly-in (02b style)
CH4_02A_N_LABEL_REVEAL = 2 if _CH3_DRAFT else max(2, _smooth_n(1))
CH4_02A_N_LABEL_HOLD = 4 if _CH3_DRAFT else max(14, _smooth_n(10))
CH4_02A_N_LABEL_FADE = 4 if _CH3_DRAFT else max(28, _smooth_n(22))
CH4_02A_N_3D_REVEAL = 4 if _CH3_DRAFT else max(32, _smooth_n(24))
CH4_02A_N_HOLD_END = 4 if _CH3_DRAFT else max(10, CH3_LIK_W12_N_HOLD // 4)

_CH4_02A_MESH = None


def _ch4_02a_opening_mesh():
    global _CH4_02A_MESH
    if _CH4_02A_MESH is not None:
        return _CH4_02A_MESH
    lo = float(CH3_LIK_W12_3D_LO)
    hi = float(CH3_LIK_W12_3D_HI)
    b0 = float(CH3_LIK_W12_B0)
    mesh = ch3_lik_w12_mesh_pack(
        study_sep, exam_sep, y_sep, b0,
        w1_lo=lo, w1_hi=hi, w2_lo=lo, w2_hi=hi,
        quadrant_fine=True,
    )
    z_lim = ch3_lik_w12_z_limits(mesh["Z"], scale=1.0)
    _CH4_02A_MESH = (mesh, z_lim)
    return _CH4_02A_MESH


def _ch4_02a_compose(plot_img):
    return _ch4_02_compose_plot(plot_img)


def _ch4_02a_lerp_3d_rect(u, r_full):
    u = float(ch3_knob_smoothstep(np.clip(float(u), 0.0, 1.0)))
    if u <= 1e-5:
        return None
    x0, y0, w, h = r_full
    w = float(w) * u
    x0 = float(x0) + float(r_full[2]) * (1.0 - u)
    return (x0, float(y0), w, float(h))


def _ch4_02a_label_color(col, alpha):
    rgba = mpl.colors.to_rgba(col)
    a = float(np.clip(float(alpha), 0.0, 1.0))
    return (rgba[0], rgba[1], rgba[2], rgba[3] * a)


def _ch4_02a_draw_pct_label(ax, text, x, y, color, *, dx=0.20, dy=0.22, ha="left", va="bottom", alpha=1.0):
    ax.annotate(
        str(text),
        (float(x), float(y)),
        xytext=(float(x) + float(dx), float(y) + float(dy)),
        textcoords="data",
        ha=str(ha),
        va=str(va),
        fontsize=float(CH3_PROB_DATA_LABEL_FS),
        fontweight="bold",
        color=_ch4_02a_label_color(color, alpha),
        zorder=25,
    )


def _ch4_02a_draw_opening_3d(ax3d, mesh_pack, z_lim):
    w1_lo, w1_hi = mesh_pack["w1_lo"], mesh_pack["w1_hi"]
    w2_lo, w2_hi = mesh_pack["w2_lo"], mesh_pack["w2_hi"]
    ax3d.cla()
    ax3d.set_xlim(w1_lo, w1_hi)
    ax3d.set_ylim(w2_lo, w2_hi)
    ax3d.set_zlim(float(z_lim[0]), float(z_lim[1]))
    ax3d.set_xlabel("Knob 1", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax3d.set_ylabel("Knob 2", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax3d.set_zlabel("Likelihood", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax3d.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
    ax3d.grid(True)
    ax3d.view_init(elev=float(CH3_LIK_W12_ELEV_W1), azim=float(CH3_LIK_W12_AZIM_W1))


def _ch4_02a_prob_frame(
    w_st,
    w_el,
    b,
    study,
    exam,
    y,
    *,
    slide_u=0.0,
    panel3d_u=0.0,
    cmap_alpha=0.0,
    knob_grow_u=0.0,
    data_labels=None,
    label_alpha=1.0,
):
    cmap_full = float(_CH3_SIGMA_CONTOUR_ALPHA)
    ca = float(np.clip(float(cmap_alpha), 0.0, cmap_full))
    wide_data, wide_knobs = _ch4_02b_wide_layout()
    duo_data, duo_knobs, duo_3d = _ch4_02b_duo_layout()
    su = float(np.clip(float(slide_u), 0.0, 1.0))
    data_r = _ch4_02b_lerp_rect(su, wide_data, duo_data)
    knob_rs = tuple(_ch4_02b_lerp_rect(su, wide_knobs[i], duo_knobs[i]) for i in range(3))

    fig = plt.figure(figsize=CH4_DUO_FIGSIZE)
    fig.patch.set_facecolor("white")
    axd = fig.add_axes(data_r)
    leg = legend_linear_equation_values_bold_param(float(w_st), float(w_el), float(b), None)
    stg, elg = ST_KNOB, EL_KNOB
    sigma_Z = sigmoid(logits_plane(float(w_st), float(w_el), float(b), stg, elg))
    ch3_draw_left_panel(
        axd,
        float(w_st),
        float(w_el),
        float(b),
        study,
        exam,
        y,
        leg,
        show_colormap=ca > 1e-5,
        highlight_mistakes_flag=False,
        sigma_Z=sigma_Z if ca > 1e-5 else None,
        colormap_alpha=ca,
    )
    axd.set_xlim(*xlim)
    axd.set_ylim(*ylim)
    finalize_style_legend_tex(axd)
    if data_labels:
        la = float(np.clip(float(label_alpha), 0.0, 1.0))
        if la > 1e-4:
            for item in data_labels:
                if item is None:
                    continue
                txt, tx, ty = item[0], float(item[1]), float(item[2])
                col = item[4] if len(item) > 4 else CH3_PASS_LABEL_COLOR
                dx = float(item[5]) if len(item) > 5 else CH3_PROB_LABEL_DX
                dy = float(item[6]) if len(item) > 6 else CH3_PROB_LABEL_DY
                ha = item[7] if len(item) > 7 else "left"
                va = item[8] if len(item) > 8 else "bottom"
                _ch4_02a_draw_pct_label(axd, txt, tx, ty, col, dx=dx, dy=dy, ha=ha, va=va, alpha=la)

    _ch4_02b_place_knobs_flyin(fig, axd, data_r, knob_rs, w_st, w_el, b, grow_u=knob_grow_u)

    pu = float(np.clip(float(panel3d_u), 0.0, 1.0))
    if pu > 1e-5:
        mesh_pack, z_lim = _ch4_02a_opening_mesh()
        r3d = _ch4_02a_lerp_3d_rect(pu, duo_3d)
        if r3d is not None:
            ax3d = fig.add_axes(r3d, projection="3d")
            _ch4_02a_draw_opening_3d(ax3d, mesh_pack, z_lim)

    fig.canvas.draw()
    return fig_to_image(fig, dpi=CH3_ANIM_DPI)


def _ch4_02a_opening_frame():
    mesh, z_lim = _ch4_02a_opening_mesh()
    return ch3_lik_w12_frame_opening(
        mesh, z_lim, w_st=CH4_02A_W_ST, w_el=CH4_02A_W_EL,
    )


def _ch4_02a_append_hold(frames, im, n=None):
    n = int(CH4_02A_N_HOLD if n is None else n)
    for _ in range(max(1, n)):
        frames.append(im.copy() if hasattr(im, "copy") else im)


def ch4_build_frames_prob_labels_slide(with_noise=False):
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    w_st, w_el, b = CH4_02A_W_ST, CH4_02A_W_EL, CH4_02A_B
    cmap_full = float(_CH3_SIGMA_CONTOUR_ALPHA)
    frames = []

    im0 = _ch4_02a_compose(_ch4_02a_prob_frame(
        w_st, w_el, b, study, exam, y,
        slide_u=0.0, cmap_alpha=0.0, knob_grow_u=0.0,
    ))
    _ch4_02a_append_hold(frames, im0)

    for tv in np.linspace(0.0, 1.0, int(CH4_02A_N_CMAP_REVEAL), endpoint=True):
        u = float(ch3_knob_smoothstep(float(tv)))
        frames.append(_ch4_02a_compose(_ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=0.0, cmap_alpha=u * cmap_full, knob_grow_u=0.0,
        )))

    for tv in np.linspace(0.0, 1.0, int(CH4_02A_N_SLIDE), endpoint=True):
        u = float(ch3_knob_smoothstep(float(tv)))
        frames.append(_ch4_02a_compose(_ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=u, cmap_alpha=cmap_full, knob_grow_u=u,
        )))
    _ch4_02a_append_hold(frames, frames[-1])

    pass_idx, fail_idx = ch3_student_label_indices(study, exam, y)
    n_rev = int(CH4_02A_N_LABEL_REVEAL)

    def _append_labeled(rows):
        im = _ch4_02a_compose(_ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0, data_labels=ch3_rows_to_data_labels(rows),
        ))
        for _ in range(n_rev):
            frames.append(im)

    shown = []
    for i in pass_idx:
        shown.append(ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, fail_red=False))
        _append_labeled(shown)
    for i in fail_idx:
        shown.append(ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, fail_red=False))
        _append_labeled(shown)

    labels_pre = ch3_rows_to_data_labels(shown)
    im_hold = _ch4_02a_compose(_ch4_02a_prob_frame(
        w_st, w_el, b, study, exam, y,
        slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0, data_labels=labels_pre,
    ))
    _ch4_02a_append_hold(frames, im_hold, CH4_02A_N_LABEL_HOLD)

    for tv in np.linspace(0.0, 1.0, int(CH4_02A_N_LABEL_FADE), endpoint=True):
        u = float(ch3_knob_smoothstep(float(tv)))
        alpha = 1.0 - u
        frames.append(_ch4_02a_compose(_ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0,
            data_labels=labels_pre, label_alpha=alpha,
        )))

    for tv in np.linspace(0.0, 1.0, int(CH4_02A_N_3D_REVEAL), endpoint=True):
        frames.append(_ch4_02a_compose(_ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0, panel3d_u=float(tv),
        )))

    opening = _ch4_02a_compose(_ch4_02a_opening_frame())
    _ch4_02a_append_hold(frames, opening, CH4_02A_N_HOLD_END)
    return frames


def ch4_export_prob_labels_slide(with_noise=False):
    frames = ch4_build_frames_prob_labels_slide(with_noise)
    fn = "ch4_02a_prob_labels_slide.mp4"
    save_mp4(frames, fn, duration=int(CH4_02A_MS))
    del frames
    gc.collect()
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def _ch4_02a_all_prob_labels(study, exam, y, w_st, w_el, b):
    pass_idx, fail_idx = ch3_student_label_indices(study, exam, y)
    shown = []
    for i in pass_idx:
        shown.append(ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, fail_red=False))
    for i in fail_idx:
        shown.append(ch3_label_row_for_index(study, exam, y, w_st, w_el, b, i, fail_red=False))
    return ch3_rows_to_data_labels(shown)


def ch4_preview_prob_labels_slide_frame(*, phase="wide", with_noise=False):
    """Single ch4_02a frame — ``wide`` | ``cmap`` | ``knobs`` | ``duo`` | ``labels`` | ``fade`` | ``sig3d`` | ``end``."""
    study = study_real if with_noise else study_sep
    exam = exam_real if with_noise else exam_sep
    y = y_real if with_noise else y_sep
    w_st, w_el, b = CH4_02A_W_ST, CH4_02A_W_EL, CH4_02A_B
    cmap_full = float(_CH3_SIGMA_CONTOUR_ALPHA)
    phase = str(phase)
    labels = _ch4_02a_all_prob_labels(study, exam, y, w_st, w_el, b)

    if phase == "wide":
        im = _ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=0.0, cmap_alpha=0.0, knob_grow_u=0.0,
        )
    elif phase == "cmap":
        im = _ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=0.0, cmap_alpha=cmap_full, knob_grow_u=0.0,
        )
    elif phase == "knobs":
        im = _ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=0.5, cmap_alpha=cmap_full, knob_grow_u=0.5,
        )
    elif phase in ("duo", "slide_end"):
        im = _ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0,
        )
    elif phase == "labels":
        im = _ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0,
            data_labels=labels, label_alpha=1.0,
        )
    elif phase == "fade":
        im = _ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0,
            data_labels=labels, label_alpha=0.0,
        )
    elif phase == "sig3d":
        im = _ch4_02a_prob_frame(
            w_st, w_el, b, study, exam, y,
            slide_u=1.0, cmap_alpha=cmap_full, knob_grow_u=1.0, panel3d_u=1.0,
        )
    elif phase == "end":
        return _ch4_02a_compose(_ch4_02a_opening_frame())
    else:
        raise ValueError(f"unknown ch4_02a preview phase: {phase!r}")
    return _ch4_02a_compose(im)


# --- ch4_03/04: likelihood ch4 notation/NLL morph → 3D measurements + trajectory ---

CH3_LIK_CH4_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_CH4_W_LO = -4.0
CH3_LIK_CH4_W_HI = 4.0
CH3_LIK_CH4_PLANE_B_MID = 0.0
CH3_LIK_CH4_PLANE_B = -4.0
CH3_LIK_CH4_SURFACE_ALPHA = 0.6
CH3_LIK_CH4_CT_ELEV = 24.0
CH3_LIK_CH4_CT_AZIM = -128.0
# Shared CT slice mesh (matches ch4_05a axis sweeps + voxel cut planes).
CH3_LIK_CT_GRID = 18 if _CH3_DRAFT else 32
CH3_LIK_CT_PLANE_ALPHA = 0.5
CH3_LIK_CT_VIEW_BOUNDS = (-4.0, 4.0, -4.0, 4.0, -4.0, 4.0)
CH3_LIK_3D_CAM_AZIM0 = CH3_LIK_CH4_CT_AZIM
CH3_LIK_CH4_N_HEATMAP_REVEAL = 8 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_CH4_N_SQUISH_PLANE = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_PLANE_DROP = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_3D_MS = 90 if not _CH3_DRAFT else 110
CH3_LIK_CH4_N_MORPH = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_LOG = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_NLL = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_NOTATION_MOVE = 8 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_CH4_N_CORNER_WRITE = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_KNOB_SWAP = 8 if _CH3_DRAFT else max(20, _smooth_n(14))
CH3_LIK_CH4_N_LOG_HOLD = 8 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_CH4_N_NLL_HOLD = 8 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_CH4_N_PROB_EARLY_WRITE = 8 if _CH3_DRAFT else max(40, _smooth_n(32))
CH3_LIK_CH4_N_PROB_EARLY_ERASE = 8 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_CH4_N_WEIGHTS_NOTATION = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_LIK_FORMULA = 8 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_CH4_N_YIXI_NOTATION = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_PROB_WRITE = 8 if _CH3_DRAFT else max(40, _smooth_n(32))
CH3_LIK_3D_N_INTRO_ZOOM = 6 if _CH3_DRAFT else max(22, _smooth_n(16))
CH3_LIK_3D_N_INTRO_SPIN = 10 if _CH3_DRAFT else max(64, _smooth_n(48))
CH3_LIK_3D_N_INTRO_ZOOM_OUT = 6 if _CH3_DRAFT else max(22, _smooth_n(16))
CH3_LIK_3D_PATH_START = (-0.5, 0.33, -0.5)
CH3_LIK_CH4_GD_START = CH3_LIK_3D_PATH_START
CH3_LIK_CH4_VIEW_POSE = (
    float(CH3_LIK_CH4_W_HI),
    float(CH3_LIK_CH4_W_LO),
    float(CH3_LIK_CH4_PLANE_B_MID),
)
CH3_LIK_CH4_N_BRIDGE_TO_GD = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_BRIDGE_TO_CORNER = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_BRIDGE_TO_GD2 = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_BRIDGE_ZOOM = CH3_LIK_3D_N_INTRO_ZOOM
CH3_LIK_CH4_N_BRIDGE_SPIN = CH3_LIK_3D_N_INTRO_SPIN
CH3_LIK_CH4_N_BRIDGE_ZOOM_OUT = CH3_LIK_3D_N_INTRO_ZOOM_OUT
CH3_LIK_CH4_N_BRIDGE_TO_CORNER2 = 8 if _CH3_DRAFT else max(28, _smooth_n(22))
CH3_LIK_CH4_N_BRIDGE_MARKER_FADE = 8 if _CH3_DRAFT else max(16, _smooth_n(12))
CH3_LIK_CH4_BRIDGE_XY_ZOOM_FRAC = 0.09
CH3_LIK_3D_N_RIGHT_TITLE = 6 if _CH3_DRAFT else max(14, _smooth_n(10))
CH3_LIK_3D_N_RIGHT_WRITE = 8 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_3D_N_TRANS = 10 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_3D_N_PATH = 24 if _CH3_DRAFT else max(120, _smooth_n(90))
CH3_LIK_3D_N_COLOR = 8 if _CH3_DRAFT else max(40, _smooth_n(32))
CH3_LIK_3D_CAM_PATH_ROT = 90.0
CH3_LIK_3D_N_05_HANDOFF = 8 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_3D_BALL_NSAMPLE = 16 if _CH3_DRAFT else 40
CH3_LIK_3D_BALL_R_SCALE = 0.085
CH3_LIK_3D_BALL_QUIVER_LEN = 0.075
CH3_LIK_3D_MS_BALL = 110 if not _CH3_DRAFT else 130
CH3_LIK_3D_AXIS_LABEL_SCALE = 1.45
CH3_LIK_3D_POINT_COLOR = FAIL_COLOR
CH3_LIK_GD_PATH_WAYPOINT_COLOR = "#111111"
# ch4_03 bridge marker only — split-complementary gold + navy edge.
CH4_03_MARKER_COLOR = "#ffd166"
CH4_03_MARKER_EDGE = "#1b2631"
CH3_LIK_GD_POINT_S = 160.0
CH3_LIK_GD_STEP = 0.06
CH3_LIK_GD_N_ITERS = 10 if not _CH3_DRAFT else 3
CH3_LIK_GD_N_HOLD_ARROWS = 4 if _CH3_DRAFT else 12
CH3_LIK_GD_N_PARAM_STEP = 4 if _CH3_DRAFT else 10
CH3_LIK_GD_N_COMBINE = 4 if _CH3_DRAFT else 16
CH3_LIK_GD_SUBSTEPS = 6 if _CH3_DRAFT else 12
CH3_LIK_3D_MS_GD = 120 if not _CH3_DRAFT else 140

# ch4_02 bookends + ch4_03 step 1 share layout_u=0; ch4_03 morphs the plot into the slot.
CH4_LIK_PLOT_START_RECT = (0.0, 0.0, 1.0, 1.0)


def _ch3_lik_w12_z_limits_signed(Z, *, pad_frac=0.10):
    z_lo = float(np.nanmin(Z))
    z_hi = float(np.nanmax(Z))
    span = max(z_hi - z_lo, 1e-9)
    pad = float(pad_frac) * span
    return z_lo - pad, z_hi + pad


def _ch3_lik_w12_z_morph_limits(Zlik, Zlog, Znll, log_u, nll_u):
    """Blend z-axis limits: ℒ (0…max) → log ℒ (min…max) → NLL (0…max)."""
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    lim_lik = ch3_lik_w12_z_limits(Zlik)
    lim_log = _ch3_lik_w12_z_limits_signed(Zlog)
    lim_nll = ch3_lik_w12_z_limits(Znll)
    if mu_nll > 1e-9:
        lim_a, lim_b, mu = lim_log, lim_nll, mu_nll
    elif mu_log > 1e-9:
        lim_a, lim_b, mu = lim_lik, lim_log, mu_log
    else:
        return lim_lik
    return (
        (1.0 - mu) * lim_a[0] + mu * lim_b[0],
        (1.0 - mu) * lim_a[1] + mu * lim_b[1],
    )


def _ch3_lik_w12_z_morph_surface(Zlik, Zlog, Znll, log_u, nll_u, z_lim):
    """Morph surface height in normalized z, with axis limits lerped separately."""
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    lim_lik = ch3_lik_w12_z_limits(Zlik)
    lim_log = _ch3_lik_w12_z_limits_signed(Zlog)
    lim_nll = ch3_lik_w12_z_limits(Znll)
    zlo, zhi = float(z_lim[0]), float(z_lim[1])

    def _norm(Z, lo, hi):
        return (np.asarray(Z, dtype=float) - float(lo)) / max(float(hi) - float(lo), 1e-9)

    def _denorm(t):
        return t * (zhi - zlo) + zlo

    if mu_nll >= 1.0 - 1e-9:
        return np.asarray(Znll, dtype=float)
    if mu_log >= 1.0 - 1e-9 and mu_nll <= 1e-9:
        return np.asarray(Zlog, dtype=float)
    if mu_log <= 1e-9 and mu_nll <= 1e-9:
        return np.asarray(Zlik, dtype=float)
    if mu_nll > 1e-9:
        t = (1.0 - mu_nll) * _norm(Zlog, *lim_log) + mu_nll * _norm(Znll, *lim_nll)
    elif mu_log > 1e-9:
        t = (1.0 - mu_log) * _norm(Zlik, *lim_lik) + mu_log * _norm(Zlog, *lim_log)
    else:
        t = _norm(Zlik, *lim_lik)
    return _denorm(t)


def _ch3_lik86_terminal_state():
    """Ch4_03 story state — handoff from ch4_02: (w1, w2, b) = (3, -3, 0)."""
    st = _ch4_lik_02_03_handoff_state()
    return {
        **st,
        "w1_lo": float(CH3_LIK_W12_W1_LO),
        "w1_hi": float(CH3_LIK_W12_W1_HI),
        "w2_lo": float(CH3_LIK_W12_W2_LO),
        "w2_hi": float(CH3_LIK_W12_W2_HI),
    }


def _ch3_lik_w12_loglik_nll_grids(study, exam, y, w1m, w2m, b):
    """Σ log p and Σ −log p on a mesh — avoid ``log(likelihood product)`` underflow."""
    w1f = np.asarray(w1m, dtype=np.float64).ravel()
    w2f = np.asarray(w2m, dtype=np.float64).ravel()
    bf = np.full(w1f.size, float(b), dtype=np.float64)
    sh = np.asarray(w1m, dtype=np.float64).shape
    zlog = _ch3_loglik_on_flat_w12_grid(study, exam, y, w1f, w2f, bf).reshape(sh)
    znll = _ch3_nll_sum_on_flat_grid(study, exam, y, w1f, w2f, bf).reshape(sh)
    return zlog, znll


def _ch3_lik_ch4_plane_mesh(state, *, b=None):
    """±3 (w_ST, w_EL) grid at fixed ``b`` — same NLL grid as ch4_05a ``b`` slices."""
    study, exam, y = state["study"], state["exam"], state["y"]
    bb = float(CH3_LIK_CH4_PLANE_B if b is None else b)
    mesh = ch3_lik_w12_mesh_pack(
        study, exam, y, bb,
        w1_lo=float(CH3_LIK_CH4_W_LO), w1_hi=float(CH3_LIK_CH4_W_HI),
        w2_lo=float(CH3_LIK_CH4_W_LO), w2_hi=float(CH3_LIK_CH4_W_HI),
        grid_n=int(CH3_LIK_CT_GRID),
    )
    W1m, W2m = mesh["W1m"], mesh["W2m"]
    _, nll = _ch3_lik_w12_loglik_nll_grids(study, exam, y, W1m, W2m, bb)
    mesh["nll"] = nll
    return mesh


def ch4_lik_ct_view_init(ax3d, *, cam_azim_u=0.0, cam_spin_deg=0.0):
    """Canonical CT camera — ch4_03 plane-drop end pose when idle."""
    u = float(np.clip(float(cam_azim_u), 0.0, 1.0))
    spin = float(cam_spin_deg)
    if abs(spin) > 1e-9:
        azim = _ch3_lik_cam_azim(u, total_deg=spin)
    elif abs(u) > 1e-9:
        azim = _ch3_lik_cam_azim(u, total_deg=0.0)
    else:
        azim = float(CH3_LIK_CH4_CT_AZIM)
    ax3d.view_init(elev=float(CH3_LIK_CH4_CT_ELEV), azim=float(azim))


def _ch4_nll_heatmap_plot_surface(ax3d, w1, w2, z, nll, *, alpha=None, zorder=6):
    """Shared NLL heatmap surface coloring — matches ch4_05a CT (canonical)."""
    from ch4_layout import ch4_nll_heatmap_cmap

    al = float(CH3_LIK_CT_PLANE_ALPHA if alpha is None else alpha)
    lo, hi = ch4_nll_global_scale()
    cmap = ch4_nll_heatmap_cmap()
    span = max(float(hi) - float(lo), 1e-9)
    arr = np.asarray(nll, dtype=float)
    normed = np.clip((arr - float(lo)) / span, 0.0, 1.0)
    face = cmap(normed)
    ax3d.plot_surface(
        w1, w2, z,
        facecolors=face,
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=False,
        shade=False,
        alpha=float(al),
        zorder=int(zorder),
    )


def _ch3_lik_w12_plot_crop(img):
    """Crop the right 3D panel from a split-screen ch3_86 frame."""
    img = img.convert("RGB")
    w, h = img.size
    x0 = int(round(w * 0.405))
    return img.crop((x0, int(h * 0.04), w - int(w * 0.02), h - int(h * 0.05)))


def ch3_frame_lik_w12_single_surface(
    state,
    *,
    log_u=0.0,
    nll_u=0.0,
    heatmap_u=0.0,
    squish_u=0.0,
    plane_drop_u=0.0,
    keep_layers=1,
    elev=None,
    azim=None,
    show_axis_labels=True,
    knob_labeled_blend=None,
    landscape_reveal=1.0,
    landscape_reveal_origin="lo_lo",
    display_ws=None,
    display_we=None,
    display_b=None,
    marker=False,
    marker_ws=None,
    marker_we=None,
    marker_z=None,
    marker_color=None,
    marker_edgecolors="white",
    here_annotation=False,
    here_label="WE ARE HERE",
    here_label_fig=None,
    ax3d_xlim=None,
    ax3d_ylim=None,
):
    """One NLL surface; 3-D axes ±3, fixed CT camera (z ticks left), 2-D knobs unchanged."""
    study, exam, y = state["study"], state["exam"], state["y"]
    ws = float(display_ws if display_ws is not None else state["w_st"])
    we = float(display_we if display_we is not None else state["w_el"])
    bb = float(display_b if display_b is not None else state["b"])
    mesh = state["mesh"]
    W1m, W2m, Z = mesh["W1m"], mesh["W2m"], mesh["Z"]
    Zlik = np.asarray(Z, dtype=float)
    mu_log = float(np.clip(log_u, 0.0, 1.0))
    mu_nll = float(np.clip(nll_u, 0.0, 1.0))
    su = ch3_knob_smoothstep(float(np.clip(float(squish_u), 0.0, 1.0)))
    pu = ch3_knob_smoothstep(float(np.clip(float(plane_drop_u), 0.0, 1.0)))
    b_mid = float(CH3_LIK_CH4_PLANE_B_MID)
    b_end = float(CH3_LIK_CH4_PLANE_B)
    z_flat = (1.0 - pu) * b_mid + pu * b_end
    if su > 1e-6:
        plane = _ch3_lik_ch4_plane_mesh(state, b=z_flat)
        W1m, W2m = plane["W1m"], plane["W2m"]
        Zlik_p = np.asarray(plane["Z"], dtype=float)
        Zlog_p, Znll = _ch3_lik_w12_loglik_nll_grids(study, exam, y, W1m, W2m, z_flat)
        z_lim_nll = _ch3_lik_w12_z_morph_limits(Zlik_p, Zlog_p, Znll, mu_log, mu_nll)
        Zmix_p = _ch3_lik_w12_z_morph_surface(Zlik_p, Zlog_p, Znll, mu_log, mu_nll, z_lim_nll)
        Zplot = (1.0 - su) * Zmix_p + su * np.full_like(Zmix_p, float(z_flat))
    else:
        Zlog, Znll = _ch3_lik_w12_loglik_nll_grids(study, exam, y, W1m, W2m, bb)
        z_lim_nll = _ch3_lik_w12_z_morph_limits(Zlik, Zlog, Znll, mu_log, mu_nll)
        Zmix = _ch3_lik_w12_z_morph_surface(Zlik, Zlog, Znll, mu_log, mu_nll, z_lim_nll)
        Zplot = np.asarray(Zmix, dtype=float)
    z_lo_b, z_hi_b = float(CH3_LIK_CH4_W_LO), float(CH3_LIK_CH4_W_HI)
    z_lim = (
        (1.0 - su) * float(z_lim_nll[0]) + su * z_lo_b,
        (1.0 - su) * float(z_lim_nll[1]) + su * z_hi_b,
    )
    if su >= 0.5:
        z_lab = r"$b$"
    elif mu_nll > 0.5:
        z_lab = "NLL"
    elif mu_log > 0.5:
        z_lab = "log likelihood"
    else:
        z_lab = "Likelihood"
    hu = ch3_knob_smoothstep(float(np.clip(float(heatmap_u), 0.0, 1.0)))
    surf_alpha = float(
        CH3_LIK_CT_PLANE_ALPHA if su > 1e-6 else CH3_LIK_CH4_SURFACE_ALPHA
    )
    rgba_red = mpl.colors.to_rgba(CH4_LIK_SURFACE_COLOR)
    fc_red = np.empty(W1m.shape + (4,), dtype=float)
    fc_red[..., :3] = rgba_red[:3]
    fc_red[..., 3] = surf_alpha
    if hu > 1e-6:
        from ch4_layout import ch4_nll_heatmap_facecolors

        g_lo, g_hi = ch4_nll_global_scale()
        fc_heat = ch4_nll_heatmap_facecolors(
            Znll, vmin=g_lo, vmax=g_hi, alpha=surf_alpha,
        )
        fc = fc_red * (1.0 - hu) + fc_heat * hu
    else:
        fc = fc_red
    rev_u = float(np.clip(float(landscape_reveal), 0.0, 1.0))
    if rev_u < 1.0 - 1e-6:
        w1_lo, w1_hi = float(mesh["w1_lo"]), float(mesh["w1_hi"])
        w2_lo, w2_hi = float(mesh["w2_lo"]), float(mesh["w2_hi"])
        rev_fc = ch3_lik_w12_facecolors_diag(
            W1m, W2m, rev_u,
            w1_lo=w1_lo, w1_hi=w1_hi, w2_lo=w2_lo, w2_hi=w2_hi,
            rgba=(CH4_LIK_SURFACE_COLOR, surf_alpha),
            origin=str(landscape_reveal_origin),
        )
        fc = np.asarray(fc, dtype=float).copy()
        fc[..., 3] = rev_fc[..., 3]
    el = float(CH3_LIK_CH4_CT_ELEV if elev is None else elev)
    az = float(CH3_LIK_CH4_CT_AZIM if azim is None else azim)
    knob_pack = _ch4_lik_resolve_knob_pack(knob_labeled_blend)
    use_weight_axis_labels = (
        knob_labeled_blend is not None
        and all(float(x) >= 1.0 - 1e-6 for x in knob_labeled_blend)
    )
    b_show = float(z_flat) if su > 1e-4 else float(bb)
    if su > 1e-6 and hu > 1.0 - 1e-6:
        flat_surface = {"W1m": W1m, "W2m": W2m, "Z": Zplot, "nll_heatmap": Znll}
    else:
        flat_surface = {"W1m": W1m, "W2m": W2m, "Z": Zplot, "facecolors": fc}
    return ch3_frame_lik_w12_3d(
        study, exam, y, ws, we, b_show,
        mesh_pack=mesh, z_lim=z_lim, curves=[],
        elev=el, azim=az, emphasize_knob="all",
        landscape_reveal=0.0, show_curves=False, marker=marker,
        stack_layers=None, stack_n_layers=1.0, slice_reveal=1.0,
        show_axis_labels=show_axis_labels,
        flat_surface=flat_surface,
        z_label=z_lab,
        knob_pack=knob_pack,
        knob_scales=[1.0, 1.0, 1.0],
        weight_axis_labels=use_weight_axis_labels,
        marker_ws=marker_ws,
        marker_we=marker_we,
        marker_z=marker_z,
        marker_color=marker_color,
        marker_edgecolors=marker_edgecolors,
        here_annotation=here_annotation,
        here_label=here_label,
        here_label_fig=here_label_fig,
        ax3d_xlim=ax3d_xlim,
        ax3d_ylim=ax3d_ylim,
    )


def _ch4_lik_03_opening_plot(*, knob_labeled_blend=None):
    """Plot panel for ch4_03 frame 0 — same pose/camera as ch4_02 end."""
    return _ch4_lik_02_03_handoff_plot(knob_labeled_blend=knob_labeled_blend)


def _ch4_lik_03_opening_compose(plot_img, *, layout_u=0.0):
    """Compose wrapper for ch4_03 opening / ch4_02 bookends (no rails yet)."""
    from ch4_layout import compose_tutorial

    return compose_tutorial(
        plot_img,
        right_blocks=[],
        bottom_blocks=[],
        layout_u=float(layout_u),
        panel_u=0.0,
        title_write_progress=0.0,
        write_progress=0.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )


def _ch3_lik_stage_tex(stages, frame_i, n_frames):
    """Pick one of ``stages`` for frame index (hard cut, no crossfade)."""
    n = max(int(n_frames), 1)
    idx = min(int(frame_i * len(stages) / n), len(stages) - 1)
    return stages[idx]


def _ch3_lik_emit_ch4_03_formulas(
    plot_img,
    *,
    right_blocks,
    bottom_prog,
    bottom_blocks=None,
    right_title="Notation",
    bottom_title="Formulas",
    corner_blocks=None,
    corner_title=None,
    right_blocks_empty=False,
    progress_override=None,
):
    """Compose ch4_03 with three-column handwritten formulas."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        ch4_formula_blocks_ch4_03_three_col,
        compose_tutorial,
    )

    blocks = bottom_blocks if bottom_blocks is not None else ch4_formula_blocks_ch4_03_three_col()
    prog = {"bottom": bottom_prog}
    if progress_override:
        prog.update(progress_override)
    kw = dict(
        plot_img=plot_img,
        right_blocks=[] if right_blocks_empty else right_blocks,
        bottom_blocks=blocks,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        progress_override=prog,
        theme="classic_light",
    )
    if corner_blocks is not None:
        kw.update(
            corner_blocks=corner_blocks,
            bottom_title=bottom_title or CH4_FORMULAS_SECTION_TITLE,
            corner_title=corner_title or CH4_NOTATION_SECTION_TITLE,
        )
    else:
        kw.update(right_title=right_title, bottom_title=bottom_title)
    return compose_tutorial(**kw)


_CH4_LIK_KNOB_BLEND_FULL = (1.0, 1.0, 1.0)


def _ch4_lik_knob_blend_per_line(u, n_lines=3):
    """Numbered → labeled knob blend as each notation line finishes writing."""
    u = float(np.clip(float(u), 0.0, 1.0))
    n = max(int(n_lines), 1)
    blends = []
    for slot in range(n):
        t0 = slot / n
        t1 = (slot + 1) / n
        if u <= t0:
            blends.append(0.0)
        elif u >= t1:
            blends.append(1.0)
        else:
            blends.append(ch3_knob_smoothstep((u - t0) / (t1 - t0)))
    return tuple(blends)


def _ch4_lik_03_bridge_xy_zoom(ws, we, *, frac=CH3_LIK_CH4_BRIDGE_XY_ZOOM_FRAC):
    """Tight x/y limits around a weight-space point; z limits stay unchanged."""
    span = float(CH3_LIK_CH4_W_HI) - float(CH3_LIK_CH4_W_LO)
    half = max(float(frac) * span, 0.08)
    return (float(ws) - half, float(ws) + half), (float(we) - half, float(we) + half)


def _ch4_lik_03_bridge_plot_likelihood(
    ws,
    we,
    bb,
    *,
    marker=False,
    marker_ws=None,
    marker_we=None,
    marker_alpha=1.0,
    ax3d_xlim=None,
    ax3d_ylim=None,
    azim=None,
    elev=None,
):
    """GD tour on the likelihood landscape (before log/NLL morph)."""
    mcol = CH4_03_MARKER_COLOR
    if float(marker_alpha) < 1.0 - 1e-6:
        import matplotlib.colors as mcolors

        rgba = mcolors.to_rgba(mcol, alpha=float(marker_alpha))
        mcol = rgba
    return _ch4_lik_02_03_handoff_plot(
        knob_labeled_blend=_CH4_LIK_KNOB_BLEND_FULL,
        display_ws=float(ws),
        display_we=float(we),
        display_b=float(bb),
        marker=bool(marker) and float(marker_alpha) > 1e-3,
        marker_ws=marker_ws,
        marker_we=marker_we,
        marker_color=mcol,
        marker_edgecolors=CH4_03_MARKER_EDGE,
        ax3d_xlim=ax3d_xlim,
        ax3d_ylim=ax3d_ylim,
        elev=elev,
        azim=azim,
    )


def _ch4_lik_03_prob_compose(
    plot_img,
    *,
    right_exp,
    bottom_prob,
    prob_u,
    right_prog=None,
):
    from ch4_layout import ch4_bottom_prog_ch4_03_prob_interlude

    kw = dict(
        plot_img=plot_img,
        right_blocks=right_exp,
        bottom_blocks=bottom_prob,
        bottom_prog=ch4_bottom_prog_ch4_03_prob_interlude(lik_u=1.0, prob_u=float(prob_u)),
    )
    if right_prog is not None:
        kw["progress_override"] = {"right": right_prog}
    return _ch3_lik_emit_ch4_03_formulas(**kw)


def _ch4_lik_03_story_hold(frames, *, n=None):
    if not frames:
        return frames
    hold_n = max(8, CH3_SCRIPT_N_HOLD // 4) if n is None else int(n)
    last = frames[-1]
    for _ in range(hold_n):
        frames.append(last.copy())
    return frames


def _ch4_lik_03_lerp_pose(pose_a, pose_b, u):
    u = ch3_knob_smoothstep(float(u))
    a = tuple(float(v) for v in pose_a)
    b = tuple(float(v) for v in pose_b)
    return tuple(x + (y - x) * u for x, y in zip(a, b))


def ch3_build_frames_likelihood_ch4_nll_story():
    from ch4_layout import (
        CH4_COMPOSER,
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NOTATION_YI_LINE_IDX,
        ch4_blend_images,
        ch4_bottom_prog_ch4_03_three_col,
        ch4_bottom_prog_ch4_03_prob_interlude,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_3d_story,
        ch4_formula_blocks_ch4_03_three_col,
        ch4_formula_blocks_ch4_03_prob_interlude,
        ch4_formula_blocks_nll_story,
        ch4_blocks_write_from_slot,
        ch4_bottom_per_block_progress,
        ch4_group_write_from_slot,
        ch4_notation_blocks_basic,
        ch4_notation_blocks_expanded,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    frames = []
    style = CH4_COMPOSER.handwrite_style()
    bottom_3col = ch4_formula_blocks_ch4_03_three_col()

    def plot_surface(
        *,
        log_u=0.0,
        nll_u=0.0,
        heatmap_u=0.0,
        squish_u=0.0,
        plane_drop_u=0.0,
        knob_labeled_blend=_CH4_LIK_KNOB_BLEND_FULL,
    ):
        return ch3_frame_lik_w12_single_surface(
            state,
            log_u=log_u,
            nll_u=nll_u,
            heatmap_u=heatmap_u,
            squish_u=squish_u,
            plane_drop_u=plane_drop_u,
            show_axis_labels=True,
            knob_labeled_blend=knob_labeled_blend,
        )

    def plot_handoff(*, knob_labeled_blend=None):
        blend = _CH4_LIK_KNOB_BLEND_FULL if knob_labeled_blend is None else knob_labeled_blend
        return _ch4_lik_03_opening_plot(knob_labeled_blend=blend)

    def emit(
        plot_img,
        *,
        layout_u=1.0,
        panel_u=1.0,
        title_write_progress=None,
        write_progress=0.0,
        right_blocks=None,
        bottom_blocks=None,
        right_write_progress=None,
        bottom_write_progress=None,
        progress_override=None,
    ):
        frames.append(
            compose_tutorial(
                plot_img,
                right_blocks=right_blocks if right_blocks is not None else ch4_notation_blocks_basic(),
                bottom_blocks=bottom_blocks if bottom_blocks is not None else bottom_3col,
                right_title="Notation",
                bottom_title="Formulas",
                layout_u=layout_u,
                panel_u=panel_u,
                title_write_progress=title_write_progress,
                write_progress=write_progress,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                right_write_progress=right_write_progress,
                bottom_write_progress=bottom_write_progress,
                progress_override=progress_override,
                theme="classic_light",
            )
        )

    plot0 = _ch4_lik_03_opening_plot()
    n_titles = max(14, _smooth_n(12))
    basic_blocks = ch4_notation_blocks_basic()
    right_exp = ch4_notation_blocks_expanded()
    right_full = ch4_blocks_write_from_slot(basic_blocks, 0, 1.0, style=style)
    bottom_empty = ch4_bottom_prog_ch4_03_three_col(lik_u=0.0)
    bottom_lik_only = ch4_bottom_prog_ch4_03_three_col(lik_u=1.0)

    # 1 — full ch4_02 figure resizes into the template plot slot (no rails yet)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_MORPH, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(plot0, layout_u=u, panel_u=0.0, title_write_progress=0.0, write_progress=0.0)

    # 2 — section titles appear and stay (rails visible, no block text yet)
    for tv in np.linspace(0.0, 1.0, n_titles, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(plot0, layout_u=1.0, panel_u=1.0, title_write_progress=u, write_progress=0.0)

    # 3 — knob notation only (w_ST, w_EL, b); numbered → labeled knobs line-by-line
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_WEIGHTS_NOTATION, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        knob_blend = _ch4_lik_knob_blend_per_line(u, 3)
        plot_k = plot_handoff(knob_labeled_blend=knob_blend)
        emit(
            plot_k,
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=basic_blocks,
            progress_override={
                "right": ch4_blocks_write_from_slot(basic_blocks, 0, u, style=style),
                "bottom": bottom_empty,
            },
        )

    plot_labeled = plot_handoff()

    # 4 — likelihood formula only (notation stays fully written)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_LIK_FORMULA, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit(
            plot_labeled,
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=basic_blocks,
            progress_override={
                "right": right_full,
                "bottom": ch4_bottom_prog_ch4_03_three_col(lik_u=u),
            },
        )

    # 5 — y_i / x_i notation only (weights stay written; no new formulas)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_YIXI_NOTATION, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        yi_xi_prog = ch4_group_write_from_slot(
            [right_exp], int(CH4_NOTATION_YI_LINE_IDX), u, style=style,
        )[0]
        emit(
            plot_labeled,
            layout_u=1.0,
            panel_u=1.0,
            title_write_progress=1.0,
            write_progress=1.0,
            right_blocks=right_exp,
            progress_override={
                "right": yi_xi_prog,
                "bottom": bottom_lik_only,
            },
        )

    # 6 — handwrite p(y_i | x_i) in the log column (part 1 ends here)
    bottom_prob = ch4_formula_blocks_ch4_03_prob_interlude()
    right_exp_full = ch4_blocks_write_from_slot(right_exp, 0, 1.0, style=style)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_PROB_EARLY_WRITE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_ch4_lik_03_prob_compose(
            plot_labeled,
            right_exp=right_exp,
            bottom_prob=bottom_prob,
            prob_u=u,
            right_prog=right_exp_full,
        ))

    return _ch4_lik_03_story_hold(frames)


def _ch4_lik_03_frozen_compose(
    plot_img,
    *,
    bottom_3d,
    corner,
    style,
    right_blocks=None,
    right_title=None,
    right_title_single_line=None,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        ch4_blocks_write_from_slot,
        ch4_bottom_per_block_progress,
        compose_tutorial,
    )

    return compose_tutorial(
        plot_img,
        right_blocks=[] if right_blocks is None else right_blocks,
        bottom_blocks=bottom_3d,
        corner_blocks=corner,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title=right_title,
        right_title_single_line=right_title_single_line,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        progress_override={
            "bottom": ch4_bottom_per_block_progress(bottom_3d, {0: 1.0, 1: 1.0}),
            "corner": ch4_blocks_write_from_slot(corner, 0, 1.0, style=style),
        },
        theme="classic_light",
    )


def _ch4_lik_03_lerp_axis_limits(lim_a, lim_b, u):
    u = ch3_knob_smoothstep(float(u))
    a0, a1 = float(lim_a[0]), float(lim_a[1])
    b0, b1 = float(lim_b[0]), float(lim_b[1])
    return (a0 + (b0 - a0) * u, a1 + (b1 - a1) * u)


def ch3_build_frames_likelihood_ch4_nll_bridge():
    """Part 2 — GD start tour on likelihood surface; erase p(y_i|x_i) at end."""
    from ch4_layout import (
        CH4_COMPOSER,
        ch4_formula_blocks_ch4_03_prob_interlude,
        ch4_notation_blocks_expanded,
        ch4_blocks_write_from_slot,
    )

    style = CH4_COMPOSER.handwrite_style()
    right_exp = ch4_notation_blocks_expanded()
    bottom_prob = ch4_formula_blocks_ch4_03_prob_interlude()
    right_full = ch4_blocks_write_from_slot(right_exp, 0, 1.0, style=style)
    frames = []
    gd = tuple(float(v) for v in CH3_LIK_CH4_GD_START)
    view = CH3_LIK_CH4_VIEW_POSE
    elev0 = float(CH3_LIK_CH4_CT_ELEV)
    az0 = float(CH3_LIK_CH4_CT_AZIM)
    wide_x = (float(CH3_LIK_CH4_W_LO), float(CH3_LIK_CH4_W_HI))
    wide_y = wide_x
    tight_x, tight_y = _ch4_lik_03_bridge_xy_zoom(gd[0], gd[1])

    def emit_prob(plot_img, *, prob_u=1.0):
        frames.append(_ch4_lik_03_prob_compose(
            plot_img,
            right_exp=right_exp,
            bottom_prob=bottom_prob,
            prob_u=prob_u,
            right_prog=right_full,
        ))

    # 1–2 — decision boundary at GD start (b = 0); black point tracks (w_ST, w_EL)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_BRIDGE_TO_GD, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws, we, bb = _ch4_lik_03_lerp_pose(view, gd, u)
        emit_prob(_ch4_lik_03_bridge_plot_likelihood(
            ws, we, 0.0,
            marker=True, marker_alpha=u,
            elev=elev0, azim=az0,
        ))

    # 3 — move smoothly to the (4, −4, 0) view pose
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_BRIDGE_TO_CORNER, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws, we, bb = _ch4_lik_03_lerp_pose(gd, view, u)
        emit_prob(_ch4_lik_03_bridge_plot_likelihood(
            ws, we, bb,
            marker=True,
            elev=elev0, azim=az0,
        ))

    # 4 — return smoothly to the GD starting point
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_BRIDGE_TO_GD2, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws, we, bb = _ch4_lik_03_lerp_pose(view, gd, u)
        emit_prob(_ch4_lik_03_bridge_plot_likelihood(
            ws, we, 0.0,
            marker=True,
            elev=elev0, azim=az0,
        ))

    # 5 — zoom x/y around GD start (z limits unchanged)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_BRIDGE_ZOOM, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        xlim = _ch4_lik_03_lerp_axis_limits(wide_x, tight_x, u)
        ylim = _ch4_lik_03_lerp_axis_limits(wide_y, tight_y, u)
        emit_prob(_ch4_lik_03_bridge_plot_likelihood(
            gd[0], gd[1], 0.0,
            marker=True,
            ax3d_xlim=xlim, ax3d_ylim=ylim,
            elev=elev0, azim=az0,
        ))

    # 6 — rotate 360°
    spin_n = max(int(CH3_LIK_CH4_N_BRIDGE_SPIN), 2)
    for tv in np.linspace(0.0, 1.0, spin_n, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit_prob(_ch4_lik_03_bridge_plot_likelihood(
            gd[0], gd[1], 0.0,
            marker=True,
            ax3d_xlim=tight_x, ax3d_ylim=tight_y,
            elev=elev0, azim=az0 + 360.0 * float(u),
        ))

    # 7 — zoom back out
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_BRIDGE_ZOOM_OUT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        xlim = _ch4_lik_03_lerp_axis_limits(tight_x, wide_x, u)
        ylim = _ch4_lik_03_lerp_axis_limits(tight_y, wide_y, u)
        emit_prob(_ch4_lik_03_bridge_plot_likelihood(
            gd[0], gd[1], 0.0,
            marker=True,
            ax3d_xlim=xlim, ax3d_ylim=ylim,
            elev=elev0, azim=az0,
        ))

    # 8 — return to (4, −4, 0); point fades out
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_BRIDGE_TO_CORNER2, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws, we, bb = _ch4_lik_03_lerp_pose(gd, view, u)
        emit_prob(_ch4_lik_03_bridge_plot_likelihood(
            ws, we, bb,
            marker=True,
            marker_alpha=max(0.0, 1.0 - u),
            elev=elev0, azim=az0,
        ))

    # 9 — erase p(y_i | x_i) (handoff to part 3 log-likelihood morph)
    plot_view = _ch4_lik_02_03_handoff_plot(knob_labeled_blend=_CH4_LIK_KNOB_BLEND_FULL)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_PROB_EARLY_ERASE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        emit_prob(plot_view, prob_u=1.0 - u)

    return _ch4_lik_03_story_hold(frames)


def ch3_build_frames_likelihood_ch4_nll_story_part2():
    """Part 3 — log/NLL morph through heatmap, squish, plane drop."""
    from ch4_layout import (
        CH4_COMPOSER,
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_blend_images,
        ch4_bottom_prog_ch4_03_three_col,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_3d_story,
        ch4_formula_blocks_nll_story,
        ch4_blocks_write_from_slot,
        ch4_bottom_per_block_progress,
        ch4_notation_blocks_expanded,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    style = CH4_COMPOSER.handwrite_style()
    right_exp = ch4_notation_blocks_expanded()
    right_full = ch4_blocks_write_from_slot(right_exp, 0, 1.0, style=style)
    corner = ch4_cached_notation_corner_blocks()
    bottom_nll = ch4_formula_blocks_nll_story()
    bottom_3d = ch4_formula_blocks_3d_story()
    nll_legend = ch4_nll_global_legend_blocks()
    frames = []

    def plot_surface(
        *,
        log_u=0.0,
        nll_u=0.0,
        heatmap_u=0.0,
        squish_u=0.0,
        plane_drop_u=0.0,
    ):
        return ch3_frame_lik_w12_single_surface(
            state,
            log_u=log_u,
            nll_u=nll_u,
            heatmap_u=heatmap_u,
            squish_u=squish_u,
            plane_drop_u=plane_drop_u,
            show_axis_labels=True,
            knob_labeled_blend=_CH4_LIK_KNOB_BLEND_FULL,
        )

    def emit_3col(plot_img, *, bottom_prog):
        frames.append(_ch3_lik_emit_ch4_03_formulas(
            plot_img,
            right_blocks=right_exp,
            bottom_prog=bottom_prog,
            progress_override={"right": right_full},
        ))

    def emit_ch4_03_end(plot_img, *, right_blocks=None, right_title=None, right_title_single_line=None):
        frames.append(_ch4_lik_03_frozen_compose(
            plot_img,
            bottom_3d=bottom_3d,
            corner=corner,
            style=style,
            right_blocks=right_blocks,
            right_title=right_title,
            right_title_single_line=right_title_single_line,
        ))

    # 5 — surface → log; handwrite log ℒ column (single line, two-phase reveal)
    n_log = CH3_LIK_CH4_N_LOG
    for i, tv in enumerate(np.linspace(0.0, 1.0, n_log, endpoint=True)):
        u = ch3_knob_smoothstep(float(tv))
        plot_u = plot_surface(log_u=u, nll_u=0.0)
        if u < 0.5:
            log_us = (u * 2.0, 0.0)
        else:
            log_us = (1.0, (u - 0.5) * 2.0)
        emit_3col(
            plot_u,
            bottom_prog=ch4_bottom_prog_ch4_03_three_col(
                lik_u=1.0, log_line_us=log_us, nll_u=0.0,
            ),
        )

    for _ in range(int(CH3_LIK_CH4_N_LOG_HOLD)):
        emit_3col(
            plot_surface(log_u=1.0, nll_u=0.0),
            bottom_prog=ch4_bottom_prog_ch4_03_three_col(
                lik_u=1.0, log_line_us=(1.0, 1.0), nll_u=0.0,
            ),
        )

    # 6 — surface → NLL; handwrite NLL in column 3
    n_nll = CH3_LIK_CH4_N_NLL
    for i, tv in enumerate(np.linspace(0.0, 1.0, n_nll, endpoint=True)):
        u = ch3_knob_smoothstep(float(tv))
        plot_u = plot_surface(log_u=1.0, nll_u=u)
        emit_3col(
            plot_u,
            bottom_prog=ch4_bottom_prog_ch4_03_three_col(
                lik_u=1.0, log_line_us=(1.0, 1.0), nll_u=u,
            ),
        )

    for _ in range(int(CH3_LIK_CH4_N_NLL_HOLD)):
        emit_3col(
            plot_surface(log_u=1.0, nll_u=1.0),
            bottom_prog=ch4_bottom_prog_ch4_03_three_col(
                lik_u=1.0, log_line_us=(1.0, 1.0), nll_u=1.0,
            ),
        )

    plot_nll = plot_surface(log_u=1.0, nll_u=1.0)

    # 7 — notation → corner; erase ℒ+log columns, NLL moves to left column
    frame_right_3col = _ch3_lik_emit_ch4_03_formulas(
        plot_nll,
        right_blocks=right_exp,
        bottom_prog=ch4_bottom_prog_ch4_03_three_col(
            lik_u=1.0, log_line_us=(1.0, 1.0), nll_u=1.0,
        ),
        progress_override={"right": right_full},
    )
    frame_corner_nll = compose_tutorial(
        plot_nll,
        right_blocks=[],
        bottom_blocks=bottom_nll,
        corner_blocks=corner,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        progress_override={"corner": ch4_blocks_write_from_slot(corner, 0, 0.0, style=style)},
        theme="classic_light",
    )
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_NOTATION_MOVE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_right_3col, frame_corner_nll, u))

    plot_corner = plot_surface(log_u=1.0, nll_u=1.0)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_CORNER_WRITE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        corner_prog = ch4_blocks_write_from_slot(corner, 0, u, style=style)
        frames.append(
            compose_tutorial(
                plot_corner,
                right_blocks=[],
                bottom_blocks=bottom_nll,
                corner_blocks=corner,
                bottom_title=CH4_FORMULAS_SECTION_TITLE,
                corner_title=CH4_NOTATION_SECTION_TITLE,
                layout_u=1.0,
                panel_u=1.0,
                write_progress=1.0,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                progress_override={"corner": corner_prog},
                theme="classic_light",
            )
        )

    # 8 — handwrite p(y_i | x_i) below NLL (same mathtext style as ch4_04)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_PROB_WRITE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        prog = ch4_bottom_per_block_progress(bottom_3d, {0: 1.0, 1: u})
        corner_full = ch4_blocks_write_from_slot(corner, 0, 1.0, style=style)
        frames.append(
            compose_tutorial(
                plot_corner,
                right_blocks=[],
                bottom_blocks=bottom_3d,
                corner_blocks=corner,
                bottom_title=CH4_FORMULAS_SECTION_TITLE,
                corner_title=CH4_NOTATION_SECTION_TITLE,
                layout_u=1.0,
                panel_u=1.0,
                write_progress=1.0,
                plot_start_rect=CH4_LIK_PLOT_START_RECT,
                progress_override={"bottom": prog, "corner": corner_full},
                theme="classic_light",
            )
        )

    # 9 — heatmap reveal on NLL surface + NLL color scale legend
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_HEATMAP_REVEAL, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        plot_h = plot_surface(log_u=1.0, nll_u=1.0, heatmap_u=u)
        emit_ch4_03_end(
            plot_h,
            right_blocks=nll_legend if u > 0.02 else [],
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE if u > 0.02 else None,
            right_title_single_line=True,
        )

    # 10 — collapse NLL surface → flat plane at b = 0
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_SQUISH_PLANE, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        plot_s = plot_surface(
            log_u=1.0, nll_u=1.0, heatmap_u=1.0, squish_u=u, plane_drop_u=0.0,
        )
        emit_ch4_03_end(
            plot_s,
            right_blocks=nll_legend,
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
            right_title_single_line=True,
        )

    # 11 — slide plane from b = 0 down to b = −4 (handoff to ch4_05a CT scan)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CH4_N_PLANE_DROP, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        plot_d = plot_surface(
            log_u=1.0, nll_u=1.0, heatmap_u=1.0, squish_u=1.0, plane_drop_u=u,
        )
        emit_ch4_03_end(
            plot_d,
            right_blocks=nll_legend,
            right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
            right_title_single_line=True,
        )

    return _ch4_lik_03_story_hold(frames)


def _ch3_lik_margin_bounds(bounds, *, frac=0.07):
    """Inset axis limits so zig-zag waypoints stay comfortably inside."""
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    out = []
    for lo, hi in ((dlo1, dhi1), (dlo2, dhi2), (dlob, dhib)):
        span = max(float(hi) - float(lo), 1e-9)
        m = float(frac) * span
        out.extend([float(lo) + m, float(hi) - m])
    return tuple(out)


def _ch3_lik_zigzag_path_3d(bounds, *, end, n_pts=None):
    """Zig-zag through weight space inside ``bounds``, ending at ``end``."""
    n_pts = int(CH3_LIK_3D_N_PATH if n_pts is None else n_pts)
    end = np.asarray(end, dtype=float).reshape(3)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch3_lik_margin_bounds(bounds)
    mid1 = 0.5 * (dlo1 + dhi1)
    mid2 = 0.5 * (dlo2 + dhi2)
    midb = 0.5 * (dlob + dhib)
    waypoints = np.array([
        [dhi1, dlo2, dhib],
        [dlo1, dhi2, dlob],
        [dhi1, dhi2, midb],
        [mid1, dlo2, dhib],
        [dlo1, mid2, dlob],
        [mid1, dhi2, midb],
        end,
    ], dtype=float)
    seg = np.linalg.norm(np.diff(waypoints, axis=0), axis=1)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    total = float(cum[-1])
    if total < 1e-12:
        return np.tile(end, (n_pts, 1)).astype(float)
    targets = np.linspace(0.0, total, n_pts, endpoint=True)
    pts = np.empty((n_pts, 3), dtype=float)
    for j, ut in enumerate(targets):
        i = int(np.searchsorted(cum, ut, side="right") - 1)
        i = min(max(i, 0), len(waypoints) - 2)
        t = float((ut - cum[i]) / max(seg[i], 1e-12))
        pts[j] = (1.0 - t) * waypoints[i] + t * waypoints[i + 1]
    pts[-1] = end
    return pts.astype(float)


def _ch3_lik_fun_path_3d(n_pts=None, start=None, bounds=None):
    """Path in (w_ST, w_EL, b); ends at ``start`` (default ``CH3_LIK_3D_PATH_START``)."""
    end = CH3_LIK_3D_PATH_START if start is None else start
    if bounds is None:
        t = np.linspace(0.0, 1.0, int(CH3_LIK_3D_N_PATH if n_pts is None else n_pts), endpoint=True)
        w1_c, w2_c, b_c = (float(end[0]), float(end[1]), float(end[2]))
        w1 = w1_c + 1.8 * np.sin(2.0 * np.pi * t) * (0.35 + 0.65 * t)
        w2 = w2_c + 1.6 * np.cos(2.4 * np.pi * t + 0.6) * (0.35 + 0.65 * t)
        b = b_c + 0.55 * np.sin(4.0 * np.pi * t + 1.1)
        return np.column_stack([w1, w2, b]).astype(float)
    return _ch3_lik_zigzag_path_3d(bounds, end=end, n_pts=n_pts)


def _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib):
    """3-D axis labels (+45%) and sparser z ticks."""
    from matplotlib.ticker import MaxNLocator

    fs = float(AXIS_LABEL_SIZE) * float(CH3_LIK_3D_AXIS_LABEL_SCALE)
    ax3d.set_xlim(dlo1, dhi1)
    ax3d.set_ylim(dlo2, dhi2)
    ax3d.set_zlim(dlob, dhib)
    ax3d.set_xlabel(r"$w_{\mathrm{ST}}$", fontsize=fs, labelpad=8)
    ax3d.set_ylabel(r"$w_{\mathrm{EL}}$", fontsize=fs, labelpad=8)
    ax3d.set_zlabel(r"$b$", fontsize=fs, labelpad=8)
    ax3d.zaxis.set_major_locator(MaxNLocator(nbins=5))


def _ch3_lik_ax3d_fixed_bounds(ref_lo1, hi1, lo2, hi2, lob, hib, path_xyz, *, extra_pts=None):
    """Axis limits that contain the full path (and grid), with padding."""
    ref_lo = np.array([float(ref_lo1), float(lo2), float(lob)], dtype=float)
    ref_hi = np.array([float(hi1), float(hi2), float(hib)], dtype=float)
    ref_span = np.maximum(ref_hi - ref_lo, 1e-9)
    pts = [np.asarray(path_xyz, dtype=float).reshape(-1, 3)]
    if extra_pts is not None:
        pts.append(np.asarray(extra_pts, dtype=float).reshape(-1, 3))
    P = np.vstack(pts)
    P = P[np.isfinite(P).all(axis=1)]
    if P.shape[0] == 0:
        return float(ref_lo1), float(hi1), float(lo2), float(hi2), float(lob), float(hib)
    lo = np.minimum(P.min(axis=0), ref_lo)
    hi = np.maximum(P.max(axis=0), ref_hi)
    span = np.maximum(hi - lo, CH3_KERAS_NLL3D_ZOOM_MIN_SPAN_FRAC * ref_span)
    center = 0.5 * (lo + hi)
    lo = center - 0.5 * span
    hi = center + 0.5 * span
    pad = CH3_KERAS_NLL3D_VIEW_PAD_FRAC * (hi - lo)
    return (
        float(lo[0] - pad[0]),
        float(hi[0] + pad[0]),
        float(lo[1] - pad[1]),
        float(hi[1] + pad[1]),
        float(lo[2] - pad[2]),
        float(hi[2] + pad[2]),
    )


def _ch3_lik_cam_azim(cam_u, *, total_deg=CH3_LIK_3D_CAM_PATH_ROT, base=CH3_LIK_3D_CAM_AZIM0):
    u = float(np.clip(float(cam_u), 0.0, 1.0))
    return float(base + float(total_deg) * u)


def _ch3_lik_lerp_bounds(bounds_a, bounds_b, u):
    u = float(np.clip(float(u), 0.0, 1.0))
    a = tuple(float(v) for v in bounds_a)
    b = tuple(float(v) for v in bounds_b)
    return tuple(float(x + (y - x) * u) for x, y in zip(a, b))


def _ch3_lik_bounds_span(bounds):
    return np.array(
        [bounds[1] - bounds[0], bounds[3] - bounds[2], bounds[5] - bounds[4]],
        dtype=float,
    )


def _ch3_lik_ball_zoom_bounds(ws, we, bb, wide_bounds, *, r_scale=CH3_LIK_3D_BALL_R_SCALE):
    """Tight axis limits framing the NLL ball around one weight-space point."""
    center = np.array([float(ws), float(we), float(bb)], dtype=float)
    span_ref = float(np.max(_ch3_lik_bounds_span(wide_bounds)))
    R = float(r_scale) * span_ref
    half = max(R * 2.35, span_ref * 0.055)
    lo = center - half
    hi = center + half
    pad = 0.08 * (hi - lo)
    return (
        float(lo[0] - pad[0]), float(hi[0] + pad[0]),
        float(lo[1] - pad[1]), float(hi[1] + pad[1]),
        float(lo[2] - pad[2]), float(hi[2] + pad[2]),
    )


def _ch3_lik_ball_vector_field(study, exam, y, ws, we, bb, wide_bounds):
    """Sample points on a sphere + negative-NLL gradient vectors at each sample."""
    span_ref = float(np.max(_ch3_lik_bounds_span(wide_bounds)))
    R = float(CH3_LIK_3D_BALL_R_SCALE) * span_ref
    offs = _ch3_ball_unit_offsets(int(CH3_LIK_3D_BALL_NSAMPLE))
    center = np.array([float(ws), float(we), float(bb)], dtype=float)
    P = center[np.newaxis, :] + offs * R
    L = _ch3_nll_sum_on_flat_grid(study, exam, y, P[:, 0], P[:, 1], P[:, 2])
    U = np.empty(P.shape[0], dtype=float)
    V = np.empty(P.shape[0], dtype=float)
    W = np.empty(P.shape[0], dtype=float)
    for i, p in enumerate(P):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, p[0], p[1], p[2])
        U[i], V[i], W[i] = -float(g1), -float(g2), -float(gb)
    return P, L, U, V, W


_BALL_FIELD_CACHE: dict[tuple, tuple] = {}


def _ch3_lik_ball_vector_field_cached(study, exam, y, ws, we, bb, wide_bounds):
    key = (
        round(float(ws), 5), round(float(we), 5), round(float(bb), 5),
        round(float(wide_bounds[0]), 4), round(float(wide_bounds[1]), 4),
        round(float(wide_bounds[2]), 4), round(float(wide_bounds[3]), 4),
        round(float(wide_bounds[4]), 4), round(float(wide_bounds[5]), 4),
        int(CH3_LIK_3D_BALL_NSAMPLE),
    )
    hit = _BALL_FIELD_CACHE.get(key)
    if hit is not None:
        return hit
    out = _ch3_lik_ball_vector_field(study, exam, y, ws, we, bb, wide_bounds)
    _BALL_FIELD_CACHE[key] = out
    return out


def _ch3_lik_ball_nll_limits(study, exam, y, path, wide_bounds):
    chunks = []
    for r in np.asarray(path, dtype=float):
        _, L, _, _, _ = _ch3_lik_ball_vector_field_cached(
            study, exam, y, float(r[0]), float(r[1]), float(r[2]), wide_bounds,
        )
        chunks.append(L)
    flat = np.concatenate(chunks) if chunks else np.array([0.0])
    return float(np.min(flat)), float(np.max(flat))


def _ch3_lik_gd_ax3d_bounds(pack, *, specs_fn=None):
    """Fixed 3-D limits for ch4_06/07: contain full GD motion + arrow tips."""
    if specs_fn is None:
        specs_fn = _ch3_lik_gd_frame_specs
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    eta = float(pack["gd_eta"])
    pts = [np.asarray(pack["path"], dtype=float).reshape(-1, 3)]
    for r in pack["path"]:
        ws, we, bb = float(r[0]), float(r[1]), float(r[2])
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        pts.append(np.array([
            [ws - eta * g1, we, bb],
            [ws, we - eta * g2, bb],
            [ws, we, bb - eta * gb],
            [ws - eta * g1, we - eta * g2, bb - eta * gb],
        ], dtype=float))
    for spec in specs_fn(pack):
        pts.append(np.array([[spec["ws"], spec["we"], spec["bb"]]], dtype=float))
    all_pts = np.vstack(pts)
    k_lo1, k_hi1 = float(pack["W1m"].min()), float(pack["W1m"].max())
    k_lo2, k_hi2 = float(pack["W2m"].min()), float(pack["W2m"].max())
    k_lob, k_hib = float(pack["Bm"].min()), float(pack["Bm"].max())
    return _ch3_lik_ax3d_fixed_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, all_pts,
    )


def _ch3_lik_gd_bounds_source(study, exam, y, W1m, W2m, Bm):
    """Minimal pack fields for ``_ch3_lik_gd_ax3d_bounds`` (matches ch4_06, unchanged)."""
    gd_trail = _ch3_lik_gd_path(
        study, exam, y, CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, CH3_LIK_GD_STEP,
    )
    return {
        "study": study, "exam": exam, "y": y,
        "W1m": W1m, "W2m": W2m, "Bm": Bm,
        "path": gd_trail,
        "gd_start": CH3_LIK_3D_PATH_START,
        "gd_n_iters": CH3_LIK_GD_N_ITERS,
        "gd_eta": CH3_LIK_GD_STEP,
    }


def _ch3_lik_story_view_bounds(pack):
    """3-D axis limits shared with ch4_06 opening (``gd_ax3d_bounds``)."""
    b = pack.get("gd_ax3d_bounds")
    if b is not None:
        return b
    return pack["ax3d_bounds"]


def _ch3_lik_gd_display_delta(raw, *, log_display=False):
    """Map a raw GD step component to display length (``log1p`` when ``log_display``)."""
    v = float(raw)
    if not log_display:
        return v
    if abs(v) < 1e-12:
        return 0.0
    return float(np.sign(v) * np.log1p(abs(v)))


def _ch3_lik_draw_gd_axis_arrow(ax3d, x0, y0, z0, dx, dy, dz, color, *, head_len, alpha=1.0):
    """Draw one GD arrow: line shaft + quiver head (works for short vectors)."""
    mag = float(np.hypot(dx, np.hypot(dy, dz)))
    if mag < 1e-12:
        return
    a = float(np.clip(float(alpha), 0.0, 1.0))
    x1, y1, z1 = float(x0 + dx), float(y0 + dy), float(z0 + dz)
    ax3d.plot(
        [x0, x1], [y0, y1], [z0, z1],
        color=color, linewidth=3.2, alpha=a * 0.95, zorder=18, solid_capstyle="round",
    )
    ux, uy, uz = dx / mag, dy / mag, dz / mag
    hl = float(min(max(float(head_len), 0.04 * mag), 0.42 * mag))
    ax3d.quiver(
        x1 - ux * hl, y1 - uy * hl, z1 - uz * hl,
        ux * hl, uy * hl, uz * hl,
        color=color, arrow_length_ratio=0.42, linewidth=2.6,
        normalize=False, alpha=a * 0.95, zorder=19,
    )


def _ch3_lik_draw_gd_combined_arrow(ax3d, ws, we, bb, grad, eta, color, *, bounds, alpha=1.0, log_display=False):
    g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
    span = float(np.max(_ch3_lik_bounds_span(bounds))) if bounds is not None else 1.0
    head_len = 0.06 * span
    _ch3_lik_draw_gd_axis_arrow(
        ax3d, float(ws), float(we), float(bb),
        _ch3_lik_gd_display_delta(-float(eta) * g1, log_display=log_display),
        _ch3_lik_gd_display_delta(-float(eta) * g2, log_display=log_display),
        _ch3_lik_gd_display_delta(-float(eta) * gb, log_display=log_display),
        color, head_len=head_len, alpha=alpha,
    )


def _ch3_lik_draw_gd_axis_deltas(ax3d, ws, we, bb, deltas, *, visible=(True, True, True), bounds=None, alpha=1.0, colors=None, log_display=False):
    """Axis-aligned arrows with explicit (Δw_ST, Δw_EL, Δb) — used for Newton morph."""
    from ch4_layout import CH4_GD_ARROW_B_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_ST_COLOR

    du, dv, dw = (float(deltas[0]), float(deltas[1]), float(deltas[2]))
    span = float(np.max(_ch3_lik_bounds_span(bounds))) if bounds is not None else 1.0
    head_len = 0.06 * span
    if colors is None:
        colors = (CH4_GD_ARROW_ST_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_B_COLOR)
    specs = (
        (du, 0.0, 0.0, colors[0], bool(visible[0])),
        (0.0, dv, 0.0, colors[1], bool(visible[1])),
        (0.0, 0.0, dw, colors[2], bool(visible[2])),
    )
    for dx, dy, dz, color, show in specs:
        if not show:
            continue
        _ch3_lik_draw_gd_axis_arrow(
            ax3d, float(ws), float(we), float(bb),
            _ch3_lik_gd_display_delta(dx, log_display=log_display),
            _ch3_lik_gd_display_delta(dy, log_display=log_display),
            _ch3_lik_gd_display_delta(dz, log_display=log_display),
            color, head_len=head_len, alpha=alpha,
        )


def _ch3_lik_draw_gd_step_arrows(ax3d, ws, we, bb, grad, eta, *, visible=(True, True, True), bounds=None, alpha=1.0, log_display=False):
    """Axis-aligned GD step arrows: length = α × ∂NLL/∂(coord), direction of the update."""
    from ch4_layout import CH4_GD_ARROW_B_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_ST_COLOR

    g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
    eta = float(eta)
    span = float(np.max(_ch3_lik_bounds_span(bounds))) if bounds is not None else 1.0
    head_len = 0.06 * span
    specs = (
        (-eta * g1, 0.0, 0.0, CH4_GD_ARROW_ST_COLOR, bool(visible[0])),
        (0.0, -eta * g2, 0.0, CH4_GD_ARROW_EL_COLOR, bool(visible[1])),
        (0.0, 0.0, -eta * gb, CH4_GD_ARROW_B_COLOR, bool(visible[2])),
    )
    for du, dv, dw, color, show in specs:
        if not show:
            continue
        _ch3_lik_draw_gd_axis_arrow(
            ax3d, float(ws), float(we), float(bb),
            _ch3_lik_gd_display_delta(du, log_display=log_display),
            _ch3_lik_gd_display_delta(dv, log_display=log_display),
            _ch3_lik_gd_display_delta(dw, log_display=log_display),
            color, head_len=head_len, alpha=alpha,
        )


def _ch3_lik_draw_gd_arrows_for_spec(
    ax3d, ws, we, bb, grad, eta, *, bounds, arrow_mode, visible, transition_u=0.0,
    newton_split_vec=None, log_display=False,
):
    from ch4_layout import CH4_GD_GRADIENT_COLOR

    bounds = bounds
    if arrow_mode == "none":
        return
    if arrow_mode == "newton_morph":
        nv = newton_split_vec
        if nv is None:
            return
        u = float(np.clip(float(transition_u), 0.0, 1.0))
        g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
        eta = float(eta)
        dw, de, db = (float(nv[0]), float(nv[1]), float(nv[2]))
        # Descent direction: θ ← θ − H⁻¹∇NLL (lerp GD split steps → Newton step).
        deltas = (
            (1.0 - u) * (-eta * g1) + u * (-dw),
            (1.0 - u) * (-eta * g2) + u * (-de),
            (1.0 - u) * (-eta * gb) + u * (-db),
        )
        _ch3_lik_draw_gd_axis_deltas(
            ax3d, ws, we, bb, deltas, visible=visible, bounds=bounds, alpha=1.0,
            log_display=log_display,
        )
        return
    if arrow_mode == "split":
        _ch3_lik_draw_gd_step_arrows(
            ax3d, ws, we, bb, grad, eta, visible=visible, bounds=bounds,
            log_display=log_display,
        )
        return
    if arrow_mode == "combined":
        _ch3_lik_draw_gd_combined_arrow(
            ax3d, ws, we, bb, grad, eta, CH4_GD_GRADIENT_COLOR, bounds=bounds,
            log_display=log_display,
        )
        return
    if arrow_mode == "transition":
        u = float(np.clip(float(transition_u), 0.0, 1.0))
        if u < 1.0 - 1e-9:
            _ch3_lik_draw_gd_step_arrows(
                ax3d, ws, we, bb, grad, eta, visible=visible, bounds=bounds, alpha=1.0 - u,
                log_display=log_display,
            )
        if u > 1e-9:
            _ch3_lik_draw_gd_combined_arrow(
                ax3d, ws, we, bb, grad, eta, CH4_GD_GRADIENT_COLOR, bounds=bounds, alpha=u,
                log_display=log_display,
            )


def _ch3_lik_gd_frame_specs(pack):
    """Frame descriptors for sequential per-parameter GD (10 iterations)."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = (float(pack["gd_start"][0]), float(pack["gd_start"][1]), float(pack["gd_start"][2]))
    eta = float(pack["gd_eta"])
    n_iters = int(pack["gd_n_iters"])
    hold_n = max(int(CH3_LIK_GD_N_HOLD_ARROWS), 1)
    step_n = max(int(CH3_LIK_GD_N_PARAM_STEP), 2)
    specs = []

    def _append(ws_i, we_i, bb_i, grad, *, arrows, bold, arrow_mode="split", grad_red=False, bold_all=False, transition_u=0.0):
        specs.append({
            "ws": float(ws_i), "we": float(we_i), "bb": float(bb_i),
            "grad": (float(grad[0]), float(grad[1]), float(grad[2])),
            "eta": eta,
            "arrows": tuple(bool(v) for v in arrows),
            "bold": bold,
            "bold_all": bool(bold_all),
            "grad_red": bool(grad_red),
            "arrow_mode": str(arrow_mode),
            "transition_u": float(transition_u),
        })

    for _ in range(n_iters):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (g1, g2, gb)
        for _ in range(hold_n):
            _append(ws, we, bb, grad, arrows=(True, True, True), bold=None)

        ws0 = ws
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws0 - u * eta * g1, we, bb, grad,
                arrows=(False, True, True), bold=0,
            )
        ws = ws0 - eta * g1

        we0 = we
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws, we0 - u * eta * g2, bb, grad,
                arrows=(False, False, True), bold=1,
            )
        we = we0 - eta * g2

        bb0 = bb
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws, we, bb0 - u * eta * gb, grad,
                arrows=(False, False, False), bold=2,
            )
        bb = bb0 - eta * gb

    return specs


def _ch3_lik_gd_combined_frame_specs(pack):
    """ch4_07: split arrows once, then red combined gradient arrow + simultaneous GD."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = (float(pack["gd_start"][0]), float(pack["gd_start"][1]), float(pack["gd_start"][2]))
    eta = float(pack["gd_eta"])
    n_iters = int(pack["gd_n_iters"])
    hold_n = max(int(CH3_LIK_GD_N_HOLD_ARROWS), 1)
    step_n = max(int(CH3_LIK_GD_N_PARAM_STEP), 2)
    combine_n = max(int(CH3_LIK_GD_N_COMBINE), 2)
    specs = []

    def _append(ws_i, we_i, bb_i, grad, *, arrow_mode, grad_red=False, bold_all=False, transition_u=0.0):
        specs.append({
            "ws": float(ws_i), "we": float(we_i), "bb": float(bb_i),
            "grad": (float(grad[0]), float(grad[1]), float(grad[2])),
            "eta": eta,
            "arrows": (True, True, True),
            "bold": None,
            "bold_all": bool(bold_all),
            "grad_red": bool(grad_red),
            "arrow_mode": str(arrow_mode),
            "transition_u": float(transition_u),
        })

    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    grad = (g1, g2, gb)
    for _ in range(hold_n):
        _append(ws, we, bb, grad, arrow_mode="split", grad_red=False)

    for si in range(combine_n):
        u = ch3_knob_smoothstep(float(si) / float(combine_n - 1))
        _append(
            ws, we, bb, grad,
            arrow_mode="transition",
            transition_u=u,
            grad_red=u >= 0.5,
        )

    for _ in range(n_iters):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (g1, g2, gb)
        for _ in range(hold_n):
            _append(ws, we, bb, grad, arrow_mode="combined", grad_red=True)

        ws0, we0, bb0 = ws, we, bb
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            _append(
                ws0 - u * eta * g1, we0 - u * eta * g2, bb0 - u * eta * gb,
                grad,
                arrow_mode="none",
                grad_red=True,
                bold_all=True,
            )
        ws = ws0 - eta * g1
        we = we0 - eta * g2
        bb = bb0 - eta * gb

    return specs


def _ch3_lik_draw_ball_vectors(
    ax3d, study, exam, y, ws, we, bb, wide_bounds, *,
    vmin, vmax, cmap=None, alpha=0.86,
    ball_field=None,
):
    if ball_field is None:
        ball_field = _ch3_lik_ball_vector_field_cached(study, exam, y, ws, we, bb, wide_bounds)
    P, L, U, V, W = ball_field
    gn = np.sqrt(U * U + V * V + W * W)
    keep = gn > 1e-14
    if not np.any(keep):
        return
    P = P[keep]
    L = L[keep]
    U = U[keep]
    V = V[keep]
    W = W[keep]
    from ch4_layout import ch4_nll_heatmap_cmap

    cmap = ch4_nll_heatmap_cmap() if cmap is None else cmap
    span = max(float(vmax) - float(vmin), 1e-9)
    cols = cmap((L - float(vmin)) / span)
    ax3d.scatter(
        P[:, 0], P[:, 1], P[:, 2],
        c=L, cmap=cmap, vmin=float(vmin), vmax=float(vmax),
        s=22.0, alpha=float(alpha) * 0.55, linewidths=0, depthshade=False, zorder=6,
    )
    ax3d.quiver(
        P[:, 0], P[:, 1], P[:, 2],
        U, V, W,
        colors=cols,
        length=float(CH3_LIK_3D_BALL_QUIVER_LEN),
        normalize=True,
        alpha=float(alpha),
        linewidth=0.95,
        arrow_length_ratio=0.34,
        zorder=7,
    )


def _ch3_lik_3d_measurements_pack(*, ball_colormap_limits=False):
    state = _ch3_lik86_terminal_state()
    study, exam, y = state["study"], state["exam"], state["y"]
    ws, we, bb = state["w_st"], state["w_el"], state["b"]
    gn = 10 if _CH3_DRAFT else 14
    w1g = np.linspace(float(state["w1_lo"]), float(state["w1_hi"]), gn)
    w2g = np.linspace(float(state["w2_lo"]), float(state["w2_hi"]), gn)
    bg = np.linspace(-float(CH3_LIK_W12_B_HALF), float(CH3_LIK_W12_B_HALF), gn)
    W1m, W2m, Bm = np.meshgrid(w1g, w2g, bg, indexing="ij")
    Lf = _ch3_nll_sum_on_flat_grid(study, exam, y, W1m.ravel(), W2m.ravel(), Bm.ravel()).reshape(W1m.shape)
    vmin, vmax = float(np.nanmin(Lf)), float(np.nanmax(Lf))
    k_lo1, k_hi1 = float(W1m.min()), float(W1m.max())
    k_lo2, k_hi2 = float(W2m.min()), float(W2m.max())
    k_lob, k_hib = float(Bm.min()), float(Bm.max())
    gd_ax3d_bounds = _ch3_lik_gd_ax3d_bounds(
        _ch3_lik_gd_bounds_source(study, exam, y, W1m, W2m, Bm),
    )
    path = _ch3_lik_fun_path_3d(start=CH3_LIK_3D_PATH_START, bounds=gd_ax3d_bounds)
    ax3d_bounds = _ch3_lik_ax3d_fixed_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, path,
        extra_pts=np.array([[path[0, 0], path[0, 1], path[0, 2]]], dtype=float),
    )
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in path
    ], dtype=float)
    if ball_colormap_limits:
        ball_vmin, ball_vmax = _ch3_lik_ball_nll_limits(study, exam, y, path, gd_ax3d_bounds)
    else:
        ball_vmin, ball_vmax = float(vmin), float(vmax)
    span = max(float(ball_vmax) - float(ball_vmin), 1e-9)
    from ch4_layout import ch4_nll_heatmap_cmap

    nll_cmap = ch4_nll_heatmap_cmap()
    path_colors = [nll_cmap(float((v - ball_vmin) / span)) for v in path_nll]
    n_slow = max(CH3_LIK_3D_N_PATH // 3, 20)
    n_fast = CH3_LIK_3D_N_PATH - n_slow
    path_us = list(np.linspace(0.02, 0.35, n_slow, endpoint=True)) + list(
        np.linspace(0.35, 1.0, n_fast, endpoint=True)
    )
    return {
        "study": study, "exam": exam, "y": y,
        "W1m": W1m, "W2m": W2m, "Bm": Bm, "Lf": Lf,
        "vmin": vmin, "vmax": vmax,
        "path": path, "path_colors": path_colors,
        "ax3d_bounds": ax3d_bounds,
        "gd_ax3d_bounds": gd_ax3d_bounds,
        "path_us": path_us, "n_slow": n_slow, "n_fast": n_fast,
        "ball_vmin": ball_vmin, "ball_vmax": ball_vmax,
    }


_CH4_NLL_GLOBAL_SCALE = None


def ch4_nll_global_scale():
    """Canonical NLL heatmap limits — same as ch4_05a CT scan ``pack['vmin'/'vmax']``."""
    global _CH4_NLL_GLOBAL_SCALE
    if _CH4_NLL_GLOBAL_SCALE is None:
        pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
        _CH4_NLL_GLOBAL_SCALE = (float(pack["vmin"]), float(pack["vmax"]))
    return _CH4_NLL_GLOBAL_SCALE


def ch4_nll_global_legend_blocks(*, stacked=False, pre_gap_pt=None):
    """Right-rail NLL color scale — shared range and labels everywhere."""
    from ch4_layout import ch4_nll_heatmap_legend_blocks

    lo, hi = ch4_nll_global_scale()
    return ch4_nll_heatmap_legend_blocks(lo, hi)


def ch3_frame_lik_weight3d_measurements(
    study, exam, y, ws, we, bb, *,
    W1m, W2m, Bm, Lf, vmin, vmax,
    path_xyz=None,
    path_colors=None,
    path_u=1.0,
    notation_condensed=False,
    measurements=None,
    write_progress=1.0,
    plot_alpha=1.0,
    ax3d_bounds=None,
    show_weight_grid=False,
    show_ball_vectors=False,
    wide_bounds=None,
    ball_vmin=None,
    ball_vmax=None,
    ball_field=None,
    elev=CH3_LIK_CH4_CT_ELEV,
    azim=None,
    cam_azim_u=0.0,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    grad=None,
    step_size=None,
    gd_formulas=False,
    newton_formulas=False,
    gd_arrows_grad=None,
    gd_arrows_visible=None,
    gd_bold_update_idx=None,
    gd_bold_all_updates=False,
    gd_grad_red=False,
    gd_arrow_mode="split",
    gd_arrow_transition_u=0.0,
    gd_newton_split_vec=None,
    gd_ghosts=None,
    gd_arrow_log_display=False,
    newton_step_vec=None,
    newton_arrow_color=None,
    path_trail_xyz=None,
    path_trail_linecolor=None,
    path_trail_extension_xyz=None,
    path_trail_extension_color=None,
    path_trail_extension_linestyle=None,
    path_trail_extension_alpha=None,
    path_trail_extension_waypoint_alpha=None,
    gd_bottom_blocks=None,
    progress_override=None,
    right_write_progress=None,
    title_write_progress=None,
    bottom_write_progress=None,
    show_path_line=True,
    voxel_draw=None,
    point_s=None,
    point_color=None,
    right_title=None,
    right_title_single_line=None,
    hide_duo_left=False,
    panel_xlim=None,
    panel_ylim=None,
    panel_xlabel=None,
    panel_sigma_stg=None,
    panel_sigma_elg=None,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_HERE_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        ch4_cached_formula_blocks_3d_story,
        ch4_cached_formula_blocks_gd_story,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_gd_story,
        ch4_formula_blocks_newton_story,
        ch4_knob_asset_pack,
        ch4_nll_heatmap_cmap,
        ch4_rails_cache_key,
        ch4_rails_cache_key_gd,
        ch4_rails_cache_key_newton,
        ch4_we_are_here_blocks,
        compose_tutorial,
    )

    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    if not hide_duo_left:
        leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
        panel_kw = dict(show_colormap=True, highlight_mistakes_flag=False)
        if panel_sigma_stg is not None and panel_sigma_elg is not None:
            panel_kw["sigma_stg"] = panel_sigma_stg
            panel_kw["sigma_elg"] = panel_sigma_elg
        try:
            ch3_draw_left_panel(ax_data, ws, we, bb, study, exam, y, leg, **panel_kw)
        except TypeError:
            ch3_draw_left_panel(
                ax_data, ws, we, bb, study, exam, y, leg,
                show_colormap=True, highlight_mistakes_flag=False,
            )
        ax_data.set_xlim(*(panel_xlim if panel_xlim is not None else xlim))
        ax_data.set_ylim(*(panel_ylim if panel_ylim is not None else ylim))
        if panel_xlabel is not None:
            ax_data.set_xlabel(str(panel_xlabel), fontsize=AXIS_LABEL_SIZE, labelpad=10)
        finalize_style_legend_tex(ax_data)
        knob_rgbs, canvas_sides = ch4_knob_asset_pack()
        ch3_draw_knob_row(
            fig, axes_k, ws, we, bb, "st", knob_rgbs, canvas_sides,
            rot_strip_deg=0.0, strip_scale=1.0,
            knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
        )
    else:
        ax_data.set_visible(False)
        for ax in axes_k:
            ax.set_visible(False)
    if show_weight_grid:
        ax3d.scatter(
            W1m.ravel(), W2m.ravel(), Bm.ravel(),
            c=Lf.ravel(), cmap=ch4_nll_heatmap_cmap(), vmin=float(vmin), vmax=float(vmax),
            s=28.0, alpha=0.45, linewidths=0, depthshade=False, zorder=1,
        )
    k_lo1, k_hi1 = float(W1m.min()), float(W1m.max())
    k_lo2, k_hi2 = float(W2m.min()), float(W2m.max())
    k_lob, k_hib = float(Bm.min()), float(Bm.max())
    if ax3d_bounds is not None:
        dlo1, dhi1, dlo2, dhi2, dlob, dhib = ax3d_bounds
    else:
        dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch3_keras_ax3d_zoom_bounds(
            k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, np.array([[ws, we, bb]], dtype=float),
        )
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    if azim is None:
        if abs(float(cam_rot_deg)) > 1e-9 or abs(float(cam_azim_u)) > 1e-9:
            azim = _ch3_lik_cam_azim(cam_azim_u, total_deg=float(cam_rot_deg))
        else:
            azim = float(CH3_LIK_CH4_CT_AZIM)
    ax3d.view_init(elev=float(elev), azim=float(azim))
    ref_bounds = wide_bounds if wide_bounds is not None else (
        ax3d_bounds if ax3d_bounds is not None else (dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    )
    if gd_arrows_grad is not None:
        vis = gd_arrows_visible if gd_arrows_visible is not None else (True, True, True)
        _ch3_lik_draw_gd_arrows_for_spec(
            ax3d, ws, we, bb, gd_arrows_grad, float(step_size if step_size is not None else CH3_LIK_GD_STEP),
            bounds=(dlo1, dhi1, dlo2, dhi2, dlob, dhib),
            arrow_mode=str(gd_arrow_mode),
            visible=vis,
            transition_u=float(gd_arrow_transition_u),
            newton_split_vec=gd_newton_split_vec,
            log_display=bool(gd_arrow_log_display),
        )
    pt_s = float(CH3_LIK_GD_POINT_S if point_s is None else point_s)
    pt_c = CH3_LIK_3D_POINT_COLOR if point_color is None else str(point_color)
    if gd_ghosts:
        g_eta = float(step_size if step_size is not None else CH3_LIK_GD_STEP)
        for gspec in gd_ghosts:
            gws = float(gspec["ws"])
            gwe = float(gspec["we"])
            gbb = float(gspec["bb"])
            ggrad = gspec["grad"]
            ga = float(np.clip(float(gspec.get("alpha", 0.38)), 0.0, 1.0))
            g_vis = gspec.get("arrows", (True, True, True))
            g_mode = str(gspec.get("arrow_mode", "combined"))
            ax3d.scatter(
                [gws], [gwe], [gbb],
                s=max(pt_s * 0.42, 18.0), c=["#c62828"], edgecolors="white",
                linewidths=1.2, depthshade=False, alpha=min(ga * 0.95, 0.78), zorder=18,
            )
            if g_mode == "split":
                _ch3_lik_draw_gd_step_arrows(
                    ax3d, gws, gwe, gbb, ggrad, g_eta,
                    visible=tuple(bool(v) for v in g_vis),
                    bounds=(dlo1, dhi1, dlo2, dhi2, dlob, dhib),
                    alpha=min(ga * 0.90, 0.75),
                    log_display=bool(gd_arrow_log_display),
                )
            else:
                from ch4_layout import CH4_GD_GRADIENT_COLOR
                _ch3_lik_draw_gd_combined_arrow(
                    ax3d, gws, gwe, gbb, ggrad, g_eta, CH4_GD_GRADIENT_COLOR,
                    bounds=(dlo1, dhi1, dlo2, dhi2, dlob, dhib),
                    alpha=min(ga * 0.90, 0.75),
                    log_display=bool(gd_arrow_log_display),
                )
    if newton_step_vec is not None:
        nv = np.asarray(newton_step_vec, dtype=float).ravel()
        nc = str(newton_arrow_color or "#7e57c2")
        if float(np.linalg.norm(nv)) > 1e-14:
            _ch3_lik_draw_gd_axis_arrow(
                ax3d, float(ws), float(we), float(bb),
                float(nv[0]), float(nv[1]), float(nv[2]),
                nc,
                head_len=0.06 * float(np.max(_ch3_lik_bounds_span((dlo1, dhi1, dlo2, dhi2, dlob, dhib)))),
                alpha=0.95,
            )
    elif show_ball_vectors:
        _ch3_lik_draw_ball_vectors(
            ax3d, study, exam, y, ws, we, bb, ref_bounds,
            vmin=float(ball_vmin if ball_vmin is not None else vmin),
            vmax=float(ball_vmax if ball_vmax is not None else vmax),
            ball_field=ball_field,
        )
    if voxel_draw is not None:
        xs = voxel_draw.get("xs")
        if xs is not None and len(xs):
            ax3d.bar3d(
                voxel_draw["xs"], voxel_draw["ys"], voxel_draw["zs"],
                float(voxel_draw["dx"]), float(voxel_draw["dy"]), float(voxel_draw["dz"]),
                color=voxel_draw["colors"],
                shade=False,
                linewidth=0.0,
                edgecolor=(0.0, 0.0, 0.0, 0.0),
                zorder=6,
            )
    ax3d.scatter(
        [ws], [we], [bb], s=pt_s, c=[pt_c], edgecolors="white",
        linewidths=2.0, depthshade=False, zorder=20,
    )
    if path_trail_xyz is not None and len(path_trail_xyz) >= 2:
        Pk = np.asarray(path_trail_xyz, dtype=float)
        trail_c = str(path_trail_linecolor or CH3_LIK_3D_POINT_COLOR)
        trail_lw = 6.0 if path_trail_linecolor is not None else 3.0
        ax3d.plot(
            Pk[:, 0], Pk[:, 1], Pk[:, 2],
            color=trail_c, linewidth=trail_lw, alpha=0.95, zorder=14,
        )
        if Pk.shape[0] > 1:
            wp_s = max(pt_s * (0.55 if path_trail_linecolor is not None else 0.38), 28.0)
            ax3d.scatter(
                Pk[:, 0], Pk[:, 1], Pk[:, 2],
                s=wp_s, c=[CH3_LIK_GD_PATH_WAYPOINT_COLOR if path_trail_linecolor is None else trail_c],
                edgecolors="white",
                linewidths=1.4, depthshade=False, alpha=0.98, zorder=15,
            )
    if path_trail_extension_xyz is not None and len(path_trail_extension_xyz) >= 2:
        Pe = np.asarray(path_trail_extension_xyz, dtype=float)
        ext_c = str(path_trail_extension_color or CH3_LIK_3D_POINT_COLOR)
        ext_ls = str(path_trail_extension_linestyle or "-")
        ext_a = float(path_trail_extension_alpha) if path_trail_extension_alpha is not None else 0.92
        wp_a = (
            float(path_trail_extension_waypoint_alpha)
            if path_trail_extension_waypoint_alpha is not None
            else ext_a * 0.90
        )
        ax3d.plot(
            Pe[:, 0], Pe[:, 1], Pe[:, 2],
            color=ext_c, linewidth=3.4, alpha=ext_a, zorder=11, linestyle=ext_ls,
        )
        if Pe.shape[0] > 1:
            ax3d.scatter(
                Pe[:, 0], Pe[:, 1], Pe[:, 2],
                s=max(pt_s * 0.34, 20.0), c=[ext_c], edgecolors="white",
                linewidths=1.0, depthshade=False, alpha=wp_a, zorder=12,
            )
    if show_path_line and path_xyz is not None and len(path_xyz) >= 2:
        P = np.asarray(path_xyz, dtype=float)
        n_keep = max(2, int(round(float(path_u) * (P.shape[0] - 1))) + 1)
        Pk = P[:n_keep]
        if path_colors is not None and len(path_colors) >= n_keep:
            cols = np.asarray(path_colors[:n_keep])
            for i in range(Pk.shape[0] - 1):
                ax3d.plot(
                    Pk[i:i + 2, 0], Pk[i:i + 2, 1], Pk[i:i + 2, 2],
                    color=cols[i], linewidth=3.2, alpha=0.95,
                )
        else:
            ax3d.plot(Pk[:, 0], Pk[:, 1], Pk[:, 2], color=CH3_LIK_3D_POINT_COLOR, linewidth=3.0, alpha=0.9)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    nll = float(-loss_log_likelihood(ws, we, bb, study, exam, y))
    bvmin = float(ball_vmin if ball_vmin is not None else vmin)
    bvmax = float(ball_vmax if ball_vmax is not None else vmax)
    if measurements is not None:
        right = measurements
    else:
        right = ch4_we_are_here_blocks(
            ws, we, bb, nll,
            nll_vmin=bvmin, nll_vmax=bvmax,
            point_color=CH3_LIK_3D_POINT_COLOR,
            grad=grad, step_size=step_size,
        )
    if gd_bottom_blocks is not None:
        bottom = gd_bottom_blocks
        rails_key = None
    elif newton_formulas:
        bottom = ch4_formula_blocks_newton_story(
            highlight_update_idx=gd_bold_update_idx,
            highlight_all_updates=gd_bold_all_updates,
        )
        rails_key = (
            ch4_rails_cache_key_newton(
                highlight_update_idx=gd_bold_update_idx,
                highlight_all=gd_bold_all_updates,
            )
            if float(write_progress) >= 1.0 - 1e-9
            else None
        )
    elif gd_formulas:
        bottom = ch4_formula_blocks_gd_story(
            highlight_update_idx=gd_bold_update_idx,
            highlight_all_updates=gd_bold_all_updates,
            grad_red=gd_grad_red,
        )
        rails_key = (
            ch4_rails_cache_key_gd(
                highlight_update_idx=gd_bold_update_idx,
                highlight_all=gd_bold_all_updates,
                grad_red=gd_grad_red,
            )
            if float(write_progress) >= 1.0 - 1e-9
            else None
        )
    elif notation_condensed:
        bottom = ch4_cached_formula_blocks_3d_story()
        rails_key = ch4_rails_cache_key(gd_formulas=False) if float(write_progress) >= 1.0 - 1e-9 else None
    else:
        bottom = ch4_cached_formula_blocks_3d_story()
        rails_key = ch4_rails_cache_key(gd_formulas=False) if float(write_progress) >= 1.0 - 1e-9 else None
    if notation_condensed:
        rt = CH4_HERE_SECTION_TITLE if right_title is None else right_title
        kw = dict(
            plot_img=plot_img,
            right_blocks=right,
            bottom_blocks=bottom,
            corner_blocks=ch4_cached_notation_corner_blocks(),
            right_title=rt,
            bottom_title=CH4_FORMULAS_SECTION_TITLE,
            corner_title=CH4_NOTATION_SECTION_TITLE,
            right_title_color=CH3_LIK_3D_POINT_COLOR,
            write_progress=write_progress,
            plot_alpha=plot_alpha,
            theme="classic_light",
            rails_cache_key=rails_key,
            progress_override=progress_override,
            right_write_progress=right_write_progress,
            title_write_progress=title_write_progress,
            bottom_write_progress=bottom_write_progress,
        )
        if right_title_single_line is not None:
            kw["right_title_single_line"] = bool(right_title_single_line)
        return compose_tutorial(**kw)
    return compose_tutorial(
        plot_img,
        right_blocks=right,
        bottom_blocks=bottom,
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_HERE_SECTION_TITLE,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title_color=CH3_LIK_3D_POINT_COLOR,
        write_progress=write_progress,
        plot_alpha=plot_alpha,
        theme="classic_light",
        rails_cache_key=rails_key,
        progress_override=progress_override,
        right_write_progress=right_write_progress,
        title_write_progress=title_write_progress,
        bottom_write_progress=bottom_write_progress,
    )


def _ch3_lik_we_are_here_at(pack, ws, we, bb):
    from ch4_layout import ch4_we_are_here_blocks

    nll = float(-loss_log_likelihood(ws, we, bb, pack["study"], pack["exam"], pack["y"]))
    return ch4_we_are_here_blocks(
        ws, we, bb, nll,
        nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
        point_color=CH3_LIK_3D_POINT_COLOR,
    )


def _ch3_lik_append_path_frames(
    frames, pack, *,
    show_ball_vectors=False,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    gd_formulas=False,
):
    path = pack["path"]
    path_us = pack["path_us"]
    n_path = max(len(path_us) - 1, 1)
    cols = pack["path_colors"]
    bounds = _ch3_lik_story_view_bounds(pack)
    for i, pu in enumerate(path_us):
        pu = float(pu)
        idx = min(len(path) - 1, max(0, int(round(pu * (len(path) - 1)))))
        ws_i, we_i, bb_i = path[idx]
        frames.append(
            ch3_frame_lik_weight3d_measurements(
                pack["study"], pack["exam"], pack["y"], ws_i, we_i, bb_i,
                W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
                vmin=pack["vmin"], vmax=pack["vmax"],
                path_xyz=path, path_colors=cols, path_u=pu,
                notation_condensed=True,
                measurements=_ch3_lik_we_are_here_at(pack, ws_i, we_i, bb_i),
                ax3d_bounds=bounds,
                wide_bounds=bounds,
                show_ball_vectors=show_ball_vectors,
                ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
                cam_azim_u=float(i) / float(n_path),
                cam_rot_deg=float(cam_rot_deg),
                gd_formulas=gd_formulas,
            )
        )


def _ch3_lik_append_04_right_intro(frames, pack):
    """Handwrite We-are-here title + weights + NLL before the path animation."""
    path = pack["path"]
    ws0, we0, bb0 = path[0]
    right = _ch3_lik_we_are_here_at(pack, ws0, we0, bb0)
    bounds = _ch3_lik_story_view_bounds(pack)

    def _emit(*, title_u=1.0, right_u=1.0):
        frames.append(ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_colors=pack["path_colors"], path_u=0.0,
            notation_condensed=True,
            measurements=right,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            show_ball_vectors=False,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            cam_azim_u=0.0,
            gd_formulas=False,
            write_progress=1.0,
            title_write_progress=title_u,
            right_write_progress=right_u,
        ))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_RIGHT_TITLE, endpoint=True):
        _emit(title_u=ch3_knob_smoothstep(float(tv)), right_u=0.0)
    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_RIGHT_WRITE, endpoint=True):
        _emit(title_u=1.0, right_u=ch3_knob_smoothstep(float(tv)))


def ch3_build_frames_likelihood_3d_measurements_story():
    pack = _ch3_lik_3d_measurements_pack()
    frames = []
    _ch3_lik_append_04_right_intro(frames, pack)
    _ch3_lik_append_path_frames(frames, pack, show_ball_vectors=False)
    if frames:
        last = frames[-1]
        for _ in range(max(10, CH3_SCRIPT_N_HOLD // 3)):
            frames.append(last.copy())
    return frames


def _ch3_lik_gd_path(study, exam, y, start, n_steps, eta):
    """Gradient-descent trajectory from ``start`` in (w_ST, w_EL, b)."""
    ws, we, bb = float(start[0]), float(start[1]), float(start[2])
    pts = [[ws, we, bb]]
    eta = float(eta)
    for _ in range(int(n_steps)):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        ws -= eta * float(g1)
        we -= eta * float(g2)
        bb -= eta * float(gb)
        pts.append([ws, we, bb])
    return np.asarray(pts, dtype=float)


def _ch3_lik_3d_gd_pack():
    pack = dict(_ch3_lik_3d_measurements_pack(ball_colormap_limits=False))
    trail_path = _ch3_lik_gd_path(
        pack["study"], pack["exam"], pack["y"],
        CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, CH3_LIK_GD_STEP,
    )
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in trail_path
    ], dtype=float)
    pack.update({
        "gd_start": CH3_LIK_3D_PATH_START,
        "gd_n_iters": CH3_LIK_GD_N_ITERS,
        "gd_eta": CH3_LIK_GD_STEP,
        "ball_vmin": float(np.min(path_nll)),
        "ball_vmax": float(np.max(path_nll)),
        "path": trail_path,
    })
    pack["gd_ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    pack["ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    return pack


def _ch3_lik_3d_gd_combined_pack():
    pack = _ch3_lik_3d_gd_pack()
    pack["gd_ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    pack["ax3d_bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    return pack


def _ch3_lik_gd_render_frame(pack, spec, *, cam_azim_u=0.0):
    from ch4_layout import (
        ch4_formula_blocks_newton_story,
        ch4_rails_cache_key_newton,
        ch4_we_are_here_blocks,
        ch4_we_are_here_grad_line_colors,
    )

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = float(spec["ws"]), float(spec["we"]), float(spec["bb"])
    grad = spec["grad"]
    eta = float(spec["eta"])
    grad_red = bool(spec.get("grad_red", False))
    nll = float(-loss_log_likelihood(ws, we, bb, study, exam, y))
    grad_colors = ch4_we_are_here_grad_line_colors(
        grad_red=grad_red,
        transition_u=float(spec.get("transition_u", 0.0)),
    )
    here_mode = str(spec.get("here_grad_mode", "partial"))
    newton_step = spec.get("newton_step_display")
    if newton_step is None and spec.get("newton_split_vec") is not None:
        newton_step = spec.get("newton_split_vec")
    right = ch4_we_are_here_blocks(
        ws, we, bb, nll,
        nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
        point_color=CH3_LIK_3D_POINT_COLOR,
        grad=grad, step_size=eta,
        grad_line_colors=grad_colors,
        here_grad_mode=here_mode,
        newton_step=newton_step,
        morph_u=float(spec.get("transition_u", 0.0)),
    )
    view_bounds = spec.get("view_bounds")
    if view_bounds is None:
        view_bounds = pack.get("gd_view_bounds", CH3_LIK_CT_VIEW_BOUNDS)
    arrow_mode = str(spec.get("arrow_mode", "split"))
    show_grad = arrow_mode != "none"
    path_trail = spec.get("path_trail")
    cam_u = float(spec["cam_azim_u"]) if "cam_azim_u" in spec else float(cam_azim_u)
    cam_rot = float(spec.get("cam_rot_deg", CH3_LIK_3D_CAM_PATH_ROT))
    newton_formulas = bool(spec.get("newton_formulas", False))
    hide_duo_left = bool(spec.get("hide_duo_left", pack.get("hide_duo_left", False)))
    return ch3_frame_lik_weight3d_measurements(
        study, exam, y, ws, we, bb,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        notation_condensed=True,
        measurements=right,
        ax3d_bounds=view_bounds,
        wide_bounds=view_bounds,
        show_ball_vectors=False,
        ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
        cam_azim_u=cam_u,
        cam_rot_deg=cam_rot,
        grad=grad,
        step_size=eta,
        gd_formulas=not newton_formulas,
        newton_formulas=newton_formulas,
        gd_arrows_grad=grad if show_grad else None,
        gd_arrows_visible=spec.get("arrows", (True, True, True)),
        gd_bold_update_idx=spec.get("bold"),
        gd_bold_all_updates=bool(spec.get("bold_all", False)),
        gd_grad_red=grad_red,
        gd_arrow_mode=arrow_mode,
        gd_arrow_transition_u=float(spec.get("transition_u", 0.0)),
        gd_newton_split_vec=spec.get("newton_split_vec"),
        gd_ghosts=spec.get("ghosts"),
        gd_arrow_log_display=bool(spec.get("gd_arrow_log_display", pack.get("gd_arrow_log_display", False))),
        newton_step_vec=spec.get("newton_step_vec"),
        newton_arrow_color=spec.get("newton_arrow_color"),
        path_trail_xyz=path_trail,
        path_trail_linecolor=spec.get("path_trail_linecolor"),
        path_trail_extension_xyz=spec.get("path_trail_extension"),
        path_trail_extension_color=spec.get("path_trail_extension_color"),
        path_trail_extension_linestyle=spec.get("path_trail_extension_linestyle"),
        path_trail_extension_alpha=spec.get("path_trail_extension_alpha"),
        path_trail_extension_waypoint_alpha=spec.get("path_trail_extension_waypoint_alpha"),
        show_path_line=bool(spec.get("show_path_line", path_trail is None)),
        voxel_draw=spec.get("voxel_draw"),
        hide_duo_left=hide_duo_left,
        panel_xlim=pack.get("panel_xlim"),
        panel_ylim=pack.get("panel_ylim"),
        panel_xlabel=pack.get("panel_xlabel"),
        panel_sigma_stg=pack.get("panel_sigma_stg"),
        panel_sigma_elg=pack.get("panel_sigma_elg"),
    )


def _ch3_lik_ball_intro_frames(pack, *, gd_formulas=False):
    """Zoom → 360° spin → zoom out; shared by ch4_05 and ch4_06."""
    _BALL_FIELD_CACHE.clear()
    path = pack["path"]
    ws0, we0, bb0 = path[0]
    wide = _ch3_lik_story_view_bounds(pack)
    tight = _ch3_lik_ball_zoom_bounds(ws0, we0, bb0, wide)
    intro_ball = _ch3_lik_ball_vector_field_cached(
        pack["study"], pack["exam"], pack["y"], ws0, we0, bb0, wide,
    )
    frames = []

    def _intro_frame(bounds, azim, *, path_u=0.0):
        if gd_formulas:
            from ch4_layout import ch4_we_are_here_blocks
            g1, g2, gb = _ch3_nll_sum_grad_at_point(
                pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            )
            nll0 = float(-loss_log_likelihood(ws0, we0, bb0, pack["study"], pack["exam"], pack["y"]))
            right = ch4_we_are_here_blocks(
                ws0, we0, bb0, nll0,
                nll_vmin=pack["ball_vmin"], nll_vmax=pack["ball_vmax"],
                point_color=CH3_LIK_3D_POINT_COLOR,
                grad=(g1, g2, gb), step_size=CH3_LIK_GD_STEP,
            )
        else:
            right = _ch3_lik_we_are_here_at(pack, ws0, we0, bb0)
        return ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_u=path_u,
            notation_condensed=True,
            measurements=right,
            ax3d_bounds=bounds,
            wide_bounds=wide,
            show_ball_vectors=True,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            ball_field=intro_ball,
            azim=float(azim),
            gd_formulas=gd_formulas,
        )

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_INTRO_ZOOM, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(_ch3_lik_lerp_bounds(wide, tight, u), CH3_LIK_3D_CAM_AZIM0))

    spin_n = max(int(CH3_LIK_3D_N_INTRO_SPIN), 2)
    for tv in np.linspace(0.0, 1.0, spin_n, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(tight, CH3_LIK_3D_CAM_AZIM0 + 360.0 * float(u)))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_INTRO_ZOOM_OUT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_intro_frame(_ch3_lik_lerp_bounds(tight, wide, u), CH3_LIK_3D_CAM_AZIM0))

    return frames


def _ch3_lik_append_gd_frames(
    frames, pack, *,
    cam_rot_deg=CH3_LIK_3D_CAM_PATH_ROT,
    specs_fn=None,
    render_fn=None,
):
    del cam_rot_deg  # camera pan is encoded in render_frame cam_azim_u
    if specs_fn is None:
        specs_fn = _ch3_lik_gd_frame_specs
    if render_fn is None:
        render_fn = _ch3_lik_gd_render_frame
    from ch4_export_pipeline import build_tutorial_frames

    rendered = build_tutorial_frames(
        pack,
        specs_fn(pack),
        render_fn,
        prewarm="gd",
        progress_label="ch4_gd",
    )
    frames.extend(rendered)


def _ch3_lik_append_gd_combined_frames(frames, pack):
    _ch3_lik_append_gd_frames(frames, pack, specs_fn=_ch3_lik_gd_combined_frame_specs)


def _ch3_lik_story_hold(frames):
    if frames:
        last = frames[-1]
        for _ in range(max(10, CH3_SCRIPT_N_HOLD // 3)):
            frames.append(last.copy())
    return frames


def _ch3_lik_pack_path_frame(pack, path_index=0, *, show_ball_vectors=False, gd_formulas=False, **frame_kw):
    """Render one path frame from a measurements/GD pack (no full story build)."""
    path = pack["path"]
    path_us = pack["path_us"]
    i = int(np.clip(path_index, 0, len(path_us) - 1))
    pu = float(path_us[i])
    idx = min(len(path) - 1, max(0, int(round(pu * (len(path) - 1)))))
    ws_i, we_i, bb_i = path[idx]
    n_path = max(len(path_us) - 1, 1)
    bounds = _ch3_lik_story_view_bounds(pack)
    return ch3_frame_lik_weight3d_measurements(
        pack["study"], pack["exam"], pack["y"], ws_i, we_i, bb_i,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        path_xyz=path, path_colors=pack["path_colors"], path_u=pu,
        notation_condensed=True,
        measurements=_ch3_lik_we_are_here_at(pack, ws_i, we_i, bb_i),
        ax3d_bounds=bounds,
        wide_bounds=bounds,
        show_ball_vectors=show_ball_vectors,
        ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
        cam_azim_u=float(i) / float(n_path),
        gd_formulas=gd_formulas,
        **frame_kw,
    )


def ch4_preview_likelihood_morph_surface_frame(
    *,
    log_u=0.0,
    nll_u=0.0,
    heatmap_u=0.0,
    squish_u=0.0,
    plane_drop_u=0.0,
):
    """Plot panel only — likelihood / log / NLL morph stages (ch4_03 surface checks)."""
    state = _ch3_lik86_terminal_state()
    return ch3_frame_lik_w12_single_surface(
        state,
        log_u=float(log_u),
        nll_u=float(nll_u),
        heatmap_u=float(heatmap_u),
        squish_u=float(squish_u),
        plane_drop_u=float(plane_drop_u),
        show_axis_labels=True,
        knob_labeled_blend=(1.0, 1.0, 1.0),
    )


def ch4_preview_likelihood_notation_nll_plane_frame(
    *,
    heatmap_u=1.0,
    squish_u=0.0,
    plane_drop_u=0.0,
):
    """Composed ch4_03 frame for heatmap / squish / plane-drop checks."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_3d_story,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    plot = ch3_frame_lik_w12_single_surface(
        state,
        log_u=1.0,
        nll_u=1.0,
        heatmap_u=float(heatmap_u),
        squish_u=float(squish_u),
        plane_drop_u=float(plane_drop_u),
        show_axis_labels=True,
        knob_labeled_blend=(1.0, 1.0, 1.0),
    )
    show_legend = float(heatmap_u) > 0.02
    return compose_tutorial(
        plot,
        right_blocks=ch4_nll_global_legend_blocks() if show_legend else [],
        bottom_blocks=ch4_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE if show_legend else None,
        right_title_single_line=True,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )


def ch4_preview_likelihood_notation_nll_last_frame():
    """Last ch4_03 frame for layout checks — does not build the full MP4 story."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_LIK_PLOT_START_RECT,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_notation_corner_blocks,
        ch4_formula_blocks_3d_story,
        compose_tutorial,
    )

    state = _ch3_lik86_terminal_state()
    plot = ch3_frame_lik_w12_single_surface(
        state,
        log_u=1.0,
        nll_u=1.0,
        heatmap_u=1.0,
        squish_u=1.0,
        plane_drop_u=1.0,
        show_axis_labels=True,
        knob_labeled_blend=(1.0, 1.0, 1.0),
    )
    return compose_tutorial(
        plot,
        right_blocks=ch4_nll_global_legend_blocks(),
        bottom_blocks=ch4_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
        layout_u=1.0,
        panel_u=1.0,
        write_progress=1.0,
        plot_start_rect=CH4_LIK_PLOT_START_RECT,
        theme="classic_light",
    )


def ch4_preview_likelihood_3d_measurements_frame(path_index=0):
    """Single ch4_04 frame for layout checks — does not build the full MP4 story."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    return _ch3_lik_pack_path_frame(pack, path_index, show_ball_vectors=False, gd_formulas=False)


def ch4_preview_likelihood_3d_ball_vectors_frame(*, intro=True, path_index=0, gd_start=False):
    """Single ch4_05 frame — intro zoom start, one path index, or GD opening pose."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    if gd_start:
        from ch4_layout import ch4_formula_blocks_3d_story
        return _ch3_lik_emit_05_formula_frame(
            pack,
            bottom_blocks=ch4_formula_blocks_3d_story(),
            bottom_prog={0: 1.0, 1: 1.0},
            at_gd_start=True,
        )
    if intro:
        path = pack["path"]
        ws0, we0, bb0 = path[0]
        wide = _ch3_lik_story_view_bounds(pack)
        intro_ball = _ch3_lik_ball_vector_field_cached(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0, wide,
        )
        return ch3_frame_lik_weight3d_measurements(
            pack["study"], pack["exam"], pack["y"], ws0, we0, bb0,
            W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
            vmin=pack["vmin"], vmax=pack["vmax"],
            path_xyz=path, path_u=0.0,
            notation_condensed=True,
            measurements=_ch3_lik_we_are_here_at(pack, ws0, we0, bb0),
            ax3d_bounds=wide,
            wide_bounds=wide,
            show_ball_vectors=True,
            ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
            ball_field=intro_ball,
            azim=float(CH3_LIK_3D_CAM_AZIM0),
            gd_formulas=False,
        )
    return _ch3_lik_pack_path_frame(pack, path_index, show_ball_vectors=True, gd_formulas=False)


def ch4_preview_likelihood_3d_gd_frame(*, intro=True, gd_spec_index=0):
    """Single ch4_06 frame — first hold (all arrows) or one sequential-GD frame."""
    pack = _ch3_lik_3d_gd_pack()
    specs = _ch3_lik_gd_frame_specs(pack)
    idx = 0 if intro else int(np.clip(gd_spec_index, 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch4_preview_likelihood_3d_gd_combined_frame(*, split_hold=False, combined_hold=False, gd_spec_index=0):
    """Single ch4_07 frame — split hold, combined handoff (08 start), or arbitrary index."""
    pack = _ch3_lik_3d_gd_combined_pack()
    specs = _ch3_lik_gd_combined_frame_specs(pack)
    if combined_hold:
        idx = next(i for i, s in enumerate(specs) if str(s.get("arrow_mode")) == "combined")
    elif split_hold:
        idx = 0
    else:
        idx = int(np.clip(gd_spec_index, 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch3_build_frames_likelihood_3d_gd_combined_story():
    pack = _ch3_lik_3d_gd_combined_pack()
    frames = []
    _ch3_lik_append_gd_combined_frames(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_export_likelihood_3d_gd_combined():
    from ch4_export_pipeline import export_mp4_from_specs

    pack = _ch3_lik_3d_gd_combined_pack()
    return export_mp4_from_specs(
        pack,
        _ch3_lik_gd_combined_frame_specs(pack),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename="ch4_07_likelihood_3d_gd_combined.mp4",
        duration_ms=int(CH3_LIK_3D_MS_GD),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label="ch4_07",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


def _ch3_lik_05_end_path_index(pack):
    return max(len(pack["path_us"]) - 1, 0)


def _ch3_lik_append_05_gd_handoff(frames, pack):
    """Move ch4_05 terminal pose to ch4_06 opening (position, camera, no ball vectors)."""
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch3_lik_story_view_bounds(pack)
    cols = pack["path_colors"]
    n_path = max(len(path_us) - 1, 1)
    i_end = _ch3_lik_05_end_path_index(pack)
    pu_end = float(path_us[i_end])
    idx_end = min(len(path) - 1, max(0, int(round(pu_end * (len(path) - 1)))))
    ws_e, we_e, bb_e = path[idx_end]
    cam_u_end = float(i_end) / float(n_path)
    ws_t, we_t, bb_t = (float(CH3_LIK_3D_PATH_START[0]), float(CH3_LIK_3D_PATH_START[1]), float(CH3_LIK_3D_PATH_START[2]))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_05_HANDOFF, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws = ws_e + u * (ws_t - ws_e)
        we = we_e + u * (we_t - we_e)
        bb = bb_e + u * (bb_t - bb_e)
        cam_u = cam_u_end + u * (0.0 - cam_u_end)
        frames.append(
            ch3_frame_lik_weight3d_measurements(
                pack["study"], pack["exam"], pack["y"], ws, we, bb,
                W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
                vmin=pack["vmin"], vmax=pack["vmax"],
                path_xyz=path, path_colors=cols, path_u=pu_end,
                notation_condensed=True,
                measurements=_ch3_lik_we_are_here_at(pack, ws, we, bb),
                ax3d_bounds=bounds,
                wide_bounds=bounds,
                show_ball_vectors=float(u) < 0.35,
                ball_vmin=pack["ball_vmin"], ball_vmax=pack["ball_vmax"],
                cam_azim_u=cam_u,
                gd_formulas=False,
            )
        )


def ch3_build_frames_likelihood_3d_ball_vectors_story():
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    frames = _ch3_lik_ball_intro_frames(pack, gd_formulas=False)
    _ch3_lik_append_path_frames(frames, pack, show_ball_vectors=True, gd_formulas=False)
    _ch3_lik_append_05_gd_handoff(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch3_build_frames_likelihood_3d_gd_story():
    pack = _ch3_lik_3d_gd_pack()
    frames = []
    _ch3_lik_append_gd_frames(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_export_likelihood_notation_nll():
    frames = ch3_build_frames_likelihood_ch4_nll_story()
    fn = "ch4_03_likelihood_notation_nll.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_CH4_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_notation_nll_bridge():
    frames = ch3_build_frames_likelihood_ch4_nll_bridge()
    fn = "ch4_03b_likelihood_gd_landscape.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_CH4_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_notation_nll_part2():
    frames = ch3_build_frames_likelihood_ch4_nll_story_part2()
    fn = "ch4_03c_likelihood_nll_plane.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_CH4_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_measurements():
    frames = ch3_build_frames_likelihood_3d_measurements_story()
    fn = "ch4_04_likelihood_3d_measurements.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_3D_MS))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_ball_vectors():
    frames = ch3_build_frames_likelihood_3d_ball_vectors_story()
    fn = "ch4_05_likelihood_3d_ball_vectors.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_3D_MS_BALL))
    print("wrote", OUTPUT_DIR / fn)
    return OUTPUT_DIR / fn


def ch4_export_likelihood_3d_gd():
    from ch4_export_pipeline import export_mp4_from_specs

    pack = _ch3_lik_3d_gd_pack()
    return export_mp4_from_specs(
        pack,
        _ch3_lik_gd_frame_specs(pack),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename="ch4_06_likelihood_3d_gd.mp4",
        duration_ms=int(CH3_LIK_3D_MS_GD),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label="ch4_06",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


# --- ch4_05a: CT scan — NLL heatmap planes sweeping each 3-D axis ---

CH3_LIK_CT_N_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_CT_N_SWEEP = 8 if _CH3_DRAFT else max(48, _smooth_n(36))
CH3_LIK_CT_N_PIVOT = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_CT_MS = 100 if not _CH3_DRAFT else 120
CH3_LIK_CT_AXES = ("st", "el", "b")
# (from_axis, to_axis): pivot uses shared edge at end of ``from`` sweep → start of ``to``
CH3_LIK_CT_PIVOTS = (("st", "el"), ("el", "b"))


def _ch4_ct_axis_limits(axis, bounds):
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    if axis == "st":
        return float(dlo1), float(dhi1)
    if axis == "el":
        return float(dlo2), float(dhi2)
    return float(dlob), float(dhib)


def _ch4_ct_sweep_end_value(axis, bounds):
    _, hi = _ch4_ct_axis_limits(axis, bounds)
    return float(hi)


def _ch4_ct_sweep_start_value(axis, bounds):
    lo, _ = _ch4_ct_axis_limits(axis, bounds)
    return float(lo)


def _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b):
    return _ch3_nll_sum_on_flat_grid(
        study, exam, y,
        np.asarray(w1, dtype=np.float64).ravel(),
        np.asarray(w2, dtype=np.float64).ravel(),
        np.asarray(b, dtype=np.float64).ravel(),
    ).reshape(np.asarray(w1, dtype=np.float64).shape)


def _ch4_ct_sweep_mesh(axis, value, bounds, *, gn=None):
    """Axis-aligned slice mesh + NLL."""
    gn = int(CH3_LIK_CT_GRID if gn is None else gn)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    val = float(value)
    if axis == "st":
        g2 = np.linspace(dlo2, dhi2, gn, dtype=np.float64)
        gb = np.linspace(dlob, dhib, gn, dtype=np.float64)
        w2, b = np.meshgrid(g2, gb, indexing="ij")
        w1 = np.full_like(w2, val)
    elif axis == "el":
        g1 = np.linspace(dlo1, dhi1, gn, dtype=np.float64)
        gb = np.linspace(dlob, dhib, gn, dtype=np.float64)
        w1, b = np.meshgrid(g1, gb, indexing="ij")
        w2 = np.full_like(w1, val)
    else:
        g1 = np.linspace(dlo1, dhi1, gn, dtype=np.float64)
        g2 = np.linspace(dlo2, dhi2, gn, dtype=np.float64)
        w1, w2 = np.meshgrid(g1, g2, indexing="ij")
        b = np.full_like(w1, val)
    return w1, w2, b


def _ch4_ct_pivot_mesh(study, exam, y, from_axis, to_axis, bounds, *, theta_u, gn=None):
    """
    Rotate slice plane from end of ``from_axis`` sweep to start of ``to_axis`` sweep.
    Pivot is the shared edge between the two axis-aligned planes (90° rotation).
    NLL is sampled at each vertex of the rotated plane.
    """
    gn = int(CH3_LIK_CT_GRID if gn is None else gn)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    u = float(np.clip(float(theta_u), 0.0, 1.0))
    u = ch3_knob_smoothstep(u)
    ang = u * (np.pi / 2.0)

    if from_axis == "st" and to_axis == "el":
        # End: w_ST = hi. Start: w_EL = lo. Pivot edge: (hi, lo, b), axis ∥ b.
        hi = dhi1
        lo = dlo2
        w1s, w2s, bs = _ch4_ct_sweep_mesh("st", hi, bounds, gn=gn)
        dw = w2s - lo
        w1 = hi - np.sin(ang) * dw
        w2 = lo + np.cos(ang) * dw
        b = bs
    elif from_axis == "el" and to_axis == "b":
        # End: w_EL = hi. Start: b = lo. Pivot edge: (w_ST, hi, lo), axis ∥ w_ST.
        hi = dhi2
        lob = dlob
        w1, w2s, bs = _ch4_ct_sweep_mesh("el", hi, bounds, gn=gn)
        w2 = w2s
        db = bs - lob
        w2 = hi - np.sin(ang) * db
        b = lob + np.cos(ang) * db
    else:
        raise ValueError(f"unsupported CT pivot {from_axis!r} → {to_axis!r}")
    nll = _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b)
    return w1, w2, b, nll


def _ch4_ct_draw_mesh(ax3d, w1, w2, b, nll, vmin=None, vmax=None):
    _ch4_nll_heatmap_plot_surface(ax3d, w1, w2, b, nll)


def _ch3_lik_ct_scan_pack():
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    pack["bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    return pack


def _ch4_ct_right_blocks(*, pack=None):
    if pack is None:
        return []
    return ch4_nll_global_legend_blocks()


def ch3_frame_lik_ct_scan(
    pack, *,
    sweep_axis=None,
    plane_val=None,
    show_sweep_range=False,
    pivot_from=None,
    pivot_to=None,
    pivot_u=0.0,
    cam_azim_u=0.0,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_formula_blocks_3d_story,
        ch4_cached_notation_corner_blocks,
        ch4_knob_asset_pack,
        compose_tutorial,
    )

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    bounds = pack["bounds"]
    handoff = _ch4_lik_02_03_handoff_state()
    ws, we = handoff["w_st"], handoff["w_el"]
    if str(sweep_axis) == "b" and plane_val is not None:
        bb = float(plane_val)
    else:
        bb = float(CH3_LIK_CH4_PLANE_B)
    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, bb, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    ax3d.set_autoscale_on(False)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    ch4_lik_ct_view_init(ax3d, cam_azim_u=float(cam_azim_u))
    if pivot_from is not None and pivot_to is not None:
        w1, w2, b, nll = _ch4_ct_pivot_mesh(
            study, exam, y, str(pivot_from), str(pivot_to), bounds, theta_u=float(pivot_u),
        )
    else:
        w1, w2, b = _ch4_ct_sweep_mesh(str(sweep_axis), float(plane_val), bounds)
        nll = _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b)
    _ch4_ct_draw_mesh(ax3d, w1, w2, b, nll)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    return compose_tutorial(
        plot_img,
        right_blocks=_ch4_ct_right_blocks(pack=pack),
        bottom_blocks=ch4_cached_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        write_progress=1.0,
        theme="classic_light",
    )


def _ch4_ct_append_pivot(frames, pack, from_axis, to_axis):
    for tv in np.linspace(0.0, 1.0, CH3_LIK_CT_N_PIVOT, endpoint=True):
        frames.append(ch3_frame_lik_ct_scan(
            pack,
            pivot_from=str(from_axis),
            pivot_to=str(to_axis),
            pivot_u=float(tv),
        ))


def ch3_build_frames_likelihood_ct_scan_story():
    pack = _ch3_lik_ct_scan_pack()
    frames = []
    bounds = pack["bounds"]
    pivot_map = dict(CH3_LIK_CT_PIVOTS)
    for _ in range(CH3_LIK_CT_N_HOLD):
        frames.append(ch3_frame_lik_ct_scan(
            pack, sweep_axis="b", plane_val=float(CH3_LIK_CH4_PLANE_B), show_sweep_range=False,
        ))
    for i, axis in enumerate(CH3_LIK_CT_AXES):
        if i > 0:
            prev = CH3_LIK_CT_AXES[i - 1]
            nxt = pivot_map.get(prev)
            if nxt == axis:
                _ch4_ct_append_pivot(frames, pack, prev, axis)
        lo, hi = _ch4_ct_axis_limits(axis, bounds)
        for _ in range(CH3_LIK_CT_N_HOLD):
            frames.append(ch3_frame_lik_ct_scan(
                pack, sweep_axis=axis, plane_val=lo, show_sweep_range=True,
            ))
        for tv in np.linspace(0.0, 1.0, CH3_LIK_CT_N_SWEEP, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            val = lo + u * (hi - lo)
            frames.append(ch3_frame_lik_ct_scan(
                pack, sweep_axis=axis, plane_val=val, show_sweep_range=False,
            ))
        for _ in range(max(4, CH3_LIK_CT_N_HOLD // 3)):
            frames.append(ch3_frame_lik_ct_scan(
                pack, sweep_axis=axis, plane_val=hi, show_sweep_range=False,
            ))
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_ct_scan_frame(*, axis="st", plane_u=0.5, show_sweep_range=False, pivot_u=None):
    pack = _ch3_lik_ct_scan_pack()
    bounds = pack["bounds"]
    if pivot_u is not None:
        if axis == "el":
            return ch3_frame_lik_ct_scan(
                pack, pivot_from="st", pivot_to="el", pivot_u=float(pivot_u),
            )
        if axis == "b":
            return ch3_frame_lik_ct_scan(
                pack, pivot_from="el", pivot_to="b", pivot_u=float(pivot_u),
            )
    lo, hi = _ch4_ct_axis_limits(axis, bounds)
    val = lo + float(plane_u) * (hi - lo)
    return ch3_frame_lik_ct_scan(
        pack, sweep_axis=axis, plane_val=val, show_sweep_range=show_sweep_range,
    )


def ch4_export_likelihood_ct_scan():
    frames = ch3_build_frames_likelihood_ct_scan_story()
    fn = "ch4_05a_likelihood_3d_ct_scan.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_CT_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a2: diagonal voxel fill — checkerboard cubes swept along (−3,3,−3)→(3,−3,3) ---

CH3_LIK_VOXEL_GRID = CH3_LIK_CT_GRID
CH3_LIK_VOXEL_CELL_MULT = 1
CH3_LIK_VOXEL_N_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_VOXEL_N_SWEEP = 8 if _CH3_DRAFT else max(64, _smooth_n(48))
CH3_LIK_VOXEL_MS = 100 if not _CH3_DRAFT else 120
CH3_LIK_VOXEL_ALPHA = 0.92
# Diagonal sweep: (−4, 4, −4) → (4, −4, 4) in (w_ST, w_EL, b).
CH3_LIK_VOXEL_DIAG_START = (-4.0, 4.0, -4.0)
CH3_LIK_VOXEL_DIAG_END = (4.0, -4.0, 4.0)


def _ch4_voxel_n_cells(cell_mult=None):
    """Voxel count per axis; ``cell_mult=4`` → cubes ¼ the edge length of ``cell_mult=1``."""
    mult = int(CH3_LIK_VOXEL_CELL_MULT if cell_mult is None else cell_mult)
    return max((int(CH3_LIK_CT_GRID) - 1) * mult, 2)


def _ch4_voxel_fill_cache_key(cell_mult, gap_pitch=1):
    gap_pitch = int(gap_pitch)
    if gap_pitch <= 1:
        return f"voxel_fill_cache_x{int(cell_mult)}"
    return f"voxel_fill_cache_x{int(cell_mult)}_g{gap_pitch}"


def _ch4_voxel_checker_mask(n, *, gap_pitch=1):
    """Every-other cells; ``gap_pitch=3`` → 3× center spacing (2× face gap vs pitch 1)."""
    ii, jj, kk = np.indices((int(n), int(n), int(n)))
    if int(gap_pitch) <= 1:
        return ((ii + jj + kk) % 2) == 0
    p = int(gap_pitch)
    mid = p // 2
    bi, bj, bk = ii // p, jj // p, kk // p
    at_block_center = (ii % p == mid) & (jj % p == mid) & (kk % p == mid)
    return at_block_center & ((bi + bj + bk) % 2 == 0)


def _ch4_voxel_cell_edges(lo, hi, n_cells):
    """``n_cells`` voxels along [lo, hi] — same spacing as 05a heatmap grid lines."""
    n_cells = int(n_cells)
    return np.linspace(float(lo), float(hi), n_cells + 1, dtype=np.float64)


def _ch4_voxel_fill_warm_pack(pack, *, cell_mult=None, bounds=None):
    """Precompute NLL + colors on a 3-D checkerboard cell grid (once per export)."""
    if cell_mult is None:
        cell_mult = int(pack.get("voxel_cell_mult", CH3_LIK_VOXEL_CELL_MULT))
    else:
        cell_mult = int(cell_mult)
    gap_pitch = int(pack.get("voxel_gap_pitch", 1))
    cache_key = _ch4_voxel_fill_cache_key(cell_mult, gap_pitch)
    if bounds is not None:
        cache_key = f"{cache_key}_b{hash(tuple(float(v) for v in bounds))}"
    cache = pack.get(cache_key)
    if cache is not None:
        return cache

    from ch4_layout import ch4_nll_heatmap_cmap

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = (
        bounds if bounds is not None else CH3_LIK_CT_VIEW_BOUNDS
    )
    n = _ch4_voxel_n_cells(cell_mult)
    x_edges = _ch4_voxel_cell_edges(dlo1, dhi1, n)
    y_edges = _ch4_voxel_cell_edges(dlo2, dhi2, n)
    z_edges = _ch4_voxel_cell_edges(dlob, dhib, n)
    w1 = 0.5 * (x_edges[:-1] + x_edges[1:])
    w2 = 0.5 * (y_edges[:-1] + y_edges[1:])
    bb = 0.5 * (z_edges[:-1] + z_edges[1:])
    W1, W2, B = np.meshgrid(w1, w2, bb, indexing="ij")
    nll = _ch3_nll_sum_on_flat_grid(
        study, exam, y,
        W1.ravel(), W2.ravel(), B.ravel(),
    ).reshape(W1.shape)
    lo, hi = ch4_nll_global_scale()
    cmap = ch4_nll_heatmap_cmap()
    span = max(float(hi) - float(lo), 1e-9)
    normed = np.clip((nll - float(lo)) / span, 0.0, 1.0)
    rgba = cmap(normed)
    rgba[..., 3] = float(CH3_LIK_VOXEL_ALPHA)
    ii, jj, kk = np.indices((n, n, n))
    checker = _ch4_voxel_checker_mask(n, gap_pitch=gap_pitch)
    start = np.array(CH3_LIK_VOXEL_DIAG_START, dtype=np.float64)
    end = np.array(CH3_LIK_VOXEL_DIAG_END, dtype=np.float64)
    diag = end - start
    max_proj = float(np.dot(diag, diag))
    proj = (W1 - start[0]) * diag[0] + (W2 - start[1]) * diag[1] + (B - start[2]) * diag[2]
    cache = {
        "n": n,
        "x_edges": x_edges,
        "y_edges": y_edges,
        "z_edges": z_edges,
        "checker": checker,
        "proj": proj,
        "max_proj": max_proj,
        "diag_start": start,
        "diag": diag,
        "rgba": rgba,
        "bounds": (dlo1, dhi1, dlo2, dhi2, dlob, dhib),
        "cell_mult": cell_mult,
        "gap_pitch": gap_pitch,
        "nll_lo": float(lo),
        "nll_hi": float(hi),
        "right_blocks": ch4_nll_global_legend_blocks(),
    }
    pack[cache_key] = cache
    return cache


def _ch3_lik_voxel_fill_pack(*, cell_mult=1, gap_pitch=1):
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    pack["bounds"] = CH3_LIK_CT_VIEW_BOUNDS
    pack["voxel_cell_mult"] = int(cell_mult)
    pack["voxel_gap_pitch"] = int(gap_pitch)
    _ch4_voxel_fill_warm_pack(pack, cell_mult=cell_mult)
    return pack


def _ch4_voxel_fill_cut_plane_mesh(cache, sweep_u, *, gn=None):
    """Cutting plane orthogonal to the voxel diagonal sweep (same mesh as CT slices)."""
    gn = int(CH3_LIK_CT_GRID if gn is None else gn)
    start = cache["diag_start"]
    diag = cache["diag"]
    max_proj = float(cache["max_proj"])
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = cache["bounds"]
    u = float(np.clip(float(sweep_u), 0.0, 1.0))
    u = ch3_knob_smoothstep(u)
    target = u * max_proj
    g1 = np.linspace(dlo1, dhi1, gn, dtype=np.float64)
    g2 = np.linspace(dlo2, dhi2, gn, dtype=np.float64)
    w1, w2 = np.meshgrid(g1, g2, indexing="ij")
    b = (
        target
        - (w1 - start[0]) * diag[0]
        - (w2 - start[1]) * diag[1]
    ) / float(diag[2]) + start[2]
    return w1, w2, b


def _ch4_voxel_fill_draw_cut_plane(ax3d, cache, sweep_u):
    w1, w2, b = _ch4_voxel_fill_cut_plane_mesh(cache, sweep_u)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = cache["bounds"]
    mask = (
        (b >= dlob - 1e-9) & (b <= dhib + 1e-9)
        & (w1 >= dlo1 - 1e-9) & (w1 <= dhi1 + 1e-9)
        & (w2 >= dlo2 - 1e-9) & (w2 <= dhi2 + 1e-9)
    )
    if not np.any(mask):
        return
    face = np.zeros(w1.shape + (4,), dtype=np.float64)
    face[..., :] = (0.85, 0.88, 0.92, float(CH3_LIK_CT_PLANE_ALPHA))
    face[~mask] = (0, 0, 0, 0)
    ax3d.plot_surface(
        w1, w2, b,
        facecolors=face,
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=False,
        shade=False,
        zorder=5,
    )


def _ch4_voxel_fill_draw_voxels(ax3d, cache, sweep_u):
    u = float(np.clip(float(sweep_u), 0.0, 1.0))
    u = ch3_knob_smoothstep(u)
    front = u * float(cache["max_proj"])
    visible = (cache["proj"] <= front + 1e-9) & cache["checker"]
    if not np.any(visible):
        return
    x_edges = cache["x_edges"]
    y_edges = cache["y_edges"]
    z_edges = cache["z_edges"]
    dx = float(x_edges[1] - x_edges[0])
    dy = float(y_edges[1] - y_edges[0])
    dz = float(z_edges[1] - z_edges[0])
    idx = np.argwhere(visible)
    xs = x_edges[idx[:, 0]]
    ys = y_edges[idx[:, 1]]
    zs = z_edges[idx[:, 2]]
    colors = cache["rgba"][visible]
    ax3d.bar3d(
        xs, ys, zs, dx, dy, dz,
        color=colors,
        shade=False,
        linewidth=0.0,
        edgecolor=(0.0, 0.0, 0.0, 0.0),
        zorder=6,
    )


def _ch4_voxel_draw_all_checker(ax3d, cache):
    """Draw every checkerboard voxel in ``cache`` (full cube fill)."""
    visible = np.asarray(cache["checker"], dtype=bool)
    if not np.any(visible):
        return
    x_edges = cache["x_edges"]
    y_edges = cache["y_edges"]
    z_edges = cache["z_edges"]
    dx = float(x_edges[1] - x_edges[0])
    dy = float(y_edges[1] - y_edges[0])
    dz = float(z_edges[1] - z_edges[0])
    idx = np.argwhere(visible)
    xs = x_edges[idx[:, 0]]
    ys = y_edges[idx[:, 1]]
    zs = z_edges[idx[:, 2]]
    colors = cache["rgba"][visible]
    ax3d.bar3d(
        xs, ys, zs, dx, dy, dz,
        color=colors,
        shade=False,
        linewidth=0.0,
        edgecolor=(0.0, 0.0, 0.0, 0.0),
        zorder=6,
    )


def _ch4_voxel_draw_checker_cube(ax3d, cache, half):
    """Draw checker voxels inside axis-aligned cube ``|w_i| <= half``."""
    half = float(half)
    x_edges = cache["x_edges"]
    y_edges = cache["y_edges"]
    z_edges = cache["z_edges"]
    w1 = 0.5 * (x_edges[:-1] + x_edges[1:])
    w2 = 0.5 * (y_edges[:-1] + y_edges[1:])
    bb = 0.5 * (z_edges[:-1] + z_edges[1:])
    W1, W2, B = np.meshgrid(w1, w2, bb, indexing="ij")
    in_cube = (
        (np.abs(W1) <= half + 1e-9)
        & (np.abs(W2) <= half + 1e-9)
        & (np.abs(B) <= half + 1e-9)
    )
    visible = np.asarray(cache["checker"], dtype=bool) & in_cube
    if not np.any(visible):
        return
    dx = float(x_edges[1] - x_edges[0])
    dy = float(y_edges[1] - y_edges[0])
    dz = float(z_edges[1] - z_edges[0])
    idx = np.argwhere(visible)
    xs = x_edges[idx[:, 0]]
    ys = y_edges[idx[:, 1]]
    zs = z_edges[idx[:, 2]]
    colors = cache["rgba"][visible]
    ax3d.bar3d(
        xs, ys, zs, dx, dy, dz,
        color=colors,
        shade=False,
        linewidth=0.0,
        edgecolor=(0.0, 0.0, 0.0, 0.0),
        zorder=6,
    )


def ch3_frame_lik_voxel_fill(pack, *, sweep_u=0.0, show_cut_plane=True, cam_azim_u=0.0, cam_spin_deg=0.0, view_bounds=None):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_formula_blocks_3d_story,
        ch4_cached_notation_corner_blocks,
        ch4_knob_asset_pack,
        compose_tutorial,
    )

    cache = _ch4_voxel_fill_warm_pack(pack)
    bounds = cache["bounds"]
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    handoff = _ch4_lik_02_03_handoff_state()
    ws, we, bb = handoff["w_st"], handoff["w_el"], float(CH3_LIK_CH4_PLANE_B)
    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, bb, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = (
        view_bounds if view_bounds is not None else bounds
    )
    ax3d.set_autoscale_on(False)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    ch4_lik_ct_view_init(ax3d, cam_azim_u=float(cam_azim_u), cam_spin_deg=float(cam_spin_deg))
    _ch4_voxel_fill_draw_voxels(ax3d, cache, sweep_u)
    if show_cut_plane and float(sweep_u) < 1.0 - 1e-6:
        _ch4_voxel_fill_draw_cut_plane(ax3d, cache, sweep_u)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    return compose_tutorial(
        plot_img,
        right_blocks=cache.get("right_blocks", []),
        bottom_blocks=ch4_cached_formula_blocks_3d_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        write_progress=1.0,
        theme="classic_light",
    )


def ch3_build_frames_likelihood_voxel_fill_story():
    pack = _ch3_lik_voxel_fill_pack()
    frames = []
    for _ in range(CH3_LIK_VOXEL_N_HOLD):
        frames.append(ch3_frame_lik_voxel_fill(pack, sweep_u=0.0))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_VOXEL_N_SWEEP, endpoint=True):
        frames.append(ch3_frame_lik_voxel_fill(pack, sweep_u=float(tv)))
    for _ in range(max(6, CH3_LIK_VOXEL_N_HOLD // 2)):
        frames.append(ch3_frame_lik_voxel_fill(pack, sweep_u=1.0, show_cut_plane=False))
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_voxel_fill_frame(*, sweep_u=0.45):
    pack = _ch3_lik_voxel_fill_pack()
    return ch3_frame_lik_voxel_fill(pack, sweep_u=float(sweep_u))


def ch4_export_likelihood_voxel_fill():
    frames = ch3_build_frames_likelihood_voxel_fill_story()
    fn = "ch4_05a2_diagonal_voxel_fill.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a3: fine diagonal voxel fill (¼ edge length → 4× cells per axis) ---

CH3_LIK_VOXEL_FINE_CELL_MULT = 4
CH3_LIK_VOXEL_FINE_GAP_PITCH = 3  # 2× face gap vs default checker (gap = 2·dx, cube size unchanged)
CH3_LIK_VOXEL_FINE_MS = 100 if not _CH3_DRAFT else 120
CH3_LIK_VOXEL_FINE_N_SPIN = 8 if _CH3_DRAFT else max(24, _smooth_n(18))


def _ch3_lik_voxel_fill_fine_pack():
    return _ch3_lik_voxel_fill_pack(
        cell_mult=CH3_LIK_VOXEL_FINE_CELL_MULT,
        gap_pitch=CH3_LIK_VOXEL_FINE_GAP_PITCH,
    )


def ch3_frame_lik_voxel_fill_fine(pack, *, sweep_u=0.0, show_cut_plane=True, cam_azim_u=0.0, cam_spin_deg=0.0, view_bounds=None):
    return ch3_frame_lik_voxel_fill(
        pack,
        sweep_u=sweep_u,
        show_cut_plane=show_cut_plane,
        cam_azim_u=cam_azim_u,
        cam_spin_deg=cam_spin_deg,
        view_bounds=view_bounds,
    )


def ch3_build_frames_likelihood_voxel_fill_fine_story():
    pack = _ch3_lik_voxel_fill_fine_pack()
    frames = []
    for _ in range(CH3_LIK_VOXEL_N_HOLD):
        frames.append(ch3_frame_lik_voxel_fill_fine(pack, sweep_u=0.0))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_VOXEL_N_SWEEP, endpoint=True):
        frames.append(ch3_frame_lik_voxel_fill_fine(pack, sweep_u=float(tv)))
    for _ in range(max(6, CH3_LIK_VOXEL_N_HOLD // 2)):
        frames.append(ch3_frame_lik_voxel_fill_fine(pack, sweep_u=1.0, show_cut_plane=False))
    spin_n = max(int(CH3_LIK_VOXEL_FINE_N_SPIN), 2)
    for tv in np.linspace(0.0, 1.0, spin_n, endpoint=True):
        frames.append(ch3_frame_lik_voxel_fill_fine(
            pack,
            sweep_u=1.0,
            show_cut_plane=False,
            cam_azim_u=float(tv),
            cam_spin_deg=360.0,
        ))
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_voxel_fill_fine_frame(*, sweep_u=0.45):
    pack = _ch3_lik_voxel_fill_fine_pack()
    return ch3_frame_lik_voxel_fill_fine(pack, sweep_u=float(sweep_u))


def ch4_export_likelihood_voxel_fill_fine():
    frames = ch3_build_frames_likelihood_voxel_fill_fine_story()
    fn = "ch4_05a3_diagonal_voxel_fill_fine.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_VOXEL_FINE_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a3b: from 05a3 end pose, zoom 3D axes out to ±21 (voxels stay in ±3 cube) ---

CH3_LIK_VOXEL_ZOOM_OUT_WIDE_BOUNDS = (-21.0, 21.0, -21.0, 21.0, -21.0, 21.0)
CH3_LIK_VOXEL_ZOOM_OUT_N = 8 if _CH3_DRAFT else max(48, _smooth_n(36))
CH3_LIK_VOXEL_ZOOM_OUT_N_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_VOXEL_ZOOM_OUT_MS = 80 if not _CH3_DRAFT else 100


def _ch4_voxel_zoom_out_bounds(u):
    """Interpolate 3D axis limits only — voxel data stays on the ±3 grid."""
    return _ch3_lik_lerp_bounds(
        CH3_LIK_CT_VIEW_BOUNDS,
        CH3_LIK_VOXEL_ZOOM_OUT_WIDE_BOUNDS,
        float(u),
    )


def _ch4_voxel_fill_fine_end_pose():
    """Camera + sweep state at the end of ch4_05a3."""
    return dict(
        sweep_u=1.0,
        show_cut_plane=False,
        cam_azim_u=1.0,
        cam_spin_deg=360.0,
    )


def ch3_build_frames_likelihood_voxel_fill_fine_zoom_out_story():
    pack = _ch3_lik_voxel_fill_fine_pack()
    end_pose = _ch4_voxel_fill_fine_end_pose()
    frames = []
    for _ in range(max(4, CH3_LIK_VOXEL_N_HOLD // 3)):
        frames.append(ch3_frame_lik_voxel_fill_fine(
            pack,
            view_bounds=CH3_LIK_CT_VIEW_BOUNDS,
            **end_pose,
        ))
    for tv in np.linspace(0.0, 1.0, CH3_LIK_VOXEL_ZOOM_OUT_N, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch3_frame_lik_voxel_fill_fine(
            pack,
            view_bounds=_ch4_voxel_zoom_out_bounds(u),
            **end_pose,
        ))
    wide = CH3_LIK_VOXEL_ZOOM_OUT_WIDE_BOUNDS
    for _ in range(CH3_LIK_VOXEL_ZOOM_OUT_N_HOLD):
        frames.append(ch3_frame_lik_voxel_fill_fine(
            pack,
            view_bounds=wide,
            **end_pose,
        ))
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_voxel_fill_fine_zoom_out_frame(*, zoom_u=1.0):
    pack = _ch3_lik_voxel_fill_fine_pack()
    return ch3_frame_lik_voxel_fill_fine(
        pack,
        view_bounds=_ch4_voxel_zoom_out_bounds(float(zoom_u)),
        **_ch4_voxel_fill_fine_end_pose(),
    )


def ch4_export_likelihood_voxel_fill_fine_zoom_out():
    frames = ch3_build_frames_likelihood_voxel_fill_fine_zoom_out_story()
    fn = "ch4_05a3b_voxel_zoom_out.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_VOXEL_ZOOM_OUT_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a4: from 05a3 full fill at origin → shrink to ball cube → path to (−3, −3, −3) ---

CH3_LIK_BALL_VOXEL_N_A3_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_BALL_VOXEL_N_CUBE_SHRINK = 8 if _CH3_DRAFT else 44
CH3_LIK_BALL_VOXEL_N_POINT_SHRINK = 4 if _CH3_DRAFT else 14
CH3_LIK_BALL_VOXEL_POINT_S_LARGE = 160.0
CH3_LIK_BALL_VOXEL_POINT_S_SMALL = 14.0
CH3_LIK_BALL_VOXEL_POINT_COLOR = "#111111"
CH3_LIK_BALL_VOXEL_MS = int(CH3_LIK_3D_MS_BALL)
CH3_LIK_BALL_VOXEL_HALF_SCALE = 0.5
CH3_LIK_BALL_VOXEL_N_ROT_AFTER = 8 if _CH3_DRAFT else 40
CH3_LIK_BALL_VOXEL_A3_CAM_AZIM_U = 1.0
CH3_LIK_BALL_VOXEL_A3_CAM_SPIN_DEG = 360.0
# (w_ST, w_EL, b) — from origin through mid corner, stop at (−3, −3, −3)
CH3_LIK_BALL_VOXEL_PATH_WAYPOINTS = (
    (0.0, 0.0, 0.0),
    (3.0, 0.0, 3.0),
    (-3.0, -3.0, -3.0),
)
CH3_LIK_BALL_VOXEL_N_PATH_PER_SEG = 8 if _CH3_DRAFT else 40


def _ch4_ball_voxel_path_points(n_pts_per_seg=None):
    """Piecewise-linear path with exact waypoint hits at segment junctions."""
    n = int(CH3_LIK_BALL_VOXEL_N_PATH_PER_SEG if n_pts_per_seg is None else n_pts_per_seg)
    waypoints = np.asarray(CH3_LIK_BALL_VOXEL_PATH_WAYPOINTS, dtype=float)
    parts = []
    for i in range(len(waypoints) - 1):
        seg = np.linspace(waypoints[i], waypoints[i + 1], n, endpoint=True)
        if i > 0:
            seg = seg[1:]
        parts.append(seg)
    path = np.vstack(parts).astype(float)
    waypoint_path_indices = [0]
    offset = 0
    for part in parts:
        offset += len(part)
        waypoint_path_indices.append(offset - 1)
    return path, np.asarray(waypoint_path_indices, dtype=int)


def _ch4_ball_voxel_path_us():
    """Equal animation time on each of the three straight segments."""
    n_seg = len(CH3_LIK_BALL_VOXEL_PATH_WAYPOINTS) - 1
    n_frames = max(CH3_LIK_3D_N_PATH // n_seg, 8 if _CH3_DRAFT else 20)
    us = []
    for seg_i in range(n_seg):
        for tv in np.linspace(0.0, 1.0, n_frames, endpoint=True):
            us.append((seg_i + float(tv)) / float(n_seg))
    return us


def _ch4_ball_voxel_path_index(pack, path_u):
    """Map global path_u in [0, 1] to a path sample index (waypoints at u=0, 1/3, 2/3, 1)."""
    path_u = float(np.clip(float(path_u), 0.0, 1.0))
    wp_idx = np.asarray(pack["path_waypoint_indices"], dtype=int)
    n_seg = len(wp_idx) - 1
    seg_f = path_u * float(n_seg)
    seg_i = min(int(np.floor(seg_f)), n_seg - 1)
    t = seg_f - float(seg_i)
    i0 = int(wp_idx[seg_i])
    i1 = int(wp_idx[seg_i + 1])
    return int(round(i0 + t * float(i1 - i0)))


def _ch4_ball_voxel_measurements_pack():
    """Like ``_ch3_lik_3d_measurements_pack`` but with the 05a4 corner-to-corner path."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=False)
    bounds = _ch4_ball_voxel_view_bounds()
    path, waypoint_path_indices = _ch4_ball_voxel_path_points()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    ball_vmin = float(global_cache["nll_lo"])
    ball_vmax = float(global_cache["nll_hi"])
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    path_nll = np.array([
        float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
        for r in path
    ], dtype=float)
    span = max(float(ball_vmax) - float(ball_vmin), 1e-9)
    from ch4_layout import ch4_nll_heatmap_cmap

    nll_cmap = ch4_nll_heatmap_cmap()
    pack["path"] = path
    pack["path_waypoint_indices"] = waypoint_path_indices
    pack["path_us"] = _ch4_ball_voxel_path_us()
    pack["path_colors"] = [
        nll_cmap(float(np.clip((v - ball_vmin) / span, 0.0, 1.0)))
        for v in path_nll
    ]
    pack["ball_vmin"] = ball_vmin
    pack["ball_vmax"] = ball_vmax
    pack["ax3d_bounds"] = bounds
    pack["gd_ax3d_bounds"] = bounds
    pack.pop("ball_voxel_ready", None)
    return pack


def _ch4_ball_voxel_view_bounds():
    """Same ±3 cube as ch4_05a3."""
    return CH3_LIK_CT_VIEW_BOUNDS


def _ch4_ball_voxel_global_cache(pack):
    """05a3 fine grid: same bounds, cell size, gap pitch, and NLL colors."""
    cache = pack.get("ball_voxel_global_cache")
    if cache is not None:
        return cache
    cell_mult = int(CH3_LIK_VOXEL_FINE_CELL_MULT)
    gap_pitch = int(CH3_LIK_VOXEL_FINE_GAP_PITCH)
    pack["voxel_cell_mult"] = cell_mult
    pack["voxel_gap_pitch"] = gap_pitch
    cache = _ch4_voxel_fill_warm_pack(pack, cell_mult=cell_mult)
    pack["ball_voxel_global_cache"] = cache
    return cache


def _ch4_ball_voxel_half():
    """Ball framing half-extent using the 05a3 view span."""
    bounds = _ch4_ball_voxel_view_bounds()
    span_ref = float(np.max(_ch3_lik_bounds_span(bounds)))
    r = float(CH3_LIK_3D_BALL_R_SCALE) * span_ref
    return max(r * 2.35, span_ref * 0.055) * float(CH3_LIK_BALL_VOXEL_HALF_SCALE)


def _ch4_ball_voxel_build_cube(global_cache, center, half):
    """Ball-sized subset of the 05a3 fine checkerboard grid."""
    center = np.asarray(center, dtype=np.float64)
    half = float(half)
    x_edges = global_cache["x_edges"]
    y_edges = global_cache["y_edges"]
    z_edges = global_cache["z_edges"]
    w1 = 0.5 * (x_edges[:-1] + x_edges[1:])
    w2 = 0.5 * (y_edges[:-1] + y_edges[1:])
    bb = 0.5 * (z_edges[:-1] + z_edges[1:])
    W1, W2, B = np.meshgrid(w1, w2, bb, indexing="ij")
    in_ball = (
        (np.abs(W1 - center[0]) <= half)
        & (np.abs(W2 - center[1]) <= half)
        & (np.abs(B - center[2]) <= half)
    )
    checker = global_cache["checker"] & in_ball
    ci = int(np.argmin(np.abs(w1 - center[0])))
    cj = int(np.argmin(np.abs(w2 - center[1])))
    ck = int(np.argmin(np.abs(bb - center[2])))
    n = int(global_cache["n"])
    ii, jj, kk = np.indices((n, n, n))
    cheb = np.maximum(np.maximum(np.abs(ii - ci), np.abs(jj - cj)), np.abs(kk - ck))
    max_cheb = int(cheb[checker].max()) if np.any(checker) else 0
    return {
        "x_edges": x_edges,
        "y_edges": y_edges,
        "z_edges": z_edges,
        "checker": checker,
        "cheb": cheb,
        "max_cheb": max_cheb,
        "rgba": global_cache["rgba"],
        "dx": float(x_edges[1] - x_edges[0]),
        "dy": float(y_edges[1] - y_edges[0]),
        "dz": float(z_edges[1] - z_edges[0]),
    }


def _ch4_ball_voxel_sweep_complete_draw(global_cache, *, alpha_scale=1.0):
    """Full diagonal voxel fill — same visible set as ch4_05a3 at sweep_u=1."""
    visible = (
        (global_cache["proj"] <= float(global_cache["max_proj"]) + 1e-9)
        & np.asarray(global_cache["checker"], dtype=bool)
    )
    if not np.any(visible):
        return None
    x_edges = global_cache["x_edges"]
    y_edges = global_cache["y_edges"]
    z_edges = global_cache["z_edges"]
    idx = np.argwhere(visible)
    xs = x_edges[idx[:, 0]]
    ys = y_edges[idx[:, 1]]
    zs = z_edges[idx[:, 2]]
    colors = global_cache["rgba"][visible].copy()
    if float(alpha_scale) < 1.0 - 1e-6:
        colors[..., 3] *= float(alpha_scale)
    return {
        "xs": xs,
        "ys": ys,
        "zs": zs,
        "dx": float(x_edges[1] - x_edges[0]),
        "dy": float(y_edges[1] - y_edges[0]),
        "dz": float(z_edges[1] - z_edges[0]),
        "colors": colors,
    }


def _ch4_ball_voxel_intro_half_start(center):
    """Starting half-extent: axis-aligned cube that covers the ±3 view box at ``center``."""
    center = np.asarray(center, dtype=np.float64)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = _ch4_ball_voxel_view_bounds()
    hx = max(abs(float(dhi1) - center[0]), abs(float(dlo1) - center[0]))
    hy = max(abs(float(dhi2) - center[1]), abs(float(dlo2) - center[1]))
    hz = max(abs(float(dhib) - center[2]), abs(float(dlob) - center[2]))
    return float(max(hx, hy, hz))


def _ch4_ball_voxel_intro_shrink_draw(global_cache, center, half_start, half_end, shrink_u):
    """Shrink an axis-aligned voxel cube from ``half_start`` → ``half_end`` (no crossfade)."""
    u = float(np.clip(float(shrink_u), 0.0, 1.0))
    half = float(half_start) + u * (float(half_end) - float(half_start))
    cube = _ch4_ball_voxel_build_cube(global_cache, center, half)
    return _ch4_ball_voxel_cube_draw(cube, grow_u=1.0)


_CH4_BALL_VOXEL_INTRO_MEM_CACHE = {}


def _ch4_ball_voxel_intro_cache_key(center, target_half):
    c = np.asarray(center, dtype=np.float64)
    return (
        round(float(c[0]), 5),
        round(float(c[1]), 5),
        round(float(c[2]), 5),
        round(float(target_half), 8),
        int(CH3_LIK_BALL_VOXEL_N_A3_HOLD),
        int(CH3_LIK_BALL_VOXEL_N_CUBE_SHRINK),
        int(CH3_LIK_BALL_VOXEL_N_POINT_SHRINK),
        bool(_CH3_DRAFT),
        int(CH3_ANIM_DPI),
    )


def _ch4_ball_voxel_intro_cache_slug(key):
    cx, cy, cz, th, *_rest = key
    draft = "draft" if key[7] else "full"
    return f"cx{cx:.4f}_cy{cy:.4f}_cz{cz:.4f}_h{th:.6f}_{draft}_dpi{key[8]}"


def _ch4_ball_voxel_intro_cache_dir(key):
    return OUTPUT_DIR / "cache" / "ch4_voxel_intro" / _ch4_ball_voxel_intro_cache_slug(key)


def _ch4_ball_voxel_intro_cache_load(key):
    if os.environ.get("CH4_VOXEL_INTRO_CACHE_REBUILD", "").strip() in {"1", "true", "yes"}:
        return None
    d = _ch4_ball_voxel_intro_cache_dir(key)
    manifest = d / "manifest.txt"
    if not manifest.is_file():
        return None
    from PIL import Image

    n = int(manifest.read_text().strip())
    frames = []
    for i in range(n):
        frames.append(Image.open(d / f"frame_{i:04d}.png").convert("RGB"))
    return frames


def _ch4_ball_voxel_intro_cache_save(key, frames):
    from PIL import Image

    d = _ch4_ball_voxel_intro_cache_dir(key)
    d.mkdir(parents=True, exist_ok=True)
    for i, fr in enumerate(frames):
        if isinstance(fr, Image.Image):
            fr.save(d / f"frame_{i:04d}.png")
        else:
            Image.fromarray(fr).save(d / f"frame_{i:04d}.png")
    (d / "manifest.txt").write_text(str(len(frames)))


def _ch4_ball_voxel_cached_intro_frames(pack, target_half, *, use_cache=True):
    """Render (or load) the shared 05a3-hold → voxel shrink → point-shrink intro once per key."""
    path = pack["path"]
    center = np.asarray(path[0], dtype=np.float64)
    target_half = float(target_half)
    key = _ch4_ball_voxel_intro_cache_key(center, target_half)
    if use_cache and key in _CH4_BALL_VOXEL_INTRO_MEM_CACHE:
        return list(_CH4_BALL_VOXEL_INTRO_MEM_CACHE[key])
    if use_cache:
        disk = _ch4_ball_voxel_intro_cache_load(key)
        if disk is not None:
            _CH4_BALL_VOXEL_INTRO_MEM_CACHE[key] = disk
            print(
                "  loaded cached voxel intro:",
                _ch4_ball_voxel_intro_cache_slug(key),
                f"({len(disk)} frames)",
                flush=True,
            )
            return list(disk)
    frames = _ch3_lik_ball_voxel_intro_frames_build(
        pack, target_half=target_half, skip_prewarm=True,
    )
    if use_cache:
        _CH4_BALL_VOXEL_INTRO_MEM_CACHE[key] = frames
        _ch4_ball_voxel_intro_cache_save(key, frames)
        print(
            "  cached voxel intro:",
            _ch4_ball_voxel_intro_cache_slug(key),
            f"({len(frames)} frames)",
            flush=True,
        )
    return list(frames)


def _ch4_ball_voxel_cube_draw(
    cube, *, grow_u=1.0, alpha_scale=1.0, emphasize_cheb=None, dim_others=1.0,
):
    grow_u = float(np.clip(float(grow_u), 0.0, 1.0))
    max_cheb = float(cube["max_cheb"])
    lim = grow_u * max_cheb + 1e-9
    visible = cube["checker"] & (cube["cheb"] <= lim)
    if not np.any(visible):
        return None
    idx = np.argwhere(visible)
    xs = cube["x_edges"][idx[:, 0]]
    ys = cube["y_edges"][idx[:, 1]]
    zs = cube["z_edges"][idx[:, 2]]
    colors = cube["rgba"][visible].copy()
    if alpha_scale < 1.0 - 1e-6:
        colors[..., 3] *= float(alpha_scale)
    if emphasize_cheb is not None and float(dim_others) < 1.0 - 1e-6:
        emp = cube["cheb"][visible] == float(emphasize_cheb)
        colors[~emp, 3] *= float(dim_others)
    return {
        "xs": xs,
        "ys": ys,
        "zs": zs,
        "dx": cube["dx"],
        "dy": cube["dy"],
        "dz": cube["dz"],
        "colors": colors,
    }


def _ch4_ball_voxel_merge_draws(draws):
    draws = [d for d in draws if d is not None]
    if not draws:
        return None
    if len(draws) == 1:
        return draws[0]
    return {
        "xs": np.concatenate([d["xs"] for d in draws]),
        "ys": np.concatenate([d["ys"] for d in draws]),
        "zs": np.concatenate([d["zs"] for d in draws]),
        "dx": draws[0]["dx"],
        "dy": draws[0]["dy"],
        "dz": draws[0]["dz"],
        "colors": np.concatenate([d["colors"] for d in draws], axis=0),
    }


def _ch4_ball_voxel_prewarm_pack(pack):
    if pack.get("ball_voxel_ready"):
        return pack
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = _ch4_ball_voxel_half()
    path = np.asarray(pack["path"], dtype=float)
    start_cube = _ch4_ball_voxel_build_cube(global_cache, path[0], half)
    path_cubes = [
        _ch4_ball_voxel_build_cube(global_cache, r, half)
        for r in path
    ]
    pack["ball_voxel_half"] = half
    pack["ball_voxel_start_cube"] = start_cube
    pack["ball_voxel_path_cubes"] = path_cubes
    pack["ball_voxel_right"] = global_cache["right_blocks"]
    pack["ball_voxel_ready"] = True
    return pack


def _ch4_ball_voxel_path_draw(pack, path_index, *, grow_u=1.0, alpha_scale=1.0):
    cubes = pack["ball_voxel_path_cubes"]
    i = int(np.clip(int(path_index), 0, len(cubes) - 1))
    draws = [_ch4_ball_voxel_cube_draw(cubes[j], grow_u=1.0, alpha_scale=alpha_scale) for j in range(i + 1)]
    return _ch4_ball_voxel_merge_draws(draws)


def _ch4_ball_voxel_frame(
    pack, ws, we, bb, *,
    ax3d_bounds,
    wide_bounds,
    voxel_draw=None,
    point_s=None,
    cam_azim_u=0.0,
    cam_rot_deg=0.0,
    measurements=None,
    right_blocks=None,
    right_title=None,
    right_title_single_line=None,
):
    if right_blocks is not None:
        right = right_blocks
    elif measurements is not None:
        right = measurements
    else:
        right = pack.get("ball_voxel_right", [])
    return ch3_frame_lik_weight3d_measurements(
        pack["study"], pack["exam"], pack["y"], ws, we, bb,
        W1m=pack["W1m"], W2m=pack["W2m"], Bm=pack["Bm"], Lf=pack["Lf"],
        vmin=pack["vmin"], vmax=pack["vmax"],
        path_xyz=pack["path"],
        path_u=0.0,
        notation_condensed=True,
        measurements=right,
        ax3d_bounds=ax3d_bounds,
        wide_bounds=wide_bounds,
        show_ball_vectors=False,
        ball_vmin=pack["ball_vmin"],
        ball_vmax=pack["ball_vmax"],
        cam_azim_u=float(cam_azim_u),
        cam_rot_deg=float(cam_rot_deg),
        show_path_line=False,
        voxel_draw=voxel_draw,
        point_s=point_s,
        point_color=CH3_LIK_BALL_VOXEL_POINT_COLOR,
        write_progress=1.0,
        right_title=right_title,
        right_title_single_line=right_title_single_line,
    )


def _ch3_lik_ball_voxel_intro_frames_build(pack, *, skip_prewarm=False, intro_mode="a3_shrink", target_half=None):
    if not skip_prewarm:
        _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    center = np.asarray(path[0], dtype=np.float64)
    ws0, we0, bb0 = float(center[0]), float(center[1]), float(center[2])
    bounds = _ch4_ball_voxel_view_bounds()
    target_half = float(pack["ball_voxel_half"] if target_half is None else target_half)
    global_cache = _ch4_ball_voxel_global_cache(pack)
    end_cube = _ch4_ball_voxel_build_cube(global_cache, center, target_half)
    half_start = _ch4_ball_voxel_intro_half_start(center)
    rail = global_cache["right_blocks"]
    rail_title = CH4_NLL_HEATMAP_SECTION_TITLE
    frames = []

    if str(intro_mode) == "grow":
        start_cube = end_cube
        for _ in range(max(4, CH3_LIK_VOXEL_N_HOLD // 3)):
            frames.append(_ch4_ball_voxel_frame(
                pack, ws0, we0, bb0,
                ax3d_bounds=bounds, wide_bounds=bounds,
                voxel_draw=None,
                point_s=CH3_LIK_BALL_VOXEL_POINT_S_LARGE,
                right_blocks=rail,
                right_title=rail_title,
                right_title_single_line=True,
            ))
        for tv in np.linspace(0.0, 1.0, CH3_LIK_BALL_VOXEL_N_CUBE_SHRINK, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            frames.append(_ch4_ball_voxel_frame(
                pack, ws0, we0, bb0,
                ax3d_bounds=bounds, wide_bounds=bounds,
                voxel_draw=_ch4_ball_voxel_cube_draw(start_cube, grow_u=u),
                point_s=CH3_LIK_BALL_VOXEL_POINT_S_LARGE,
                right_blocks=rail,
                right_title=rail_title,
                right_title_single_line=True,
            ))
    else:
        cam_u = float(CH3_LIK_BALL_VOXEL_A3_CAM_AZIM_U)
        cam_spin = float(CH3_LIK_BALL_VOXEL_A3_CAM_SPIN_DEG)
        for _ in range(CH3_LIK_BALL_VOXEL_N_A3_HOLD):
            frames.append(_ch4_ball_voxel_frame(
                pack, ws0, we0, bb0,
                ax3d_bounds=bounds, wide_bounds=bounds,
                voxel_draw=_ch4_ball_voxel_sweep_complete_draw(global_cache),
                point_s=CH3_LIK_BALL_VOXEL_POINT_S_LARGE,
                cam_azim_u=cam_u,
                cam_rot_deg=cam_spin,
                right_blocks=rail,
                right_title=rail_title,
                right_title_single_line=True,
            ))
        for tv in np.linspace(0.0, 1.0, CH3_LIK_BALL_VOXEL_N_CUBE_SHRINK, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            frames.append(_ch4_ball_voxel_frame(
                pack, ws0, we0, bb0,
                ax3d_bounds=bounds, wide_bounds=bounds,
                voxel_draw=_ch4_ball_voxel_intro_shrink_draw(
                    global_cache, center, half_start, target_half, u,
                ),
                point_s=CH3_LIK_BALL_VOXEL_POINT_S_LARGE,
                cam_azim_u=cam_u,
                cam_rot_deg=cam_spin,
                right_blocks=rail,
                right_title=rail_title,
                right_title_single_line=True,
            ))

    for tv in np.linspace(0.0, 1.0, CH3_LIK_BALL_VOXEL_N_POINT_SHRINK, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ps = ch3_lerp(CH3_LIK_BALL_VOXEL_POINT_S_LARGE, CH3_LIK_BALL_VOXEL_POINT_S_SMALL, u)
        frame_kw = dict(
            ax3d_bounds=bounds, wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_cube_draw(end_cube, grow_u=1.0),
            point_s=ps,
            right_blocks=rail,
            right_title=rail_title,
            right_title_single_line=True,
        )
        if str(intro_mode) != "grow":
            frame_kw["cam_azim_u"] = float(CH3_LIK_BALL_VOXEL_A3_CAM_AZIM_U)
            frame_kw["cam_rot_deg"] = float(CH3_LIK_BALL_VOXEL_A3_CAM_SPIN_DEG)
        frames.append(_ch4_ball_voxel_frame(pack, ws0, we0, bb0, **frame_kw))

    return frames


def _ch3_lik_ball_voxel_intro_frames(pack, *, skip_prewarm=False, intro_mode="a3_shrink", target_half=None):
    if not skip_prewarm:
        _ch4_ball_voxel_prewarm_pack(pack)
    if target_half is None:
        target_half = float(pack["ball_voxel_half"])
    if str(intro_mode) == "a3_shrink":
        return _ch4_ball_voxel_cached_intro_frames(pack, target_half)
    return _ch3_lik_ball_voxel_intro_frames_build(
        pack, skip_prewarm=True, intro_mode=intro_mode, target_half=target_half,
    )


def _ch3_lik_append_ball_voxel_path_frames(frames, pack, *, rotate_during=True):
    _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch4_ball_voxel_view_bounds()
    n_path = max(len(path_us) - 1, 1)
    for i, pu in enumerate(path_us):
        pu = float(pu)
        idx = _ch4_ball_voxel_path_index(pack, pu)
        ws_i, we_i, bb_i = path[idx]
        cam_u = float(i) / float(n_path) if rotate_during else 0.0
        cam_rot = float(CH3_LIK_3D_CAM_PATH_ROT) if rotate_during else 0.0
        frames.append(_ch4_ball_voxel_frame(
            pack, float(ws_i), float(we_i), float(bb_i),
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_path_draw(pack, idx),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=cam_u,
            cam_rot_deg=cam_rot,
            measurements=_ch3_lik_we_are_here_at(pack, float(ws_i), float(we_i), float(bb_i)),
        ))


def _ch3_lik_append_ball_voxel_rot_after(frames, pack):
    """Single 360° pan after the path completes (05a4 rot-after variant)."""
    _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch4_ball_voxel_view_bounds()
    i_end = _ch4_ball_voxel_path_index(pack, float(path_us[-1]))
    ws_e, we_e, bb_e = path[i_end]
    for tv in np.linspace(0.0, 1.0, CH3_LIK_BALL_VOXEL_N_ROT_AFTER, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_ch4_ball_voxel_frame(
            pack, float(ws_e), float(we_e), float(bb_e),
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_path_draw(pack, i_end),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=u,
            cam_rot_deg=float(CH3_LIK_3D_CAM_PATH_ROT),
            measurements=_ch3_lik_we_are_here_at(pack, float(ws_e), float(we_e), float(bb_e)),
        ))


def _ch3_lik_append_05a4_handoff(frames, pack):
    path = pack["path"]
    path_us = pack["path_us"]
    bounds = _ch4_ball_voxel_view_bounds()
    n_path = max(len(path_us) - 1, 1)
    i_end = _ch3_lik_05_end_path_index(pack)
    pu_end = float(path_us[i_end])
    idx_end = _ch4_ball_voxel_path_index(pack, pu_end)
    ws_e, we_e, bb_e = path[idx_end]
    cam_u_end = float(i_end) / float(n_path)
    ws_t, we_t, bb_t = float(path[0, 0]), float(path[0, 1]), float(path[0, 2])
    for tv in np.linspace(0.0, 1.0, CH3_LIK_3D_N_05_HANDOFF, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws = ws_e + u * (ws_t - ws_e)
        we = we_e + u * (we_t - we_e)
        bb = bb_e + u * (bb_t - bb_e)
        cam_u = cam_u_end + u * (0.0 - cam_u_end)
        alpha = max(0.0, 1.0 - 1.15 * u)
        frames.append(_ch4_ball_voxel_frame(
            pack, ws, we, bb,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_path_draw(pack, idx_end, alpha_scale=alpha),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=cam_u,
            measurements=_ch3_lik_we_are_here_at(pack, ws, we, bb),
        ))


def ch3_build_frames_likelihood_ball_voxel_path_story(*, rotate_during=True):
    pack = _ch4_ball_voxel_measurements_pack()
    _ch4_ball_voxel_prewarm_pack(pack)
    frames = list(_ch4_ball_voxel_cached_intro_frames(pack, _ch4_ball_voxel_half()))
    _ch3_lik_append_ball_voxel_path_frames(frames, pack, rotate_during=rotate_during)
    if not rotate_during:
        _ch3_lik_append_ball_voxel_rot_after(frames, pack)
    _ch3_lik_append_05a4_handoff(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_ball_voxel_path_frame(*, intro_shrink_u=None, intro_grow_u=None, path_index=None, path_u=None):
    pack = _ch4_ball_voxel_measurements_pack()
    _ch4_ball_voxel_prewarm_pack(pack)
    path = pack["path"]
    ws0, we0, bb0 = float(path[0, 0]), float(path[0, 1]), float(path[0, 2])
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    center = np.asarray(path[0], dtype=np.float64)
    target_half = float(pack["ball_voxel_half"])
    half_start = _ch4_ball_voxel_intro_half_start(center)
    if intro_shrink_u is None and intro_grow_u is not None:
        intro_shrink_u = float(intro_grow_u)
    at_start = path_u is None and (path_index is None or int(path_index) <= 0)
    if intro_shrink_u is not None and at_start:
        su = float(intro_shrink_u)
        if su <= 1e-6:
            vdraw = _ch4_ball_voxel_sweep_complete_draw(global_cache)
            ps = CH3_LIK_BALL_VOXEL_POINT_S_LARGE
        elif su >= 1.0 - 1e-6:
            vdraw = _ch4_ball_voxel_cube_draw(
                _ch4_ball_voxel_build_cube(global_cache, center, target_half), grow_u=1.0,
            )
            ps = CH3_LIK_BALL_VOXEL_POINT_S_SMALL
        else:
            vdraw = _ch4_ball_voxel_intro_shrink_draw(
                global_cache, center, half_start, target_half, su,
            )
            ps = CH3_LIK_BALL_VOXEL_POINT_S_LARGE
        return _ch4_ball_voxel_frame(
            pack, ws0, we0, bb0,
            ax3d_bounds=bounds,
            wide_bounds=bounds,
            voxel_draw=vdraw,
            point_s=ps,
            cam_azim_u=float(CH3_LIK_BALL_VOXEL_A3_CAM_AZIM_U),
            cam_rot_deg=float(CH3_LIK_BALL_VOXEL_A3_CAM_SPIN_DEG),
        )
    if path_u is not None:
        idx = _ch4_ball_voxel_path_index(pack, float(path_u))
    else:
        idx = int(np.clip(int(path_index or 0), 0, len(path) - 1))
    ws_i, we_i, bb_i = path[idx]
    return _ch4_ball_voxel_frame(
        pack, float(ws_i), float(we_i), float(bb_i),
        ax3d_bounds=bounds,
        wide_bounds=bounds,
        voxel_draw=_ch4_ball_voxel_path_draw(pack, idx),
        point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
        measurements=_ch3_lik_we_are_here_at(pack, float(ws_i), float(we_i), float(bb_i)),
    )


def ch4_export_likelihood_ball_voxel_path():
    frames = ch3_build_frames_likelihood_ball_voxel_path_story(rotate_during=True)
    fn = "ch4_05a4_ball_voxel_path.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_ball_voxel_path_rot_after():
    frames = ch3_build_frames_likelihood_ball_voxel_path_story(rotate_during=False)
    fn = "ch4_05a4b_ball_voxel_path_rot_after.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05a5 / 05a6: GD voxel steps from CH3_LIK_3D_PATH_START (same style as 05a4) ---

CH3_LIK_GD_VOXEL_STEP_SMALL = 0.015
CH3_LIK_GD_VOXEL_HALF_SCALE = 0.5
CH3_LIK_GD_VOXEL_N_EMPHASIS = 8 if _CH3_DRAFT else 22
CH3_LIK_GD_VOXEL_N_STEP = 8 if _CH3_DRAFT else 20
CH3_LIK_GD_VOXEL_N_HOLD = 4 if _CH3_DRAFT else 8
CH3_LIK_GD_VOXEL_EMPHASIS_DIM = 0.32
CH3_LIK_GD_VOXEL_N_ROT_AFTER = 8 if _CH3_DRAFT else 40


def _ch4_ball_voxel_closest_grad_cheb(cube, center, grad):
    """Chebyshev layer of the voxel whose offset best aligns with ``-∇NLL``."""
    center = np.asarray(center, dtype=np.float64)
    g = np.asarray(grad, dtype=np.float64).ravel()
    gn = float(np.linalg.norm(g))
    if gn < 1e-14:
        return int(cube["max_cheb"])
    ghat = -g / gn
    x_edges = cube["x_edges"]
    y_edges = cube["y_edges"]
    z_edges = cube["z_edges"]
    w1 = 0.5 * (x_edges[:-1] + x_edges[1:])
    w2 = 0.5 * (y_edges[:-1] + y_edges[1:])
    bb = 0.5 * (z_edges[:-1] + z_edges[1:])
    visible = np.asarray(cube["checker"], dtype=bool)
    cheb = np.asarray(cube["cheb"], dtype=float)
    best_score = -np.inf
    best_cheb = int(cube["max_cheb"])
    for i, j, k in np.argwhere(visible):
        vc = np.array([w1[i], w2[j], bb[k]], dtype=np.float64) - center
        vn = float(np.linalg.norm(vc))
        if vn < 1e-14:
            continue
        score = float(np.dot(vc / vn, ghat))
        c = int(cheb[i, j, k])
        if score > best_score:
            best_score = score
            best_cheb = c
    return best_cheb


def _ch4_gd_voxel_measurements_pack(*, eta):
    """05a4-style pack with a GD trail from ``CH3_LIK_3D_PATH_START``."""
    pack = _ch4_ball_voxel_measurements_pack()
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = _ch3_lik_gd_path(
        study, exam, y,
        CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, float(eta),
    )
    pack["path"] = trail
    pack["path_waypoint_indices"] = np.arange(len(trail), dtype=int)
    pack["gd_eta"] = float(eta)
    pack.pop("ball_voxel_ready", None)
    return pack


def _ch4_gd_voxel_half():
    """Half-extent of each step's voxel ball — 05a5/05a6 use half of 05a4."""
    return float(_ch4_ball_voxel_half()) * float(CH3_LIK_GD_VOXEL_HALF_SCALE)


def _ch4_gd_voxel_prewarm_pack(pack):
    """Like ``_ch4_ball_voxel_prewarm_pack`` but with the smaller GD voxel half."""
    if pack.get("ball_voxel_ready"):
        return pack
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = _ch4_gd_voxel_half()
    path = np.asarray(pack["path"], dtype=float)
    start_cube = _ch4_ball_voxel_build_cube(global_cache, path[0], half)
    path_cubes = [
        _ch4_ball_voxel_build_cube(global_cache, r, half)
        for r in path
    ]
    pack["ball_voxel_half"] = half
    pack["ball_voxel_start_cube"] = start_cube
    pack["ball_voxel_path_cubes"] = path_cubes
    pack["ball_voxel_right"] = global_cache["right_blocks"]
    pack["ball_voxel_ready"] = True
    return pack


def _ch3_lik_gd_voxel_intro_frames(pack):
    _ch4_gd_voxel_prewarm_pack(pack)
    return list(_ch4_ball_voxel_cached_intro_frames(pack, _ch4_gd_voxel_half()))


def ch4_prewarm_ball_voxel_intro_cache():
    """Build disk+memory cache for both intro variants (05a4 + 05a5/06)."""
    pack_path = _ch4_ball_voxel_measurements_pack()
    _ch4_ball_voxel_prewarm_pack(pack_path)
    _ch4_ball_voxel_cached_intro_frames(pack_path, _ch4_ball_voxel_half())
    pack_gd = _ch4_gd_voxel_measurements_pack(eta=float(CH3_LIK_GD_STEP))
    _ch4_gd_voxel_prewarm_pack(pack_gd)
    _ch4_ball_voxel_cached_intro_frames(pack_gd, _ch4_gd_voxel_half())
    print("ch4 voxel intro cache ready", flush=True)


def _ch4_gd_voxel_step_draw(accumulated, *, grow_u=1.0, emphasize_cheb=None, emphasize_idx=-1):
    """Draw completed voxels; optionally grow/emphasize the cube at ``emphasize_idx``."""
    draws = []
    for j, c in enumerate(accumulated):
        if j == emphasize_idx:
            draws.append(_ch4_ball_voxel_cube_draw(
                c,
                grow_u=float(grow_u),
                emphasize_cheb=emphasize_cheb,
                dim_others=float(CH3_LIK_GD_VOXEL_EMPHASIS_DIM),
            ))
        else:
            draws.append(_ch4_ball_voxel_cube_draw(c, grow_u=1.0))
    return _ch4_ball_voxel_merge_draws(draws)


def _ch4_gd_voxel_append_step_frames(frames, pack, *, rotate_during=True):
    """Ten GD steps: emphasize voxel nearest ``-∇NLL``, step, keep prior voxels."""
    _ch4_gd_voxel_prewarm_pack(pack)
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    n_steps = len(trail) - 1
    n_path = max(n_steps, 1)
    rail = pack["ball_voxel_right"]
    rail_title = CH4_NLL_HEATMAP_SECTION_TITLE

    accumulated = [pack["ball_voxel_start_cube"]]

    for step_i in range(n_steps):
        ws, we, bb = (float(trail[step_i, 0]), float(trail[step_i, 1]), float(trail[step_i, 2]))
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (g1, g2, gb)
        cube = accumulated[-1]
        emp = _ch4_ball_voxel_closest_grad_cheb(cube, (ws, we, bb), grad)
        cur_idx = len(accumulated) - 1
        cam_u = float(step_i) / float(n_path) if rotate_during else 0.0
        cam_rot = float(CH3_LIK_3D_CAM_PATH_ROT) if rotate_during else 0.0

        if step_i > 0:
            for tv in np.linspace(0.0, 1.0, CH3_LIK_GD_VOXEL_N_EMPHASIS, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                frames.append(_ch4_ball_voxel_frame(
                    pack, ws, we, bb,
                    ax3d_bounds=bounds, wide_bounds=bounds,
                    voxel_draw=_ch4_gd_voxel_step_draw(
                        accumulated, grow_u=u, emphasize_cheb=emp, emphasize_idx=cur_idx,
                    ),
                    point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
                    cam_azim_u=cam_u,
                    cam_rot_deg=cam_rot,
                    right_blocks=rail,
                    right_title=rail_title,
                    right_title_single_line=True,
                ))

            for _ in range(CH3_LIK_GD_VOXEL_N_HOLD):
                frames.append(_ch4_ball_voxel_frame(
                    pack, ws, we, bb,
                    ax3d_bounds=bounds, wide_bounds=bounds,
                    voxel_draw=_ch4_gd_voxel_step_draw(
                        accumulated, grow_u=1.0, emphasize_cheb=emp, emphasize_idx=cur_idx,
                    ),
                    point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
                    cam_azim_u=cam_u,
                    cam_rot_deg=cam_rot,
                    right_blocks=rail,
                    right_title=rail_title,
                    right_title_single_line=True,
                ))

        ws1, we1, bb1 = (float(trail[step_i + 1, 0]), float(trail[step_i + 1, 1]), float(trail[step_i + 1, 2]))
        for tv in np.linspace(0.0, 1.0, CH3_LIK_GD_VOXEL_N_STEP, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            wi = ws + u * (ws1 - ws)
            ei = we + u * (we1 - we)
            bi = bb + u * (bb1 - bb)
            step_cam_u = cam_u + u / float(n_path) if rotate_during else 0.0
            frames.append(_ch4_ball_voxel_frame(
                pack, wi, ei, bi,
                ax3d_bounds=bounds, wide_bounds=bounds,
                voxel_draw=_ch4_ball_voxel_merge_draws([
                    _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
                ]),
                point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
                cam_azim_u=step_cam_u,
                cam_rot_deg=cam_rot,
                right_blocks=rail,
                right_title=rail_title,
                right_title_single_line=True,
            ))

        accumulated.append(_ch4_ball_voxel_build_cube(global_cache, trail[step_i + 1], half))

    ws_f, we_f, bb_f = (float(trail[-1, 0]), float(trail[-1, 1]), float(trail[-1, 2]))
    end_cam_u = 1.0 if rotate_during else 0.0
    end_cam_rot = float(CH3_LIK_3D_CAM_PATH_ROT) if rotate_during else 0.0
    for _ in range(max(4, CH3_LIK_VOXEL_N_HOLD // 3)):
        frames.append(_ch4_ball_voxel_frame(
            pack, ws_f, we_f, bb_f,
            ax3d_bounds=bounds, wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_merge_draws([
                _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
            ]),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=end_cam_u,
            cam_rot_deg=end_cam_rot,
            right_blocks=rail,
            right_title=rail_title,
            right_title_single_line=True,
        ))


def _ch4_gd_voxel_append_rot_after(frames, pack):
    """Combined 360° pan after all GD steps (05a5/05a6 rot-after variants)."""
    _ch4_gd_voxel_prewarm_pack(pack)
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    accumulated = [
        _ch4_ball_voxel_build_cube(global_cache, trail[j], half)
        for j in range(len(trail))
    ]
    ws_f, we_f, bb_f = (float(trail[-1, 0]), float(trail[-1, 1]), float(trail[-1, 2]))
    rail = pack["ball_voxel_right"]
    rail_title = CH4_NLL_HEATMAP_SECTION_TITLE
    for tv in np.linspace(0.0, 1.0, CH3_LIK_GD_VOXEL_N_ROT_AFTER, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(_ch4_ball_voxel_frame(
            pack, ws_f, we_f, bb_f,
            ax3d_bounds=bounds, wide_bounds=bounds,
            voxel_draw=_ch4_ball_voxel_merge_draws([
                _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
            ]),
            point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
            cam_azim_u=u,
            cam_rot_deg=float(CH3_LIK_3D_CAM_PATH_ROT),
            right_blocks=rail,
            right_title=rail_title,
            right_title_single_line=True,
        ))


def ch3_build_frames_likelihood_gd_voxel_steps_story(*, eta, rotate_during=True):
    pack = _ch4_gd_voxel_measurements_pack(eta=float(eta))
    _ch4_gd_voxel_prewarm_pack(pack)
    frames = _ch3_lik_gd_voxel_intro_frames(pack)
    _ch4_gd_voxel_append_step_frames(frames, pack, rotate_during=rotate_during)
    if not rotate_during:
        _ch4_gd_voxel_append_rot_after(frames, pack)
    return _ch3_lik_story_hold(frames)


def ch4_preview_likelihood_gd_voxel_steps_end_frame(*, eta=None):
    """Last frame — all 10 GD steps accumulated."""
    eta = float(CH3_LIK_GD_STEP if eta is None else eta)
    pack = _ch4_gd_voxel_measurements_pack(eta=eta)
    _ch4_gd_voxel_prewarm_pack(pack)
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    accumulated = [
        _ch4_ball_voxel_build_cube(global_cache, trail[j], half)
        for j in range(len(trail))
    ]
    ws_f, we_f, bb_f = (float(trail[-1, 0]), float(trail[-1, 1]), float(trail[-1, 2]))
    return _ch4_ball_voxel_frame(
        pack, ws_f, we_f, bb_f,
        ax3d_bounds=bounds, wide_bounds=bounds,
        voxel_draw=_ch4_ball_voxel_merge_draws([
            _ch4_ball_voxel_cube_draw(c, grow_u=1.0) for c in accumulated
        ]),
        point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
        cam_azim_u=1.0,
        cam_rot_deg=float(CH3_LIK_3D_CAM_PATH_ROT),
        right_blocks=pack["ball_voxel_right"],
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
    )


def ch4_preview_likelihood_gd_voxel_steps_frame(*, eta=None, step_index=0, emphasis_u=1.0):
    eta = float(CH3_LIK_GD_STEP if eta is None else eta)
    pack = _ch4_gd_voxel_measurements_pack(eta=eta)
    _ch4_gd_voxel_prewarm_pack(pack)
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = np.asarray(pack["path"], dtype=float)
    bounds = _ch4_ball_voxel_view_bounds()
    global_cache = _ch4_ball_voxel_global_cache(pack)
    half = float(pack["ball_voxel_half"])
    step_i = int(np.clip(int(step_index), 0, len(trail) - 2))
    ws, we, bb = (float(trail[step_i, 0]), float(trail[step_i, 1]), float(trail[step_i, 2]))
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    cube = _ch4_ball_voxel_build_cube(global_cache, trail[step_i], half)
    emp = _ch4_ball_voxel_closest_grad_cheb(cube, (ws, we, bb), (g1, g2, gb))
    accumulated = [
        _ch4_ball_voxel_build_cube(global_cache, trail[j], half)
        for j in range(step_i + 1)
    ]
    return _ch4_ball_voxel_frame(
        pack, ws, we, bb,
        ax3d_bounds=bounds, wide_bounds=bounds,
        voxel_draw=_ch4_gd_voxel_step_draw(
            accumulated, grow_u=float(emphasis_u), emphasize_cheb=emp, emphasize_idx=len(accumulated) - 1,
        ),
        point_s=CH3_LIK_BALL_VOXEL_POINT_S_SMALL,
        right_blocks=pack["ball_voxel_right"],
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
    )


def ch4_export_likelihood_gd_voxel_steps():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_STEP), rotate_during=True,
    )
    fn = "ch4_05a5_gd_voxel_steps.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_gd_voxel_steps_rot_after():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_STEP), rotate_during=False,
    )
    fn = "ch4_05a5b_gd_voxel_steps_rot_after.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_gd_voxel_steps_small():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_VOXEL_STEP_SMALL), rotate_during=True,
    )
    fn = "ch4_05a6_gd_voxel_steps_small.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_gd_voxel_steps_small_rot_after():
    frames = ch3_build_frames_likelihood_gd_voxel_steps_story(
        eta=float(CH3_LIK_GD_VOXEL_STEP_SMALL), rotate_during=False,
    )
    fn = "ch4_05a6b_gd_voxel_steps_small_rot_after.mp4"
    save_mp4(frames, fn, duration=int(CH3_LIK_BALL_VOXEL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_05b: partial derivative motivation (between ball vectors and GD) ---

CH3_LIK_PARTIAL_HALF_W_START = 0.38
CH3_LIK_PARTIAL_ROC_WIDE_HALF_W = float(CH3_LIK_PARTIAL_HALF_W_START) - 0.05
CH3_LIK_PARTIAL_HALF_W_END = 0.006
CH3_LIK_PARTIAL_N_CURVE = 72 if not _CH3_DRAFT else 36
CH3_LIK_PARTIAL_VECTOR_SCALE = 0.14
CH3_LIK_PARTIAL_N_HOLD = 4 if _CH3_DRAFT else max(12, _smooth_n(8))
CH3_LIK_PARTIAL_N_DRAW = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_ROC_INTRO = 4 if _CH3_DRAFT else max(16, _smooth_n(12))
CH3_LIK_PARTIAL_N_ROC_SHRINK = 10 if _CH3_DRAFT else max(64, _smooth_n(48))
CH3_LIK_PARTIAL_N_VECTORS = 6 if _CH3_DRAFT else max(28, _smooth_n(20))
CH3_LIK_PARTIAL_N_ERASE = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_PARTIAL = 6 if _CH3_DRAFT else max(54, _smooth_n(42))
CH3_LIK_PARTIAL_N_UPDATE_REVEAL = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_ALPHA = 4 if _CH3_DRAFT else max(24, _smooth_n(18))
CH3_LIK_PARTIAL_N_PLOT = 6 if _CH3_DRAFT else max(28, _smooth_n(20))
CH3_LIK_PARTIAL_PARAM_BUMP = 0.1
CH3_LIK_PARTIAL_N_PARAM_BUMP = 6 if _CH3_DRAFT else max(32, _smooth_n(24))
CH3_LIK_PARTIAL_N_PARAM_PAUSE = 4 if _CH3_DRAFT else max(16, _smooth_n(12))
CH3_LIK_PARTIAL_N_NEG_REVEAL = 6 if _CH3_DRAFT else max(36, _smooth_n(28))
CH3_LIK_PARTIAL_N_GD_DEMO = 6 if _CH3_DRAFT else max(48, _smooth_n(36))
CH3_LIK_PARTIAL_N_GD_DEMO_STEPS = 2
CH3_LIK_PARTIAL_MS = 110 if not _CH3_DRAFT else 130
CH3_LIK_PARTIAL_POINT_S = 165.0
CH3_LIK_PARTIAL_SECANT_POINT_S = 72.0
CH3_LIK_PARTIAL_VECTOR_DIM_ALPHA = 0.28
CH3_LIK_PARTIAL_VECTOR_LW = 6.0
CH3_LIK_PARTIAL_VECTOR_MUTATION = 28.0
CH3_LIK_PARTIAL_GHOST_CURVE_ALPHA = 0.38
CH3_LIK_PARTIAL_GHOST_ARROW_ALPHA = 0.32
CH3_LIK_PARTIAL_GHOST_TRAIL_N = 5
CH3_LIK_PARTIAL_GHOST_POINT_S = 72.0

_CH4_PARTIAL_PARAM_LABELS = (
    (r"$w_{\mathrm{ST}}$", "st"),
    (r"$w_{\mathrm{EL}}$", "el"),
    (r"$b$", "b"),
)
_CH4_PARTIAL_PARAM_TEX = {
    "st": r"w_{\mathrm{ST}}",
    "el": r"w_{\mathrm{EL}}",
    "b": "b",
}


def _ch4_partial_draw_delta_bracket(ax, x_lo, x_hi, y_lo_p, y_hi_p, color):
    """Horizontal Δparam at ``y_lo_p``; vertical ΔNLL at ``x_hi``."""
    c = str(color)
    ax.plot([x_lo, x_hi], [y_lo_p, y_lo_p], color=c, lw=2.4, solid_capstyle="butt", zorder=6)
    ax.plot([x_hi, x_hi], [y_lo_p, y_hi_p], color=c, lw=4.4, solid_capstyle="butt", zorder=6)


def _ch4_partial_annotate_panel(
    ax, which, x_lo, x_hi, y_lo_p, y_hi_p, color, *, label_fs,
):
    pname = _CH4_PARTIAL_PARAM_TEX[which]
    fs_h = float(label_fs) * 2.0
    fs_v = float(label_fs) * 1.5
    dparam_tex = rf"$\Delta {pname}$"
    dnll_tex = rf"$\Delta NLL$"
    y_span = max(float(ax.get_ylim()[1] - ax.get_ylim()[0]), 1e-9)
    x_span = max(float(ax.get_xlim()[1] - ax.get_xlim()[0]), 1e-9)
    mid_x = 0.5 * (float(x_lo) + float(x_hi))
    mid_y = 0.5 * (float(y_lo_p) + float(y_hi_p))
    ax.text(
        mid_x, float(y_lo_p) + 0.028 * y_span, dparam_tex,
        va="bottom", ha="center", fontsize=fs_h, color=str(color), zorder=15,
    )
    ax.text(
        float(x_hi) + 0.012 * x_span, mid_y, dnll_tex,
        va="center", ha="left", fontsize=fs_v, color=str(color), zorder=15,
    )


def _ch4_partial_panel_colors():
    from ch4_layout import CH4_GD_ARROW_B_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_ST_COLOR
    return (CH4_GD_ARROW_ST_COLOR, CH4_GD_ARROW_EL_COLOR, CH4_GD_ARROW_B_COLOR)


def _ch4_nll_scalar(ws, we, bb, study, exam, y):
    return float(-loss_log_likelihood(ws, we, bb, study, exam, y))


def _ch4_nll_param_value(which, ws, we, bb, val, study, exam, y):
    v = float(val)
    if which == "st":
        return _ch4_nll_scalar(v, we, bb, study, exam, y)
    if which == "el":
        return _ch4_nll_scalar(ws, v, bb, study, exam, y)
    return _ch4_nll_scalar(ws, we, v, study, exam, y)


def _ch4_nll_param_curve(which, ws, we, bb, half_w, study, exam, y, *, x_center=None, n=None):
    n = int(CH3_LIK_PARTIAL_N_CURVE if n is None else n)
    centers = {"st": float(ws), "el": float(we), "b": float(bb)}
    c = float(centers[which] if x_center is None else x_center)
    d = float(half_w)
    xs = np.linspace(c - d, c + d, n, dtype=np.float64)
    ys = np.array(
        [_ch4_nll_param_value(which, ws, we, bb, x, study, exam, y) for x in xs],
        dtype=np.float64,
    )
    return xs, ys


def _ch4_partial_panel_x_anchor(
    which, live_centers, ghost_centers, show_ghosts, *, tol=1e-9, center_on_live=False,
):
    """Fixed x-window anchor when this panel's parameter is changing; else center on live value."""
    xc = float(live_centers[which])
    if center_on_live:
        return xc
    if show_ghosts and abs(xc - float(ghost_centers[which])) > tol:
        return float(ghost_centers[which])
    return xc


def _ch4_partial_compute_panel_y_half(study, exam, y, ws, we, bb, plot_half_w):
    """Per-panel NLL y half-span at ``(ws, we, bb)`` for centered y-axis tracking."""
    halves = {}
    centers = {"st": float(ws), "el": float(we), "b": float(bb)}
    ph = float(plot_half_w)
    for which in ("st", "el", "b"):
        _, ys = _ch4_nll_param_curve(
            which, ws, we, bb, ph, study, exam, y, x_center=centers[which],
        )
        span = max(1e-6, float(np.nanmax(ys) - np.nanmin(ys)))
        halves[which] = 0.54 * span
    return halves


def _ch4_partial_infer_active_param(live_centers, ghost_centers, *, tol=1e-9):
    changed = [
        which for which in ("st", "el", "b")
        if abs(float(live_centers[which]) - float(ghost_centers[which])) > tol
    ]
    return changed[0] if len(changed) == 1 else None


def _ch4_partial_trail_poses(gws, gwe, gbb, ws, we, bb, n_trail=None):
    """Evenly spaced poses from ghost baseline to live (inclusive of live at the end)."""
    n_trail = int(CH3_LIK_PARTIAL_GHOST_TRAIL_N if n_trail is None else n_trail)
    if n_trail <= 0 or not _ch4_partial_pose_changed(ws, we, bb, gws, gwe, gbb):
        return []
    poses = []
    for i in range(1, n_trail + 1):
        t = float(i) / float(n_trail)
        poses.append((
            float(gws) + t * (float(ws) - float(gws)),
            float(gwe) + t * (float(we) - float(gwe)),
            float(gbb) + t * (float(bb) - float(gbb)),
        ))
    return poses


def _ch4_partial_trail_alpha(frac, base_alpha):
    return float(base_alpha) * (0.30 + 0.70 * float(np.clip(frac, 0.0, 1.0)))


def _ch4_partial_trail_ys(
    study, exam, y, trail_poses, plot_half_w, live_centers, ghost_centers, *,
    center_on_live=False,
):
    ys_all = []
    for tws, twe, tbb in trail_poses:
        for which in ("st", "el", "b"):
            anchor = _ch4_partial_panel_x_anchor(
                which, live_centers, ghost_centers, True, center_on_live=center_on_live,
            )
            _, ys = _ch4_nll_param_curve(
                which, tws, twe, tbb, plot_half_w, study, exam, y, x_center=anchor,
            )
            ys_all.append(ys)
    return ys_all


def _ch4_partial_draw_panel_ghost_trail(
    ax, which, color, trail_poses, live_centers, ghost_centers,
    study, exam, y, plot_half_w, grad_ref, *, active_param=None,
    ghost_trail_panels=None, center_on_live=False, vector_axis_scale=False,
):
    """Draw ``CH3_LIK_PARTIAL_GHOST_TRAIL_N`` muted snapshots along a knob sweep."""
    if not trail_poses:
        return
    if ghost_trail_panels is not None:
        if which not in ghost_trail_panels:
            return
    elif active_param is not None and which == active_param:
        return
    n_trail = len(trail_poses)
    for ti, (tws, twe, tbb) in enumerate(trail_poses):
        frac = float(ti + 1) / float(n_trail)
        curve_a = _ch4_partial_trail_alpha(frac, CH3_LIK_PARTIAL_GHOST_CURVE_ALPHA)
        arrow_a = _ch4_partial_trail_alpha(frac, CH3_LIK_PARTIAL_GHOST_ARROW_ALPHA)
        point_a = _ch4_partial_trail_alpha(frac, 0.92)
        z = 1 + ti
        anchor = _ch4_partial_panel_x_anchor(
            which, live_centers, ghost_centers, True, center_on_live=center_on_live,
        )
        txs, tys = _ch4_nll_param_curve(
            which, tws, twe, tbb, plot_half_w, study, exam, y, x_center=anchor,
        )
        ax.plot(txs, tys, color=str(color), lw=2.2, alpha=curve_a, zorder=z)
        trail_centers = {"st": tws, "el": twe, "b": tbb}
        txc = float(trail_centers[which])
        tyc = float(_ch4_nll_param_value(which, tws, twe, tbb, txc, study, exam, y))
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, tws, twe, tbb)
        tgrad = {"st": g1, "el": g2, "b": gb}[which]
        ax.scatter(
            [txc], [tyc], s=float(CH3_LIK_PARTIAL_GHOST_POINT_S), c=[str(color)],
            edgecolors="white", linewidths=1.0, alpha=point_a, zorder=z + 5,
        )
        _ch4_partial_draw_gradient_vector(
            ax, txc, tyc, tgrad, plot_half_w, color,
            grad_ref=grad_ref, alpha=arrow_a, vector_axis_scale=vector_axis_scale,
        )


def _ch4_partial_pose_curves(
    study, exam, y, ws, we, bb, plot_half_w, *,
    ghost_ws=None, ghost_we=None, ghost_bb=None,
    panel_axes_centered=False,
):
    """NLL slices + shared y-limits at live pose; x sampling follows panel anchor rules."""
    ph = float(plot_half_w)
    live = {"st": float(ws), "el": float(we), "b": float(bb)}
    ghost = {
        "st": float(ws if ghost_ws is None else ghost_ws),
        "el": float(we if ghost_we is None else ghost_we),
        "b": float(bb if ghost_bb is None else ghost_bb),
    }
    show_ghosts = _ch4_partial_pose_changed(ws, we, bb, ghost["st"], ghost["el"], ghost["b"])
    trail_poses = _ch4_partial_trail_poses(ghost["st"], ghost["el"], ghost["b"], ws, we, bb)
    curves = {}
    ys_all = []
    for which in ("st", "el", "b"):
        anchor = _ch4_partial_panel_x_anchor(
            which, live, ghost, show_ghosts, center_on_live=panel_axes_centered,
        )
        xs, ys = _ch4_nll_param_curve(
            which, ws, we, bb, ph, study, exam, y, x_center=anchor,
        )
        curves[which] = (xs, ys)
        ys_all.append(ys)
    if show_ghosts and trail_poses:
        ys_all.extend(_ch4_partial_trail_ys(
            study, exam, y, trail_poses, ph, live, ghost,
            center_on_live=panel_axes_centered,
        ))
    all_y = np.concatenate(ys_all)
    pad = 0.08 * max(1e-6, float(np.nanmax(all_y) - np.nanmin(all_y)))
    return (
        curves,
        float(np.nanmin(all_y) - pad),
        float(np.nanmax(all_y) + pad),
        ghost,
        show_ghosts,
        trail_poses,
    )


def _ch4_partial_pose_changed(ws, we, bb, gws, gwe, gbb, *, tol=1e-9):
    return (
        abs(float(ws) - float(gws)) > tol
        or abs(float(we) - float(gwe)) > tol
        or abs(float(bb) - float(gbb)) > tol
    )


def _ch4_nll_avg_roc(which, ws, we, bb, half_w, study, exam, y):
    centers = {"st": float(ws), "el": float(we), "b": float(bb)}
    c = centers[which]
    h = max(float(half_w), 1e-8)
    yp = _ch4_nll_param_value(which, ws, we, bb, c + h, study, exam, y)
    ym = _ch4_nll_param_value(which, ws, we, bb, c - h, study, exam, y)
    return float((yp - ym) / (2.0 * h))


def _ch4_nll_partial_triptych_ylim(ws, we, bb, half_w, study, exam, y):
    ys_all = []
    for which in ("st", "el", "b"):
        _, ys = _ch4_nll_param_curve(which, ws, we, bb, half_w, study, exam, y)
        ys_all.append(ys)
    all_y = np.concatenate(ys_all)
    pad = 0.08 * max(1e-6, float(np.nanmax(all_y) - np.nanmin(all_y)))
    return float(np.nanmin(all_y) - pad), float(np.nanmax(all_y) + pad)


def _ch4_partial_warm_curve_cache(pack, *, plot_half_w=None):
    """Precompute NLL slice curves + y-limits once (fixed axes for whole clip)."""
    plot_half_w = float(CH3_LIK_PARTIAL_HALF_W_START if plot_half_w is None else plot_half_w)
    cache = pack.setdefault("partial_curve_cache", {})
    if plot_half_w in cache:
        return cache[plot_half_w]
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"])
    we = float(pack["partial_we"])
    bb = float(pack["partial_bb"])
    curves = {
        which: _ch4_nll_param_curve(which, ws, we, bb, plot_half_w, study, exam, y)
        for which in ("st", "el", "b")
    }
    y_lo, y_hi = _ch4_nll_partial_triptych_ylim(ws, we, bb, plot_half_w, study, exam, y)
    cache[plot_half_w] = (curves, y_lo, y_hi)
    pack["partial_curve_pose"] = (ws, we, bb)
    return cache[plot_half_w]


def _ch4_partial_bottom_fully_written(bottom_prog):
    if not bottom_prog:
        return True
    return all(float(v) >= 1.0 - 1e-9 for v in bottom_prog.values())


def _ch4_partial_rails_keys(*, gd_bottom: bool, bottom_prog):
    from ch4_layout import (
        CH4_RAILS_CACHE_SHELL,
        ch4_rails_cache_key,
        ch4_rails_cache_key_gd,
    )

    if not _ch4_partial_bottom_fully_written(bottom_prog):
        return None, CH4_RAILS_CACHE_SHELL
    if gd_bottom:
        return ch4_rails_cache_key_gd(), None
    return ch4_rails_cache_key(gd_formulas=False), None


def ch3_figure_nllkeras_partial_triptych():
    """Left: dataset + knobs; right: three stacked NLL-vs-parameter panels."""
    fig = plt.figure(figsize=CH4_DUO_FIGSIZE)
    gs = fig.add_gridspec(1, 2, width_ratios=CH4_DUO_WIDTH_RATIOS, wspace=CH3_DUO_WSPACE)
    g_left = GridSpecFromSubplotSpec(
        2, 1, subplot_spec=gs[0, 0], height_ratios=CH3_LEFT_HEIGHT_RATIOS, hspace=CH3_LEFT_HSPACE
    )
    ax_data = fig.add_subplot(g_left[0, 0])
    g_k = GridSpecFromSubplotSpec(1, 3, subplot_spec=g_left[1, 0], wspace=CH3_KNOB_WSPACE)
    axes_k = tuple(fig.add_subplot(g_k[0, j]) for j in range(3))
    g_right = GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[0, 1], hspace=0.42)
    axes_partial = tuple(fig.add_subplot(g_right[i, 0]) for i in range(3))
    fig.subplots_adjust(left=0.05, right=0.97, top=0.93, bottom=0.06)
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    return fig, ax_data, axes_partial, axes_k


def _ch4_partial_draw_gradient_vector(
    ax, x0, y0, grad, half_w, color, *, grad_ref, alpha=1.0, vector_axis_scale=False,
):
    """Horizontal ∂ arrow; normalize to ``grad_ref`` or map ``|grad|`` directly to Δw on the x-axis."""
    g = float(grad)
    if abs(g) < 1e-12:
        return
    span = 2.0 * max(float(half_w), 1e-6)
    max_frac = 0.40
    max_dx = max_frac * span
    if vector_axis_scale:
        dx = float(np.sign(g) * min(abs(g) * float(CH3_LIK_PARTIAL_VECTOR_SCALE), max_dx))
    else:
        g_ref = max(abs(float(grad_ref)), 1e-12)
        dx = float(np.sign(g) * (abs(g) / g_ref) * max_dx)
    import matplotlib.colors as mcolors
    rgba = mcolors.to_rgba(str(color), alpha=float(np.clip(alpha, 0.0, 1.0)))
    ax.annotate(
        "",
        xy=(x0 + dx, y0),
        xytext=(x0, y0),
        arrowprops=dict(
            arrowstyle="-|>",
            color=rgba,
            lw=float(CH3_LIK_PARTIAL_VECTOR_LW),
            mutation_scale=float(CH3_LIK_PARTIAL_VECTOR_MUTATION),
            shrinkA=0.0,
            shrinkB=0.0,
        ),
        zorder=12,
    )


def ch3_frame_lik_partial_motivation(
    study, exam, y, ws, we, bb, *,
    half_w,
    plot_half_w=None,
    curve_u=1.0,
    show_secant=True,
    show_vectors=False,
    vector_panels_show=3,
    vector_alpha=1.0,
    neg_vectors=False,
    neg_vector_alpha=0.95,
    rocs=None,
    show_partials=False,
    show_neg_partials=False,
    partials=None,
    partial_lines_show=3,
    show_alpha=False,
    step_size=None,
    bottom_blocks=None,
    progress_override=None,
    write_progress=1.0,
    right_blocks=None,
    curve_cache=None,
    ghost_ws=None,
    ghost_we=None,
    ghost_bb=None,
    active_param=None,
    ghost_trail_panels=None,
    panel_axes_centered=False,
    panel_y_half=None,
    vector_axis_scale=False,
    rails_cache_key=None,
    shell_cache_key=None,
    gd_bottom=False,
):
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_HERE_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        ch4_cached_notation_corner_blocks,
        ch4_duo_partial_layout_tune,
        ch4_knob_asset_pack,
        ch4_we_are_here_roc_blocks,
        ch4_we_are_here_grad_line_colors,
        compose_tutorial,
    )

    ws, we, bb = float(ws), float(we), float(bb)
    half_w = float(half_w)
    plot_half_w = float(CH3_LIK_PARTIAL_HALF_W_START if plot_half_w is None else plot_half_w)
    fig, ax_data, axes_partial, axes_k = ch3_figure_nllkeras_partial_triptych()
    ch4_duo_partial_layout_tune(
        fig, ax_data, axes_partial,
        y_lift_mm=10.0, x_shift_mm=-10.0, subplot_height_frac=0.88,
    )
    leg = legend_linear_equation_values_bold_param(ws, we, bb, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, bb, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, bb), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    if curve_cache is None:
        curves, y_lo, y_hi, ghost_centers, show_ghosts, trail_poses = _ch4_partial_pose_curves(
            study, exam, y, ws, we, bb, plot_half_w,
            ghost_ws=ghost_ws, ghost_we=ghost_we, ghost_bb=ghost_bb,
            panel_axes_centered=panel_axes_centered,
        )
    else:
        curves, y_lo, y_hi = curve_cache
        gws = float(ws if ghost_ws is None else ghost_ws)
        gwe = float(we if ghost_we is None else ghost_we)
        gbb = float(bb if ghost_bb is None else ghost_bb)
        ghost_centers = {"st": gws, "el": gwe, "b": gbb}
        show_ghosts = _ch4_partial_pose_changed(ws, we, bb, gws, gwe, gbb)
        trail_poses = _ch4_partial_trail_poses(gws, gwe, gbb, ws, we, bb)
    live_centers = {"st": ws, "el": we, "b": bb}
    if active_param is None and show_ghosts:
        active_param = _ch4_partial_infer_active_param(live_centers, ghost_centers)
    nll0 = _ch4_nll_scalar(ws, we, bb, study, exam, y)
    if partials is None:
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        partials = (g1, g2, gb)
    centers = {"st": ws, "el": we, "b": bb}
    grads = {"st": partials[0], "el": partials[1], "b": partials[2]}
    cu = float(np.clip(curve_u, 0.0, 1.0))
    grad_ref = max(abs(float(partials[0])), abs(float(partials[1])), abs(float(partials[2])), 1e-12)
    label_fs = FONT_SIZE * 0.88 + 2.0
    v_show = max(0, min(int(vector_panels_show), 3))
    for pi, (ax, (_xlabel, which), color) in enumerate(
        zip(axes_partial, _CH4_PARTIAL_PARAM_LABELS, _ch4_partial_panel_colors()),
    ):
        xs, ys = curves[which]
        n_keep = max(2, int(round(cu * (len(xs) - 1))) + 1)
        xc = float(centers[which])
        yc = float(_ch4_nll_param_value(which, ws, we, bb, xc, study, exam, y))
        x_anchor = _ch4_partial_panel_x_anchor(
            which, centers, ghost_centers, show_ghosts,
            center_on_live=panel_axes_centered,
        )
        if show_ghosts and show_vectors and pi < v_show:
            _ch4_partial_draw_panel_ghost_trail(
                ax, which, color, trail_poses, live_centers, ghost_centers,
                study, exam, y, plot_half_w, grad_ref, active_param=active_param,
                ghost_trail_panels=ghost_trail_panels,
                center_on_live=panel_axes_centered,
                vector_axis_scale=vector_axis_scale,
            )
        ax.plot(xs[:n_keep], ys[:n_keep], color="#333333", lw=2.4, zorder=2)
        ax.scatter(
            [xc], [yc], s=float(CH3_LIK_PARTIAL_POINT_S), c=[color],
            edgecolors="white", linewidths=1.6, zorder=8,
        )
        ax.set_xlim(x_anchor - plot_half_w, x_anchor + plot_half_w)
        if panel_axes_centered and panel_y_half is not None:
            yh = float(panel_y_half[which])
            ax.set_ylim(yc - yh, yc + yh)
        else:
            ax.set_ylim(y_lo, y_hi)
        if show_secant and cu >= 0.99 and not show_vectors:
            h = min(float(half_w), plot_half_w)
            x_lo, x_hi = xc - h, xc + h
            y_lo_p = _ch4_nll_param_value(which, ws, we, bb, x_lo, study, exam, y)
            y_hi_p = _ch4_nll_param_value(which, ws, we, bb, x_hi, study, exam, y)
            _ch4_partial_draw_delta_bracket(ax, x_lo, x_hi, y_lo_p, y_hi_p, color)
            ax.scatter(
                [x_lo, x_hi], [y_lo_p, y_hi_p],
                s=float(CH3_LIK_PARTIAL_SECANT_POINT_S), c=[str(color)], zorder=7, alpha=0.9,
            )
            _ch4_partial_annotate_panel(
                ax, which, x_lo, x_hi, y_lo_p, y_hi_p, color, label_fs=label_fs,
            )
        if show_vectors and pi < v_show:
            _ch4_partial_draw_gradient_vector(
                ax, xc, yc, grads[which], plot_half_w, color,
                grad_ref=grad_ref, alpha=float(vector_alpha),
                vector_axis_scale=vector_axis_scale,
            )
            if neg_vectors:
                _ch4_partial_draw_gradient_vector(
                    ax, xc, yc, -grads[which], plot_half_w, color,
                    grad_ref=grad_ref, alpha=float(neg_vector_alpha),
                    vector_axis_scale=vector_axis_scale,
                )
        ax.tick_params(labelsize=FONT_SIZE * 0.82)
        for spine in ax.spines.values():
            spine.set_linewidth(0.9)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    grad_colors = ch4_we_are_here_grad_line_colors()
    if right_blocks is None:
        right = ch4_we_are_here_roc_blocks(
            ws, we, bb, nll0, rocs,
            point_color=CH3_LIK_3D_POINT_COLOR,
            show_partials=show_partials,
            show_neg_partials=show_neg_partials,
            partials=partials,
            grad_line_colors=grad_colors,
            partial_lines_show=partial_lines_show,
            show_alpha=show_alpha,
            step_size=step_size,
        )
    else:
        right = right_blocks
    if bottom_blocks is None:
        from ch4_layout import ch4_cached_formula_blocks_3d_story
        bottom = ch4_cached_formula_blocks_3d_story()
    else:
        bottom = bottom_blocks
    return compose_tutorial(
        plot_img,
        right_blocks=right,
        bottom_blocks=bottom,
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_HERE_SECTION_TITLE,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        right_title_color=CH3_LIK_3D_POINT_COLOR,
        write_progress=write_progress,
        theme="classic_light",
        progress_override=progress_override,
        rails_cache_key=rails_cache_key,
        shell_cache_key=shell_cache_key,
    )


def _ch3_lik_partial_gd_start_pack():
    """Shared pose at ``CH3_LIK_3D_PATH_START`` (ch4_05 handoff end / ch4_06 start)."""
    pack = _ch3_lik_3d_measurements_pack(ball_colormap_limits=True)
    ws = float(CH3_LIK_3D_PATH_START[0])
    we = float(CH3_LIK_3D_PATH_START[1])
    bb = float(CH3_LIK_3D_PATH_START[2])
    pack["partial_ws"] = ws
    pack["partial_we"] = we
    pack["partial_bb"] = bb
    _ch4_partial_warm_curve_cache(pack)
    return pack


def _ch3_lik_emit_partial_frame(
    pack, *,
    half_w,
    plot_half_w=None,
    curve_u=1.0,
    show_secant=True,
    show_vectors=False,
    vector_panels_show=3,
    vector_alpha=1.0,
    neg_vectors=False,
    neg_vector_alpha=0.95,
    rocs=None,
    show_partials=False,
    show_neg_partials=False,
    partials=None,
    partial_lines_show=3,
    show_alpha=False,
    step_size=None,
    bottom_blocks=None,
    bottom_prog=None,
    right_blocks=None,
    right_prog=None,
    gd_bottom=False,
    ws=None,
    we=None,
    bb=None,
    ghost_ws=None,
    ghost_we=None,
    ghost_bb=None,
    active_param=None,
    ghost_trail_panels=None,
    panel_axes_centered=False,
    panel_y_half=None,
    vector_axis_scale=False,
):
    from ch4_layout import ch4_bottom_per_block_progress

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"] if ws is None else ws)
    we = float(pack["partial_we"] if we is None else we)
    bb = float(pack["partial_bb"] if bb is None else bb)
    plot_hw = float(CH3_LIK_PARTIAL_HALF_W_START if plot_half_w is None else plot_half_w)
    cached = pack.get("partial_curve_cache", {}).get(plot_hw)
    cache_pose = pack.get("partial_curve_pose")
    curve_cache = None
    if cached is not None and cache_pose is not None:
        if not _ch4_partial_pose_changed(ws, we, bb, *cache_pose):
            curve_cache = cached
    rails_key, shell_key = _ch4_partial_rails_keys(
        gd_bottom=gd_bottom, bottom_prog=bottom_prog,
    )
    po = {}
    if bottom_blocks and bottom_prog and not _ch4_partial_bottom_fully_written(bottom_prog):
        po["bottom"] = ch4_bottom_per_block_progress(bottom_blocks, bottom_prog)
    if right_prog is not None:
        po["right"] = right_prog
    return ch3_frame_lik_partial_motivation(
        study, exam, y, ws, we, bb,
        half_w=half_w,
        plot_half_w=plot_half_w,
        curve_u=curve_u,
        show_secant=show_secant,
        show_vectors=show_vectors,
        vector_panels_show=vector_panels_show,
        vector_alpha=vector_alpha,
        neg_vectors=neg_vectors,
        neg_vector_alpha=neg_vector_alpha,
        rocs=rocs,
        show_partials=show_partials,
        show_neg_partials=show_neg_partials,
        partials=partials,
        partial_lines_show=partial_lines_show,
        show_alpha=show_alpha,
        step_size=step_size,
        bottom_blocks=bottom_blocks,
        right_blocks=right_blocks,
        progress_override=po if po else None,
        write_progress=1.0,
        curve_cache=curve_cache,
        ghost_ws=ghost_ws,
        ghost_we=ghost_we,
        ghost_bb=ghost_bb,
        active_param=active_param,
        ghost_trail_panels=ghost_trail_panels,
        panel_axes_centered=panel_axes_centered,
        panel_y_half=panel_y_half,
        vector_axis_scale=vector_axis_scale,
        rails_cache_key=rails_key,
        shell_cache_key=shell_key,
        gd_bottom=gd_bottom,
    )


def _ch3_lik_partial_render_frame(pack, spec, *, cam_azim_u=0.0):
    """Worker entry for fast parallel export."""
    return _ch3_lik_emit_partial_frame(pack, **spec)


def _ch4_partial_pose_with_delta(ws0, we0, bb0, which, delta):
    ws, we, bb = float(ws0), float(we0), float(bb0)
    d = float(delta)
    if which == "st":
        ws += d
    elif which == "el":
        we += d
    else:
        bb += d
    return ws, we, bb


def _ch4_partial_grad_tuple(study, exam, y, ws, we, bb):
    return _ch3_nll_sum_grad_at_point(study, exam, y, float(ws), float(we), float(bb))


def _ch4_partial_gd_demo_end_pose(pack):
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"])
    we = float(pack["partial_we"])
    bb = float(pack["partial_bb"])
    eta = float(CH3_LIK_GD_STEP)
    for _ in range(int(CH3_LIK_PARTIAL_N_GD_DEMO_STEPS)):
        g1, g2, gb = _ch4_partial_grad_tuple(study, exam, y, ws, we, bb)
        ws -= eta * g1
        we -= eta * g2
        bb -= eta * gb
    return ws, we, bb


def ch3_partial_motivation_build_specs(pack, *, part=1):
    """Frame specs for partial-motivation phases. ``part`` 1|2|3 splits the export."""
    from ch4_layout import ch4_formula_blocks_3d_story, ch4_formula_blocks_gd_progressive

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws0 = float(pack["partial_ws"])
    we0 = float(pack["partial_we"])
    bb0 = float(pack["partial_bb"])
    g1, g2, gb = _ch4_partial_grad_tuple(study, exam, y, ws0, we0, bb0)
    partials = (g1, g2, gb)
    half_w0 = float(CH3_LIK_PARTIAL_HALF_W_START)
    half_w1 = float(CH3_LIK_PARTIAL_HALF_W_END)
    roc_hw = float(CH3_LIK_PARTIAL_ROC_WIDE_HALF_W)
    bottom_3d = ch4_formula_blocks_3d_story()
    bottom_partial_full = ch4_formula_blocks_gd_progressive(n_grad_lines=3)
    specs = []

    def _spec(**kw):
        specs.append(kw)

    def _vec_spec(**extra):
        base = dict(
            half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=True,
            vector_panels_show=3, rocs=partials, show_partials=True, partials=partials,
            partial_lines_show=3,
            bottom_blocks=bottom_partial_full,
            bottom_prog={0: 1.0, 1: 1.0},
            gd_bottom=True,
        )
        base.update(extra)
        _spec(**base)

    def _vec_at(ws, we, bb, *, ghost_ws=None, ghost_we=None, ghost_bb=None, active_param=None, **extra):
        pt = _ch4_partial_grad_tuple(study, exam, y, ws, we, bb)
        _vec_spec(
            ws=ws, we=we, bb=bb, rocs=pt, partials=pt,
            ghost_ws=ghost_ws, ghost_we=ghost_we, ghost_bb=ghost_bb,
            active_param=active_param,
            **extra,
        )

    def _bump_param(which, delta_target):
        for seg in ("out", "back"):
            for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_PARAM_BUMP, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                if seg == "out":
                    d = float(delta_target) * u
                else:
                    d = float(delta_target) * (1.0 - u)
                ws, we, bb = _ch4_partial_pose_with_delta(ws0, we0, bb0, which, d)
                _vec_at(
                    ws, we, bb,
                    ghost_ws=ws0, ghost_we=we0, ghost_bb=bb0,
                    active_param=which,
                    vector_panels_show=3, partial_lines_show=3,
                )

    if int(part) == 1:
        for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_DRAW, endpoint=True):
            _spec(
                half_w=half_w0, curve_u=ch3_knob_smoothstep(float(tv)), show_secant=False,
                bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
            )

        for _ in range(CH3_LIK_PARTIAL_N_HOLD):
            _spec(
                half_w=half_w0, curve_u=1.0, show_secant=False,
                bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
            )

        rocs_wide = (
            _ch4_nll_avg_roc("st", ws0, we0, bb0, roc_hw, study, exam, y),
            _ch4_nll_avg_roc("el", ws0, we0, bb0, roc_hw, study, exam, y),
            _ch4_nll_avg_roc("b", ws0, we0, bb0, roc_hw, study, exam, y),
        )
        for _ in range(CH3_LIK_PARTIAL_N_ROC_INTRO):
            _spec(
                half_w=roc_hw, plot_half_w=half_w0, curve_u=1.0, show_secant=True, rocs=rocs_wide,
                bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
            )

        for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_ROC_SHRINK, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            hw = roc_hw + u * (half_w1 - roc_hw)
            rocs = (
                _ch4_nll_avg_roc("st", ws0, we0, bb0, hw, study, exam, y),
                _ch4_nll_avg_roc("el", ws0, we0, bb0, hw, study, exam, y),
                _ch4_nll_avg_roc("b", ws0, we0, bb0, hw, study, exam, y),
            )
            _spec(
                half_w=hw, curve_u=1.0, show_secant=True, show_vectors=False, rocs=rocs,
                bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
            )

        for _ in range(max(6, CH3_SCRIPT_N_HOLD // 5)):
            _spec(
                half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=False,
                rocs=rocs_wide, show_partials=False,
                bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=False,
            )

        for tv in np.linspace(1.0, 0.0, CH3_LIK_PARTIAL_N_ERASE, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            _spec(
                half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=False,
                rocs=rocs_wide, show_partials=False,
                bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: u}, gd_bottom=False,
            )

        for n_p in (1, 2, 3):
            bottom = ch4_formula_blocks_gd_progressive(n_grad_lines=n_p)
            for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_PARTIAL, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                _spec(
                    half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=True,
                    vector_panels_show=n_p, rocs=partials, show_partials=True, partials=partials,
                    partial_lines_show=n_p,
                    bottom_blocks=bottom,
                    bottom_prog={0: 1.0, 1: u},
                    gd_bottom=True,
                )

    elif int(part) == 2:
        _bump_param("st", float(CH3_LIK_PARTIAL_PARAM_BUMP))
        _bump_param("el", float(CH3_LIK_PARTIAL_PARAM_BUMP))
        for _ in range(CH3_LIK_PARTIAL_N_PARAM_PAUSE):
            _vec_at(ws0, we0, bb0, vector_panels_show=3, partial_lines_show=3)
        _bump_param("st", -float(CH3_LIK_PARTIAL_PARAM_BUMP))

        for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_NEG_REVEAL, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            dim = float(CH3_LIK_PARTIAL_VECTOR_DIM_ALPHA)
            _vec_at(
                ws0, we0, bb0,
                vector_panels_show=3,
                vector_alpha=1.0 - u * (1.0 - dim),
                neg_vectors=u >= 0.35,
                neg_vector_alpha=float(np.clip(u, 0.0, 1.0)),
                show_neg_partials=u >= 0.35,
                partial_lines_show=3,
            )

        pos = [ws0, we0, bb0]
        eta = float(CH3_LIK_GD_STEP)
        for _ in range(int(CH3_LIK_PARTIAL_N_GD_DEMO_STEPS)):
            g = _ch4_partial_grad_tuple(study, exam, y, pos[0], pos[1], pos[2])
            target = (
                pos[0] - eta * g[0],
                pos[1] - eta * g[1],
                pos[2] - eta * g[2],
            )
            ghost_pos = list(pos)
            for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_GD_DEMO, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                ws = pos[0] + u * (target[0] - pos[0])
                we = pos[1] + u * (target[1] - pos[1])
                bb = pos[2] + u * (target[2] - pos[2])
                _vec_at(
                    ws, we, bb,
                    ghost_ws=ghost_pos[0], ghost_we=ghost_pos[1], ghost_bb=ghost_pos[2],
                    vector_panels_show=3,
                    vector_alpha=dim,
                    neg_vectors=True,
                    neg_vector_alpha=0.95,
                    show_neg_partials=True,
                    partial_lines_show=3,
                )
            pos = list(target)
        pack["partial_ws"] = float(pos[0])
        pack["partial_we"] = float(pos[1])
        pack["partial_bb"] = float(pos[2])

    elif int(part) == 3:
        ws, we, bb = _ch4_partial_gd_demo_end_pose(pack)
        pt = _ch4_partial_grad_tuple(study, exam, y, ws, we, bb)
        dim = float(CH3_LIK_PARTIAL_VECTOR_DIM_ALPHA)
        base_vec = dict(
            half_w=half_w1, curve_u=1.0, show_secant=False, show_vectors=True,
            vector_panels_show=3, rocs=pt, show_partials=True, partials=pt,
            partial_lines_show=3,
            vector_alpha=dim, neg_vectors=True, neg_vector_alpha=0.95,
            show_neg_partials=True, gd_bottom=True,
            ws=ws, we=we, bb=bb,
        )

        for n_u in (1, 2, 3):
            bottom = ch4_formula_blocks_gd_progressive(n_grad_lines=3, n_update_lines=n_u)
            for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_UPDATE_REVEAL, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                blk = 2 if n_u > 0 else 1
                _spec(
                    **base_vec,
                    bottom_blocks=bottom,
                    bottom_prog={0: 1.0, 1: 1.0, blk: u},
                )

        for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_ALPHA, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            bottom = ch4_formula_blocks_gd_progressive(n_grad_lines=3, n_update_lines=3)
            _spec(
                **base_vec,
                show_alpha=u >= 0.5,
                step_size=CH3_LIK_GD_STEP,
                bottom_blocks=bottom,
                bottom_prog={0: 1.0, 1: 1.0, 2: 1.0},
            )

        bottom = ch4_formula_blocks_gd_progressive(n_grad_lines=3, n_update_lines=3)
        for _ in range(max(8, CH3_SCRIPT_N_HOLD // 4)):
            _spec(
                **base_vec,
                show_alpha=True,
                step_size=CH3_LIK_GD_STEP,
                bottom_blocks=bottom,
                bottom_prog={0: 1.0, 1: 1.0, 2: 1.0},
            )

    pack["partial_tail"] = {"partials": partials, "half_w1": half_w1}
    return specs


def _ch3_build_partial_motivation_plot_frames(pack, specs, *, parallel=None, label="ch4_05b partial"):
    from ch4_export_pipeline import build_tutorial_frames

    return build_tutorial_frames(
        pack,
        specs,
        _ch3_lik_partial_render_frame,
        prewarm="partial",
        parallel=parallel,
        progress_label=label,
        render_fn_name="_ch3_lik_partial_render_frame",
    )


def ch3_build_frames_likelihood_partial_motivation_story(*, parallel=None, part=None):
    """Motivate partial derivatives: 1-D NLL slices → ∂ demo → GD formulas → 3D."""
    from ch4_export_pipeline import build_tutorial_frames

    pack = _ch3_lik_partial_gd_start_pack()
    if part is not None:
        specs = ch3_partial_motivation_build_specs(pack, part=int(part))
        return _ch3_build_partial_motivation_plot_frames(
            pack, specs, parallel=parallel, label=f"ch4_05b part{int(part)}",
        )

    gd_pack = _ch3_lik_3d_gd_pack()
    all_specs = []
    for p in (1, 2, 3):
        all_specs.extend(ch3_partial_motivation_build_specs(pack, part=p))
    tail = CH3_LIK_PARTIAL_N_PLOT + max(8, CH3_SCRIPT_N_HOLD // 4)
    mode = "draft" if _CH3_DRAFT else "full"
    print(
        f"ch4_05b: {len(all_specs)} plot frames + {tail} crossfade/hold ({mode}); "
        f"set CH3_DRAFT_EXPORT=1 for a quick preview",
        flush=True,
    )

    frames = _ch3_build_partial_motivation_plot_frames(pack, all_specs, parallel=parallel)

    frame_gd_open = _ch3_lik_gd_render_frame(
        gd_pack, _ch3_lik_gd_frame_specs(gd_pack)[0], cam_azim_u=0.0,
    )
    frame_plot_from = frames[-1]
    for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_PLOT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_plot_from, frame_gd_open, u))

    for _ in range(max(8, CH3_SCRIPT_N_HOLD // 4)):
        frames.append(frame_gd_open.copy())
    return frames


def ch3_build_frames_likelihood_partial_motivation_part1(*, parallel=None):
    return ch3_build_frames_likelihood_partial_motivation_story(parallel=parallel, part=1)


def ch3_build_frames_likelihood_partial_motivation_part2(*, parallel=None):
    return ch3_build_frames_likelihood_partial_motivation_story(parallel=parallel, part=2)


def ch3_build_frames_likelihood_partial_motivation_part3(*, parallel=None):
    pack = _ch3_lik_partial_gd_start_pack()
    specs = ch3_partial_motivation_build_specs(pack, part=3)
    frames = _ch3_build_partial_motivation_plot_frames(
        pack, specs, parallel=parallel, label="ch4_05b part3",
    )
    gd_pack = _ch3_lik_3d_gd_pack()
    frame_gd_open = _ch3_lik_gd_render_frame(
        gd_pack, _ch3_lik_gd_frame_specs(gd_pack)[0], cam_azim_u=0.0,
    )
    frame_plot_from = frames[-1]
    for tv in np.linspace(0.0, 1.0, CH3_LIK_PARTIAL_N_PLOT, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_plot_from, frame_gd_open, u))
    for _ in range(max(8, CH3_SCRIPT_N_HOLD // 4)):
        frames.append(frame_gd_open.copy())
    return frames


def ch4_preview_likelihood_partial_motivation_frame(*, phase="roc_wide"):
    """Preview one partial-motivation frame."""
    from ch4_layout import ch4_formula_blocks_3d_story, ch4_formula_blocks_gd_story

    pack = _ch3_lik_partial_gd_start_pack()
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["partial_ws"])
    we = float(pack["partial_we"])
    bb = float(pack["partial_bb"])
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    partials = (g1, g2, gb)
    half_w0 = float(CH3_LIK_PARTIAL_HALF_W_START)
    half_w1 = float(CH3_LIK_PARTIAL_HALF_W_END)
    roc_hw = float(CH3_LIK_PARTIAL_ROC_WIDE_HALF_W)
    bottom_3d = ch4_formula_blocks_3d_story()
    bottom_gd = ch4_formula_blocks_gd_story()
    if phase == "draw":
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w0, curve_u=0.45, show_secant=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "roc_wide":
        rocs = (
            _ch4_nll_avg_roc("st", ws, we, bb, roc_hw, study, exam, y),
            _ch4_nll_avg_roc("el", ws, we, bb, roc_hw, study, exam, y),
            _ch4_nll_avg_roc("b", ws, we, bb, roc_hw, study, exam, y),
        )
        return _ch3_lik_emit_partial_frame(
            pack, half_w=roc_hw, plot_half_w=half_w0, curve_u=1.0, show_secant=True, rocs=rocs,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "roc_tight":
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_secant=True,
            rocs=partials, show_partials=True, partials=partials,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "vectors":
        rocs = (
            _ch4_nll_avg_roc("st", ws, we, bb, half_w1, study, exam, y),
            _ch4_nll_avg_roc("el", ws, we, bb, half_w1, study, exam, y),
            _ch4_nll_avg_roc("b", ws, we, bb, half_w1, study, exam, y),
        )
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_vectors=True,
            rocs=rocs, show_partials=False,
            bottom_blocks=bottom_3d, bottom_prog={0: 1.0, 1: 1.0},
        )
    if phase == "mid_grad":
        from ch4_layout import ch4_formula_blocks_gd_progressive
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_vectors=True,
            vector_panels_show=2,
            rocs=partials, show_partials=True, partials=partials, partial_lines_show=2,
            bottom_blocks=ch4_formula_blocks_gd_progressive(n_grad_lines=2),
            bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=True,
        )
    if phase == "param_bump":
        ws1, we1, bb1 = _ch4_partial_pose_with_delta(ws, we, bb, "st", 0.1)
        pt = _ch4_partial_grad_tuple(study, exam, y, ws1, we1, bb1)
        from ch4_layout import ch4_formula_blocks_gd_progressive
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_vectors=True, vector_panels_show=3,
            ws=ws1, we=we1, bb=bb1,
            ghost_ws=ws, ghost_we=we, ghost_bb=bb,
            active_param="st",
            rocs=pt, show_partials=True, partials=pt,
            partial_lines_show=3,
            bottom_blocks=ch4_formula_blocks_gd_progressive(n_grad_lines=3),
            bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=True,
        )
    if phase == "neg_grad":
        from ch4_layout import ch4_formula_blocks_gd_progressive
        return _ch3_lik_emit_partial_frame(
            pack, half_w=half_w1, curve_u=1.0, show_vectors=True, vector_panels_show=3,
            rocs=partials, show_partials=True, partials=partials, partial_lines_show=3,
            vector_alpha=CH3_LIK_PARTIAL_VECTOR_DIM_ALPHA, neg_vectors=True,
            show_neg_partials=True,
            bottom_blocks=ch4_formula_blocks_gd_progressive(n_grad_lines=3),
            bottom_prog={0: 1.0, 1: 1.0}, gd_bottom=True,
        )
    if phase == "gd_open":
        gd_pack = _ch3_lik_3d_gd_pack()
        return _ch3_lik_gd_render_frame(gd_pack, _ch3_lik_gd_frame_specs(gd_pack)[0], cam_azim_u=0.0)
    gd_pack = _ch3_lik_3d_gd_pack()
    return _ch3_lik_gd_render_frame(gd_pack, _ch3_lik_gd_frame_specs(gd_pack)[0], cam_azim_u=0.0)


def _ch4_export_partial_motivation_part(frames, fn):
    save_mp4(frames, fn, duration=int(CH3_LIK_PARTIAL_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


def ch4_export_likelihood_partial_motivation_part1(*, parallel=None):
    frames = ch3_build_frames_likelihood_partial_motivation_part1(parallel=parallel)
    return _ch4_export_partial_motivation_part(frames, "ch4_05b_partial_slices_partials.mp4")


def ch4_export_likelihood_partial_motivation_part2(*, parallel=None):
    frames = ch3_build_frames_likelihood_partial_motivation_part2(parallel=parallel)
    return _ch4_export_partial_motivation_part(frames, "ch4_05b_partial_param_demo.mp4")


def ch4_export_likelihood_partial_motivation_part3(*, parallel=None):
    frames = ch3_build_frames_likelihood_partial_motivation_part3(parallel=parallel)
    return _ch4_export_partial_motivation_part(frames, "ch4_05b_partial_gd_formulas.mp4")


def ch4_export_likelihood_partial_motivation(*, parallel=None):
    """Export full ch4_05b. Split parts: part1/2/3 for faster iteration."""
    frames = ch3_build_frames_likelihood_partial_motivation_story(parallel=parallel)
    return _ch4_export_partial_motivation_part(frames, "ch4_05b_partial_derivative_motivation.mp4")


# --- ch4_08 / 08b / 07b: Newton probe ghosts + path trails from ch4_07 combined frame ---

CH3_LIK_NEWTON_N_CYCLES = 5
CH3_LIK_NEWTON_N_PROBE = 8 if _CH3_DRAFT else 44
CH3_LIK_NEWTON_PROBE_SCALE = 0.35
CH3_LIK_NEWTON_PROBE_REF_SPAN = 8.0  # ±4 reference cube (display scaling)
CH3_LIK_NEWTON_N_BOX_GHOSTS = 14  # 6 face centers + 8 cube corners (no center)
CH3_LIK_NEWTON_GHOST_ALPHA = 0.72
CH3_LIK_NEWTON_N_STEP = 4 if _CH3_DRAFT else 14
CH3_LIK_NEWTON_N_HOLD = 4 if _CH3_DRAFT else 12
CH3_LIK_NEWTON_DAMP = 0.10
CH3_LIK_NEWTON_MS = 130 if not _CH3_DRAFT else 150
CH3_LIK_NEWTON_ARROW_COLOR = "#7e57c2"
CH3_LIK_NEWTON_OPEN_HOLD = 4 if _CH3_DRAFT else 12
CH3_LIK_GD_COMBINED_PATH_N_ROT_AFTER = 8 if _CH3_DRAFT else 40


def _ch3_nll_hessian_3(w_st, w_el, b, study, exam, y, *, h=3e-4):
    """3×3 Hessian of NLL at (w_ST, w_EL, b) via central differences."""
    theta = np.array([float(w_st), float(w_el), float(b)], dtype=np.float64)

    def nll(t):
        return float(-loss_log_likelihood(t[0], t[1], t[2], study, exam, y))

    H = np.zeros((3, 3), dtype=np.float64)
    for i in range(3):
        for j in range(i, 3):
            ei = np.zeros(3)
            ej = np.zeros(3)
            ei[i] = h
            ej[j] = h
            hij = (
                nll(theta + ei + ej) - nll(theta + ei - ej)
                - nll(theta - ei + ej) + nll(theta - ei - ej)
            ) / (4.0 * h * h)
            H[i, j] = hij
            H[j, i] = hij
    return H


def _ch3_nll_newton_step(study, exam, y, ws, we, bb, *, damp=None):
    """Damped Newton step for NLL minimization: Δθ = H⁻¹ ∇NLL."""
    damp = float(CH3_LIK_NEWTON_DAMP if damp is None else damp)
    g = np.array(
        _ch3_nll_sum_grad_at_point(study, exam, y, float(ws), float(we), float(bb)),
        dtype=np.float64,
    )
    H = _ch3_nll_hessian_3(ws, we, bb, study, exam, y)
    d = damp * float(np.trace(H) / 3.0 + 1e-6)
    try:
        step = np.linalg.solve(H + d * np.eye(3), g)
    except np.linalg.LinAlgError:
        step = g / (float(np.linalg.norm(g)) + 1e-9)
    return float(step[0]), float(step[1]), float(step[2])


def _ch3_lik_combined_hold_spec(pack):
    """Single frame spec matching ch4_07 after the split→combined transition."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws, we, bb = (float(pack["gd_start"][0]), float(pack["gd_start"][1]), float(pack["gd_start"][2]))
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    return {
        "ws": ws, "we": we, "bb": bb,
        "grad": (float(g1), float(g2), float(gb)),
        "eta": float(pack["gd_eta"]),
        "arrows": (True, True, True),
        "bold": None,
        "bold_all": False,
        "grad_red": True,
        "arrow_mode": "combined",
        "transition_u": 0.0,
    }


def _ch3_lik_newton_box_probe_offsets(*, bounds=None):
    """Fixed probe sites around the iterate: ±s on each axis + all 8 diagonal corners."""
    s = float(CH3_LIK_NEWTON_PROBE_SCALE)
    if bounds is not None:
        span = float(np.max(_ch3_lik_bounds_span(bounds)))
        s *= span / max(float(CH3_LIK_NEWTON_PROBE_REF_SPAN), 1e-9)
    axes = [
        (s, 0.0, 0.0), (-s, 0.0, 0.0),
        (0.0, s, 0.0), (0.0, -s, 0.0),
        (0.0, 0.0, s), (0.0, 0.0, -s),
    ]
    corners = [
        (sx, sy, sz)
        for sx in (-s, s) for sy in (-s, s) for sz in (-s, s)
    ]
    out = np.asarray(axes + corners, dtype=np.float64)
    assert out.shape == (int(CH3_LIK_NEWTON_N_BOX_GHOSTS), 3)
    return out


def _ch3_lik_newton_box_probe_sites(study, exam, y, ws, we, bb, *, bounds=None):
    """Precompute all box probe positions and ∇NLL (once per Newton iterate)."""
    sites = []
    for off in _ch3_lik_newton_box_probe_offsets(bounds=bounds):
        pws = float(ws) + float(off[0])
        pwe = float(we) + float(off[1])
        pbb = float(bb) + float(off[2])
        pg = _ch3_nll_sum_grad_at_point(study, exam, y, pws, pwe, pbb)
        sites.append({
            "ws": pws, "we": pwe, "bb": pbb,
            "grad": (float(pg[0]), float(pg[1]), float(pg[2])),
        })
    return sites


def _ch3_lik_newton_ghosts_reveal(sites, *, alpha=0.40, reveal_u=1.0, partial_vis=None):
    """Build ghost list from precomputed probe sites; only alpha/reveal varies per frame."""
    n_sites = len(sites)
    ghosts = []
    for oi, site in enumerate(sites):
        site_u = ch3_knob_smoothstep(float(np.clip(float(reveal_u) * n_sites - oi, 0.0, 1.0)))
        if site_u < 0.04:
            continue
        g = {
            "ws": site["ws"], "we": site["we"], "bb": site["bb"],
            "grad": site["grad"],
            "alpha": float(alpha) * site_u,
        }
        if partial_vis is not None:
            g["arrow_mode"] = "split"
            g["arrows"] = partial_vis
        ghosts.append(g)
    return ghosts


def _ch3_lik_newton_box_ghosts_at(study, exam, y, ws, we, bb, *, alpha=0.40):
    """All 14 box probe ghosts with gradients at fixed offsets from ``(ws, we, bb)``."""
    sites = _ch3_lik_newton_box_probe_sites(study, exam, y, ws, we, bb)
    return _ch3_lik_newton_ghosts_reveal(sites, alpha=float(alpha), reveal_u=1.0)


def _ch3_lik_newton_base_spec(pack, ws, we, bb, grad, *, ghosts=None, path_trail=None, **extra):
    spec = {
        "ws": float(ws), "we": float(we), "bb": float(bb),
        "grad": tuple(float(v) for v in grad),
        "eta": float(pack["gd_eta"]),
        "arrows": (True, True, True),
        "bold": None,
        "bold_all": False,
        "grad_red": True,
        "arrow_mode": "combined",
        "transition_u": 0.0,
        "ghosts": list(ghosts) if ghosts else None,
        "newton_arrow_color": CH3_LIK_NEWTON_ARROW_COLOR,
        "newton_formulas": True,
        "here_grad_mode": "partial",
    }
    if path_trail is not None and len(path_trail) >= 2:
        spec["path_trail"] = np.asarray(path_trail, dtype=float)
    spec.update(extra)
    return spec


def _ch3_lik_newton_build_trajectory(pack):
    """Precompute Newton iterates + probe ∇NLL for ch4_08/08b (cached on ``pack``)."""
    cached = pack.get("_newton_trajectory")
    if cached is not None:
        return cached
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["gd_start"][0])
    we = float(pack["gd_start"][1])
    bb = float(pack["gd_start"][2])
    cycles = []
    for _ in range(int(CH3_LIK_NEWTON_N_CYCLES)):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (float(g1), float(g2), float(gb))
        sites = _ch3_lik_newton_box_probe_sites(
            study, exam, y, ws, we, bb,
            bounds=pack.get("gd_view_bounds") or pack.get("gd_view_bounds_end"),
        )
        dw, de, db = _ch3_nll_newton_step(study, exam, y, ws, we, bb)
        ws1, we1, bb1 = float(ws - dw), float(we - de), float(bb - db)
        newton_vec = (ws1 - ws, we1 - we, bb1 - bb)
        newton_step = (float(dw), float(de), float(db))
        g1p, g2p, gbp = _ch3_nll_sum_grad_at_point(study, exam, y, ws1, we1, bb1)
        cycles.append({
            "ws": float(ws), "we": float(we), "bb": float(bb),
            "grad": grad,
            "sites": sites,
            "full_ghosts": _ch3_lik_newton_ghosts_reveal(sites, alpha=0.40, reveal_u=1.0),
            "newton_vec": newton_vec,
            "newton_step": newton_step,
            "ws1": ws1, "we1": we1, "bb1": bb1,
            "post_grad": (float(g1p), float(g2p), float(gbp)),
        })
        ws, we, bb = ws1, we1, bb1
    traj = {
        "start": (
            float(pack["gd_start"][0]),
            float(pack["gd_start"][1]),
            float(pack["gd_start"][2]),
        ),
        "cycles": cycles,
    }
    pack["_newton_trajectory"] = traj
    return traj


def _ch3_lik_newton_ghost_specs_from_trajectory(pack, trajectory, *, track_path=False):
    """Expand precomputed Newton trajectory into ch4_08 frame specs."""
    specs = []
    hold_n = max(int(CH3_LIK_NEWTON_N_HOLD), 1)
    open_spec = _ch3_lik_combined_hold_spec(pack)
    open_spec["newton_formulas"] = True
    open_spec["here_grad_mode"] = "partial"
    for _ in range(max(int(CH3_LIK_NEWTON_OPEN_HOLD), 1)):
        specs.append(dict(open_spec))

    ws0, we0, bb0 = trajectory["start"]
    path_trail = [[ws0, we0, bb0]] if track_path else None
    ghost_cache = {}

    for cycle in trajectory["cycles"]:
        ws, we, bb = cycle["ws"], cycle["we"], cycle["bb"]
        grad = cycle["grad"]
        sites = cycle["sites"]
        full_ghosts = cycle["full_ghosts"]
        newton_vec = cycle["newton_vec"]
        newton_step = cycle["newton_step"]
        ws1, we1, bb1 = cycle["ws1"], cycle["we1"], cycle["bb1"]
        post_grad = cycle["post_grad"]
        newton_display = {"newton_step_display": newton_step}

        for tv in np.linspace(0.0, 1.0, CH3_LIK_NEWTON_N_PROBE, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            key = (round(float(u), 5),)
            if key not in ghost_cache:
                ghost_cache[key] = _ch3_lik_newton_ghosts_reveal(sites, alpha=0.40, reveal_u=u)
            ghosts = ghost_cache[key]
            specs.append(_ch3_lik_newton_base_spec(
                pack, ws, we, bb, grad,
                ghosts=list(ghosts),
                path_trail=path_trail,
                arrow_mode="none" if ghosts else "combined",
            ))

        probe_hold = None
        for _ in range(max(hold_n // 2, 1)):
            if probe_hold is None:
                probe_hold = _ch3_lik_newton_base_spec(
                    pack, ws, we, bb, grad,
                    ghosts=list(full_ghosts),
                    path_trail=path_trail,
                    newton_step_vec=newton_vec,
                    arrow_mode="combined",
                    here_grad_mode="newton",
                    **newton_display,
                )
            specs.append(probe_hold)

        step_n = max(int(CH3_LIK_NEWTON_N_STEP), 2)
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            wi = ws + u * (ws1 - ws)
            ei = we + u * (we1 - we)
            bi = bb + u * (bb1 - bb)
            specs.append(_ch3_lik_newton_base_spec(
                pack, wi, ei, bi, grad,
                ghosts=list(full_ghosts),
                path_trail=path_trail,
                newton_step_vec=((1.0 - u) * newton_vec[0], (1.0 - u) * newton_vec[1], (1.0 - u) * newton_vec[2]),
                arrow_mode="none",
                here_grad_mode="newton",
                **newton_display,
            ))

        if track_path:
            path_trail.append([ws1, we1, bb1])

        post_hold = None
        for _ in range(hold_n):
            if post_hold is None:
                post_hold = _ch3_lik_newton_base_spec(
                    pack, ws1, we1, bb1, post_grad,
                    ghosts=None,
                    path_trail=path_trail,
                )
            specs.append(post_hold)

    return specs


def _ch3_lik_newton_ghost_frame_specs(pack, *, track_path=False):
    """Probe ghosts + damped Newton steps (5×), opening from ch4_07 combined hold."""
    trajectory = _ch3_lik_newton_build_trajectory(pack)
    return _ch3_lik_newton_ghost_specs_from_trajectory(pack, trajectory, track_path=bool(track_path))


def _ch3_lik_gd_combined_path_frame_specs(pack):
    """Like ch4_07 but trail follows GD iter endpoints from ``pack['path']`` (+ step tip while moving)."""
    base = _ch3_lik_gd_combined_frame_specs(pack)
    gd_path = np.asarray(pack["path"], dtype=float)
    out = []
    settled_i = 0  # index into gd_path for the last completed GD position
    for i, s in enumerate(base):
        s2 = dict(s)
        mode = str(s.get("arrow_mode"))
        pos = np.array([float(s["ws"]), float(s["we"]), float(s["bb"])], dtype=float)
        if mode == "none" and s.get("bold_all") and settled_i + 1 < len(gd_path):
            trail = np.vstack([gd_path[: settled_i + 1], pos])
        else:
            trail = gd_path[: settled_i + 1]
        if len(trail) >= 2:
            s2["path_trail"] = trail
        if (
            mode == "none"
            and s.get("bold_all")
            and i + 1 < len(base)
            and str(base[i + 1].get("arrow_mode")) == "combined"
        ):
            settled_i = min(settled_i + 1, len(gd_path) - 1)
        out.append(s2)
    return out


def _ch3_lik_gd_combined_path_end_spec(pack):
    """Canonical ch4_07b end frame: final GD pose, full path trail, red combined arrow."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    path = np.asarray(pack["path"], dtype=float)
    ws, we, bb = float(path[-1, 0]), float(path[-1, 1]), float(path[-1, 2])
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
    spec = {
        "ws": ws, "we": we, "bb": bb,
        "grad": (float(g1), float(g2), float(gb)),
        "eta": float(pack["gd_eta"]),
        "arrows": (True, True, True),
        "bold": None,
        "bold_all": False,
        "grad_red": True,
        "arrow_mode": "combined",
        "transition_u": 0.0,
    }
    if len(path) >= 2:
        spec["path_trail"] = path
    return spec


def _ch3_lik_gd_combined_path_append_rot_after_specs(specs):
    """360° pan after the GD path completes (07b rot-after variant)."""
    if not specs:
        return specs
    last = dict(specs[-1])
    for tv in np.linspace(0.0, 1.0, CH3_LIK_GD_COMBINED_PATH_N_ROT_AFTER, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frame = dict(last)
        frame["cam_azim_u"] = u
        frame["cam_rot_deg"] = float(CH3_LIK_3D_CAM_PATH_ROT)
        specs.append(frame)
    return specs


def _ch3_lik_gd_combined_path_story_specs(pack, *, rotate_during=True):
    specs = _ch3_lik_gd_combined_path_frame_specs(pack)
    if rotate_during:
        return specs
    locked = []
    for s in specs:
        s2 = dict(s)
        s2["cam_azim_u"] = 0.0
        s2["cam_rot_deg"] = 0.0
        locked.append(s2)
    return _ch3_lik_gd_combined_path_append_rot_after_specs(locked)


def ch4_preview_likelihood_3d_gd_combined_path_frame(
    *, gd_spec_index=-1, path_end=False, rotate_during=True, rot_after_end=False,
):
    """Single ch4_07b frame — path trail; ``path_end=True`` = final GD pose + full trail."""
    pack = _ch3_lik_3d_gd_combined_pack()
    if path_end:
        return _ch3_lik_gd_render_frame(
            pack, _ch3_lik_gd_combined_path_end_spec(pack), cam_azim_u=0.0,
        )
    specs = _ch3_lik_gd_combined_path_story_specs(pack, rotate_during=rotate_during)
    if rot_after_end and not rotate_during:
        idx = len(specs) - 1
    else:
        idx = int(np.clip(int(gd_spec_index), 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch4_preview_likelihood_3d_newton_ghosts_frame(*, track_path=False, cycle=0, probe_u=0.92):
    """Mid-probe frame with box ghosts (default: cycle 0, all 14 sites visible)."""
    pack = _ch3_lik_3d_gd_combined_pack()
    specs = _ch3_lik_newton_ghost_frame_specs(pack, track_path=bool(track_path))
    hold = max(int(CH3_LIK_NEWTON_OPEN_HOLD), 1)
    probe_n = max(int(CH3_LIK_NEWTON_N_PROBE), 2)
    hold_n = max(int(CH3_LIK_NEWTON_N_HOLD), 1)
    step_n = max(int(CH3_LIK_NEWTON_N_STEP), 2)
    block = probe_n + max(hold_n // 2, 1) + step_n + hold_n
    idx = hold + int(cycle) * block + int(float(probe_u) * float(probe_n - 1))
    idx = int(np.clip(idx, 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch4_export_likelihood_3d_newton_ghosts():
    from ch4_export_pipeline import export_mp4_from_specs

    pack = _ch3_lik_3d_gd_combined_pack()
    return export_mp4_from_specs(
        pack,
        _ch3_lik_newton_ghost_frame_specs(pack, track_path=False),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename="ch4_08_newton_ghosts.mp4",
        duration_ms=int(CH3_LIK_NEWTON_MS),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label="ch4_08",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


def ch4_export_likelihood_3d_newton_ghosts_path():
    from ch4_export_pipeline import export_mp4_from_specs

    pack = _ch3_lik_3d_gd_combined_pack()
    return export_mp4_from_specs(
        pack,
        _ch3_lik_newton_ghost_frame_specs(pack, track_path=True),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename="ch4_08b_newton_ghosts_path.mp4",
        duration_ms=int(CH3_LIK_NEWTON_MS),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label="ch4_08b",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


def ch4_export_likelihood_3d_gd_combined_path(*, rotate_during=True):
    from ch4_export_pipeline import export_mp4_from_specs

    pack = _ch3_lik_3d_gd_combined_pack()
    filename = (
        "ch4_07b_gd_combined_path.mp4"
        if rotate_during
        else "ch4_07b_gd_combined_path_rot_after.mp4"
    )
    progress_label = "ch4_07b" if rotate_during else "ch4_07b_rot_after"
    return export_mp4_from_specs(
        pack,
        _ch3_lik_gd_combined_path_story_specs(pack, rotate_during=rotate_during),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename=filename,
        duration_ms=int(CH3_LIK_3D_MS_GD),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label=progress_label,
        render_fn_name="_ch3_lik_gd_render_frame",
    )


def ch4_export_likelihood_3d_gd_combined_path_rot_after():
    return ch4_export_likelihood_3d_gd_combined_path(rotate_during=False)


# --- ch4_07c: 07b end ↔ partial triptych with ∂ arrows (05b style) ---

CH3_LIK_07C_N_HOLD = 4 if _CH3_DRAFT else 16
CH3_LIK_07C_N_CROSS = 6 if _CH3_DRAFT else 36
CH3_LIK_07C_N_VECTOR = 4 if _CH3_DRAFT else 18
CH3_LIK_07C_N_PARTIAL_HOLD = 4 if _CH3_DRAFT else 24
CH3_LIK_07C_N_COUPLED = 6 if _CH3_DRAFT else 48
CH3_LIK_07C_N_COUPLED_HOLD = 4 if _CH3_DRAFT else 12
CH3_LIK_07C_MS = 110 if not _CH3_DRAFT else 130
CH3_LIK_07C_WST_TARGET = 4.0
CH3_LIK_07C_WEL_TARGET = -4.0
CH3_LIK_07C_PATH_EXT_COLOR = "#111111"
CH3_LIK_07C_PATH_EXT_LS = "--"
CH3_LIK_07C_PATH_EXT_N = 8 if _CH3_DRAFT else 32
CH3_LIK_07C_N_3D_EXT = 6 if _CH3_DRAFT else 48


def _ch4_07c_path_end_pose(pack):
    path = np.asarray(pack["path"], dtype=float)
    tip = path[-1]
    return float(tip[0]), float(tip[1]), float(tip[2])


def _ch4_07c_07b_end_spec(pack):
    return _ch3_lik_gd_combined_path_end_spec(pack)


def _ch4_07c_07b_end_frame(pack):
    return _ch3_lik_gd_render_frame(pack, _ch4_07c_07b_end_spec(pack), cam_azim_u=0.0)


def _ch4_07c_coupled_pose(ws0, we0, bb0, t):
    """Coupled sweep: Δw_ST = +δ, Δw_EL = −δ (same magnitude), then finish w_EL to target."""
    u = float(np.clip(t, 0.0, 1.0))
    ws_tgt = float(CH3_LIK_07C_WST_TARGET)
    we_tgt = float(CH3_LIK_07C_WEL_TARGET)
    ws0, we0, bb0 = float(ws0), float(we0), float(bb0)
    rise = max(0.0, ws_tgt - ws0)
    drop = max(0.0, we0 - we_tgt)
    if rise <= 1e-12 and drop <= 1e-12:
        return ws0, we0, bb0
    phase1 = min(rise, drop)
    rest = max(0.0, drop - phase1)
    span = phase1 + rest
    s = u * span
    if s <= phase1 + 1e-12:
        d = s
        ws = ws0 + d
        we = we0 - d
    else:
        ws = ws_tgt
        we = we0 - phase1 - (s - phase1)
    return ws, we, bb0


def _ch4_07c_coupled_target_pose(ws0, we0, bb0):
    return _ch4_07c_coupled_pose(ws0, we0, bb0, 1.0)


def _ch4_07c_path_extension_xyz(gd_pack, ws, we, bb, ext_u):
    """Smooth segment from the last GD waypoint to ``(ws, we, bb)``."""
    ext_u = float(np.clip(ext_u, 0.0, 1.0))
    if ext_u <= 1e-9:
        return None
    gd_path = np.asarray(gd_pack["path"], dtype=float)
    p0 = gd_path[-1]
    p1 = np.array([float(ws), float(we), float(bb)], dtype=float)
    n = max(3, int(round(float(CH3_LIK_07C_PATH_EXT_N) * ext_u)))
    ts = np.linspace(0.0, ext_u, n, dtype=np.float64)
    return p0 + ts[:, None] * (p1 - p0)[None, :]


def _ch4_07c_3d_spec(gd_pack, ws, we, bb, *, ext_u=1.0):
    """3D frame: GD path + optional black dashed extension to ``(ws, we, bb)``."""
    study, exam, y = gd_pack["study"], gd_pack["exam"], gd_pack["y"]
    spec = _ch3_lik_gd_combined_path_end_spec(gd_pack)
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, float(ws), float(we), float(bb))
    spec["ws"] = float(ws)
    spec["we"] = float(we)
    spec["bb"] = float(bb)
    spec["grad"] = (float(g1), float(g2), float(gb))
    ext = _ch4_07c_path_extension_xyz(gd_pack, ws, we, bb, ext_u)
    if ext is not None and len(ext) >= 2:
        spec["path_trail_extension"] = ext
        spec["path_trail_extension_color"] = CH3_LIK_07C_PATH_EXT_COLOR
        spec["path_trail_extension_linestyle"] = CH3_LIK_07C_PATH_EXT_LS
    return spec


def _ch4_07c_3d_frame(gd_pack, ws, we, bb, *, ext_u=1.0):
    return _ch3_lik_gd_render_frame(
        gd_pack, _ch4_07c_3d_spec(gd_pack, ws, we, bb, ext_u=ext_u), cam_azim_u=0.0,
    )


def _ch4_07c_partial_pack(gd_pack, ws, we, bb):
    pack = {
        "study": gd_pack["study"],
        "exam": gd_pack["exam"],
        "y": gd_pack["y"],
        "partial_ws": float(ws),
        "partial_we": float(we),
        "partial_bb": float(bb),
    }
    _ch4_partial_warm_curve_cache(pack)
    return pack


def _ch4_07c_emit_partial_frame(gd_pack, ws, we, bb, *, ghost_ws=None, ghost_we=None, ghost_bb=None, **extra):
    from ch4_layout import ch4_formula_blocks_gd_progressive

    study, exam, y = gd_pack["study"], gd_pack["exam"], gd_pack["y"]
    half_w1 = float(CH3_LIK_PARTIAL_HALF_W_END)
    pt = _ch4_partial_grad_tuple(study, exam, y, ws, we, bb)
    pack = _ch4_07c_partial_pack(gd_pack, ws, we, bb)
    base = dict(
        half_w=half_w1,
        curve_u=1.0,
        show_secant=False,
        ws=float(ws),
        we=float(we),
        bb=float(bb),
        ghost_ws=ghost_ws,
        ghost_we=ghost_we,
        ghost_bb=ghost_bb,
        rocs=pt,
        show_partials=True,
        partials=pt,
        bottom_blocks=ch4_formula_blocks_gd_progressive(n_grad_lines=3),
        bottom_prog={0: 1.0, 1: 1.0},
        gd_bottom=True,
        vector_axis_scale=True,
    )
    base.update(extra)
    return _ch3_lik_emit_partial_frame(pack, **base)


def ch3_build_frames_likelihood_07c_partial_bridge(*, parallel=None):
    """Hold 07b end → partial ∂ arrows → coupled w_ST/w_EL sweep → 3D + dashed extension."""
    if parallel is not None:
        print("ch4_07c: parallel export not used (small crossfade clip)", flush=True)

    gd_pack = _ch3_lik_3d_gd_combined_pack()
    ws0, we0, bb0 = _ch4_07c_path_end_pose(gd_pack)
    ws1, we1, bb1 = _ch4_07c_coupled_target_pose(ws0, we0, bb0)
    frame_07b = _ch4_07c_07b_end_frame(gd_pack)
    frames = []

    for _ in range(int(CH3_LIK_07C_N_HOLD)):
        frames.append(frame_07b.copy())

    frame_partial_open = _ch4_07c_emit_partial_frame(
        gd_pack, ws0, we0, bb0,
        show_vectors=False,
        partial_lines_show=3,
    )
    for tv in np.linspace(0.0, 1.0, int(CH3_LIK_07C_N_CROSS), endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_07b, frame_partial_open, u))

    frame_partial_full = frame_partial_open
    for n_p in (1, 2, 3):
        frame_partial_full = _ch4_07c_emit_partial_frame(
            gd_pack, ws0, we0, bb0,
            show_vectors=True,
            vector_panels_show=n_p,
            partial_lines_show=n_p,
        )
        for _ in range(int(CH3_LIK_07C_N_VECTOR)):
            frames.append(frame_partial_full.copy())

    for _ in range(int(CH3_LIK_07C_N_PARTIAL_HOLD)):
        frames.append(frame_partial_full.copy())

    frame_coupled_end = frame_partial_full
    plot_hw = float(CH3_LIK_PARTIAL_HALF_W_START)
    panel_y_half = _ch4_partial_compute_panel_y_half(
        gd_pack["study"], gd_pack["exam"], gd_pack["y"], ws0, we0, bb0, plot_hw,
    )
    for tv in np.linspace(0.0, 1.0, int(CH3_LIK_07C_N_COUPLED), endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws, we, bb = _ch4_07c_coupled_pose(ws0, we0, bb0, u)
        frame_coupled_end = _ch4_07c_emit_partial_frame(
            gd_pack, ws, we, bb,
            ghost_ws=ws0, ghost_we=we0, ghost_bb=bb0,
            ghost_trail_panels=("st", "el"),
            panel_axes_centered=True,
            panel_y_half=panel_y_half,
            plot_half_w=plot_hw,
            show_vectors=True,
            vector_panels_show=3,
            partial_lines_show=3,
        )
        frames.append(frame_coupled_end.copy())

    for _ in range(int(CH3_LIK_07C_N_COUPLED_HOLD)):
        frames.append(frame_coupled_end.copy())

    # Crossfade to 3D at the GD end pose — point and path stay fixed until fade completes.
    frame_3d_gd_end = _ch4_07c_3d_frame(gd_pack, ws0, we0, bb0, ext_u=0.0)
    for tv in np.linspace(0.0, 1.0, int(CH3_LIK_07C_N_CROSS), endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frames.append(ch4_blend_images(frame_coupled_end, frame_3d_gd_end, u))

    frame_3d_end = _ch4_07c_3d_frame(gd_pack, ws1, we1, bb1, ext_u=1.0)
    for tv in np.linspace(0.0, 1.0, int(CH3_LIK_07C_N_3D_EXT), endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        ws, we, bb = _ch4_07c_coupled_pose(ws0, we0, bb0, u)
        frame_3d = _ch4_07c_3d_frame(gd_pack, ws, we, bb, ext_u=u)
        frames.append(frame_3d.copy())

    for _ in range(int(CH3_LIK_07C_N_COUPLED_HOLD)):
        frames.append(frame_3d_end.copy())

    for _ in range(int(CH3_LIK_07C_N_HOLD)):
        frames.append(frame_3d_end.copy())

    mode = "draft" if _CH3_DRAFT else "full"
    print(
        f"ch4_07c: {len(frames)} frames ({mode}); "
        f"GD end ({ws0:.3f}, {we0:.3f}, {bb0:.3f}) → coupled ({ws1:.3f}, {we1:.3f}, {bb1:.3f})",
        flush=True,
    )
    return frames


def ch4_preview_likelihood_07c_partial_bridge_frame(*, phase="partial"):
    """Preview one frame from the 07b ↔ partial bridge."""
    gd_pack = _ch3_lik_3d_gd_combined_pack()
    ws0, we0, bb0 = _ch4_07c_path_end_pose(gd_pack)
    ws1, we1, bb1 = _ch4_07c_coupled_target_pose(ws0, we0, bb0)
    phase = str(phase)
    if phase == "07b":
        return _ch4_07c_07b_end_frame(gd_pack)
    if phase == "coupled":
        ws, we, bb = _ch4_07c_coupled_pose(ws0, we0, bb0, 0.72)
        plot_hw = float(CH3_LIK_PARTIAL_HALF_W_START)
        panel_y_half = _ch4_partial_compute_panel_y_half(
            gd_pack["study"], gd_pack["exam"], gd_pack["y"], ws0, we0, bb0, plot_hw,
        )
        return _ch4_07c_emit_partial_frame(
            gd_pack, ws, we, bb,
            ghost_ws=ws0, ghost_we=we0, ghost_bb=bb0,
            ghost_trail_panels=("st", "el"),
            panel_axes_centered=True,
            panel_y_half=panel_y_half,
            plot_half_w=plot_hw,
            show_vectors=True,
            vector_panels_show=3,
            partial_lines_show=3,
        )
    if phase == "3d_ext":
        return _ch4_07c_3d_frame(gd_pack, ws1, we1, bb1, ext_u=1.0)
    return _ch4_07c_emit_partial_frame(
        gd_pack, ws0, we0, bb0,
        show_vectors=True,
        vector_panels_show=3,
        partial_lines_show=3,
    )


def ch4_export_likelihood_07c_partial_bridge(*, parallel=None):
    frames = ch3_build_frames_likelihood_07c_partial_bridge(parallel=parallel)
    save_mp4(frames, "ch4_07c_partial_derivative_bridge.mp4", duration=int(CH3_LIK_07C_MS))
    print("wrote", OUTPUT_DIR / "ch4_07c_partial_derivative_bridge.mp4", f"({len(frames)} frames)")
    return OUTPUT_DIR / "ch4_07c_partial_derivative_bridge.mp4"


# --- ch4_08c / 08d / 08e / 08f: decomposed partial ghosts → Newton-morph split → Newton step ---

CH3_LIK_NEWTON_PARTIAL_N_CYCLES = 8
CH3_LIK_NEWTON_PARTIAL_OPEN_HOLD = 4 if _CH3_DRAFT else 10
CH3_LIK_NEWTON_PARTIAL_REVEAL = 6 if _CH3_DRAFT else 28
CH3_LIK_NEWTON_PARTIAL_HOLD = 2 if _CH3_DRAFT else 8
CH3_LIK_NEWTON_PARTIAL_ERASE = 6 if _CH3_DRAFT else 22
CH3_LIK_NEWTON_PARTIAL_MORPH = 4 if _CH3_DRAFT else 20
CH3_LIK_NEWTON_PARTIAL_MORPH_HOLD = 2 if _CH3_DRAFT else 8
CH3_LIK_NEWTON_PARTIAL_STEP = 4 if _CH3_DRAFT else 14
CH3_LIK_NEWTON_PARTIAL_POST_HOLD = 4 if _CH3_DRAFT else 10
CH3_LIK_NEWTON_PARTIAL_MS = 130 if not _CH3_DRAFT else 150
CH3_LIK_NEWTON_PARTIAL_N_ROT_AFTER = 8 if _CH3_DRAFT else 40
CH3_LIK_NEWTON_PARTIAL_VOXEL_HALF_SCALE = 0.85

_CH3_NEWTON_PARTIAL_ARROW_VIS = (
    (True, False, False),
    (False, True, False),
    (False, False, True),
)
_NEWTON_PARTIAL_CMAPS = ("Blues", "Oranges", "Greens")


def _ch4_export_pipeline_mod():
    """Reload so notebook picks up ``ch4_export_pipeline.py`` edits without kernel restart."""
    import importlib
    import ch4_export_pipeline as mod
    return importlib.reload(mod)


def _ch3_lik_newton_partial_cam_extra(cycle_idx, *, rotate_during=True):
    """Camera azimuth for one Newton cycle (rotate during steps) or fixed (rot-after variant)."""
    n = int(CH3_LIK_NEWTON_PARTIAL_N_CYCLES)
    if rotate_during:
        u = float(int(cycle_idx)) / float(max(n - 1, 1))
        return {"cam_azim_u": u, "cam_rot_deg": float(CH3_LIK_3D_CAM_PATH_ROT)}
    return {"cam_azim_u": 0.0, "cam_rot_deg": 0.0}


def _ch3_lik_split_hold_spec(pack, ws, we, bb, *, cam_extra=None):
    """Decomposed GD arrows at ``(ws, we, bb)`` — entry frame for ch4_08c."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, float(ws), float(we), float(bb))
    spec = {
        "ws": float(ws), "we": float(we), "bb": float(bb),
        "grad": (float(g1), float(g2), float(gb)),
        "eta": float(pack["gd_eta"]),
        "arrows": (True, True, True),
        "bold": None,
        "bold_all": False,
        "grad_red": False,
        "arrow_mode": "split",
        "transition_u": 0.0,
        "newton_formulas": True,
        "here_grad_mode": "partial",
    }
    if cam_extra:
        spec.update(cam_extra)
    return spec


def _ch3_lik_partial_move_target(ws, we, bb, grad, partial_idx, eta):
    """One-axis GD probe move for partial ``partial_idx`` (0=st, 1=el, 2=b)."""
    g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
    eta = float(eta)
    if int(partial_idx) == 0:
        return float(ws - eta * g1), float(we), float(bb)
    if int(partial_idx) == 1:
        return float(ws), float(we - eta * g2), float(bb)
    return float(ws), float(we), float(bb - eta * gb)


def _ch4_newton_partial_voxel_half():
    return float(CH3_LIK_NEWTON_PROBE_SCALE) * float(CH3_LIK_NEWTON_PARTIAL_VOXEL_HALF_SCALE)


def _ch4_newton_partial_voxel_prewarm(pack):
    if pack.get("newton_partial_voxel_ready"):
        return
    pack["_newton_partial_voxel_global"] = _ch4_ball_voxel_global_cache(pack)
    pack["newton_partial_voxel_half"] = _ch4_newton_partial_voxel_half()
    pack["newton_partial_voxel_ready"] = True


def _ch4_partial_voxel_tint_cube_draw(
    cube, partial_idx, partial_val, grad_ref, *, grow_u=1.0, alpha_scale=1.0,
):
    """Small voxel cube tinted by partial-component magnitude (Blues / Oranges / Greens)."""
    import matplotlib.pyplot as plt

    draw = _ch4_ball_voxel_cube_draw(cube, grow_u=grow_u, alpha_scale=alpha_scale)
    if draw is None:
        return None
    cmap = plt.colormaps[_NEWTON_PARTIAL_CMAPS[int(partial_idx)]]
    ref = max(float(grad_ref), 1e-9)
    norm = float(np.clip(abs(float(partial_val)) / ref, 0.0, 1.0))
    rgb = cmap(0.30 + 0.65 * norm)[:3]
    draw["colors"] = draw["colors"].copy()
    draw["colors"][..., :3] = rgb
    return draw


def _ch3_lik_partial_box_ghosts_from_sites(sites, partial_idx, *, alpha=0.40, reveal_u=1.0):
    """Partial box ghosts from precomputed probe sites (no extra ∇NLL calls)."""
    vis = _CH3_NEWTON_PARTIAL_ARROW_VIS[int(partial_idx)]
    ghosts = _ch3_lik_newton_ghosts_reveal(
        sites, alpha=float(alpha), reveal_u=float(reveal_u), partial_vis=vis,
    )
    return ghosts


def _ch3_lik_partial_box_ghosts_at(
    study, exam, y, ws, we, bb, partial_idx, *, alpha=0.40, reveal_u=1.0,
):
    """Box probe ghosts showing only one decomposed partial gradient component."""
    sites = _ch3_lik_newton_box_probe_sites(study, exam, y, ws, we, bb)
    return _ch3_lik_partial_box_ghosts_from_sites(
        sites, partial_idx, alpha=float(alpha), reveal_u=float(reveal_u),
    )


def _ch3_lik_newton_partial_voxel_cubes_for_sites(pack, sites):
    """Build voxel cubes at all probe sites once per Newton iterate."""
    _ch4_newton_partial_voxel_prewarm(pack)
    global_cache = pack["_newton_partial_voxel_global"]
    half = float(pack["newton_partial_voxel_half"])
    return [
        _ch4_ball_voxel_build_cube(global_cache, (site["ws"], site["we"], site["bb"]), half)
        for site in sites
    ]


def _ch3_lik_partial_voxel_ghosts_draw_from_sites(
    sites, voxel_cubes, partial_idx, *, alpha=0.42, reveal_u=1.0, grad_ref=None,
):
    """Voxel-cube ghosts from cached probe sites + cubes (no ∇NLL or cube rebuild)."""
    pi = int(partial_idx)
    if grad_ref is None:
        grad_ref = 1e-9
    draws = []
    n_sites = len(sites)
    for oi, (site, cube) in enumerate(zip(sites, voxel_cubes)):
        site_u = ch3_knob_smoothstep(float(np.clip(float(reveal_u) * n_sites - oi, 0.0, 1.0)))
        if site_u < 0.04:
            continue
        pg = site["grad"]
        draw = _ch4_partial_voxel_tint_cube_draw(
            cube, pi, float(pg[pi]), grad_ref,
            grow_u=site_u, alpha_scale=float(alpha) * site_u,
        )
        if draw is not None:
            draws.append(draw)
    return _ch4_ball_voxel_merge_draws(draws)


def _ch3_lik_partial_voxel_ghosts_draw(
    study, exam, y, ws, we, bb, partial_idx, pack, *,
    alpha=0.42, reveal_u=1.0, grad_ref=None, sites=None, voxel_cubes=None,
):
    """Voxel-cube ghosts at box probe sites — heatmapped by active partial component."""
    if sites is None:
        sites = _ch3_lik_newton_box_probe_sites(
            study, exam, y, ws, we, bb,
            bounds=pack.get("gd_view_bounds") or pack.get("gd_view_bounds_end"),
        )
    if voxel_cubes is None:
        voxel_cubes = _ch3_lik_newton_partial_voxel_cubes_for_sites(pack, sites)
    if grad_ref is None:
        g = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad_ref = max(abs(float(g[int(partial_idx)])), 1e-9)
    return _ch3_lik_partial_voxel_ghosts_draw_from_sites(
        sites, voxel_cubes, partial_idx,
        alpha=float(alpha), reveal_u=float(reveal_u), grad_ref=float(grad_ref),
    )


def _ch3_lik_newton_partial_base_spec(
    pack, ws, we, bb, grad, *, ghosts=None, voxel_draw=None, path_trail=None, cam_extra=None, **extra,
):
    spec = {
        "ws": float(ws), "we": float(we), "bb": float(bb),
        "grad": tuple(float(v) for v in grad),
        "eta": float(pack["gd_eta"]),
        "arrows": (True, True, True),
        "bold": None,
        "bold_all": False,
        "grad_red": False,
        "arrow_mode": "split",
        "transition_u": 0.0,
        "ghosts": list(ghosts) if ghosts else None,
        "voxel_draw": voxel_draw,
        "newton_arrow_color": CH3_LIK_NEWTON_ARROW_COLOR,
        "newton_formulas": True,
        "here_grad_mode": "partial",
    }
    if cam_extra:
        spec.update(cam_extra)
    if path_trail is not None and len(path_trail) >= 2:
        spec["path_trail"] = np.asarray(path_trail, dtype=float)
    spec.update(extra)
    return spec


def _ch3_lik_newton_partial_build_trajectory(pack, *, with_voxel_cubes=False):
    """Precompute all Newton partial iterates, probe ∇NLL, and optional voxel cubes."""
    cached = pack.get("_newton_partial_trajectory")
    if cached is not None:
        if with_voxel_cubes and not cached.get("voxel_ready"):
            _ch4_newton_partial_voxel_prewarm(pack)
            for cycle in cached["cycles"]:
                cycle["voxel_cubes"] = _ch3_lik_newton_partial_voxel_cubes_for_sites(
                    pack, cycle["sites"],
                )
            cached["voxel_ready"] = True
        return cached

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    ws = float(pack["gd_start"][0])
    we = float(pack["gd_start"][1])
    bb = float(pack["gd_start"][2])
    cycles = []
    if with_voxel_cubes:
        _ch4_newton_partial_voxel_prewarm(pack)

    for cycle_idx in range(int(pack.get("newton_n_cycles", CH3_LIK_NEWTON_PARTIAL_N_CYCLES))):
        g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
        grad = (float(g1), float(g2), float(gb))
        grad_ref = max(abs(float(g1)), abs(float(g2)), abs(float(gb)), 1e-9)
        sites = _ch3_lik_newton_box_probe_sites(
            study, exam, y, ws, we, bb,
            bounds=pack.get("gd_view_bounds") or pack.get("gd_view_bounds_end"),
        )
        cycle = {
            "cycle_idx": int(cycle_idx),
            "ws": float(ws), "we": float(we), "bb": float(bb),
            "grad": grad,
            "grad_ref": float(grad_ref),
            "sites": sites,
        }
        if with_voxel_cubes:
            cycle["voxel_cubes"] = _ch3_lik_newton_partial_voxel_cubes_for_sites(pack, sites)
        dw, de, db = _ch3_nll_newton_step(
            study, exam, y, ws, we, bb, damp=pack.get("newton_damp"),
        )
        cycle["newton_step"] = (float(dw), float(de), float(db))
        cycle["newton_descent"] = (-float(dw), -float(de), -float(db))
        ws1, we1, bb1 = float(ws - dw), float(we - de), float(bb - db)
        cycle["ws1"], cycle["we1"], cycle["bb1"] = ws1, we1, bb1
        g1p, g2p, gbp = _ch3_nll_sum_grad_at_point(study, exam, y, ws1, we1, bb1)
        cycle["post_grad"] = (float(g1p), float(g2p), float(gbp))
        cycles.append(cycle)
        ws, we, bb = ws1, we1, bb1

    traj = {
        "start": (
            float(pack["gd_start"][0]),
            float(pack["gd_start"][1]),
            float(pack["gd_start"][2]),
        ),
        "cycles": cycles,
        "voxel_ready": bool(with_voxel_cubes),
    }
    pack["_newton_partial_trajectory"] = traj
    return traj


def _ch3_lik_newton_partial_partial_ghosts(
    cycle, pack, *, partial_idx, use_voxel_ghosts, alpha, reveal_u,
):
    """Build box or voxel partial ghosts for one cycle using cached probe data."""
    vis = _CH3_NEWTON_PARTIAL_ARROW_VIS[int(partial_idx)]
    if use_voxel_ghosts:
        vdraw = _ch3_lik_partial_voxel_ghosts_draw_from_sites(
            cycle["sites"], cycle["voxel_cubes"], partial_idx,
            alpha=float(alpha), reveal_u=float(reveal_u), grad_ref=cycle["grad_ref"],
        )
        return None, vdraw
    ghosts = _ch3_lik_partial_box_ghosts_from_sites(
        cycle["sites"], partial_idx, alpha=float(alpha), reveal_u=float(reveal_u),
    )
    return ghosts, None


def _ch3_lik_newton_partial_specs_from_trajectory(
    pack, trajectory, *, track_path=False, rotate_during=True, use_voxel_ghosts=False,
    skip_open_hold=False,
):
    """Expand precomputed partial-Newton trajectory into ch4_08c frame specs."""
    if use_voxel_ghosts and not trajectory.get("voxel_ready"):
        trajectory = _ch3_lik_newton_partial_build_trajectory(pack, with_voxel_cubes=True)
    specs = []
    ghost_cache = {}
    ws0, we0, bb0 = trajectory["start"]
    path_trail = [[ws0, we0, bb0]] if track_path else None
    open_cam = _ch3_lik_newton_partial_cam_extra(0, rotate_during=False)
    open_spec = _ch3_lik_split_hold_spec(pack, ws0, we0, bb0, cam_extra=open_cam)

    if not skip_open_hold:
        for _ in range(max(int(CH3_LIK_NEWTON_PARTIAL_OPEN_HOLD), 1)):
            specs.append(dict(open_spec))

    def _partial_ghosts(cycle, pi, alpha, reveal_u):
        key = (cycle["cycle_idx"], int(pi), round(float(reveal_u), 5), round(float(alpha), 5))
        if key in ghost_cache:
            return ghost_cache[key]
        out = _ch3_lik_newton_partial_partial_ghosts(
            cycle, pack, partial_idx=pi, use_voxel_ghosts=use_voxel_ghosts,
            alpha=float(alpha), reveal_u=float(reveal_u),
        )
        ghost_cache[key] = out
        return out

    def _append_hold(cycle, pi, vis, ws, we, bb, grad, cam_extra, n_hold):
        hold_spec = None
        for _ in range(n_hold):
            if hold_spec is None:
                ghosts, vdraw = _partial_ghosts(cycle, pi, 0.42, 1.0)
                hold_spec = _ch3_lik_newton_partial_base_spec(
                    pack, ws, we, bb, grad,
                    ghosts=ghosts,
                    voxel_draw=vdraw,
                    path_trail=path_trail,
                    cam_extra=cam_extra,
                    arrow_mode="none",
                    arrows=vis,
                )
            specs.append(hold_spec)

    for cycle in trajectory["cycles"]:
        ws, we, bb = cycle["ws"], cycle["we"], cycle["bb"]
        grad = cycle["grad"]
        cam_extra = _ch3_lik_newton_partial_cam_extra(
            cycle["cycle_idx"], rotate_during=rotate_during,
        )
        newton_step = cycle["newton_step"]
        newton_descent = cycle["newton_descent"]
        ws1, we1, bb1 = cycle["ws1"], cycle["we1"], cycle["bb1"]
        post_grad = cycle["post_grad"]
        newton_display = {"newton_step_display": newton_step}

        for pi in range(3):
            vis = _CH3_NEWTON_PARTIAL_ARROW_VIS[pi]

            for tv in np.linspace(0.0, 1.0, CH3_LIK_NEWTON_PARTIAL_REVEAL, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                ghosts, vdraw = _partial_ghosts(cycle, pi, 0.42, u)
                specs.append(_ch3_lik_newton_partial_base_spec(
                    pack, ws, we, bb, grad,
                    ghosts=ghosts,
                    voxel_draw=vdraw,
                    path_trail=path_trail,
                    cam_extra=cam_extra,
                    arrow_mode="none",
                    arrows=vis,
                ))

            _append_hold(
                cycle, pi, vis, ws, we, bb, grad, cam_extra,
                max(int(CH3_LIK_NEWTON_PARTIAL_HOLD), 1),
            )

            for tv in np.linspace(0.0, 1.0, CH3_LIK_NEWTON_PARTIAL_ERASE, endpoint=True):
                u = ch3_knob_smoothstep(float(tv))
                fade = 1.0 - u
                if fade > 0.08:
                    ghosts, vdraw = _partial_ghosts(cycle, pi, 0.42 * fade, fade)
                else:
                    ghosts, vdraw = None, None
                specs.append(_ch3_lik_newton_partial_base_spec(
                    pack, ws, we, bb, grad,
                    ghosts=ghosts,
                    voxel_draw=vdraw,
                    path_trail=path_trail,
                    cam_extra=cam_extra,
                    arrow_mode="none" if fade > 0.08 else "split",
                    arrows=vis,
                ))

        for tv in np.linspace(0.0, 1.0, CH3_LIK_NEWTON_PARTIAL_MORPH, endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            specs.append(_ch3_lik_newton_partial_base_spec(
                pack, ws, we, bb, grad,
                path_trail=path_trail,
                cam_extra=cam_extra,
                arrow_mode="newton_morph",
                arrows=(True, True, True),
                transition_u=u,
                newton_split_vec=newton_step,
                here_grad_mode="morph",
                **newton_display,
            ))

        morph_hold = None
        for _ in range(max(int(CH3_LIK_NEWTON_PARTIAL_MORPH_HOLD), 1)):
            if morph_hold is None:
                morph_hold = _ch3_lik_newton_partial_base_spec(
                    pack, ws, we, bb, grad,
                    path_trail=path_trail,
                    cam_extra=cam_extra,
                    arrow_mode="newton_morph",
                    arrows=(True, True, True),
                    transition_u=1.0,
                    newton_split_vec=newton_step,
                    here_grad_mode="newton",
                    **newton_display,
                )
            specs.append(morph_hold)

        step_n = max(int(CH3_LIK_NEWTON_PARTIAL_STEP), 2)
        for si in range(step_n):
            u = ch3_knob_smoothstep(float(si) / float(step_n - 1))
            wi = ws + u * (ws1 - ws)
            ei = we + u * (we1 - we)
            bi = bb + u * (bb1 - bb)
            specs.append(_ch3_lik_newton_partial_base_spec(
                pack, wi, ei, bi, grad,
                path_trail=path_trail,
                cam_extra=cam_extra,
                arrow_mode="none",
                here_grad_mode="newton",
                newton_step_vec=(
                    (1.0 - u) * newton_descent[0],
                    (1.0 - u) * newton_descent[1],
                    (1.0 - u) * newton_descent[2],
                ),
                **newton_display,
            ))

        if track_path:
            path_trail.append([ws1, we1, bb1])

        post_hold = None
        for _ in range(max(int(CH3_LIK_NEWTON_PARTIAL_POST_HOLD), 1)):
            if post_hold is None:
                post_hold = _ch3_lik_newton_partial_base_spec(
                    pack, ws1, we1, bb1, post_grad,
                    path_trail=path_trail,
                    cam_extra=cam_extra,
                    arrow_mode="split",
                    arrows=(True, True, True),
                )
            specs.append(post_hold)

    return specs


def _ch3_lik_newton_partial_ghost_frame_specs(
    pack, *, track_path=False, rotate_during=True, use_voxel_ghosts=False,
    skip_open_hold=False,
):
    """Per cycle: 3× partial box/voxel ghosts → Newton-morph split → damped Newton step."""
    trajectory = _ch3_lik_newton_partial_build_trajectory(
        pack, with_voxel_cubes=bool(use_voxel_ghosts),
    )
    return _ch3_lik_newton_partial_specs_from_trajectory(
        pack, trajectory,
        track_path=bool(track_path),
        rotate_during=bool(rotate_during),
        use_voxel_ghosts=bool(use_voxel_ghosts),
        skip_open_hold=bool(skip_open_hold),
    )


def _ch3_lik_newton_partial_append_rot_after_specs(specs):
    """360° pan after all Newton cycles (rot-after variants)."""
    if not specs:
        return specs
    last = dict(specs[-1])
    for tv in np.linspace(0.0, 1.0, CH3_LIK_NEWTON_PARTIAL_N_ROT_AFTER, endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        frame = dict(last)
        frame["cam_azim_u"] = u
        frame["cam_rot_deg"] = float(CH3_LIK_3D_CAM_PATH_ROT)
        specs.append(frame)
    return specs


def _ch3_lik_newton_partial_story_specs(
    pack, *, track_path=False, rotate_during=True, use_voxel_ghosts=False,
    skip_open_hold=False,
):
    specs = _ch3_lik_newton_partial_ghost_frame_specs(
        pack,
        track_path=bool(track_path),
        rotate_during=bool(rotate_during),
        use_voxel_ghosts=bool(use_voxel_ghosts),
        skip_open_hold=bool(skip_open_hold),
    )
    if not rotate_during:
        _ch3_lik_newton_partial_append_rot_after_specs(specs)
    return specs


def ch4_preview_likelihood_3d_newton_partial_ghosts_frame(
    *, track_path=False, cycle=0, partial=0, phase="hold",
    rotate_during=True, use_voxel_ghosts=False,
):
    """Preview one ch4_08c frame (``partial`` 0=st, 1=el, 2=b; ``phase`` reveal|hold|morph|step)."""
    pack = _ch3_lik_3d_gd_combined_pack()
    specs = _ch3_lik_newton_partial_story_specs(
        pack,
        track_path=bool(track_path),
        rotate_during=bool(rotate_during),
        use_voxel_ghosts=bool(use_voxel_ghosts),
    )
    open_h = max(int(CH3_LIK_NEWTON_PARTIAL_OPEN_HOLD), 1)
    block = (
        3 * (
            int(CH3_LIK_NEWTON_PARTIAL_REVEAL)
            + max(int(CH3_LIK_NEWTON_PARTIAL_HOLD), 1)
            + int(CH3_LIK_NEWTON_PARTIAL_ERASE)
        )
        + int(CH3_LIK_NEWTON_PARTIAL_MORPH)
        + max(int(CH3_LIK_NEWTON_PARTIAL_MORPH_HOLD), 1)
        + max(int(CH3_LIK_NEWTON_PARTIAL_STEP), 2)
        + max(int(CH3_LIK_NEWTON_PARTIAL_POST_HOLD), 1)
    )
    c = int(np.clip(int(cycle), 0, int(CH3_LIK_NEWTON_PARTIAL_N_CYCLES) - 1))
    pi = int(np.clip(int(partial), 0, 2))
    base = open_h + c * block
    if phase == "reveal":
        idx = base + pi * (
            int(CH3_LIK_NEWTON_PARTIAL_REVEAL)
            + max(int(CH3_LIK_NEWTON_PARTIAL_HOLD), 1)
            + int(CH3_LIK_NEWTON_PARTIAL_ERASE)
        ) + int(CH3_LIK_NEWTON_PARTIAL_REVEAL) // 2
    elif phase == "hold":
        idx = base + pi * (
            int(CH3_LIK_NEWTON_PARTIAL_REVEAL)
            + max(int(CH3_LIK_NEWTON_PARTIAL_HOLD), 1)
            + int(CH3_LIK_NEWTON_PARTIAL_ERASE)
        ) + int(CH3_LIK_NEWTON_PARTIAL_REVEAL) + 1
    elif phase == "morph":
        idx = base + 3 * (
            int(CH3_LIK_NEWTON_PARTIAL_REVEAL)
            + max(int(CH3_LIK_NEWTON_PARTIAL_HOLD), 1)
            + int(CH3_LIK_NEWTON_PARTIAL_ERASE)
        ) + int(CH3_LIK_NEWTON_PARTIAL_MORPH) // 2
    elif phase == "step":
        idx = base + 3 * (
            int(CH3_LIK_NEWTON_PARTIAL_REVEAL)
            + max(int(CH3_LIK_NEWTON_PARTIAL_HOLD), 1)
            + int(CH3_LIK_NEWTON_PARTIAL_ERASE)
        ) + int(CH3_LIK_NEWTON_PARTIAL_MORPH) + max(int(CH3_LIK_NEWTON_PARTIAL_MORPH_HOLD), 1) + 1
    elif phase == "post":
        idx = base + 3 * (
            int(CH3_LIK_NEWTON_PARTIAL_REVEAL)
            + max(int(CH3_LIK_NEWTON_PARTIAL_HOLD), 1)
            + int(CH3_LIK_NEWTON_PARTIAL_ERASE)
        ) + int(CH3_LIK_NEWTON_PARTIAL_MORPH) + max(int(CH3_LIK_NEWTON_PARTIAL_MORPH_HOLD), 1) + max(int(CH3_LIK_NEWTON_PARTIAL_STEP), 2) + 1
    else:
        idx = base
    rot_tail = 0 if rotate_during else int(CH3_LIK_NEWTON_PARTIAL_N_ROT_AFTER)
    idx = int(np.clip(idx, 0, len(specs) - rot_tail - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch4_preview_likelihood_3d_newton_partial_text_frame(*, text_phase="formulas"):
    """Preview ch4_08 bottom/right text rails for one story beat.

    ``text_phase``: formulas | here_partial | here_morph | here_newton | here_post
    """
    phase = str(text_phase)
    if phase == "formulas":
        pack = _ch3_lik_3d_gd_combined_pack()
        ws, we, bb = (float(pack["gd_start"][0]), float(pack["gd_start"][1]), float(pack["gd_start"][2]))
        return _ch3_lik_gd_render_frame(pack, _ch3_lik_split_hold_spec(pack, ws, we, bb), cam_azim_u=0.0)
    phase_map = {
        "here_partial": ("hold", 0, 0),
        "here_morph": ("morph", 0, 0),
        "here_newton": ("step", 0, 0),
        "here_post": ("post", 0, 0),
    }
    if phase not in phase_map:
        raise ValueError(f"unknown text_phase {text_phase!r}")
    p, c, pi = phase_map[phase]
    return ch4_preview_likelihood_3d_newton_partial_ghosts_frame(
        cycle=c, partial=pi, phase=p, rotate_during=True, use_voxel_ghosts=False,
    )


def ch4_preview_likelihood_3d_newton_08_text_frame(*, text_phase="formulas"):
    """Preview ch4_08/08b text rails (probe ghosts family)."""
    pack = _ch3_lik_3d_gd_combined_pack()
    phase = str(text_phase)
    if phase == "formulas":
        spec = _ch3_lik_combined_hold_spec(pack)
        spec["newton_formulas"] = True
        spec["here_grad_mode"] = "partial"
        return _ch3_lik_gd_render_frame(pack, spec, cam_azim_u=0.0)
    specs = _ch3_lik_newton_ghost_frame_specs(pack, track_path=False)
    if phase == "here_partial":
        idx = max(int(CH3_LIK_NEWTON_OPEN_HOLD), 1) + int(CH3_LIK_NEWTON_N_PROBE) // 2
    elif phase == "here_newton":
        hold = max(int(CH3_LIK_NEWTON_OPEN_HOLD), 1)
        probe_n = max(int(CH3_LIK_NEWTON_N_PROBE), 2)
        hold_n = max(int(CH3_LIK_NEWTON_N_HOLD), 1)
        step_n = max(int(CH3_LIK_NEWTON_N_STEP), 2)
        block = probe_n + max(hold_n // 2, 1) + step_n + hold_n
        idx = hold + max(hold_n // 2, 1) + 1
    elif phase == "here_post":
        hold = max(int(CH3_LIK_NEWTON_OPEN_HOLD), 1)
        probe_n = max(int(CH3_LIK_NEWTON_N_PROBE), 2)
        hold_n = max(int(CH3_LIK_NEWTON_N_HOLD), 1)
        step_n = max(int(CH3_LIK_NEWTON_N_STEP), 2)
        block = probe_n + max(hold_n // 2, 1) + step_n + hold_n
        idx = hold + block - 1
    else:
        raise ValueError(f"unknown text_phase {text_phase!r}")
    idx = int(np.clip(idx, 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def _ch3_lik_newton_partial_export(
    *,
    track_path=False,
    rotate_during=True,
    use_voxel_ghosts=False,
    filename,
    progress_label,
):
    mod = _ch4_export_pipeline_mod()
    pack = _ch3_lik_3d_gd_combined_pack()
    return mod.export_mp4_from_specs(
        pack,
        _ch3_lik_newton_partial_story_specs(
            pack,
            track_path=bool(track_path),
            rotate_during=bool(rotate_during),
            use_voxel_ghosts=bool(use_voxel_ghosts),
        ),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        filename=filename,
        duration_ms=int(CH3_LIK_NEWTON_PARTIAL_MS),
        story_hold_fn=_ch3_lik_story_hold,
        prewarm="gd",
        progress_label=progress_label,
        render_fn_name="_ch3_lik_gd_render_frame",
    )


def ch4_export_likelihood_3d_newton_partial_ghosts(*, rotate_during=True):
    fn = (
        "ch4_08c_newton_partial_ghosts.mp4"
        if rotate_during
        else "ch4_08c_newton_partial_ghosts_rot_after.mp4"
    )
    label = "ch4_08c" if rotate_during else "ch4_08c_rot_after"
    return _ch3_lik_newton_partial_export(
        track_path=False,
        rotate_during=rotate_during,
        use_voxel_ghosts=False,
        filename=fn,
        progress_label=label,
    )


def ch4_export_likelihood_3d_newton_partial_ghosts_rot_after():
    return ch4_export_likelihood_3d_newton_partial_ghosts(rotate_during=False)


def ch4_export_likelihood_3d_newton_partial_ghosts_path(*, rotate_during=True):
    fn = (
        "ch4_08d_newton_partial_ghosts_path.mp4"
        if rotate_during
        else "ch4_08d_newton_partial_ghosts_path_rot_after.mp4"
    )
    label = "ch4_08d" if rotate_during else "ch4_08d_rot_after"
    return _ch3_lik_newton_partial_export(
        track_path=True,
        rotate_during=rotate_during,
        use_voxel_ghosts=False,
        filename=fn,
        progress_label=label,
    )


def ch4_export_likelihood_3d_newton_partial_ghosts_path_rot_after():
    return ch4_export_likelihood_3d_newton_partial_ghosts_path(rotate_during=False)


def ch4_export_likelihood_3d_newton_partial_voxel_ghosts(*, rotate_during=True):
    fn = (
        "ch4_08e_newton_partial_voxel_ghosts.mp4"
        if rotate_during
        else "ch4_08e_newton_partial_voxel_ghosts_rot_after.mp4"
    )
    label = "ch4_08e" if rotate_during else "ch4_08e_rot_after"
    return _ch3_lik_newton_partial_export(
        track_path=False,
        rotate_during=rotate_during,
        use_voxel_ghosts=True,
        filename=fn,
        progress_label=label,
    )


def ch4_export_likelihood_3d_newton_partial_voxel_ghosts_rot_after():
    return ch4_export_likelihood_3d_newton_partial_voxel_ghosts(rotate_during=False)


def ch4_export_likelihood_3d_newton_partial_voxel_ghosts_path(*, rotate_during=True):
    fn = (
        "ch4_08f_newton_partial_voxel_ghosts_path.mp4"
        if rotate_during
        else "ch4_08f_newton_partial_voxel_ghosts_path_rot_after.mp4"
    )
    label = "ch4_08f" if rotate_during else "ch4_08f_rot_after"
    return _ch3_lik_newton_partial_export(
        track_path=True,
        rotate_during=rotate_during,
        use_voxel_ghosts=True,
        filename=fn,
        progress_label=label,
    )


def ch4_export_likelihood_3d_newton_partial_voxel_ghosts_path_rot_after():
    return ch4_export_likelihood_3d_newton_partial_voxel_ghosts_path(rotate_during=False)


_CH4_NEWTON_PARTIAL_VARIANTS = (
    dict(
        track_path=False, rotate_during=True, use_voxel_ghosts=False,
        filename="ch4_08c_newton_partial_ghosts.mp4", progress_label="ch4_08c",
    ),
    dict(
        track_path=False, rotate_during=False, use_voxel_ghosts=False,
        filename="ch4_08c_newton_partial_ghosts_rot_after.mp4", progress_label="ch4_08c_rot_after",
    ),
    dict(
        track_path=True, rotate_during=True, use_voxel_ghosts=False,
        filename="ch4_08d_newton_partial_ghosts_path.mp4", progress_label="ch4_08d",
    ),
    dict(
        track_path=True, rotate_during=False, use_voxel_ghosts=False,
        filename="ch4_08d_newton_partial_ghosts_path_rot_after.mp4", progress_label="ch4_08d_rot_after",
    ),
    dict(
        track_path=False, rotate_during=True, use_voxel_ghosts=True,
        filename="ch4_08e_newton_partial_voxel_ghosts.mp4", progress_label="ch4_08e",
    ),
    dict(
        track_path=False, rotate_during=False, use_voxel_ghosts=True,
        filename="ch4_08e_newton_partial_voxel_ghosts_rot_after.mp4", progress_label="ch4_08e_rot_after",
    ),
    dict(
        track_path=True, rotate_during=True, use_voxel_ghosts=True,
        filename="ch4_08f_newton_partial_voxel_ghosts_path.mp4", progress_label="ch4_08f",
    ),
    dict(
        track_path=True, rotate_during=False, use_voxel_ghosts=True,
        filename="ch4_08f_newton_partial_voxel_ghosts_path_rot_after.mp4", progress_label="ch4_08f_rot_after",
    ),
)


def _ch4_newton_partial_batch_jobs(pack):
    """Build export job dicts for all ch4_08c–08f variants from one shared trajectory."""
    need_voxel = any(bool(cfg["use_voxel_ghosts"]) for cfg in _CH4_NEWTON_PARTIAL_VARIANTS)
    trajectory = _ch3_lik_newton_partial_build_trajectory(
        pack, with_voxel_cubes=need_voxel,
    )
    jobs = []
    for cfg in _CH4_NEWTON_PARTIAL_VARIANTS:
        specs = _ch3_lik_newton_partial_specs_from_trajectory(
            pack, trajectory,
            track_path=bool(cfg["track_path"]),
            rotate_during=bool(cfg["rotate_during"]),
            use_voxel_ghosts=bool(cfg["use_voxel_ghosts"]),
        )
        if not cfg["rotate_during"]:
            _ch3_lik_newton_partial_append_rot_after_specs(specs)
        jobs.append({
            "filename": cfg["filename"],
            "progress_label": cfg["progress_label"],
            "specs": specs,
            "duration_ms": int(CH3_LIK_NEWTON_PARTIAL_MS),
            "story_hold_fn": _ch3_lik_story_hold,
        })
    return jobs


def ch4_export_likelihood_3d_newton_partial_all(*, parallel=None):
    """Batch-export 08c–08f (8 clips): dedupe shared frames, parallel unique renders."""
    mod = _ch4_export_pipeline_mod()
    pack = _ch3_lik_3d_gd_combined_pack()
    return mod.export_mp4_batch_from_jobs(
        pack,
        _ch4_newton_partial_batch_jobs(pack),
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        prewarm="gd",
        parallel=parallel,
        progress_label="ch4_08c–08f batch",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


def ch4_export_likelihood_3d_newton_08_family_all(*, parallel=None):
    """Batch-export ch4_08/08b + 08c–08f (10 clips) with one shared frame cache."""
    mod = _ch4_export_pipeline_mod()
    pack = _ch3_lik_3d_gd_combined_pack()
    newton_traj = _ch3_lik_newton_build_trajectory(pack)
    jobs = [
        {
            "filename": "ch4_08_newton_ghosts.mp4",
            "progress_label": "ch4_08",
            "specs": _ch3_lik_newton_ghost_specs_from_trajectory(
                pack, newton_traj, track_path=False,
            ),
            "duration_ms": int(CH3_LIK_NEWTON_MS),
            "story_hold_fn": _ch3_lik_story_hold,
        },
        {
            "filename": "ch4_08b_newton_ghosts_path.mp4",
            "progress_label": "ch4_08b",
            "specs": _ch3_lik_newton_ghost_specs_from_trajectory(
                pack, newton_traj, track_path=True,
            ),
            "duration_ms": int(CH3_LIK_NEWTON_MS),
            "story_hold_fn": _ch3_lik_story_hold,
        },
    ]
    jobs.extend(_ch4_newton_partial_batch_jobs(pack))
    return mod.export_mp4_batch_from_jobs(
        pack,
        jobs,
        _ch3_lik_gd_render_frame,
        save_mp4=save_mp4,
        prewarm="gd",
        parallel=parallel,
        progress_label="ch4_08 family batch",
        render_fn_name="_ch3_lik_gd_render_frame",
    )


# --- ch4_02b: wide 2D → duo layout → σ(ST,EL) 3D copy → knob demo → NLL/CT → ch4_02 opening ---

CH4_02B_W_START = (1.0, -1.0, 0.0)
CH4_02B_W_KNOB_END = (2.0, 3.0, 0.0)
CH4_02B_W_MID = (0.0, 0.0, 0.0)
CH4_02B_W_NLL = (4.0, -4.0, 0.0)
CH4_02B_W_DEMO = (
    (2.0, 0.5, -0.3),
    (0.8, -0.8, 0.15),
    (0.2, 0.6, -0.8),
)
CH4_02B_MS = 68 if not _CH3_DRAFT else 110
CH4_02B_N_HOLD_WIDE = 8 if _CH3_DRAFT else 16
CH4_02B_N_INTRO = 14 if _CH3_DRAFT else max(90, _smooth_n(72))
CH4_02B_N_COPY = 6 if _CH3_DRAFT else max(52, _smooth_n(40))
CH4_02B_N_SIG3D = 6 if _CH3_DRAFT else max(44, _smooth_n(34))
CH4_02B_N_CAMERA = 8 if _CH3_DRAFT else max(54, _smooth_n(42))
CH4_02B_N_KNOB_SEG = 4 if _CH3_DRAFT else max(36, _smooth_n(28))
CH4_02B_N_SIG_ERASE = 4 if _CH3_DRAFT else max(28, _smooth_n(22))
CH4_02B_N_NLL_REVEAL = 4 if _CH3_DRAFT else max(28, _smooth_n(22))
CH4_02B_N_MARKER_PATH = 6 if _CH3_DRAFT else max(48, _smooth_n(36))
CH4_02B_N_HEATMAP = 4 if _CH3_DRAFT else max(52, _smooth_n(40))
CH4_02B_N_SQUISH = 4 if _CH3_DRAFT else max(48, _smooth_n(36))
CH4_02B_N_PLANE_DROP = 4 if _CH3_DRAFT else max(48, _smooth_n(36))
CH4_02B_N_CT_HOLD = 2 if _CH3_DRAFT else max(16, _smooth_n(12))
CH4_02B_N_CT_SWEEP = 4 if _CH3_DRAFT else max(64, _smooth_n(48))
CH4_02B_N_CT_PIVOT = 3 if _CH3_DRAFT else max(48, _smooth_n(36))
CH4_02B_N_ERASE_OPEN = 4 if _CH3_DRAFT else max(28, _smooth_n(22))
CH4_02B_N_RELOC_CAM = 4 if _CH3_DRAFT else max(36, _smooth_n(28))
CH4_02B_N_HOLD_END = 4 if _CH3_DRAFT else 14
CH4_02B_ELEV_TOP = 89.0
CH4_02B_ELEV_VIEW = 26.0
CH4_02B_AZIM_TOP = -90.0
CH4_02B_AZIM_VIEW = -135.0
CH4_02B_SIG_REVEAL_ORIGIN = "hi_hi"
CH4_02B_COPY_END_SCALE = 1.10
CH4_02B_COPY_FADE_START = 0.64
CH4_02B_COPY_FADE_END = 0.90
CH4_02B_SIG3D_GRID = 22 if _CH3_DRAFT else 56
CH4_02B_LIK_MESH_GRID = 28 if _CH3_DRAFT else 68
CH4_02B_CT_MESH_GRID = 18 if _CH3_DRAFT else 48
CH4_02B_2D_CONTOUR_LEVELS = 26 if _CH3_DRAFT else 58
CH4_02B_ANIM_DPI = min(int(EXPORT_DPI), 78 if _CH3_DRAFT else 120)
CH4_02B_WIDE_DATA = (0.06, 0.14, 0.88, 0.72)
CH4_02B_WIDE_KNOB_ROW = 0.34

_CH4_02B_DUO_RECTS = None
_CH4_02B_SIG_MESH = None
_CH4_02B_LIK_STATE = None
_CH4_02B_QUALITY = None


def _ch4_02b_anim_dpi():
    return float(CH4_02B_ANIM_DPI)


def _ch4_02b_ct_grid():
    return int(CH4_02B_CT_MESH_GRID)


def _ch4_02b_quality_enter():
    """Boost mesh / contour resolution for this export only."""
    global _CH4_02B_QUALITY, _CH4_02B_LIK_STATE, _CH4_02B_SIG_MESH
    global _CH3_SIGMA_CONTOUR_LEVELS, CH3_LIK_CT_GRID, CH3_ANIM_DPI
    if _CH4_02B_QUALITY is not None:
        return
    _CH4_02B_LIK_STATE = None
    _CH4_02B_SIG_MESH = None
    _CH4_02B_QUALITY = {
        "contour_levels": int(_CH3_SIGMA_CONTOUR_LEVELS),
        "ct_grid": int(CH3_LIK_CT_GRID),
        "anim_dpi": float(CH3_ANIM_DPI),
    }
    if not _CH3_DRAFT:
        _CH3_SIGMA_CONTOUR_LEVELS = int(CH4_02B_2D_CONTOUR_LEVELS)
        CH3_LIK_CT_GRID = int(CH4_02B_CT_MESH_GRID)
        CH3_ANIM_DPI = float(CH4_02B_ANIM_DPI)


def _ch4_02b_quality_exit():
    global _CH4_02B_QUALITY, _CH3_SIGMA_CONTOUR_LEVELS, CH3_LIK_CT_GRID, CH3_ANIM_DPI
    if _CH4_02B_QUALITY is None:
        return
    saved = _CH4_02B_QUALITY
    _CH3_SIGMA_CONTOUR_LEVELS = int(saved["contour_levels"])
    CH3_LIK_CT_GRID = int(saved["ct_grid"])
    CH3_ANIM_DPI = float(saved["anim_dpi"])
    _CH4_02B_QUALITY = None


def _ch4_02b_lik_state():
    """NLL story state with a finer w₁×w₂ mesh for ch4_02b."""
    global _CH4_02B_LIK_STATE
    if _CH4_02B_LIK_STATE is not None:
        return _CH4_02B_LIK_STATE
    st = _ch3_lik86_terminal_state()
    study, exam, y = st["study"], st["exam"], st["y"]
    bb = float(st["b"])
    mesh = ch3_lik_w12_mesh_pack(
        study, exam, y, bb,
        w1_lo=float(st["w1_lo"]), w1_hi=float(st["w1_hi"]),
        w2_lo=float(st["w2_lo"]), w2_hi=float(st["w2_hi"]),
        grid_n=int(CH4_02B_LIK_MESH_GRID),
    )
    _CH4_02B_LIK_STATE = {**st, "mesh": mesh}
    return _CH4_02B_LIK_STATE


def _ch4_02b_rect(ax_or_tuple):
    if isinstance(ax_or_tuple, tuple):
        return tuple(float(v) for v in ax_or_tuple)
    pr = ax_or_tuple.get_position()
    return (float(pr.x0), float(pr.y0), float(pr.width), float(pr.height))


def _ch4_02b_lerp_rect(u, a, b):
    u = float(ch3_knob_smoothstep(np.clip(float(u), 0.0, 1.0)))
    return tuple(float(a[i] + u * (float(b[i]) - float(a[i]))) for i in range(4))


def _ch4_02b_rect_center_wh(r):
    x0, y0, w, h = r
    return float(x0 + 0.5 * w), float(y0 + 0.5 * h), float(w), float(h)


def _ch4_02b_copy_rect(cu, data_r, duo_3d):
    """Slide the 2-D copy; grow to ``CH4_02B_COPY_END_SCALE`` × left panel size at landing."""
    cu = float(ch3_knob_smoothstep(np.clip(float(cu), 0.0, 1.0)))
    cx0, cy0, w, h = _ch4_02b_rect_center_wh(data_r)
    scale = 1.0 + cu * (float(CH4_02B_COPY_END_SCALE) - 1.0)
    w *= scale
    h *= scale
    cx1, cy1, _, _ = _ch4_02b_rect_center_wh(duo_3d)
    cx = cx0 + cu * (cx1 - cx0)
    cy = cy0 + cu * (cy1 - cy0)
    return (cx - 0.5 * w, cy - 0.5 * h, w, h)


def _ch4_02b_copy_panel_alpha(cu):
    """Fade the sliding 2-D copy out before it reaches its landing position."""
    cu = float(np.clip(float(cu), 0.0, 1.0))
    t0 = float(CH4_02B_COPY_FADE_START)
    t1 = float(CH4_02B_COPY_FADE_END)
    if cu <= t0:
        return 1.0
    if cu >= t1:
        return 0.0
    u = (cu - t0) / max(t1 - t0, 1e-9)
    return 1.0 - float(ch3_knob_smoothstep(u))


def _ch4_02b_sig3d_panel_alpha(cu):
    """Complementary fade-in for σ 3-D — same timing/rate as the 2-D copy fade-out."""
    return 1.0 - float(_ch4_02b_copy_panel_alpha(cu))


def _ch4_02b_ax_panel_alpha(ax, alpha):
    """Best-effort uniform fade on an axes (icons/lines may not respond)."""
    a = float(np.clip(float(alpha), 0.0, 1.0))
    if a >= 1.0 - 1e-6:
        return
    ax.patch.set_alpha(a)
    for line in ax.lines:
        base = line.get_alpha()
        line.set_alpha(a if base is None else float(base) * a)
    for coll in ax.collections:
        try:
            base = coll.get_alpha()
            if isinstance(base, (float, int)) or base is None:
                coll.set_alpha(a if base is None else float(base) * a)
            else:
                coll.set_alpha(a)
        except (AttributeError, TypeError, ValueError):
            pass
    for patch in ax.patches:
        base = patch.get_alpha()
        patch.set_alpha(a if base is None else float(base) * a)
    for art in ax.artists:
        try:
            base = art.get_alpha()
            art.set_alpha(a if base is None else float(base) * a)
        except (AttributeError, TypeError, ValueError):
            pass
        for child in getattr(art, "get_children", lambda: [])():
            try:
                base = child.get_alpha()
                child.set_alpha(a if base is None else float(base) * a)
            except (AttributeError, TypeError, ValueError):
                pass
    for spine in ax.spines.values():
        spine.set_alpha(a)
    for text in ax.get_xticklabels() + ax.get_yticklabels():
        text.set_alpha(a)
    ax.xaxis.label.set_alpha(a)
    ax.yaxis.label.set_alpha(a)


def _ch4_02b_draw_2d_panel_content(ax, ws, we, bb, study, exam, y, *, cmap_alpha=None, show_legend=True):
    leg = legend_linear_equation_values_bold_param(float(ws), float(we), float(bb), "all")
    ca = float(_CH3_SIGMA_CONTOUR_ALPHA if cmap_alpha is None else cmap_alpha)
    ch3_draw_left_panel(
        ax, float(ws), float(we), float(bb), study, exam, y, leg,
        show_colormap=ca > 1e-5,
        highlight_mistakes_flag=False,
        colormap_alpha=ca,
        show_legend=bool(show_legend),
    )
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    finalize_style_legend_tex(ax)


def _ch4_02b_draw_2d_panel(ax, ws, we, bb, study, exam, y, *, cmap_alpha=None, show_legend=True, panel_alpha=1.0):
    _ch4_02b_draw_2d_panel_content(
        ax, ws, we, bb, study, exam, y,
        cmap_alpha=cmap_alpha, show_legend=show_legend,
    )
    _ch4_02b_ax_panel_alpha(ax, panel_alpha)


def _ch4_02b_panel_rgba_faded(img, panel_alpha):
    """Apply a uniform fade to every pixel (background, colormap, icons, labels)."""
    pa = float(np.clip(float(panel_alpha), 0.0, 1.0))
    if pa >= 1.0 - 1e-6:
        return img.convert("RGBA")
    arr = np.asarray(img.convert("RGBA"), dtype=np.float32)
    arr[:, :, 3] *= pa
    return Image.fromarray(np.clip(arr, 0.0, 255.0).astype(np.uint8), mode="RGBA")


def _ch4_02b_render_2d_panel_image(ws, we, bb, study, exam, y, *, width_in, height_in, dpi):
    sub_fig = plt.figure(figsize=(float(width_in), float(height_in)), dpi=float(dpi), facecolor="white")
    sub_ax = sub_fig.add_axes([0.0, 0.0, 1.0, 1.0], facecolor="white")
    _ch4_02b_draw_2d_panel_content(sub_ax, ws, we, bb, study, exam, y, show_legend=False)
    img = fig_to_image(sub_fig, dpi=float(dpi), transparent=False).convert("RGBA")
    plt.close(sub_fig)
    return img


def _ch4_02b_blit_rgba_panel(fig, rect, rgba, *, zorder=10):
    host = fig.add_axes(rect)
    host.patch.set_alpha(0.0)
    host.imshow(
        rgba,
        aspect="auto",
        interpolation="bilinear",
        zorder=int(zorder),
        extent=(0.0, float(rgba.width), float(rgba.height), 0.0),
    )
    host.set_xlim(0.0, float(rgba.width))
    host.set_ylim(float(rgba.height), 0.0)
    host.axis("off")


def _ch4_02b_panel_rect_size_in(fig, rect):
    _x0, _y0, rw, rh = rect
    fig_w, fig_h = fig.get_size_inches()
    return (
        max(float(rw) * float(fig_w), 0.05),
        max(float(rh) * float(fig_h), 0.05),
        float(CH4_02B_ANIM_DPI),
    )


def _ch4_02b_draw_2d_copy_panel(fig, copy_r, ws, we, bb, study, exam, y, *, panel_alpha=1.0):
    """Draw the sliding 2-D copy; rasterize and fade the whole panel uniformly."""
    pa = float(np.clip(float(panel_alpha), 0.0, 1.0))
    sub_w, sub_h, dpi = _ch4_02b_panel_rect_size_in(fig, copy_r)
    rgba = _ch4_02b_render_2d_panel_image(
        ws, we, bb, study, exam, y,
        width_in=sub_w, height_in=sub_h, dpi=dpi,
    )
    if pa < 1.0 - 1e-6:
        rgba = _ch4_02b_panel_rgba_faded(rgba, pa)
    _ch4_02b_blit_rgba_panel(fig, copy_r, rgba, zorder=10)


def _ch4_02b_render_sig3d_panel_image(
    ws, we, bb, study, exam, y, *,
    width_in, height_in, dpi, elev, azim, morph_u=1.0, sig_erase_u=0.0,
):
    sub_fig = plt.figure(figsize=(float(width_in), float(height_in)), dpi=float(dpi), facecolor="white")
    sub_ax = sub_fig.add_axes([0.0, 0.0, 1.0, 1.0], projection="3d", facecolor="white")
    _ch4_02b_draw_sig3d_panel(
        sub_ax, ws, we, bb, study, exam, y,
        morph_u=float(morph_u), elev=elev, azim=azim,
        sig_erase_u=sig_erase_u, cmap_instant=True,
    )
    img = fig_to_image(sub_fig, dpi=float(dpi), transparent=False).convert("RGBA")
    plt.close(sub_fig)
    return img


def _ch4_02b_draw_sig3d_fade_panel(
    fig, sig_r, ws, we, bb, study, exam, y, *,
    panel_alpha=1.0, elev, azim, morph_u=1.0, sig_erase_u=0.0,
):
    """Draw σ 3-D in the duo slot; rasterize when fading in."""
    pa = float(np.clip(float(panel_alpha), 0.0, 1.0))
    if pa >= 1.0 - 1e-6:
        ax3d = fig.add_axes(sig_r, projection="3d")
        _ch4_02b_draw_sig3d_panel(
            ax3d, ws, we, bb, study, exam, y,
            morph_u=float(morph_u), elev=elev, azim=azim,
            sig_erase_u=sig_erase_u, cmap_instant=True,
        )
        return
    sub_w, sub_h, dpi = _ch4_02b_panel_rect_size_in(fig, sig_r)
    rgba = _ch4_02b_render_sig3d_panel_image(
        ws, we, bb, study, exam, y,
        width_in=sub_w, height_in=sub_h, dpi=dpi,
        elev=elev, azim=azim, morph_u=morph_u, sig_erase_u=sig_erase_u,
    )
    rgba = _ch4_02b_panel_rgba_faded(rgba, pa)
    _ch4_02b_blit_rgba_panel(fig, sig_r, rgba, zorder=5)


def _ch4_02b_sig3d_rect(_sig_u, _copy_r, duo_3d):
    """Right panel for σ 3-D — always the full duo slot (surface morph is separate)."""
    return duo_3d


def _ch4_02b_wide_layout():
    x0, y0, w, h = CH4_02B_WIDE_DATA
    kw = w / 3.2
    kh = CH4_02B_WIDE_KNOB_ROW
    gap = 0.02
    total = 3 * kw + 2 * gap
    x_start = x0 + 0.5 * (w - total)
    knobs = []
    for j in range(3):
        knobs.append((x_start + j * (kw + gap), y0 - kh - 0.02, kw, kh))
    return (x0, y0, w, h), tuple(knobs)


def _ch4_02b_duo_layout():
    global _CH4_02B_DUO_RECTS
    if _CH4_02B_DUO_RECTS is not None:
        return _CH4_02B_DUO_RECTS
    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    fig.canvas.draw()
    data_r = _ch4_02b_rect(ax_data)
    knob_rs = tuple(_ch4_02b_rect(ax) for ax in axes_k)
    r3d = _ch4_02b_rect(ax3d)
    plt.close(fig)
    _CH4_02B_DUO_RECTS = (data_r, knob_rs, r3d)
    return _CH4_02B_DUO_RECTS


def _ch4_02b_sig_mesh():
    global _CH4_02B_SIG_MESH
    if _CH4_02B_SIG_MESH is not None:
        return _CH4_02B_SIG_MESH
    n = int(CH4_02B_SIG3D_GRID)
    st = np.linspace(float(xlim[0]), float(xlim[1]), n, dtype=np.float64)
    el = np.linspace(float(ylim[0]), float(ylim[1]), n, dtype=np.float64)
    ST, EL = np.meshgrid(st, el, indexing="xy")
    _CH4_02B_SIG_MESH = (ST, EL)
    return _CH4_02B_SIG_MESH


def _ch4_02b_sigma_grid(ws, we, bb):
    ST, EL = _ch4_02b_sig_mesh()
    Z = sigmoid(logits_plane(float(ws), float(we), float(bb), ST, EL))
    return ST, EL, Z


def _ch4_02b_st_el_unit(ST, EL, *, origin="lo_lo"):
    xr = max(float(xlim[1]) - float(xlim[0]), 1e-9)
    yr = max(float(ylim[1]) - float(ylim[0]), 1e-9)
    u1 = (ST - float(xlim[0])) / xr
    if str(origin) == "lo_hi":
        u2 = (float(ylim[1]) - EL) / yr
    else:
        u2 = (EL - float(ylim[0])) / yr
    return u1, u2


def _ch4_02b_diag_mask(u1, u2, t, *, erase=False):
    t = float(ch3_knob_smoothstep(np.clip(float(t), 0.0, 1.0)))
    s = u1 + u2
    if erase:
        return s > 2.0 * t + 1e-9
    return s <= 2.0 * t + 1e-9


def _ch4_02b_is_top_view(elev):
    return float(elev) >= float(CH4_02B_ELEV_TOP) - 12.0


def _ch4_02b_sig_facecolors(ST, EL, Zsig, morph_u, *, erase_u=0.0, origin=None):
    if origin is None:
        origin = CH4_02B_SIG_REVEAL_ORIGIN
    mu = float(ch3_knob_smoothstep(np.clip(float(morph_u), 0.0, 1.0)))
    u1, u2 = _ch4_02b_st_el_unit(ST, EL, origin=origin)
    morph_mask = _ch4_02b_diag_mask(u1, u2, mu, erase=False)
    erase_mask = _ch4_02b_diag_mask(u1, u2, erase_u, erase=True)
    fc = np.empty(ST.shape + (4,), dtype=float)
    base_a = float(_CH3_SIGMA_CONTOUR_ALPHA)
    for i in range(ST.shape[0]):
        for j in range(ST.shape[1]):
            fc[i, j] = mpl.colors.to_rgba(CMAP_GD(float(Zsig[i, j])))
            fc[i, j, 3] = base_a * float(morph_mask[i, j]) * float(erase_mask[i, j])
    return fc


def _ch4_02b_legend_three_starts(fig, axd, renderer, data_r):
    leg = axd.get_legend()
    if leg is None or not leg.get_texts():
        ddx, ddy, ddw, ddh = data_r
        s1 = (ddx + 0.08 * ddw, ddy + 0.88 * ddh)
        s2 = (ddx + 0.30 * ddw, ddy + 0.88 * ddh)
        s3 = (2.0 * s2[0] - s1[0], 2.0 * s2[1] - s1[1])
        return s1, s2, s3
    bb = leg.get_window_extent(renderer=renderer)
    inv = fig.transFigure.inverted()
    y_disp = float(bb.y0 + 0.42 * bb.height)
    x_lo = float(bb.x0 + 0.10 * bb.width)
    x_hi = float(bb.x1 - 0.10 * bb.width)
    out = []
    for alpha in (0.18, 0.52, 0.86):
        xf, yf = inv.transform((x_lo + alpha * (x_hi - x_lo), y_disp))
        out.append((float(xf), float(yf)))
    return tuple(out)


def _ch4_02b_knob_side_from_slot(slot_rect):
    _, _, w, h = slot_rect
    return float(min(w, h))


def _ch4_02b_knob_target_rects(data_r, knob_row_band):
    """Canonical knob slots scaled to the current data panel (matches ``ch3_draw_knob_row``)."""
    data_canon, knob_canon = ch3_duo_left_layout_rects()
    dx0, _, dw, _ = data_canon
    x0d, _, wd, _ = data_r
    wd = max(float(wd), 1e-9)
    dw = max(float(dw), 1e-9)
    y0 = min(float(r[1]) for r in knob_row_band)
    y1 = max(float(r[1]) + float(r[3]) for r in knob_row_band)
    h = max(y1 - y0, 1e-6)
    out = []
    for i in range(3):
        x0k, _, wk, _ = knob_canon[i]
        rel_x0 = (float(x0k) - float(dx0)) / dw
        rel_w = float(wk) / dw
        w = rel_w * wd
        x0 = float(x0d) + rel_x0 * wd
        out.append((x0, y0, w, h))
    return tuple(out)


def _ch4_02b_add_knob_rect(fig, img, x0, y0, w, h):
    axk = fig.add_axes((float(x0), float(y0), float(w), float(h)))
    axk.imshow(np.asarray(img), interpolation="nearest")
    axk.axis("off")


def _ch4_02b_add_knob(fig, img, cx, cy, side):
    side = float(side)
    _ch4_02b_add_knob_rect(fig, img, cx - 0.5 * side, cy - 0.5 * side, side, side)


def _ch4_02b_place_knobs_row(fig, ax_left, data_r, knob_rs, ws, we, bb):
    """Draw full-size numbered knobs — same geometry as the NLL duo segment."""
    targets = _ch4_02b_knob_target_rects(data_r, knob_rs)
    axes_k = tuple(fig.add_axes(r) for r in targets)
    for ax in axes_k:
        ax.axis("off")
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, float(ws), float(we), float(bb), "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(float(ws), float(we), float(bb)),
        knob_scales=[1.0, 1.0, 1.0], ax_data=ax_left,
    )


def _ch4_02b_place_knobs_flyin(fig, ax_left, data_r, knob_rs, ws, we, bb, *, grow_u=1.0):
    """Fly all three knobs from legend params down into their slots (ch2/ch3 style)."""
    ch3_ensure_knob_pngs()
    knob_rgbs, canvas_sides = ch3_knob_asset_pack()
    rots = ch3_k1_knob_rots_at(float(ws), float(we), float(bb))
    grow_u = float(np.clip(float(grow_u), 0.0, 1.0))
    if grow_u <= 1e-4:
        return
    if grow_u >= 1.0 - 1e-6:
        _ch4_02b_place_knobs_row(fig, ax_left, data_r, knob_rs, ws, we, bb)
        return
    u = float(ch3_knob_smoothstep(grow_u))
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    s1c, s2c, s3c = _ch4_02b_legend_three_starts(fig, ax_left, renderer, data_r)
    starts = (s1c, s2c, s3c)
    targets = _ch4_02b_knob_target_rects(data_r, knob_rs)
    tiny_w = max(0.006, 0.02 * (1.0 - u) + 0.001)
    for i, (sc, tgt) in enumerate(zip(starts, targets)):
        tx0, ty0, tw, th = tgt
        tcx = tx0 + 0.5 * tw
        tcy = ty0 + 0.5 * th
        sx, sy = sc
        cx = sx + u * (tcx - sx)
        cy = sy + u * (tcy - sy)
        rw = float(tiny_w + u * (tw - tiny_w))
        rh = float(tiny_w + u * (th - tiny_w))
        arr = np.asarray(
            _ch3_knob_pil_rotated_square(knob_rgbs[i], float(rots[i]), canvas_sides[i]),
            dtype=np.uint8,
        )
        _ch4_02b_add_knob_rect(fig, Image.fromarray(arr), cx - 0.5 * rw, cy - 0.5 * rh, rw, rh)


def _ch4_02b_draw_3d_icons(ax, study, exam, y, prob_z, *, icon_span=None, morph_u=1.0, erase_u=0.0, elev=None):
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection

    morph_u = float(ch3_knob_smoothstep(np.clip(float(morph_u), 0.0, 1.0)))
    erase_u = float(ch3_knob_smoothstep(np.clip(float(erase_u), 0.0, 1.0)))
    if erase_u >= 1.0 - 1e-6:
        return
    xr = float(xlim[1] - xlim[0])
    yr = float(ylim[1] - ylim[0])
    span = float(icon_span if icon_span is not None else min(xr, yr) * 0.045)
    nx, ny = 16, 16
    polys = []
    fcols = []
    study = np.asarray(study, dtype=float)
    exam = np.asarray(exam, dtype=float)
    y = np.asarray(y, dtype=int)
    prob_z = np.asarray(prob_z, dtype=float)
    for k in range(len(study)):
        s = float(study[k])
        e = float(exam[k])
        z = float(prob_z[k])
        u1 = (s - float(xlim[0])) / max(xr, 1e-9)
        if str(CH4_02B_SIG_REVEAL_ORIGIN) == "hi_hi":
            u2 = (float(ylim[1]) - e) / max(yr, 1e-9)
        else:
            u2 = (e - float(ylim[0])) / max(yr, 1e-9)
        if not _ch4_02b_diag_mask(u1, u2, morph_u, erase=False):
            continue
        if not _ch4_02b_diag_mask(u1, u2, erase_u, erase=True):
            continue
        img_arr = CHECK_ICON if int(y[k]) == 1 else CROSS_ICON
        im = Image.fromarray(np.asarray(img_arr, dtype=np.uint8), mode="RGBA")
        if int(y[k]) == 1:
            im = im.transpose(Image.FLIP_TOP_BOTTOM)
        im = im.resize((nx, ny), Image.LANCZOS)
        pix = np.asarray(im).astype(float) / 255.0
        for jj in range(ny):
            for ii in range(nx):
                rgba = pix[jj, ii]
                if rgba[3] < 0.06:
                    continue
                x0 = s - 0.5 * span + (ii / nx) * span
                x1 = s - 0.5 * span + ((ii + 1) / nx) * span
                y0 = e - 0.5 * span + (jj / ny) * span
                y1 = e - 0.5 * span + ((jj + 1) / ny) * span
                polys.append([[x0, y0, z], [x1, y0, z], [x1, y1, z], [x0, y1, z]])
                fcols.append((rgba[0], rgba[1], rgba[2], float(rgba[3]) * 0.97))
    if polys:
        ax.add_collection3d(
            Poly3DCollection(polys, facecolors=fcols, edgecolors="none", linewidths=0.0, shade=False)
        )


def _ch4_02b_style_sig3d_ax(ax, *, elev, azim, morph_u, hide_z=False):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(0.0, 1.0)
    ax.view_init(elev=float(elev), azim=float(azim))
    ax.set_xlabel("Study time (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    ax.set_ylabel("Exam length (hours)", fontsize=AXIS_LABEL_SIZE, labelpad=10)
    at_top = _ch4_02b_is_top_view(elev)
    if hide_z or at_top:
        ax.set_zlabel("")
        ax.set_zticks([])
        ax.set_zticklabels([])
    elif float(morph_u) >= 0.55:
        ax.set_zlabel(r"$P(\mathrm{pass}\mid x)$", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    else:
        ax.set_zlabel("")
        ax.set_zticks([])
        ax.set_zticklabels([])
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE - 1)


def _ch4_02b_draw_sig3d_panel(ax, ws, we, bb, study, exam, y, *, morph_u, elev, azim, sig_erase_u=0.0, cmap_instant=False):
    morph_u = float(np.clip(float(morph_u), 0.0, 1.0))
    mu = ch3_knob_smoothstep(morph_u)
    cmap_u = 1.0 if bool(cmap_instant) else morph_u
    ST, EL, Zsig = _ch4_02b_sigma_grid(ws, we, bb)
    Z = Zsig
    fc = _ch4_02b_sig_facecolors(ST, EL, Zsig, cmap_u, erase_u=sig_erase_u)
    ax.plot_surface(
        ST, EL, Z,
        facecolors=fc,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    prob = sigmoid(logits_plane(float(ws), float(we), float(bb), study, exam))
    z_pts = prob
    icon_u = 1.0 if bool(cmap_instant) else morph_u
    _ch4_02b_draw_3d_icons(ax, study, exam, y, z_pts, morph_u=icon_u, erase_u=sig_erase_u, elev=elev)
    bxy = boundary_line_xy(float(ws), float(we), float(bb), float(xlim[0]), float(xlim[1]), float(ylim[0]), float(ylim[1]))
    erase_u = float(ch3_knob_smoothstep(np.clip(float(sig_erase_u), 0.0, 1.0)))
    if bxy is not None and mu > 0.08 and erase_u < 1.0 - 1e-6:
        bx, by = bxy
        zline = np.full_like(bx, 0.5)
        ax.plot(bx, by, zline, color="0.35", linestyle="--", linewidth=1.2, zorder=4, alpha=max(0.0, 1.0 - erase_u))
    _ch4_02b_style_sig3d_ax(ax, elev=elev, azim=azim, morph_u=mu)


def _ch4_02b_fig(
    ws, we, bb, study, exam, y, *,
    layout_u=0.0,
    knob_grow_u=0.0,
    copy_u=0.0,
    sig3d_u=0.0,
    sig_erase_u=0.0,
    elev=CH4_02B_ELEV_TOP,
    azim=CH4_02B_AZIM_TOP,
):
    wide_data, wide_knobs = _ch4_02b_wide_layout()
    duo_data, duo_knobs, duo_3d = _ch4_02b_duo_layout()
    lu = float(np.clip(float(layout_u), 0.0, 1.0))
    data_r = _ch4_02b_lerp_rect(lu, wide_data, duo_data)
    knob_rs = tuple(_ch4_02b_lerp_rect(lu, wide_knobs[i], duo_knobs[i]) for i in range(3))

    fig = plt.figure(figsize=CH4_DUO_FIGSIZE)
    fig.patch.set_facecolor("white")
    ax_left = fig.add_axes(data_r)
    _ch4_02b_draw_2d_panel(ax_left, ws, we, bb, study, exam, y)
    _ch4_02b_place_knobs_flyin(fig, ax_left, data_r, knob_rs, ws, we, bb, grow_u=knob_grow_u)

    cu = float(np.clip(float(copy_u), 0.0, 1.0))
    su = float(np.clip(float(sig3d_u), 0.0, 1.0))
    copy_alpha = _ch4_02b_copy_panel_alpha(cu)
    sig3d_alpha = 1.0 if su > 1e-4 else _ch4_02b_sig3d_panel_alpha(cu)
    sig_r = _ch4_02b_sig3d_rect(su, None, duo_3d)
    show_copy = cu > 1e-4 and copy_alpha > 1e-4
    show_3d = sig3d_alpha > 1e-4
    if show_3d:
        _ch4_02b_draw_sig3d_fade_panel(
            fig, sig_r, ws, we, bb, study, exam, y,
            panel_alpha=sig3d_alpha,
            morph_u=su if su > 1e-4 else 1.0,
            elev=elev, azim=azim, sig_erase_u=sig_erase_u,
        )
    if show_copy:
        copy_r = _ch4_02b_copy_rect(cu, data_r, duo_3d)
        _ch4_02b_draw_2d_copy_panel(
            fig, copy_r, ws, we, bb, study, exam, y, panel_alpha=copy_alpha,
        )
    fig.canvas.draw()
    return fig_to_image(fig, dpi=_ch4_02b_anim_dpi())


def _ch4_02b_compose(plot_img):
    return _ch4_lik_03_opening_compose(plot_img, layout_u=0.0)


def _ch4_02b_lerp_weights(w_a, w_b, t):
    t = float(np.clip(float(t), 0.0, 1.0))
    u = ch3_knob_smoothstep(t)
    return tuple(float(w_a[i] + u * (float(w_b[i]) - float(w_a[i]))) for i in range(3))


def _ch4_02b_nll_weights():
    ws, we, bb = CH4_02B_W_NLL
    return float(ws), float(we), float(bb)


def _ch4_02b_knob_end_weights():
    ws, we, bb = CH4_02B_W_KNOB_END
    return float(ws), float(we), float(bb)


def _ch4_02b_nll_at(ws, we, bb):
    return float(loss_neg_log_likelihood(
        float(ws), float(we), float(bb), study_sep, exam_sep, y_sep,
    ))


def _ch4_02b_marker_path_point(u):
    """Piecewise path: knob end → origin → final NLL weights."""
    u = float(np.clip(float(u), 0.0, 1.0))
    w_a = CH4_02B_W_KNOB_END
    w_b = CH4_02B_W_MID
    w_c = CH4_02B_W_NLL
    if u <= 0.5:
        ws, we, bb = _ch4_02b_lerp_weights(w_a, w_b, u / 0.5)
    else:
        ws, we, bb = _ch4_02b_lerp_weights(w_b, w_c, (u - 0.5) / 0.5)
    return float(ws), float(we), float(bb), _ch4_02b_nll_at(ws, we, bb)


def _ch4_02b_ch03_plot(**kw):
    """Duo plot with ch4_03 NLL surface morph on the right (no tutorial rails)."""
    show_here = bool(kw.pop("show_here_marker", False))
    if show_here:
        kw["marker"] = True
        kw["here_annotation"] = True
        kw.setdefault("here_label", "WE ARE HERE")
        kw.setdefault("marker_color", CH4_03_MARKER_COLOR)
        kw.setdefault("marker_edgecolors", CH4_03_MARKER_EDGE)
    disp_ws = kw.pop("display_ws", None)
    disp_we = kw.pop("display_we", None)
    disp_b = kw.pop("display_b", None)
    ws, we, bb = _ch4_02b_nll_weights()
    if disp_ws is not None:
        ws = float(disp_ws)
    if disp_we is not None:
        we = float(disp_we)
    if disp_b is not None:
        bb = float(disp_b)
    if show_here:
        kw.setdefault("marker_ws", float(kw.get("marker_ws", ws)))
        kw.setdefault("marker_we", float(kw.get("marker_we", we)))
        if (
            "marker_z" not in kw
            and float(kw.get("nll_u", 1.0)) >= 1.0 - 1e-6
            and float(kw.get("squish_u", 0.0)) <= 1e-6
        ):
            kw["marker_z"] = _ch4_02b_nll_at(
                float(kw["marker_ws"]), float(kw["marker_we"]), bb,
            )
    state = _ch4_02b_lik_state()
    plot = ch3_frame_lik_w12_single_surface(
        state,
        knob_labeled_blend=(1.0, 1.0, 1.0),
        display_ws=ws,
        display_we=we,
        display_b=bb,
        **kw,
    )
    return plot


def _ch4_02b_weight_diag_mask(w1, w2, bounds, t, *, erase=False):
    dlo1, dhi1, dlo2, dhi2, _, _ = bounds
    u1 = (np.asarray(w1, dtype=float) - float(dlo1)) / max(float(dhi1) - float(dlo1), 1e-9)
    u2 = (np.asarray(w2, dtype=float) - float(dlo2)) / max(float(dhi2) - float(dlo2), 1e-9)
    return _ch4_02b_diag_mask(u1, u2, t, erase=erase)


def _ch4_02b_draw_ct_mesh(ax3d, w1, w2, b, nll, bounds, *, surface_erase_u=0.0):
    from ch4_layout import ch4_nll_heatmap_cmap

    lo, hi = ch4_nll_global_scale()
    cmap = ch4_nll_heatmap_cmap()
    span = max(float(hi) - float(lo), 1e-9)
    arr = np.asarray(nll, dtype=float)
    normed = np.clip((arr - float(lo)) / span, 0.0, 1.0)
    face = np.asarray(cmap(normed), dtype=float)
    if face.shape[-1] == 3:
        face = np.dstack([face, np.ones(face.shape[:2], dtype=float)])
    al = float(CH3_LIK_CT_PLANE_ALPHA)
    visible = _ch4_02b_weight_diag_mask(w1, w2, bounds, surface_erase_u, erase=True)
    face[..., 3] = visible.astype(float) * al
    ax3d.plot_surface(
        w1, w2, b,
        facecolors=face,
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=False,
        shade=False,
        zorder=6,
    )


def _ch4_02b_plot_ct_scan(pack, *, sweep_axis=None, plane_val=None, pivot_from=None, pivot_to=None, pivot_u=0.0, cam_azim_u=0.0):
    """CT scan duo frame — plot only, no rails (matches ch4_05a geometry)."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    bounds = pack["bounds"]
    ws, we, bb = _ch4_02b_nll_weights()
    if str(sweep_axis) == "b" and plane_val is not None:
        b_show = float(plane_val)
    else:
        b_show = float(CH3_LIK_CH4_PLANE_B)
    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, b_show, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, b_show, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, b_show, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, b_show), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds
    ax3d.set_autoscale_on(False)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    ch4_lik_ct_view_init(ax3d, cam_azim_u=float(cam_azim_u))
    ct_gn = _ch4_02b_ct_grid()
    if pivot_from is not None and pivot_to is not None:
        w1, w2, b, nll = _ch4_ct_pivot_mesh(
            study, exam, y, str(pivot_from), str(pivot_to), bounds,
            theta_u=float(pivot_u), gn=ct_gn,
        )
    else:
        w1, w2, b = _ch4_ct_sweep_mesh(str(sweep_axis), float(plane_val), bounds, gn=ct_gn)
        nll = _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b)
    _ch4_02b_draw_ct_mesh(ax3d, w1, w2, b, nll, bounds, surface_erase_u=0.0)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    plot_img = fig_to_image(fig, dpi=_ch4_02b_anim_dpi())
    plt.close(fig)
    return plot_img


def _ch4_02b_plot_opening_step(pack, *, surface_erase_u=0.0, relabel_u=0.0, cam_u=0.0):
    """Erase CT surface, relabel axes, rotate camera to the ch4_02 opening — no crossfade."""
    from ch4_layout import ch4_knob_asset_pack

    study, exam, y = pack["study"], pack["exam"], pack["y"]
    bounds = pack["bounds"]
    ws, we, bb = _ch4_02b_nll_weights()
    b_show = float(CH3_LIK_CH4_PLANE_B)
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = bounds

    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    leg = legend_linear_equation_values_bold_param(ws, we, b_show, "all")
    ch3_draw_left_panel(
        ax_data, ws, we, b_show, study, exam, y, leg,
        show_colormap=True, highlight_mistakes_flag=False,
    )
    ax_data.set_xlim(*xlim)
    ax_data.set_ylim(*ylim)
    finalize_style_legend_tex(ax_data)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, b_show, "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(ws, we, b_show), knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )

    erase_u = float(ch3_knob_smoothstep(np.clip(float(surface_erase_u), 0.0, 1.0)))
    rel_u = float(ch3_knob_smoothstep(np.clip(float(relabel_u), 0.0, 1.0)))
    cam_s = float(ch3_knob_smoothstep(np.clip(float(cam_u), 0.0, 1.0)))
    elev = float(CH3_LIK_CH4_CT_ELEV + cam_s * (CH3_LIK_W12_ELEV_W1 - CH3_LIK_CH4_CT_ELEV))
    azim = float(_ch3_lik_w12_lerp_azim_shortest(CH3_LIK_CH4_CT_AZIM, CH3_LIK_W12_AZIM_W1, cam_s))

    ax3d.set_autoscale_on(False)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    ax3d.view_init(elev=elev, azim=azim)
    if erase_u < 1.0 - 1e-6:
        w1, w2, b = _ch4_ct_sweep_mesh(
            "b", float(CH3_LIK_CH4_PLANE_B), bounds, gn=_ch4_02b_ct_grid(),
        )
        nll = _ch4_ct_nll_at_grid(study, exam, y, w1, w2, b)
        _ch4_02b_draw_ct_mesh(ax3d, w1, w2, b, nll, bounds, surface_erase_u=erase_u)
    if rel_u > 0.5:
        ax3d.set_zlabel("Likelihood", fontsize=AXIS_LABEL_SIZE, labelpad=8)
    else:
        ax3d.set_zlabel(r"$b$", fontsize=AXIS_LABEL_SIZE, labelpad=8)

    plot_img = fig_to_image(fig, dpi=_ch4_02b_anim_dpi())
    plt.close(fig)
    return plot_img


def _ch4_02b_plot_02_opening(*, elev=None, azim=None):
    """Empty 3-D at the ch4_02 opening pose (knob-1 side view)."""
    from ch4_layout import ch4_knob_asset_pack

    handoff = _ch4_lik_02_03_handoff_state()
    mesh0 = handoff["mesh"]
    z_lim = handoff["z_lim"]
    ws, we, bb = _ch4_02b_nll_weights()
    el = float(CH3_LIK_W12_ELEV_W1 if elev is None else elev)
    az = float(CH3_LIK_W12_AZIM_W1 if azim is None else azim)
    return ch3_frame_lik_w12_3d(
        study_sep, exam_sep, y_sep,
        ws, we, bb,
        mesh_pack=mesh0, z_lim=z_lim, curves=[],
        elev=el, azim=az, emphasize_knob="st",
        landscape_reveal=0.0, show_curves=False, marker=False,
        knob_pack=ch4_knob_asset_pack(),
        knob_scales=[1.0, 1.0, 1.0],
        weight_axis_labels=True,
        z_label="Likelihood",
    )


def _ch4_02b_append_ct_scan(frames, pack):
    """Full CT scan — same axis order / holds / pivots as ch4_05a."""
    bounds = pack["bounds"]
    pivot_map = dict(CH3_LIK_CT_PIVOTS)
    for _ in range(int(CH4_02B_N_CT_HOLD)):
        frames.append(_ch4_02b_compose(_ch4_02b_plot_ct_scan(
            pack, sweep_axis="b", plane_val=float(CH3_LIK_CH4_PLANE_B),
        )))
    for i, axis in enumerate(CH3_LIK_CT_AXES):
        if i > 0:
            prev = CH3_LIK_CT_AXES[i - 1]
            nxt = pivot_map.get(prev)
            if nxt == axis:
                for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_CT_PIVOT), endpoint=True):
                    frames.append(_ch4_02b_compose(_ch4_02b_plot_ct_scan(
                        pack, pivot_from=prev, pivot_to=axis, pivot_u=float(tv),
                    )))
        lo, hi = _ch4_ct_axis_limits(axis, bounds)
        for _ in range(int(CH4_02B_N_CT_HOLD)):
            frames.append(_ch4_02b_compose(_ch4_02b_plot_ct_scan(
                pack, sweep_axis=axis, plane_val=lo,
            )))
        for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_CT_SWEEP), endpoint=True):
            u = ch3_knob_smoothstep(float(tv))
            val = lo + u * (hi - lo)
            frames.append(_ch4_02b_compose(_ch4_02b_plot_ct_scan(
                pack, sweep_axis=axis, plane_val=val,
            )))
        for _ in range(max(4, int(CH4_02B_N_CT_HOLD) // 3)):
            frames.append(_ch4_02b_compose(_ch4_02b_plot_ct_scan(
                pack, sweep_axis=axis, plane_val=hi,
            )))


def _ch4_02b_append_opening_transition(frames, ct_pack):
    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_ERASE_OPEN), endpoint=True):
        frames.append(_ch4_02b_compose(_ch4_02b_plot_opening_step(
            ct_pack, surface_erase_u=float(tv), relabel_u=0.0, cam_u=0.0,
        )))
    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_RELOC_CAM), endpoint=True):
        u = float(tv)
        frames.append(_ch4_02b_compose(_ch4_02b_plot_opening_step(
            ct_pack, surface_erase_u=1.0, relabel_u=u, cam_u=u,
        )))
    opening = _ch4_02b_compose(_ch4_02b_plot_02_opening())
    _ch4_02b_append_hold(frames, opening, CH4_02B_N_HOLD_END)


def _ch4_02b_append_nll_tail(frames, ws, we, bb):
    kw_end = CH4_02B_W_KNOB_END
    znll_end = _ch4_02b_nll_at(*kw_end)
    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_SIG_ERASE), endpoint=True):
        frames.append(_ch4_02b_compose(_ch4_02b_fig(
            ws, we, bb, study_sep, exam_sep, y_sep,
            layout_u=1.0, knob_grow_u=1.0, copy_u=1.0, sig3d_u=1.0,
            sig_erase_u=float(tv),
            elev=CH4_02B_ELEV_VIEW, azim=CH4_02B_AZIM_VIEW,
        )))

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_NLL_REVEAL), endpoint=True):
        u = float(ch3_knob_smoothstep(float(tv)))
        frames.append(_ch4_02b_compose(_ch4_02b_ch03_plot(
            log_u=1.0, nll_u=1.0, heatmap_u=0.0, landscape_reveal=u,
            show_here_marker=True,
            display_ws=kw_end[0], display_we=kw_end[1], display_b=kw_end[2],
            marker_ws=kw_end[0], marker_we=kw_end[1], marker_z=znll_end,
        )))

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_MARKER_PATH), endpoint=True):
        mw, me, mb, mz = _ch4_02b_marker_path_point(float(tv))
        frames.append(_ch4_02b_compose(_ch4_02b_ch03_plot(
            log_u=1.0, nll_u=1.0, heatmap_u=0.0, landscape_reveal=1.0,
            show_here_marker=True,
            display_ws=mw, display_we=me, display_b=mb,
            marker_ws=mw, marker_we=me, marker_z=mz,
        )))
    _ch4_02b_append_hold(frames, frames[-1], max(int(CH4_02B_N_HOLD_WIDE) // 4, 2))

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_HEATMAP), endpoint=True):
        u = float(ch3_knob_smoothstep(float(tv)))
        frames.append(_ch4_02b_compose(_ch4_02b_ch03_plot(
            log_u=1.0, nll_u=1.0, heatmap_u=u, landscape_reveal=1.0,
            show_here_marker=True,
        )))

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_SQUISH), endpoint=True):
        u = float(ch3_knob_smoothstep(float(tv)))
        frames.append(_ch4_02b_compose(_ch4_02b_ch03_plot(
            log_u=1.0, nll_u=1.0, heatmap_u=1.0, squish_u=u, plane_drop_u=0.0,
            landscape_reveal=1.0, show_here_marker=True,
        )))

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_PLANE_DROP), endpoint=True):
        u = float(ch3_knob_smoothstep(float(tv)))
        frames.append(_ch4_02b_compose(_ch4_02b_ch03_plot(
            log_u=1.0, nll_u=1.0, heatmap_u=1.0, squish_u=1.0, plane_drop_u=u,
            landscape_reveal=1.0, show_here_marker=True,
        )))

    ct_pack = _ch3_lik_ct_scan_pack()
    _ch4_02b_append_ct_scan(frames, ct_pack)
    _ch4_02b_append_opening_transition(frames, ct_pack)


def _ch4_02b_append_hold(frames, im, n):
    for _ in range(max(int(n), 1)):
        frames.append(im.copy() if hasattr(im, "copy") else im)


def ch4_build_frames_sigmoid_duo_bridge_story():
    study, exam, y = study_sep, exam_sep, y_sep
    ws, we, bb = CH4_02B_W_START
    frames = []

    im = _ch4_02b_compose(_ch4_02b_fig(ws, we, bb, study, exam, y, layout_u=0.0, knob_grow_u=0.0))
    _ch4_02b_append_hold(frames, im, CH4_02B_N_HOLD_WIDE)

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_INTRO), endpoint=True):
        iu = float(tv)
        frames.append(_ch4_02b_compose(
            _ch4_02b_fig(ws, we, bb, study, exam, y, layout_u=iu, knob_grow_u=iu),
        ))
    _ch4_02b_append_hold(frames, frames[-1], max(int(CH4_02B_N_HOLD_WIDE) // 2, 2))

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_COPY), endpoint=True):
        cu = float(tv)
        frames.append(_ch4_02b_compose(
            _ch4_02b_fig(ws, we, bb, study, exam, y, layout_u=1.0, knob_grow_u=1.0, copy_u=cu),
        ))
    _ch4_02b_append_hold(frames, frames[-1], max(int(CH4_02B_N_HOLD_WIDE) // 3, 2))

    for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_SIG3D), endpoint=True):
        su = float(tv)
        frames.append(_ch4_02b_compose(
            _ch4_02b_fig(
                ws, we, bb, study, exam, y,
                layout_u=1.0, knob_grow_u=1.0, copy_u=1.0, sig3d_u=su,
                elev=CH4_02B_ELEV_TOP, azim=CH4_02B_AZIM_TOP,
            ),
        ))
    _ch4_02b_append_hold(frames, frames[-1], max(int(CH4_02B_N_HOLD_WIDE) // 3, 2))

    cam_spec = np.linspace(0.0, 1.0, int(CH4_02B_N_CAMERA), endpoint=True)
    for tv in cam_spec:
        u = float(tv)
        elev = float(CH4_02B_ELEV_TOP + u * (CH4_02B_ELEV_VIEW - CH4_02B_ELEV_TOP))
        azim = float(CH4_02B_AZIM_TOP + u * (CH4_02B_AZIM_VIEW - CH4_02B_AZIM_TOP))
        frames.append(_ch4_02b_compose(
            _ch4_02b_fig(
                ws, we, bb, study, exam, y,
                layout_u=1.0, knob_grow_u=1.0, copy_u=1.0, sig3d_u=1.0,
                elev=elev, azim=azim,
            ),
        ))
    _ch4_02b_append_hold(frames, frames[-1], max(int(CH4_02B_N_HOLD_WIDE) // 3, 2))

    w_seq = [CH4_02B_W_START] + list(CH4_02B_W_DEMO) + [CH4_02B_W_KNOB_END]
    for seg in range(len(w_seq) - 1):
        w0 = w_seq[seg]
        w1 = w_seq[seg + 1]
        for tv in np.linspace(0.0, 1.0, int(CH4_02B_N_KNOB_SEG), endpoint=True):
            ws, we, bb = _ch4_02b_lerp_weights(w0, w1, float(tv))
            frames.append(_ch4_02b_compose(
                _ch4_02b_fig(
                    ws, we, bb, study, exam, y,
                    layout_u=1.0, knob_grow_u=1.0, copy_u=1.0, sig3d_u=1.0,
                    elev=CH4_02B_ELEV_VIEW, azim=CH4_02B_AZIM_VIEW,
                ),
            ))
        _ch4_02b_append_hold(frames, frames[-1], max(int(CH4_02B_N_HOLD_WIDE) // 4, 2))

    ws, we, bb = CH4_02B_W_KNOB_END
    _ch4_02b_append_nll_tail(frames, ws, we, bb)
    return frames


def ch4_preview_sigmoid_duo_bridge_frame(*, phase="wide"):
    study, exam, y = study_sep, exam_sep, y_sep
    phase = str(phase)
    if phase == "wide":
        im = _ch4_02b_fig(*CH4_02B_W_START, study, exam, y, layout_u=0.0, knob_grow_u=0.0)
    elif phase == "duo":
        im = _ch4_02b_fig(*CH4_02B_W_START, study, exam, y, layout_u=1.0, knob_grow_u=1.0)
    elif phase == "sig3d":
        im = _ch4_02b_fig(
            *CH4_02B_W_START, study, exam, y,
            layout_u=1.0, knob_grow_u=1.0, copy_u=1.0, sig3d_u=1.0,
            elev=CH4_02B_ELEV_VIEW, azim=CH4_02B_AZIM_VIEW,
        )
    elif phase == "end":
        return _ch4_02b_compose(_ch4_02b_plot_02_opening())
    else:
        raise ValueError(f"unknown ch4_02b preview phase: {phase!r}")
    if phase != "end":
        return _ch4_02b_compose(im)
    return im


def ch4_export_sigmoid_duo_bridge():
    _ch4_02b_quality_enter()
    try:
        frames = ch4_build_frames_sigmoid_duo_bridge_story()
        fn = "ch4_02b_sigmoid_duo_bridge.mp4"
        save_mp4(frames, fn, duration=int(CH4_02B_MS))
        print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
        return OUTPUT_DIR / fn
    finally:
        _ch4_02b_quality_exit()


# --- ch4_09: from ch4_10 end → voxel sweep (05a3) → study rescale back to hours ---

CH4_STUDY_MINUTES_SCALE = 60.0
CH4_STUDY_MINUTES_N_HOLD_END = 4 if _CH3_DRAFT else 10
CH4_STUDY_MINUTES_N_SWEEP = int(CH3_LIK_VOXEL_N_SWEEP)
CH4_STUDY_MINUTES_N_UNWARP = 6 if _CH3_DRAFT else 40
CH4_STUDY_MINUTES_N_HOLD_HOURS = 4 if _CH3_DRAFT else 12
CH4_STUDY_MINUTES_MS = 110 if not _CH3_DRAFT else 130
CH4_STUDY_MINUTES_VOXEL_CELL_MULT = 4
CH4_STUDY_MINUTES_VOXEL_GAP = 3


def _ch4_study_minutes_base_weights():
    handoff = _ch4_lik_02_03_handoff_state()
    return float(handoff["w_st"]), float(handoff["w_el"]), float(handoff["b"])


def _ch4_study_minutes_unit_fields(unit_u):
    """
    Study axis scale: ``unit_u=0`` hours, ``unit_u=1`` minutes (×60).
    Weights default to handoff opening pose unless overridden by caller.
    """
    u = float(np.clip(float(unit_u), 0.0, 1.0))
    smax = float(CH4_STUDY_MINUTES_SCALE)
    study_scale = 1.0 + u * (smax - 1.0)
    study = np.asarray(study_sep, dtype=float) * study_scale
    exam = np.asarray(exam_sep, dtype=float)
    ws0, we0, bb0 = _ch4_study_minutes_base_weights()
    ws = float(ws0)
    we = float(we0)
    bb = float(bb0)
    x0, x1 = float(xlim[0]), float(xlim[1])
    y0, y1 = float(ylim[0]), float(ylim[1])
    data_xlim = (x0, x0 + (x1 - x0) * study_scale)
    data_ylim = (y0, y1)
    st_line = np.linspace(data_xlim[0], data_xlim[1], int(ST_KNOB.shape[1]))
    el_line = np.linspace(data_ylim[0], data_ylim[1], int(EL_KNOB.shape[0]))
    sigma_stg, sigma_elg = np.meshgrid(st_line, el_line)
    if u < 0.04:
        xlabel = "Study time (hours)"
    elif u > 0.96:
        xlabel = "Study time (minutes)"
    else:
        xlabel = "Study time"
    return dict(
        unit_u=u,
        study_scale=study_scale,
        study=study,
        exam=exam,
        y=y_sep,
        ws=ws,
        we=we,
        bb=bb,
        data_xlim=data_xlim,
        data_ylim=data_ylim,
        sigma_stg=sigma_stg,
        sigma_elg=sigma_elg,
        xlabel=xlabel,
    )


def _ch4_study_gd_eight_end_state():
    """Pose, minutes-scale study fields, and view bounds at the end of ch4_10."""
    pack = _ch4_gd_eight_steps_pack()
    path = np.asarray(pack["path"], dtype=float)
    ws, we, bb = (float(path[-1, 0]), float(path[-1, 1]), float(path[-1, 2]))
    st = _ch4_study_minutes_unit_fields(1.0)
    st["ws"] = ws
    st["we"] = we
    st["bb"] = bb
    return st, pack["gd_view_bounds_end"]


def _ch4_study_minutes_voxel_pack(study, exam, y, *, bounds=None):
    pack = {
        "study": np.asarray(study, dtype=float),
        "exam": np.asarray(exam, dtype=float),
        "y": np.asarray(y, dtype=int),
        "voxel_cell_mult": int(CH4_STUDY_MINUTES_VOXEL_CELL_MULT),
        "voxel_gap_pitch": int(CH4_STUDY_MINUTES_VOXEL_GAP),
    }
    b = CH3_LIK_CT_VIEW_BOUNDS if bounds is None else bounds
    return _ch4_voxel_fill_warm_pack(pack, cell_mult=CH4_STUDY_MINUTES_VOXEL_CELL_MULT, bounds=b)


def _ch4_study_minutes_draw_2d(ax, st):
    leg = legend_linear_equation_values_bold_param(st["ws"], st["we"], st["bb"], "all")
    panel_kw = dict(
        show_colormap=True,
        highlight_mistakes_flag=False,
        sigma_stg=st["sigma_stg"],
        sigma_elg=st["sigma_elg"],
    )
    try:
        ch3_draw_left_panel(
            ax, st["ws"], st["we"], st["bb"],
            st["study"], st["exam"], st["y"], leg, **panel_kw,
        )
    except TypeError:
        ch3_draw_left_panel(
            ax, st["ws"], st["we"], st["bb"],
            st["study"], st["exam"], st["y"], leg,
            show_colormap=True, highlight_mistakes_flag=False,
        )
    ax.set_xlim(*st["data_xlim"])
    ax.set_ylim(*st["data_ylim"])
    ax.set_xlabel(str(st["xlabel"]), fontsize=AXIS_LABEL_SIZE, labelpad=10)
    finalize_style_legend_tex(ax)


def ch4_frame_study_minutes_warp(
    st, *,
    view_bounds=None,
    voxel_mode="empty",
    voxel_cache=None,
    sweep_u=0.0,
    show_cut_plane=True,
    cam_azim_u=0.35,
    cam_spin_deg=0.0,
):
    """Duo layout: 2D roster + 3D voxels (empty | sweep | full | grow)."""
    from ch4_layout import (
        CH4_FORMULAS_SECTION_TITLE,
        CH4_NOTATION_SECTION_TITLE,
        CH4_NLL_HEATMAP_SECTION_TITLE,
        ch4_cached_formula_blocks_gd_story,
        ch4_cached_notation_corner_blocks,
        ch4_knob_asset_pack,
        ch4_rails_cache_key,
        compose_tutorial,
    )

    if voxel_cache is None and str(voxel_mode) != "empty":
        voxel_cache = _ch4_study_minutes_voxel_pack(st["study"], st["exam"], st["y"])
    if view_bounds is None:
        view_bounds = CH3_LIK_CT_VIEW_BOUNDS

    fig, ax_data, ax3d, axes_k = ch4_figure_duo_weight3d()
    _ch4_study_minutes_draw_2d(ax_data, st)
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, st["ws"], st["we"], st["bb"], "all",
        knob_rgbs, canvas_sides,
        rot_strip_deg=0.0, strip_scale=1.0,
        knob_rots=ch3_k1_knob_rots_at(st["ws"], st["we"], st["bb"]),
        knob_scales=[1.0, 1.0, 1.0], ax_data=ax_data,
    )
    dlo1, dhi1, dlo2, dhi2, dlob, dhib = view_bounds
    ax3d.set_autoscale_on(False)
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    ch4_lik_ct_view_init(
        ax3d, cam_azim_u=float(cam_azim_u), cam_spin_deg=float(cam_spin_deg),
    )
    mode = str(voxel_mode)
    if mode == "sweep":
        _ch4_voxel_fill_draw_voxels(ax3d, voxel_cache, float(sweep_u))
        if show_cut_plane and float(sweep_u) < 1.0 - 1e-6:
            _ch4_voxel_fill_draw_cut_plane(ax3d, voxel_cache, float(sweep_u))
    elif mode == "full":
        _ch4_voxel_draw_all_checker(ax3d, voxel_cache)
    elif mode == "grow":
        half = 3.0 + float(sweep_u) * 27.0
        _ch4_voxel_draw_checker_cube(ax3d, voxel_cache, half)
    ax3d.scatter(
        [st["ws"]], [st["we"]], [st["bb"]],
        s=float(CH3_LIK_GD_POINT_S), c=[CH3_LIK_3D_POINT_COLOR], edgecolors="white",
        linewidths=2.0, depthshade=False, zorder=20,
    )
    _ch3_lik_style_ax3d(ax3d, dlo1, dhi1, dlo2, dhi2, dlob, dhib)
    plot_img = fig_to_image(fig, dpi=CH3_ANIM_DPI)
    plt.close(fig)
    right_blocks = voxel_cache.get("right_blocks", []) if voxel_cache is not None else []
    return compose_tutorial(
        plot_img,
        right_blocks=right_blocks,
        bottom_blocks=ch4_cached_formula_blocks_gd_story(),
        corner_blocks=ch4_cached_notation_corner_blocks(),
        right_title=CH4_NLL_HEATMAP_SECTION_TITLE,
        right_title_single_line=True,
        bottom_title=CH4_FORMULAS_SECTION_TITLE,
        corner_title=CH4_NOTATION_SECTION_TITLE,
        write_progress=1.0,
        rails_cache_key=ch4_rails_cache_key(gd_formulas=True),
        theme="classic_light",
    )


def ch4_build_frames_study_minutes_warp_story():
    """ch4_10 end (no path) → 05a3 voxel sweep → study minutes→hours with voxel morph."""
    frames = []
    st_end, view_end = _ch4_study_gd_eight_end_state()
    view_cube = CH3_LIK_CT_VIEW_BOUNDS
    end_ws, end_we, end_bb = st_end["ws"], st_end["we"], st_end["bb"]

    def _with_end_pose(st):
        st = dict(st)
        st["ws"] = float(end_ws)
        st["we"] = float(end_we)
        st["bb"] = float(end_bb)
        return st

    def emit(st, *, view_bounds=None, voxel_mode="empty", voxel_cache=None, sweep_u=0.0, show_cut_plane=True):
        frames.append(ch4_frame_study_minutes_warp(
            st,
            view_bounds=view_bounds,
            voxel_mode=voxel_mode,
            voxel_cache=voxel_cache,
            sweep_u=float(sweep_u),
            show_cut_plane=bool(show_cut_plane),
        ))

    for _ in range(max(int(CH4_STUDY_MINUTES_N_HOLD_END), 1)):
        emit(st_end, view_bounds=view_end, voxel_mode="empty")

    cache_minutes = _ch4_study_minutes_voxel_pack(st_end["study"], st_end["exam"], st_end["y"])
    for tv in np.linspace(0.0, 1.0, int(CH4_STUDY_MINUTES_N_SWEEP), endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        bounds = _ch3_lik_lerp_bounds(view_end, view_cube, u)
        emit(
            st_end,
            view_bounds=bounds,
            voxel_mode="sweep",
            voxel_cache=cache_minutes,
            sweep_u=u,
            show_cut_plane=True,
        )

    for tv in np.linspace(0.0, 1.0, int(CH4_STUDY_MINUTES_N_UNWARP), endpoint=True):
        u = 1.0 - ch3_knob_smoothstep(float(tv))
        st = _with_end_pose(_ch4_study_minutes_unit_fields(u))
        cache_u = _ch4_study_minutes_voxel_pack(st["study"], st["exam"], st["y"])
        emit(
            st,
            view_bounds=view_cube,
            voxel_mode="sweep",
            voxel_cache=cache_u,
            sweep_u=1.0,
            show_cut_plane=False,
        )

    st_hours = _with_end_pose(_ch4_study_minutes_unit_fields(0.0))
    cache_hours = _ch4_study_minutes_voxel_pack(st_hours["study"], st_hours["exam"], st_hours["y"])
    for _ in range(max(int(CH4_STUDY_MINUTES_N_HOLD_HOURS), 1)):
        emit(
            st_hours,
            view_bounds=view_cube,
            voxel_mode="sweep",
            voxel_cache=cache_hours,
            sweep_u=1.0,
            show_cut_plane=False,
        )

    return _ch3_lik_story_hold(frames)


def ch4_preview_study_minutes_warp_frame(*, unit_u=1.0, sweep_u=0.0, phase="end"):
    """Preview ch4_09 — ``phase`` end|sweep|unwarp|hours."""
    phase = str(phase)
    st_end, view_end = _ch4_study_gd_eight_end_state()
    end_ws, end_we, end_bb = st_end["ws"], st_end["we"], st_end["bb"]

    def _with_end_pose(st):
        st = dict(st)
        st["ws"], st["we"], st["bb"] = float(end_ws), float(end_we), float(end_bb)
        return st

    if phase == "end":
        return ch4_frame_study_minutes_warp(st_end, view_bounds=view_end, voxel_mode="empty")
    if phase == "sweep":
        cache = _ch4_study_minutes_voxel_pack(st_end["study"], st_end["exam"], st_end["y"])
        bounds = _ch3_lik_lerp_bounds(view_end, CH3_LIK_CT_VIEW_BOUNDS, float(sweep_u))
        return ch4_frame_study_minutes_warp(
            st_end,
            view_bounds=bounds,
            voxel_mode="sweep",
            voxel_cache=cache,
            sweep_u=float(sweep_u),
        )
    if phase in ("unwarp", "hours"):
        u = 0.0 if phase == "hours" else float(unit_u)
        st = _with_end_pose(_ch4_study_minutes_unit_fields(u))
        cache = _ch4_study_minutes_voxel_pack(st["study"], st["exam"], st["y"])
        return ch4_frame_study_minutes_warp(
            st,
            view_bounds=CH3_LIK_CT_VIEW_BOUNDS,
            voxel_mode="sweep",
            voxel_cache=cache,
            sweep_u=1.0,
            show_cut_plane=False,
        )
    st = _with_end_pose(_ch4_study_minutes_unit_fields(float(unit_u)))
    cache = _ch4_study_minutes_voxel_pack(st["study"], st["exam"], st["y"])
    return ch4_frame_study_minutes_warp(st, voxel_mode="sweep", voxel_cache=cache, sweep_u=float(sweep_u))


def ch4_export_study_minutes_warp():
    frames = ch4_build_frames_study_minutes_warp_story()
    fn = "ch4_09_study_minutes_warp.mp4"
    save_mp4(frames, fn, duration=int(CH4_STUDY_MINUTES_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_10 / ch4_11 shared bridge: ch4_07 end → study ×60 → axis zoom → 8-step run ---

CH4_EIGHT_HOURS_GHOST_ALPHA = 0.42
CH4_EIGHT_HOURS_GHOST_WP_ALPHA = 0.38
CH4_EIGHT_DISPLAY_BOUNDS_SCALE = 2.0
CH4_EIGHT_N_HOLD_07 = 4 if _CH3_DRAFT else 10
CH4_EIGHT_N_WARP = 6 if _CH3_DRAFT else 40
CH4_EIGHT_N_HOLD_START = 4 if _CH3_DRAFT else 8


def _ch4_eight_reference_gd_path(pack):
    """Full ch4_06/07 GD polyline (all ``CH3_LIK_GD_N_ITERS`` steps) in display coordinates."""
    hours = np.asarray(pack["hours_gd_path"], dtype=float)
    if float(pack.get("study_unit_u", 0.0)) < 0.04:
        return hours
    scale = float(pack.get("study_scale", 1.0))
    return _ch4_eight_study_morphed_path(hours, scale)


def _ch4_eight_hours_ghost_fields(path):
    """Fixed ch4_06/07 GD path — red, low-alpha step ghosts (07c style)."""
    if path is None or len(path) < 2:
        return {}
    pts = np.asarray(path, dtype=float)
    return {
        "path_trail_extension": pts,
        "path_trail_extension_color": CH3_LIK_3D_POINT_COLOR,
        "path_trail_extension_linestyle": "-",
        "path_trail_extension_alpha": float(CH4_EIGHT_HOURS_GHOST_ALPHA),
        "path_trail_extension_waypoint_alpha": float(CH4_EIGHT_HOURS_GHOST_WP_ALPHA),
    }


def _ch4_eight_panel_fields(st):
    return {
        "panel_xlim": st["data_xlim"],
        "panel_ylim": st["data_ylim"],
        "panel_xlabel": st["xlabel"],
        "panel_sigma_stg": st["sigma_stg"],
        "panel_sigma_elg": st["sigma_elg"],
    }


def _ch4_eight_lerp_pose(p0, p1, u):
    u = float(np.clip(float(u), 0.0, 1.0))
    return tuple(float(p0[i] + u * (float(p1[i]) - float(p0[i]))) for i in range(3))


def _ch4_eight_grad_tuple(study, exam, y, ws, we, bb):
    g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, float(ws), float(we), float(bb))
    return (float(g1), float(g2), float(gb))


def _ch4_eight_study_morphed_path(hours_path, study_scale):
    """NLL-preserving path morph: only ``w_ST`` divides by ``study_scale``."""
    path = np.asarray(hours_path, dtype=float).copy()
    s = max(float(study_scale), 1e-9)
    path[:, 0] = path[:, 0] / s
    return path


def _ch4_eight_study_morphed_point(ws, we, bb, study_scale):
    s = max(float(study_scale), 1e-9)
    return float(ws) / s, float(we), float(bb)


def _ch4_eight_expand_display_bounds(bounds, scale=None):
    """Widen axis limits symmetrically (display only)."""
    factor = float(CH4_EIGHT_DISPLAY_BOUNDS_SCALE if scale is None else scale)
    lo = np.array([bounds[0], bounds[2], bounds[4]], dtype=float)
    hi = np.array([bounds[1], bounds[3], bounds[5]], dtype=float)
    center = 0.5 * (lo + hi)
    half = 0.5 * (hi - lo) * factor
    lo = center - half
    hi = center + half
    return (
        float(lo[0]), float(hi[0]),
        float(lo[1]), float(hi[1]),
        float(lo[2]), float(hi[2]),
    )


def _ch4_eight_pack_display_bounds(pack, bounds):
    """Apply optional 2× display widen (ch4_10); ch4_11 keeps raw bounds."""
    if pack.get("_eight_bounds_expand", True):
        return _ch4_eight_expand_display_bounds(bounds)
    return bounds


def _ch4_eight_hours_gd_start():
    return (
        float(CH3_LIK_3D_PATH_START[0]),
        float(CH3_LIK_3D_PATH_START[1]),
        float(CH3_LIK_3D_PATH_START[2]),
    )


def _ch4_eight_minutes_gd_start(study_scale):
    ws0, we0, bb0 = _ch4_eight_hours_gd_start()
    return _ch4_eight_study_morphed_point(ws0, we0, bb0, study_scale)


def _ch4_eight_study_scaled_eta(study_scale):
    """``α / study_scale`` — matches the ``w_ST ÷ study_scale`` axis rescaling."""
    return float(CH3_LIK_GD_STEP) / max(float(study_scale), 1.0)


def _ch4_eight_apply_study_unit(pack, unit_u):
    """Update ``pack`` in place for study-scale ``unit_u`` (0=hours, 1=minutes)."""
    st = _ch4_study_minutes_unit_fields(float(unit_u))
    study, exam, y = st["study"], st["exam"], st["y"]
    study_scale = float(st["study_scale"])
    gd_start = _ch4_eight_minutes_gd_start(study_scale)
    gd_eta = _ch4_eight_study_scaled_eta(study_scale)
    Lf = _ch3_nll_sum_on_flat_grid(
        study, exam, y, pack["W1m"].ravel(), pack["W2m"].ravel(), pack["Bm"].ravel(),
    ).reshape(pack["W1m"].shape)
    morphed_path = _ch4_eight_study_morphed_path(pack["hours_gd_path"], study_scale)
    vmin = float(np.nanmin(Lf))
    vmax = float(np.nanmax(Lf))
    eta_override = pack.get("_eight_gd_eta_override")
    if eta_override is not None:
        gd_eta = float(eta_override)
    arrow_log = pack.get("_eight_arrow_log_display")
    if arrow_log is None:
        arrow_log = True
    if bool(pack.get("use_path_ball_limits", True)):
        path_nll = np.array([
            float(-loss_log_likelihood(float(r[0]), float(r[1]), float(r[2]), study, exam, y))
            for r in morphed_path
        ], dtype=float)
        ball_vmin = float(np.min(path_nll))
        ball_vmax = float(np.max(path_nll))
    else:
        ball_vmin, ball_vmax = vmin, vmax
    pack.update({
        "study": study,
        "exam": exam,
        "y": y,
        "Lf": Lf,
        "vmin": vmin,
        "vmax": vmax,
        "scaled_gd_path": morphed_path,
        "ball_vmin": ball_vmin,
        "ball_vmax": ball_vmax,
        "study_unit_u": float(unit_u),
        "study_scale": study_scale,
        "gd_start": gd_start,
        "gd_eta": gd_eta,
        "gd_arrow_log_display": bool(arrow_log),
    })
    pack.update(_ch4_eight_panel_fields(st))
    return st


def _ch4_eight_base_pack():
    """Shared measurements + fixed hours reference path from ch4_06/07."""
    pack = dict(_ch3_lik_3d_measurements_pack(ball_colormap_limits=False))
    hours_pack = _ch3_lik_3d_gd_combined_pack()
    hours_path = np.asarray(hours_pack["path"], dtype=float)
    ws7, we7, bb7 = (float(hours_path[-1, 0]), float(hours_path[-1, 1]), float(hours_path[-1, 2]))
    pack.update({
        "hours_gd_path": hours_path,
        "hours_gd_n_iters": int(CH3_LIK_GD_N_ITERS),
        "hours_end_pose": (ws7, we7, bb7),
        "gd_eta_hours": float(CH3_LIK_GD_STEP),
        "gd_view_bounds_start": CH3_LIK_CT_VIEW_BOUNDS,
    })
    _ch4_eight_apply_study_unit(pack, 0.0)
    return pack


def _ch4_eight_newton_entry_spec(
    pack, ws, we, bb, *, path_trail, view_bounds, show_ghost=True, **extra,
):
    """Minutes-scale Newton entry: split partials + Hessian bottom rail (ch4_08c style)."""
    spec = _ch3_lik_split_hold_spec(pack, float(ws), float(we), float(bb))
    spec.update({
        "show_path_line": False,
        "view_bounds": view_bounds,
        "gd_arrow_log_display": bool(pack.get("gd_arrow_log_display", False)),
    })
    if path_trail is not None and len(path_trail) >= 2:
        spec["path_trail"] = np.asarray(path_trail, dtype=float)
    if show_ghost:
        spec.update(_ch4_eight_hours_ghost_fields(_ch4_eight_reference_gd_path(pack)))
    spec.update(extra)
    return spec


def _ch4_eight_bridge_spec(
    pack, ws, we, bb, grad, *, path_trail, view_bounds,
    show_ghost=False, grad_red=True, arrow_mode="combined", **extra,
):
    spec = {
        "ws": float(ws),
        "we": float(we),
        "bb": float(bb),
        "grad": tuple(float(v) for v in grad),
        "eta": float(pack["gd_eta"]),
        "arrows": (True, True, True),
        "bold": None,
        "bold_all": False,
        "grad_red": bool(grad_red),
        "arrow_mode": str(arrow_mode),
        "transition_u": 0.0,
        "show_path_line": False,
        "view_bounds": view_bounds,
    }
    if path_trail is not None and len(path_trail) >= 2:
        spec["path_trail"] = np.asarray(path_trail, dtype=float)
    if show_ghost:
        spec.update(_ch4_eight_hours_ghost_fields(_ch4_eight_reference_gd_path(pack)))
    spec.update(extra)
    return spec


def _ch4_eight_path_view_bounds(pack, path, *, include_hours_ghost=False):
    """3-D axis limits that fit ``path`` (and optionally the fixed hours ghost)."""
    pts = [np.asarray(path, dtype=float).reshape(-1, 3)]
    if include_hours_ghost:
        hp = pack.get("hours_gd_path")
        if hp is not None and len(hp) >= 1:
            pts.append(np.asarray(hp, dtype=float).reshape(-1, 3))
    all_pts = np.vstack(pts)
    bounds = _ch3_lik_ax3d_fixed_bounds(
        float(pack["W1m"].min()), float(pack["W1m"].max()),
        float(pack["W2m"].min()), float(pack["W2m"].max()),
        float(pack["Bm"].min()), float(pack["Bm"].max()),
        all_pts,
    )
    return _ch4_eight_pack_display_bounds(pack, bounds)


def _ch4_eight_bridge_view_bounds(pack, path, u, *, end_bounds, start_bounds, show_ghost=False):
    """Lerp 3-D limits in sync with study warp ``u`` (2-D axis uses the same ``u``)."""
    if not pack.get("_eight_bounds_expand", True):
        return _ch4_eight_pack_display_bounds(pack, end_bounds)
    u = float(np.clip(float(u), 0.0, 1.0))
    lo = _ch4_eight_pack_display_bounds(pack, start_bounds)
    path_fit = _ch4_eight_path_view_bounds(pack, path, include_hours_ghost=bool(show_ghost))
    hi = _ch4_eight_pack_display_bounds(pack, end_bounds)
    target = _ch3_lik_lerp_bounds(path_fit, hi, u)
    return _ch3_lik_lerp_bounds(lo, target, u)


def _ch4_eight_bridge_frames(pack, *, render_fn=None, entry="gd"):
    """07 end (hours) → study ×60 + path morph + 3-D zoom (same ``u`` as 2-D) → hold.

    ``entry``: ``"gd"`` for ch4_10 (GD formulas at minutes scale) or ``"newton"`` for
    ch4_11 (Hessian + split partials like ch4_08c).
    """
    if render_fn is None:
        render_fn = _ch3_lik_gd_render_frame
    entry = str(entry)
    newton_entry = entry == "newton"
    frames = []
    ws7, we7, bb7 = pack["hours_end_pose"]
    ws0, we0, bb0 = _ch4_eight_hours_gd_start()
    start_bounds = pack["gd_view_bounds_start"]
    end_bounds = pack.get("gd_view_bounds_end", start_bounds)

    def _emit(spec):
        frames.append(render_fn(pack, spec, cam_azim_u=0.0))

    _ch4_eight_apply_study_unit(pack, 0.0)
    grad7 = _ch4_eight_grad_tuple(pack["study"], pack["exam"], pack["y"], ws7, we7, bb7)
    hold07 = _ch4_eight_bridge_spec(
        pack, ws7, we7, bb7, grad7,
        path_trail=pack["hours_gd_path"],
        view_bounds=_ch4_eight_pack_display_bounds(pack, start_bounds),
        show_ghost=False,
    )
    for _ in range(max(int(CH4_EIGHT_N_HOLD_07), 1)):
        _emit(hold07)

    for tv in np.linspace(0.0, 1.0, int(CH4_EIGHT_N_WARP), endpoint=True):
        u = ch3_knob_smoothstep(float(tv))
        st = _ch4_eight_apply_study_unit(pack, u)
        study_scale = float(st["study_scale"])
        pose_u = ch3_knob_smoothstep(min(float(u) / 0.28, 1.0))
        ws_h, we_h, bb_h = _ch4_eight_lerp_pose((ws7, we7, bb7), (ws0, we0, bb0), pose_u)
        ws, we, bb = _ch4_eight_study_morphed_point(ws_h, we_h, bb_h, study_scale)
        grad = _ch4_eight_grad_tuple(pack["study"], pack["exam"], pack["y"], ws, we, bb)
        path = pack["scaled_gd_path"]
        ref = _ch4_eight_reference_gd_path(pack)
        bounds = _ch4_eight_bridge_view_bounds(
            pack, path, u,
            end_bounds=end_bounds,
            start_bounds=start_bounds,
            show_ghost=(float(u) > 0.04),
        )
        at_start = pose_u > 0.96
        if newton_entry and at_start:
            _emit(_ch4_eight_newton_entry_spec(
                pack, ws, we, bb,
                path_trail=ref,
                view_bounds=bounds,
                show_ghost=(float(u) > 0.04),
            ))
        else:
            _emit(_ch4_eight_bridge_spec(
                pack, ws, we, bb, grad,
                path_trail=ref,
                view_bounds=bounds,
                show_ghost=(float(u) > 0.04),
                grad_red=(not at_start),
                arrow_mode="split" if at_start else "combined",
            ))

    _ch4_eight_apply_study_unit(pack, 1.0)
    ws, we, bb = pack["gd_start"]
    grad0 = _ch4_eight_grad_tuple(pack["study"], pack["exam"], pack["y"], ws, we, bb)
    if newton_entry:
        start_spec = _ch4_eight_newton_entry_spec(
            pack, ws, we, bb,
            path_trail=_ch4_eight_reference_gd_path(pack),
            view_bounds=_ch4_eight_pack_display_bounds(pack, end_bounds),
            show_ghost=True,
        )
    else:
        start_spec = _ch4_eight_bridge_spec(
            pack, ws, we, bb, grad0,
            path_trail=_ch4_eight_reference_gd_path(pack),
            view_bounds=_ch4_eight_pack_display_bounds(pack, end_bounds),
            show_ghost=True,
            grad_red=False,
            arrow_mode="split",
        )
    for _ in range(max(int(CH4_EIGHT_N_HOLD_START), 1)):
        _emit(start_spec)
    return frames


def ch4_preview_eight_bridge_frame(pack, phase):
    """Single bridge frame — ``phase`` is ``before_scale`` or ``after_scale``."""
    phase = str(phase)
    ws7, we7, bb7 = pack["hours_end_pose"]
    if phase == "before_scale":
        _ch4_eight_apply_study_unit(pack, 0.0)
        grad7 = _ch4_eight_grad_tuple(pack["study"], pack["exam"], pack["y"], ws7, we7, bb7)
        spec = _ch4_eight_bridge_spec(
            pack, ws7, we7, bb7, grad7,
            path_trail=pack["hours_gd_path"],
            view_bounds=_ch4_eight_pack_display_bounds(pack, pack["gd_view_bounds_start"]),
            show_ghost=False,
        )
        return _ch3_lik_gd_render_frame(pack, spec, cam_azim_u=0.0)
    if phase == "after_scale":
        _ch4_eight_apply_study_unit(pack, 1.0)
        ws, we, bb = pack["gd_start"]
        bounds = _ch4_eight_bridge_view_bounds(
            pack, pack["scaled_gd_path"], 1.0,
            end_bounds=pack.get("gd_view_bounds_end", pack["gd_view_bounds_start"]),
            start_bounds=pack["gd_view_bounds_start"],
            show_ghost=True,
        )
        spec = _ch4_eight_bridge_spec(
            pack, ws, we, bb,
            _ch4_eight_grad_tuple(pack["study"], pack["exam"], pack["y"], ws, we, bb),
            path_trail=_ch4_eight_reference_gd_path(pack),
            view_bounds=bounds,
            show_ghost=True,
            grad_red=False,
            arrow_mode="split",
        )
        return _ch3_lik_gd_render_frame(pack, spec, cam_azim_u=0.0)
    raise ValueError(f"unknown eight-bridge preview phase: {phase!r}")


def ch4_preview_eight_newton_entry_frame(pack):
    """Minutes-scale Newton entry (Hessian formulas + split partials), ch4_11 bridge end."""
    _ch4_eight_apply_study_unit(pack, 1.0)
    ws, we, bb = pack["gd_start"]
    bounds = _ch4_eight_pack_display_bounds(
        pack,
        pack.get("gd_view_bounds_end", pack["gd_view_bounds_start"]),
    )
    spec = _ch4_eight_newton_entry_spec(
        pack, ws, we, bb,
        path_trail=_ch4_eight_reference_gd_path(pack),
        view_bounds=bounds,
        show_ghost=True,
    )
    return _ch3_lik_gd_render_frame(pack, spec, cam_azim_u=0.0)


# --- ch4_10: 07 end → study ×60 → zoom; 8 GD steps + hours-path ghost ---

CH4_GD_EIGHT_N_ITERS = 8 if not _CH3_DRAFT else 2
CH4_GD_EIGHT_MS = 100 if not _CH3_DRAFT else 120


def _ch4_gd_eight_full_gd_path(study, exam, y):
    """Full ch4_06/07 GD endpoint path at the current study scale."""
    return _ch3_lik_gd_path(
        study, exam, y, CH3_LIK_3D_PATH_START, CH3_LIK_GD_N_ITERS, CH3_LIK_GD_STEP,
    )


def _ch4_gd_eight_raw_frame_specs(pack):
    """ch4_06 sequential partials + growing trail + hours ghost (no fixed bounds)."""
    base = _ch3_lik_gd_frame_specs(pack)
    gd_path = np.asarray(pack["path"], dtype=float)
    ghost = _ch4_eight_hours_ghost_fields(_ch4_eight_reference_gd_path(pack))
    hold_n = max(int(CH3_LIK_GD_N_HOLD_ARROWS), 1)
    step_n = max(int(CH3_LIK_GD_N_PARAM_STEP), 2)
    iter_len = hold_n + 3 * step_n
    out = []
    for i, s in enumerate(base):
        s2 = dict(s)
        s2.update(ghost)
        iter_i = int(i // iter_len)
        settled_i = min(iter_i, len(gd_path) - 1)
        pos = np.array([float(s["ws"]), float(s["we"]), float(s["bb"])], dtype=float)
        settled = gd_path[: settled_i + 1]
        if np.linalg.norm(pos - settled[-1]) > 1e-9:
            trail = np.vstack([settled, pos])
        else:
            trail = settled
        if len(trail) >= 2:
            s2["path_trail"] = trail
        s2["show_path_line"] = False
        out.append(s2)
    return out


def _ch4_gd_eight_final_view_bounds(pack):
    """3-D limits for minutes-scale 8-step run + hours ghost + partial probes."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    eta = float(pack["gd_eta"])
    pts = [
        np.asarray(pack["hours_gd_path"], dtype=float).reshape(-1, 3),
        np.asarray(pack["path"], dtype=float).reshape(-1, 3),
        np.asarray(pack.get("scaled_gd_path", pack["path"]), dtype=float).reshape(-1, 3),
    ]
    for path in (pack["hours_gd_path"], pack["path"]):
        for r in path:
            ws, we, bb = float(r[0]), float(r[1]), float(r[2])
            g1, g2, gb = _ch3_nll_sum_grad_at_point(study, exam, y, ws, we, bb)
            pts.append(np.array([
                [ws - eta * g1, we, bb],
                [ws, we - eta * g2, bb],
                [ws, we, bb - eta * gb],
            ], dtype=float))
    for spec in _ch4_gd_eight_raw_frame_specs(pack):
        pts.append(np.array([[float(spec["ws"]), float(spec["we"]), float(spec["bb"])]], dtype=float))
    all_pts = np.vstack(pts)
    k_lo1, k_hi1 = float(pack["W1m"].min()), float(pack["W1m"].max())
    k_lo2, k_hi2 = float(pack["W2m"].min()), float(pack["W2m"].max())
    k_lob, k_hib = float(pack["Bm"].min()), float(pack["Bm"].max())
    return _ch3_lik_ax3d_fixed_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, all_pts,
    )


def _ch4_gd_eight_steps_pack():
    """Minutes-scale working pack for the 8-step GD segment."""
    pack = _ch4_eight_base_pack()
    _ch4_eight_apply_study_unit(pack, 1.0)
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = _ch3_lik_gd_path(
        study, exam, y, pack["gd_start"], CH4_GD_EIGHT_N_ITERS, pack["gd_eta"],
    )
    pack.update({
        "gd_n_iters": int(CH4_GD_EIGHT_N_ITERS),
        "path": trail,
    })
    pack["gd_view_bounds_end"] = _ch4_eight_expand_display_bounds(_ch4_gd_eight_final_view_bounds(pack))
    pack["gd_view_bounds"] = pack["gd_view_bounds_end"]
    pack["gd_arrow_log_display"] = True
    return pack


def _ch4_gd_eight_path_frame_specs(pack):
    specs = _ch4_gd_eight_raw_frame_specs(pack)
    end_bounds = pack["gd_view_bounds_end"]
    for s in specs:
        s["view_bounds"] = end_bounds
    return specs


def ch4_build_frames_gd_eight_steps_story():
    pack = _ch4_gd_eight_steps_pack()
    frames = []
    frames.extend(_ch4_eight_bridge_frames(pack))
    _ch3_lik_append_gd_frames(
        frames, pack,
        specs_fn=_ch4_gd_eight_path_frame_specs,
    )
    return _ch3_lik_story_hold(frames)


def ch4_preview_gd_eight_steps_frame(*, gd_spec_index=-1, phase=None):
    """Preview one ch4_10 frame.

    ``phase``: ``before_scale`` | ``after_scale`` | ``end`` (or omit and use ``gd_spec_index``).
    """
    if phase is not None:
        phase = str(phase)
        pack = _ch4_gd_eight_steps_pack()
        if phase in ("before_scale", "after_scale"):
            return ch4_preview_eight_bridge_frame(pack, phase)
        if phase == "end":
            specs = _ch4_gd_eight_path_frame_specs(pack)
            return _ch3_lik_gd_render_frame(pack, specs[-1], cam_azim_u=0.0)
        raise ValueError(f"unknown ch4_10 preview phase: {phase!r}")
    pack = _ch4_gd_eight_steps_pack()
    if int(gd_spec_index) == 0:
        _ch4_eight_apply_study_unit(pack, 0.0)
        ws7, we7, bb7 = pack["hours_end_pose"]
        grad7 = _ch4_eight_grad_tuple(pack["study"], pack["exam"], pack["y"], ws7, we7, bb7)
        spec = _ch4_eight_bridge_spec(
            pack, ws7, we7, bb7, grad7,
            path_trail=pack["hours_gd_path"],
            view_bounds=pack["gd_view_bounds_start"],
            show_ghost=False,
        )
        return _ch3_lik_gd_render_frame(pack, spec, cam_azim_u=0.0)
    if int(gd_spec_index) < 0:
        specs = _ch4_gd_eight_path_frame_specs(pack)
        return _ch3_lik_gd_render_frame(pack, specs[-1], cam_azim_u=0.0)
    specs = _ch4_gd_eight_path_frame_specs(pack)
    idx = int(np.clip(int(gd_spec_index), 0, len(specs) - 1))
    return _ch3_lik_gd_render_frame(pack, specs[idx], cam_azim_u=0.0)


def ch4_export_gd_eight_steps():
    frames = ch4_build_frames_gd_eight_steps_story()
    fn = "ch4_10_gd_eight_steps.mp4"
    save_mp4(frames, fn, duration=int(CH4_GD_EIGHT_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


# --- ch4_11: same bridge as ch4_10, then 8 Newton partial cycles (08d_rot_after) ---

CH4_NEWTON_EIGHT_N_ITERS = 8 if not _CH3_DRAFT else 2
CH4_NEWTON_EIGHT_ETA = 0.06  # partial-arrow label only
CH4_NEWTON_EIGHT_DAMP = 1.45e-4  # LM damp tuned to match minutes-scale ghost GD basin
CH4_NEWTON_EIGHT_MS = int(CH3_LIK_NEWTON_PARTIAL_MS)
CH4_NEWTON_EIGHT_VIEW_BOUNDS = CH3_LIK_CT_VIEW_BOUNDS  # (-4, 4) on w_ST, w_EL, b

_CH4_NEWTON_EIGHT_TRAIL_EXT_KEYS = (
    "path_trail_extension",
    "path_trail_extension_color",
    "path_trail_extension_linestyle",
    "path_trail_extension_alpha",
    "path_trail_extension_waypoint_alpha",
)


def _ch4_newton_eight_configure_pack(pack):
    """Minutes-scale Newton: α = 0.06 on partial arrows; ``CH4_NEWTON_EIGHT_DAMP`` on steps."""
    pack["_eight_arrow_log_display"] = False
    pack["_eight_gd_eta_override"] = float(CH4_NEWTON_EIGHT_ETA)
    pack["use_path_ball_limits"] = False
    pack["newton_damp"] = float(CH4_NEWTON_EIGHT_DAMP)
    pack["gd_eta"] = float(CH4_NEWTON_EIGHT_ETA)
    pack["gd_arrow_log_display"] = False
    pack["_eight_bounds_expand"] = False
    pack["gd_view_bounds_end"] = CH4_NEWTON_EIGHT_VIEW_BOUNDS
    return pack


def _ch4_newton_path(study, exam, y, start, n_iters, *, damp=None):
    """Newton iterate positions from ``start``."""
    damp = float(CH4_NEWTON_EIGHT_DAMP if damp is None else damp)
    trail = [np.asarray(start, dtype=float).reshape(3)]
    ws, we, bb = (float(start[0]), float(start[1]), float(start[2]))
    for _ in range(int(n_iters)):
        dw, de, db = _ch3_nll_newton_step(study, exam, y, ws, we, bb, damp=damp)
        ws, we, bb = float(ws - dw), float(we - de), float(bb - db)
        trail.append(np.array([ws, we, bb], dtype=float))
    return np.asarray(trail, dtype=float)


def _ch4_newton_eight_start_view_bounds(pack):
    """Fixed ±4 cube for the minutes Newton segment."""
    return CH4_NEWTON_EIGHT_VIEW_BOUNDS


def _ch4_newton_eight_path_nll_table(pack, path=None):
    """Return ``(step, w_ST, w_EL, b, NLL)`` rows for each point on the Newton path."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    pts = np.asarray(pack["path"] if path is None else path, dtype=float)
    rows = []
    for i, (ws, we, bb) in enumerate(pts):
        nll = float(-loss_log_likelihood(float(ws), float(we), float(bb), study, exam, y))
        rows.append((int(i), float(ws), float(we), float(bb), nll))
    return rows


def _ch4_newton_eight_print_path_nll(pack, path=None):
    rows = _ch4_newton_eight_path_nll_table(pack, path=path)
    print("ch4_11 Newton path NLL (damp={:.6g}):".format(float(pack.get("newton_damp", CH4_NEWTON_EIGHT_DAMP))))
    print(f"{'step':>4}  {'w_ST':>10}  {'w_EL':>10}  {'b':>10}  {'NLL':>12}")
    for step, ws, we, bb, nll in rows:
        print(f"{step:4d}  {ws:10.6f}  {we:10.6f}  {bb:10.6f}  {nll:12.6f}")
    return rows


def _ch4_newton_eight_strip_hours_ghost(spec):
    for k in _CH4_NEWTON_EIGHT_TRAIL_EXT_KEYS:
        spec.pop(k, None)
    return spec


def _ch4_newton_eight_end_spec(pack, specs):
    """Final frame: hours ghost + purple Newton trail (ch4_10 layout)."""
    spec = dict(specs[-1])
    end_bounds = pack.get(
        "gd_view_bounds_end", pack.get("gd_view_bounds", CH3_LIK_CT_VIEW_BOUNDS),
    )
    spec["path_trail"] = np.asarray(pack["path"], dtype=float)
    spec["show_path_line"] = False
    spec["view_bounds"] = end_bounds
    spec.update(_ch4_eight_hours_ghost_fields(_ch4_eight_reference_gd_path(pack)))
    return spec


def _ch4_newton_eight_ax3d_bounds(pack, specs):
    """3-D limits for minutes Newton run + hours ghost + partial probes."""
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    pts = [
        np.asarray(pack["hours_gd_path"], dtype=float).reshape(-1, 3),
        np.asarray(pack["path"], dtype=float).reshape(-1, 3),
    ]
    for spec in specs:
        ws, we, bb = float(spec["ws"]), float(spec["we"]), float(spec["bb"])
        pts.append(np.array([[ws, we, bb]], dtype=float))
        nsv = spec.get("newton_step_vec")
        if nsv is not None:
            pts.append(np.array([[
                ws + float(nsv[0]), we + float(nsv[1]), bb + float(nsv[2]),
            ]], dtype=float))
        nsp = spec.get("newton_split_vec")
        if nsp is not None:
            pts.append(np.array([[
                ws - float(nsp[0]), we - float(nsp[1]), bb - float(nsp[2]),
            ]], dtype=float))
        grad = spec.get("grad")
        if grad is not None and spec.get("arrow_mode") == "split":
            g1, g2, gb = (float(grad[0]), float(grad[1]), float(grad[2]))
            eta = float(spec.get("eta", pack.get("gd_eta", CH3_LIK_GD_STEP)))
            pts.append(np.array([
                [ws - eta * g1, we, bb],
                [ws, we - eta * g2, bb],
                [ws, we, bb - eta * gb],
            ], dtype=float))
    all_pts = np.vstack(pts)
    k_lo1, k_hi1 = float(pack["W1m"].min()), float(pack["W1m"].max())
    k_lo2, k_hi2 = float(pack["W2m"].min()), float(pack["W2m"].max())
    k_lob, k_hib = float(pack["Bm"].min()), float(pack["Bm"].max())
    return _ch3_lik_ax3d_fixed_bounds(
        k_lo1, k_hi1, k_lo2, k_hi2, k_lob, k_hib, all_pts,
    )


def _ch4_newton_eight_steps_pack():
    """Minutes-scale working pack for the 8-cycle Newton segment."""
    pack = _ch4_eight_base_pack()
    _ch4_newton_eight_configure_pack(pack)
    _ch4_eight_apply_study_unit(pack, 1.0)
    study, exam, y = pack["study"], pack["exam"], pack["y"]
    trail = _ch4_newton_path(
        study, exam, y, pack["gd_start"], CH4_NEWTON_EIGHT_N_ITERS,
        damp=pack["newton_damp"],
    )
    pack.pop("_newton_partial_trajectory", None)
    pack.update({
        "newton_n_cycles": int(CH4_NEWTON_EIGHT_N_ITERS),
        "path": trail,
    })
    pack["gd_view_bounds_end"] = _ch4_newton_eight_start_view_bounds(pack)
    pack["gd_view_bounds"] = pack["gd_view_bounds_end"]
    return pack


def _ch4_newton_eight_story_specs(pack):
    specs = _ch3_lik_newton_partial_story_specs(
        pack,
        track_path=True,
        rotate_during=False,
        use_voxel_ghosts=False,
        skip_open_hold=True,
    )
    ghost = _ch4_eight_hours_ghost_fields(_ch4_eight_reference_gd_path(pack))
    end_bounds = pack.get("gd_view_bounds_end", pack.get("gd_view_bounds", CH3_LIK_CT_VIEW_BOUNDS))
    out = []
    for i, s in enumerate(specs):
        s2 = dict(s)
        s2.update(ghost)
        s2["view_bounds"] = end_bounds
        s2["cam_azim_u"] = 0.0
        s2["cam_rot_deg"] = 0.0
        s2["show_path_line"] = False
        if i == len(specs) - 1:
            s2["path_trail"] = np.asarray(pack["path"], dtype=float)
            s2.pop("path_trail_linecolor", None)
        out.append(s2)
    return out


def _ch4_newton_eight_pack_with_specs():
    pack = _ch4_newton_eight_steps_pack()
    pack.pop("_newton_partial_trajectory", None)
    return pack, _ch4_newton_eight_story_specs(pack)


def ch4_build_frames_newton_eight_steps_story():
    pack = _ch4_eight_base_pack()
    _ch4_newton_eight_configure_pack(pack)
    frames = []
    frames.extend(_ch4_eight_bridge_frames(pack, entry="newton"))
    pack = _ch4_newton_eight_steps_pack()
    pack.pop("_newton_partial_trajectory", None)
    from ch4_export_pipeline import build_tutorial_frames
    frames.extend(build_tutorial_frames(
        pack,
        _ch4_newton_eight_story_specs(pack),
        _ch3_lik_gd_render_frame,
        prewarm="gd",
        progress_label="ch4_11",
    ))
    return _ch3_lik_story_hold(frames)


def ch4_preview_newton_eight_steps_frame(*, spec_index=-1, phase=None):
    """Preview one ch4_11 frame.

    ``phase``: ``before_scale`` | ``after_scale`` | ``end`` (or omit and use ``spec_index``).
    """
    if phase is not None:
        phase = str(phase)
        if phase == "before_scale":
            pack = _ch4_eight_base_pack()
            _ch4_newton_eight_configure_pack(pack)
            return ch4_preview_eight_bridge_frame(pack, phase)
        if phase == "after_scale":
            pack = _ch4_newton_eight_steps_pack()
            return ch4_preview_eight_newton_entry_frame(pack)
        if phase == "end":
            pack, specs = _ch4_newton_eight_pack_with_specs()
            return _ch3_lik_gd_render_frame(
                pack, _ch4_newton_eight_end_spec(pack, specs), cam_azim_u=0.0,
            )
        raise ValueError(f"unknown ch4_11 preview phase: {phase!r}")
    if int(spec_index) == 0:
        pack = _ch4_eight_base_pack()
        _ch4_eight_apply_study_unit(pack, 0.0)
        ws7, we7, bb7 = pack["hours_end_pose"]
        grad7 = _ch4_eight_grad_tuple(pack["study"], pack["exam"], pack["y"], ws7, we7, bb7)
        spec = _ch4_eight_bridge_spec(
            pack, ws7, we7, bb7, grad7,
            path_trail=pack["hours_gd_path"],
            view_bounds=pack["gd_view_bounds_start"],
            show_ghost=False,
        )
        return _ch3_lik_gd_render_frame(pack, spec, cam_azim_u=0.0)
    pack, specs = _ch4_newton_eight_pack_with_specs()
    idx = len(specs) - 1 if int(spec_index) < 0 else int(np.clip(int(spec_index), 0, len(specs) - 1))
    spec = _ch4_newton_eight_end_spec(pack, specs) if idx == len(specs) - 1 else specs[idx]
    return _ch3_lik_gd_render_frame(pack, spec, cam_azim_u=0.0)


def ch4_export_newton_eight_steps():
    pack = _ch4_newton_eight_steps_pack()
    _ch4_newton_eight_print_path_nll(pack)
    frames = ch4_build_frames_newton_eight_steps_story()
    fn = "ch4_11_newton_eight_steps.mp4"
    save_mp4(frames, fn, duration=int(CH4_NEWTON_EIGHT_MS))
    print("wrote", OUTPUT_DIR / fn, f"({len(frames)} frames)")
    return OUTPUT_DIR / fn


Chapter 4 layout OK — handwriting: Patrick Hand
Chapter 3 setup OK — 20 clean, 26 with noise.


logistic-regression-chap3.ipynb:580: SyntaxWarning: invalid escape sequence '\s'
  "    highlight_mistakes_flag=False,\n",
logistic-regression-chap3.ipynb:584: SyntaxWarning: invalid escape sequence '\s'
  "    if show_colormap and w_st is not None:\n",


### Likelihood story clips (ch4_02–07)

One export cell per clip — run **`ch4-likelihood-02`** through **`ch4-likelihood-07`** in order.

1. **ch4_02a** — wide 2D prob labels → slide to duo-left → fade → ch4_02 opening
2. **ch4_02** — likelihood w₁₂ landscape (knob labels)
3. **ch4_03** — template + notation + ℒ column + first p(y|x) (part 1)
2b. **ch4_02b** — wide 2D → duo σ(ST,EL) 3D + knobs → NLL heatmap + CT scan → ch4_02 opening
4. **ch4_03b** — GD point tour on likelihood surface; erase p(y|x) (part 2)
5. **ch4_03c** — log → NLL morph, corner layout, heatmap, plane drop (part 3)
6. **ch4_04** — 3D weight space + We are here + NLL trajectory (90° camera pan)
7. **ch4_05a** — CT scan: NLL heatmap planes sweep w_ST, w_EL, b
5. **ch4_05a2** — diagonal voxel fill: checkerboard cubes swept corner → corner
6. **ch4_05a3** — same as 05a2 with ¼-size voxels (4× resolution per axis)
7. **ch4_05a3b** — from 05a3 end: zoom 3D axes out to ±21 (voxels stay in ±3 cube)
8. **ch4_05a4** — from 05a3 fill at origin: shrink to ball cube, path to (−3, −3, −3)
9. **ch4_05a4b** — same as 05a4 but camera rotates after path completes
10. **ch4_05a5** — GD voxel steps (η = ch4_06), emphasize voxel nearest −∇NLL, keep trail
11. **ch4_05a5b** — same as 05a5 but camera rotates after all GD steps
12. **ch4_05a6** — same as 05a5 with smaller step size
13. **ch4_05a6b** — same as 05a6 but camera rotates after all GD steps
14. **ch4_05** — ball of heatmapped gradient vectors + intro zoom/spin
15. **ch4_05b** — partial slices → ∂ equations (part 1)
15b. **ch4_05b part2** — param bump demo, −∂ arrows, 2 GD steps
15c. **ch4_05b part3** — update formulas, α, crossfade to ch4_06
16. **ch4_06** — sequential GD: colored axis arrows, per-parameter updates (10 steps)
17. **ch4_07** — like ch4_06 opening, then red combined gradient arrow + simultaneous GD
18. **ch4_07b** — same as 07 with piecewise-linear path trail
19. **ch4_07b_rot_after** — same as 07b but camera rotates after the path completes
20. **ch4_07c** — 07b end ↔ partial triptych with ∂ arrows, then back to 07b
21. **ch4_08** — from 07 combined frame: local probe ghosts + 5 Newton steps (no formulas)
22. **ch4_08b** — same as 08 with accumulated Newton path
23. **ch4_08c** — decomposed partial box ghosts → Newton-morph split → Newton step (×8, −∇)
24. **ch4_08c_rot_after** — same as 08c but camera rotates after all cycles
25. **ch4_08d** — same as 08c with Newton path trail
26. **ch4_08d_rot_after** — same as 08d but camera rotates after all cycles
27. **ch4_08e** — voxel-cube partial ghosts (Blues/Oranges/Greens heatmap)
28. **ch4_08e_rot_after** — same as 08e but camera rotates after all cycles
29. **ch4_08f** — same as 08e with Newton path trail
30. **ch4_08f_rot_after** — same as 08f but camera rotates after all cycles
31. **ch4_08-batch** — batch export 08/08b + 08c–08f (deduped frames + parallel render)
32. **ch4_09** — from ch4_10 end: voxel sweep (05a3), study rescale minutes→hours
33. **ch4_10** — from ch4_07 end: study ×60, axis zoom, 8 GD steps + hours-path ghost
34. **ch4_11** — same opening as 10, then 8 Newton partial cycles (08d_rot_after)


In [ ]:
# ch4_02a — wide prob labels → duo-left slide → fade → ch4_02 opening
ch4_export_prob_labels_slide()


In [53]:
# ch4_02 — likelihood w₁₂ landscape (knob labels)
ch4_export_likelihood_w12_landscape()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_02_likelihood_w12_landscape.mp4


PosixPath('renders/ch4_02_likelihood_w12_landscape.mp4')

In [ ]:
# ch4_02b — wide 2D → duo σ 3D → NLL/CT → ch4_02 opening
ch4_export_sigmoid_duo_bridge()


In [2]:
# ch4_03 — part 1: notation + ℒ + first p(y|x)
ch4_export_likelihood_notation_nll()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_03_likelihood_notation_nll.mp4


PosixPath('renders/ch4_03_likelihood_notation_nll.mp4')

In [55]:
# ch4_03b — part 2: GD tour on likelihood; erase p(y|x)
ch4_export_likelihood_notation_nll_bridge()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_03b_likelihood_gd_landscape.mp4


PosixPath('renders/ch4_03b_likelihood_gd_landscape.mp4')

In [56]:
# ch4_03c — part 3: log/NLL morph through plane drop
ch4_export_likelihood_notation_nll_part2()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_03c_likelihood_nll_plane.mp4


PosixPath('renders/ch4_03c_likelihood_nll_plane.mp4')

In [32]:
# ch4_04 — 3D (w_ST, w_EL, b) measurements + colormap path
ch4_export_likelihood_3d_measurements()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_04_likelihood_3d_measurements.mp4


PosixPath('renders/ch4_04_likelihood_3d_measurements.mp4')

In [33]:
# ch4_05a — CT scan: NLL heatmap planes along each axis
ch4_export_likelihood_ct_scan()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a_likelihood_3d_ct_scan.mp4 (830 frames)


PosixPath('renders/ch4_05a_likelihood_3d_ct_scan.mp4')

In [34]:
# ch4_05a2 — diagonal checkerboard voxel fill (−3…3 cube)
ch4_export_likelihood_voxel_fill()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a2_diagonal_voxel_fill.mp4 (256 frames)


PosixPath('renders/ch4_05a2_diagonal_voxel_fill.mp4')

In [35]:
# ch4_05a3 — fine voxel fill (¼ cube size, 4× cells per axis)
ch4_export_likelihood_voxel_fill_fine()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a3_diagonal_voxel_fill_fine.mp4 (328 frames)


PosixPath('renders/ch4_05a3_diagonal_voxel_fill_fine.mp4')

In [36]:
# ch4_05a3b — zoom 3D axes from ±3 to ±21 after 05a3 end pose
ch4_export_likelihood_voxel_fill_fine_zoom_out()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a3b_voxel_zoom_out.mp4 (202 frames)


PosixPath('renders/ch4_05a3b_voxel_zoom_out.mp4')

In [37]:
# ch4_05a4 — ball-sized voxel cube grows at path start, accumulates along path
ch4_export_likelihood_ball_voxel_path()


logistic-regression-chap3.ipynb:529: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  "execution_count": null,


  cached voxel intro: cx0.0000_cy0.0000_cz0.0000_h0.799000_full_dpi110 (90 frames)


logistic-regression-chap3.ipynb:529: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  "execution_count": null,
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a4_ball_voxel_path.mp4 (578 frames)


PosixPath('renders/ch4_05a4_ball_voxel_path.mp4')

In [38]:
# ch4_05a4b — same path; camera pan after path completes
ch4_export_likelihood_ball_voxel_path_rot_after()


logistic-regression-chap3.ipynb:529: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  "execution_count": null,
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a4b_ball_voxel_path_rot_after.mp4 (618 frames)


PosixPath('renders/ch4_05a4b_ball_voxel_path_rot_after.mp4')

In [39]:
# ch4_05a5 — GD voxel steps (same η as ch4_06), accumulate voxels
ch4_export_likelihood_gd_voxel_steps()


  cached voxel intro: cx-0.5000_cy0.3300_cz-0.5000_h0.399500_full_dpi110 (90 frames)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a5_gd_voxel_steps.mp4 (586 frames)


PosixPath('renders/ch4_05a5_gd_voxel_steps.mp4')

In [40]:
# ch4_05a5b — same GD steps; camera pan after all steps
ch4_export_likelihood_gd_voxel_steps_rot_after()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a5b_gd_voxel_steps_rot_after.mp4 (626 frames)


PosixPath('renders/ch4_05a5b_gd_voxel_steps_rot_after.mp4')

In [41]:
# ch4_05a6 — GD voxel steps (small η), accumulate voxels
ch4_export_likelihood_gd_voxel_steps_small()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a6_gd_voxel_steps_small.mp4 (586 frames)


PosixPath('renders/ch4_05a6_gd_voxel_steps_small.mp4')

In [42]:
# ch4_05a6b — same small-step GD; camera pan after all steps
ch4_export_likelihood_gd_voxel_steps_small_rot_after()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05a6b_gd_voxel_steps_small_rot_after.mp4 (626 frames)


PosixPath('renders/ch4_05a6b_gd_voxel_steps_small_rot_after.mp4')

In [ ]:
# ch4_05 — ball of NLL-colored gradient vectors + intro zoom/spin
ch4_export_likelihood_3d_ball_vectors()


In [3]:
# ch4_05b — part 1: NLL slices, Δ brackets, ∂ equations
ch4_export_likelihood_partial_motivation_part1()


ch3:583: SyntaxWarning: invalid escape sequence '\s'
ch3:587: SyntaxWarning: invalid escape sequence '\s'


Chapter 3 setup OK — 20 clean, 26 with noise.
  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_05b part1: 1/1009  (4s)
  ch4_05b part1: 100/1009  (374s)
  ch4_05b part1: 200/1009  (794s)
  ch4_05b part1: 300/1009  (1246s)
  ch4_05b part1: 400/1009  (1707s)
  ch4_05b part1: 500/1009  (2310s)
  ch4_05b part1: 600/1009  (2824s)
  ch4_05b part1: 700/1009  (3343s)
  ch4_05b part1: 800/1009  (3890s)
  ch4_05b part1: 900/1009  (4458s)
  ch4_05b part1: 1000/1009  (5043s)
  ch4_05b part1: 1009/1009  (5095s)
  deduped consecutive frames: 1009 → 264


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05b_partial_slices_partials.mp4 (264 frames)


PosixPath('renders/ch4_05b_partial_slices_partials.mp4')

In [4]:
# ch4_05b part2 — param bumps, −∂ arrows, 2 GD steps
ch4_export_likelihood_partial_motivation_part2()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_05b part2: 1/1024  (5s)
  ch4_05b part2: 102/1024  (464s)
  ch4_05b part2: 204/1024  (924s)
  ch4_05b part2: 306/1024  (1387s)
  ch4_05b part2: 408/1024  (1845s)
  ch4_05b part2: 510/1024  (2307s)
  ch4_05b part2: 612/1024  (2768s)
  ch4_05b part2: 714/1024  (3227s)
  ch4_05b part2: 816/1024  (3710s)
  ch4_05b part2: 918/1024  (4204s)
  ch4_05b part2: 1020/1024  (4704s)
  ch4_05b part2: 1024/1024  (4722s)
  deduped consecutive frames: 1024 → 962


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05b_partial_param_demo.mp4 (962 frames)


PosixPath('renders/ch4_05b_partial_param_demo.mp4')

In [5]:
# ch4_05b part3 — update formulas, α, crossfade to ch4_06
ch4_export_likelihood_partial_motivation_part3()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_05b part3: 1/420  (6s)
  ch4_05b part3: 42/420  (254s)
  ch4_05b part3: 84/420  (509s)
  ch4_05b part3: 126/420  (764s)
  ch4_05b part3: 168/420  (1027s)
  ch4_05b part3: 210/420  (1289s)
  ch4_05b part3: 252/420  (1556s)
  ch4_05b part3: 294/420  (1827s)
  ch4_05b part3: 336/420  (2096s)
  ch4_05b part3: 378/420  (2294s)
  ch4_05b part3: 420/420  (2504s)
  deduped consecutive frames: 420 → 7


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05b_partial_gd_formulas.mp4 (99 frames)


PosixPath('renders/ch4_05b_partial_gd_formulas.mp4')

In [57]:
# ch4_05b full — all parts + crossfade (slow; use split exports to iterate)
ch4_export_likelihood_partial_motivation()


ch4_05b: 1330 plot frames + 92 crossfade/hold (full); set CH3_DRAFT_EXPORT=1 for a quick preview
  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_05b partial: 1/1330  (4s)
  ch4_05b partial: 133/1330  (498s)
  ch4_05b partial: 266/1330  (1088s)
  ch4_05b partial: 399/1330  (1684s)
  ch4_05b partial: 532/1330  (2372s)
  ch4_05b partial: 665/1330  (3136s)
  ch4_05b partial: 798/1330  (3888s)
  ch4_05b partial: 931/1330  (4686s)
  ch4_05b partial: 1064/1330  (5502s)
  ch4_05b partial: 1197/1330  (6341s)
  ch4_05b partial: 1330/1330  (7075s)
  deduped consecutive frames: 1330 → 278


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_05b_partial_derivative_motivation.mp4 (370 frames)


PosixPath('renders/ch4_05b_partial_derivative_motivation.mp4')

In [58]:
# ch4_06 — gradient descent on NLL with animated weight updates
ch4_export_likelihood_3d_gd()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_06: 1/420  (5s)
  ch4_06: 42/420  (215s)
  ch4_06: 84/420  (432s)
  ch4_06: 126/420  (649s)
  ch4_06: 168/420  (866s)
  ch4_06: 210/420  (1082s)
  ch4_06: 252/420  (1298s)
  ch4_06: 294/420  (1515s)
  ch4_06: 336/420  (1729s)
  ch4_06: 378/420  (1939s)
  ch4_06: 420/420  (2148s)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_06_likelihood_3d_gd.mp4 (436 frames)


PosixPath('renders/ch4_06_likelihood_3d_gd.mp4')

In [59]:
# ch4_07 — combined gradient arrow GD (all params at once)
ch4_export_likelihood_3d_gd_combined()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_07: 1/248  (5s)
  ch4_07: 24/248  (119s)
  ch4_07: 48/248  (237s)
  ch4_07: 72/248  (356s)
  ch4_07: 96/248  (474s)
  ch4_07: 120/248  (592s)
  ch4_07: 144/248  (710s)
  ch4_07: 168/248  (829s)
  ch4_07: 192/248  (947s)
  ch4_07: 216/248  (1066s)
  ch4_07: 240/248  (1185s)
  ch4_07: 248/248  (1225s)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_07_likelihood_3d_gd_combined.mp4 (264 frames)


PosixPath('renders/ch4_07_likelihood_3d_gd_combined.mp4')

In [60]:
# ch4_07b — ch4_07 with piecewise-linear path trail
ch4_export_likelihood_3d_gd_combined_path()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_07b: 1/248  (5s)
  ch4_07b: 24/248  (119s)
  ch4_07b: 48/248  (237s)
  ch4_07b: 72/248  (356s)
  ch4_07b: 96/248  (474s)
  ch4_07b: 120/248  (592s)
  ch4_07b: 144/248  (711s)
  ch4_07b: 168/248  (829s)
  ch4_07b: 192/248  (948s)
  ch4_07b: 216/248  (1066s)
  ch4_07b: 240/248  (1186s)
  ch4_07b: 248/248  (1225s)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_07b_gd_combined_path.mp4 (264 frames)


PosixPath('renders/ch4_07b_gd_combined_path.mp4')

In [61]:
# ch4_07b_rot_after — same path trail; camera pan after path completes
ch4_export_likelihood_3d_gd_combined_path_rot_after()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_07b_rot_after: 1/288  (5s)
  ch4_07b_rot_after: 28/288  (138s)
  ch4_07b_rot_after: 56/288  (276s)
  ch4_07b_rot_after: 84/288  (414s)
  ch4_07b_rot_after: 112/288  (553s)
  ch4_07b_rot_after: 140/288  (691s)
  ch4_07b_rot_after: 168/288  (829s)
  ch4_07b_rot_after: 196/288  (968s)
  ch4_07b_rot_after: 224/288  (1106s)
  ch4_07b_rot_after: 252/288  (1244s)
  ch4_07b_rot_after: 280/288  (1382s)
  ch4_07b_rot_after: 288/288  (1422s)
  deduped consecutive frames: 288 → 164


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_07b_gd_combined_path_rot_after.mp4 (180 frames)


PosixPath('renders/ch4_07b_gd_combined_path_rot_after.mp4')

In [11]:
# ch4_07c — 07b end ↔ partial slices with ∂ arrows, return to 07b
ch4_export_likelihood_07c_partial_bridge()


ch4_07c: 302 frames (full); GD end (2.742, -2.632, -0.496) → coupled (4.000, -4.000, -0.496)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_07c_partial_derivative_bridge.mp4 (302 frames)


PosixPath('renders/ch4_07c_partial_derivative_bridge.mp4')

In [ ]:
# ch4_08 — Newton probe ghosts from ch4_07 combined hold (5 steps)
ch4_export_likelihood_3d_newton_ghosts()


In [ ]:
# ch4_08b — same as 08 with Newton path trail
ch4_export_likelihood_3d_newton_ghosts_path()


In [ ]:
# ch4_08c — partial box ghosts → Newton-morph split → Newton step (×8)
ch4_export_likelihood_3d_newton_partial_ghosts()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_08c: 114/1140  (536s)
  ch4_08c: 228/1140  (1059s)
  ch4_08c: 342/1140  (1595s)
  ch4_08c: 456/1140  (2114s)
  ch4_08c: 570/1140  (2633s)
  ch4_08c: 684/1140  (3148s)
  ch4_08c: 798/1140  (3667s)
  ch4_08c: 912/1140  (4182s)
  ch4_08c: 1026/1140  (4701s)
  ch4_08c: 1140/1140  (5217s)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_08c_newton_partial_ghosts.mp4 (1156 frames)


PosixPath('renders/ch4_08c_newton_partial_ghosts.mp4')

In [ ]:
# ch4_08c_rot_after — same as 08c but camera rotates after all cycles
ch4_export_likelihood_3d_newton_partial_ghosts_rot_after()


In [ ]:
# ch4_08d — same as 08c with Newton path trail
ch4_export_likelihood_3d_newton_partial_ghosts_path()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_08d: 114/1140  (513s)
  ch4_08d: 228/1140  (1026s)
  ch4_08d: 342/1140  (1540s)
  ch4_08d: 456/1140  (2052s)
  ch4_08d: 570/1140  (2565s)
  ch4_08d: 684/1140  (3078s)
  ch4_08d: 798/1140  (3593s)
  ch4_08d: 912/1140  (4106s)
  ch4_08d: 1026/1140  (4621s)
  ch4_08d: 1140/1140  (5135s)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3000, 1900) to (3008, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_08d_newton_partial_ghosts_path.mp4 (1156 frames)


PosixPath('renders/ch4_08d_newton_partial_ghosts_path.mp4')

In [6]:
# ch4_08d_rot_after — same as 08d but camera rotates after all cycles
ch4_export_likelihood_3d_newton_partial_ghosts_path_rot_after()


ch3:583: SyntaxWarning: invalid escape sequence '\s'
ch3:587: SyntaxWarning: invalid escape sequence '\s'


Chapter 3 setup OK — 20 clean, 26 with noise.
  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_08d_rot_after: 1/1858  (6s)
  ch4_08d_rot_after: 185/1858  (961s)
  ch4_08d_rot_after: 370/1858  (1904s)
  ch4_08d_rot_after: 555/1858  (2848s)
  ch4_08d_rot_after: 740/1858  (3794s)
  ch4_08d_rot_after: 925/1858  (4738s)
  ch4_08d_rot_after: 1110/1858  (5695s)
  ch4_08d_rot_after: 1295/1858  (6669s)
  ch4_08d_rot_after: 1480/1858  (7652s)
  ch4_08d_rot_after: 1665/1858  (8632s)
  ch4_08d_rot_after: 1850/1858  (9581s)
  ch4_08d_rot_after: 1858/1858  (9623s)
  deduped consecutive frames: 1858 → 1400


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_08d_newton_partial_ghosts_path_rot_after.mp4 (1416 frames)


PosixPath('renders/ch4_08d_newton_partial_ghosts_path_rot_after.mp4')

In [ ]:
# ch4_08e — voxel-cube partial ghosts (colormap per partial)
ch4_export_likelihood_3d_newton_partial_voxel_ghosts()


In [ ]:
# ch4_08e_rot_after — same as 08e but camera rotates after all cycles
ch4_export_likelihood_3d_newton_partial_voxel_ghosts_rot_after()


In [ ]:
# ch4_08f — same as 08e with Newton path trail
ch4_export_likelihood_3d_newton_partial_voxel_ghosts_path()


In [3]:
# ch4_08f_rot_after — same as 08f but camera rotates after all cycles
ch4_export_likelihood_3d_newton_partial_voxel_ghosts_path_rot_after()


ch3:583: SyntaxWarning: invalid escape sequence '\s'
ch3:587: SyntaxWarning: invalid escape sequence '\s'


Chapter 3 setup OK — 20 clean, 26 with noise.
  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_08f_rot_after: 1/1858  (5s)
  ch4_08f_rot_after: 185/1858  (920s)
  ch4_08f_rot_after: 370/1858  (1842s)
  ch4_08f_rot_after: 555/1858  (2764s)
  ch4_08f_rot_after: 740/1858  (3686s)
  ch4_08f_rot_after: 925/1858  (4608s)
  ch4_08f_rot_after: 1110/1858  (5531s)
  ch4_08f_rot_after: 1295/1858  (6454s)
  ch4_08f_rot_after: 1480/1858  (7375s)
  ch4_08f_rot_after: 1665/1858  (8295s)
  ch4_08f_rot_after: 1850/1858  (9214s)
  ch4_08f_rot_after: 1858/1858  (9254s)
  deduped consecutive frames: 1858 → 1355


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_08f_newton_partial_voxel_ghosts_path_rot_after.mp4 (1371 frames)


PosixPath('renders/ch4_08f_newton_partial_voxel_ghosts_path_rot_after.mp4')

In [ ]:
# ch4_08 batch — 08/08b + 08c–08f in one deduped parallel render pass
ch4_export_likelihood_3d_newton_08_family_all()


In [16]:
# ch4_09 — from ch4_10 end: voxel sweep then study rescale back to hours
ch4_export_study_minutes_warp()


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_09_study_minutes_warp.mp4 (270 frames)


PosixPath('renders/ch4_09_study_minutes_warp.mp4')

In [ ]:
# ch4_10 — ch4_07 end → study ×60 → zoom → 8 GD steps + hours-path ghost
ch4_export_gd_eight_steps()


  ch4_gd: 264/336  (1364s)


In [ ]:
# ch4_11 — same opening as 10, then 8 Newton partial cycles (08d_rot_after)
ch4_export_newton_eight_steps()


  note: parallel export disabled in Jupyter (set CH4_EXPORT_WORKERS>1 from a .py script to override)
  ch4_11: 1/1848  (5s)
  ch4_11: 736/1848  (3930s)
  ch4_11: 920/1848  (5019s)
  ch4_11: 1288/1848  (7320s)
  ch4_11: 1472/1848  (8344s)
  ch4_11: 1656/1848  (9329s)
  ch4_11: 1840/1848  (10307s)
  ch4_11: 1848/1848  (10349s)
  deduped consecutive frames: 1848 → 1399


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3377, 1900) to (3392, 1904) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


wrote renders/ch4_11_newton_eight_steps.mp4 (1473 frames)


PosixPath('renders/ch4_11_newton_eight_steps.mp4')